# Replikacja potwierdzająca na VisA (preregistration_visa.md)

Otwórz ten plik w Colabie: *File → Upload notebook*. Każda komórka zaczyna się od `# KOMÓRKA n/12`.

Przed uruchomieniem:

1. *Runtime → Change runtime type → GPU* (T4 wystarczy, L4/A100 szybciej).
2. Pobierz `VisA_20220922.tar` (ok. 1,8 GB) ze strony zbioru i ustaw `VISA_TAR` na ścieżkę do tego archiwum. Oficjalna strona: https://github.com/amazon-science/spot-diff.
3. **Zamroź preregistrację przed komórką 9.** Komórki 1–8 (konwersja, podziały, cechy) wolno uruchomić wcześniej. Odległości, E0 i trajektorie nie wystartują, dopóki w `preregistration_visa.md` nie ma linii `Status: **FROZEN** on <data>`. Plik jest wbudowany w ten notatnik, więc po zamrożeniu zbuduj notatnik ponownie (`python build_colab_visa_notebook.py`). Przy pierwszym liczeniu wyników jego SHA-256 jest zapisywany w VISA_WORK; późniejsza zmiana pliku zatrzyma przebieg.
4. Edytuj tylko komórkę 4.

Wyniki (podziały, E0, trajektorie, analiza, raport) trafiają do `VISA_WORK` (domyślnie `visa_runs`). Duże pliki (obrazy, cechy ok. 32 GB, odległości ok. 12 GB) są na lokalnym dysku maszyny i znikają po zakończeniu sesji. Po przerwaniu uruchom komórki 1–7 i potem tę, na której stanęło: gotowe pliki w VISA_WORK są pomijane, a brakujące cechy i odległości liczą się od nowa (kilka minut na klasę).

In [ ]:
# KOMÓRKA 1/12 — montowanie Dysku Google
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# KOMÓRKA 2/12 — odtworzenie plików projektu na maszynie Colaba
import base64
from pathlib import Path

PROJECT = Path("/content/commissioning_project")
FILES = {"config.yaml": "IyBTY2llbnRpZmljIHBhcmFtZXRlcnMuIE1hY2hpbmUtc3BlY2lmaWMgcGF0aHMgYXJlIG5vdCBwYXJ0IG9mIHRoZSBjb25maWcgaGFzaC4NCiMgQ09NTUlTU0lPTklOR19EQVRBLCBDT01NSVNTSU9OSU5HX1dPUkssIC0tZGF0YS1yb290IGFuZCAtLXdvcmstcm9vdCBvdmVycmlkZSBwYXRocy4NCg0KcHJvdG9jb2xfdmVyc2lvbjogMQ0Kc2VlZDogNDINCm5fY2FsaWJyYXRpb246IDIwDQphbHBoYTogMC4wNQ0Kbl9yZXBlYXRzOiAyMA0Kbl9ncmlkOiBbMSwgMiwgNCwgOCwgMTYsIDMyLCA2NF0NCnN0cmF0ZWdpZXM6IFtyYW5kb20sIGtjZW50ZXJdDQphdWxjX25fbWF4OiAzMg0KcHJhY3RpY2FsX3RwcjogMC45NQ0KcHJhY3RpY2FsX2ZwcjogMC4wNQ0Kbl9zdGFyX21pbl9yZXBlYXRzOiAxNg0KZnByX2F1bGNfbWFyZ2luOiAwLjAxDQpzdG9wcGluZ19kZWx0YXM6IFswLjAyNSwgMC4wNSwgMC4xMF0NCnN0b3BwaW5nX2RlbHRhX21haW46IDAuMDUNCmJvb3RzdHJhcF9yZXBsaWNhdGVzOiAxMDAwMA0KYm9vdHN0cmFwX3NlZWQ6IDQyDQoNCmJhY2tib25lOiB3aWRlX3Jlc25ldDUwXzINCmxheWVyczogW2xheWVyMiwgbGF5ZXIzXQ0KcmVzaXplOiAyNTYNCmNlbnRlcl9jcm9wOiAyMjQNCnVwc2FtcGxlX21vZGU6IGJpbGluZWFyDQppbWFnZW5ldF9tZWFuOiBbMC40ODUsIDAuNDU2LCAwLjQwNl0NCmltYWdlbmV0X3N0ZDogWzAuMjI5LCAwLjIyNCwgMC4yMjVdDQpmZWF0dXJlX2JhdGNoX3NpemU6IDgNCmRpc3RhbmNlX3F1ZXJ5X2JhdGNoOiA4DQpkaXN0YW5jZV90cmFpbl9iYXRjaDogMTYNCg0KZTA6DQogIGF1cm9jX21lYW5fdG9sZXJhbmNlOiAwLjAwNQ0KICBzcGVhcm1hbl9taW46IDAuOTkNCiAgbnVtX25laWdoYm9yc19yZWY6IDkNCiAgbnVtX25laWdoYm9yc19ycTogMQ0KICBjb3Jlc2V0X3JhdGlvOiAwLjENCiAgIyBPcGVyYXRpb25hbCBzYW5pdHkgZmxvb3IgZm9yIHRoZSByZWZlcmVuY2UgQW5vbWFsaWIgcnVuLiBSb3RoIGV0IGFsLiByZXBvcnQgfjAuOTkxLg0KICBtZWFuX2F1cm9jX2Zsb29yOiAwLjk3DQogIHJlZmVyZW5jZV9tZWFuX2F1cm9jOiAwLjk5MQ0KDQpwYXRoczoNCiAgZGF0YV9yb290OiBkYXRhL212dGVjDQogIHdvcmtfcm9vdDogLg0KDQpvYmplY3RzOg0KICAtIGJvdHRsZQ0KICAtIGNhYmxlDQogIC0gY2Fwc3VsZQ0KICAtIGhhemVsbnV0DQogIC0gbWV0YWxfbnV0DQogIC0gcGlsbA0KICAtIHNjcmV3DQogIC0gdG9vdGhicnVzaA0KICAtIHRyYW5zaXN0b3INCiAgLSB6aXBwZXINCnRleHR1cmVzOg0KICAtIGNhcnBldA0KICAtIGdyaWQNCiAgLSBsZWF0aGVyDQogIC0gdGlsZQ0KICAtIHdvb2QNCg==", "config_visa.yaml": "IyBTY2llbnRpZmljIHBhcmFtZXRlcnMgb2YgcHJlcmVnaXN0cmF0aW9uX3Zpc2EubWQuIE1hY2hpbmUtc3BlY2lmaWMgcGF0aHMgYXJlIG5vdCBwYXJ0IG9mIHRoZSBjb25maWcgaGFzaC4NCiMgVklTQV9EQVRBLCBWSVNBX1dPUkssIFZJU0FfQ0FDSEUgYW5kIHRoZSBtYXRjaGluZyAtLWRhdGEtcm9vdCwgLS13b3JrLXJvb3QsIC0tY2FjaGUtcm9vdCBvdmVycmlkZSBwYXRocy4NCg0KcHJvdG9jb2xfdmVyc2lvbjogMg0KZGF0YXNldDogdmlzYQ0Kc2VlZDogNDINCmNsYXNzX3NlZWRfb2Zmc2V0OiAxMDAwDQp1bml2ZXJzZV9zaXplOiAzMDANCm5fY2FsaWJyYXRpb25fc21hbGw6IDIwDQpuX2NhbGlicmF0aW9uX2xhcmdlOiA2MA0KYWxwaGE6IDAuMDUNCm5fcmVwZWF0czogMjANCm5fZ3JpZDogWzEsIDIsIDQsIDgsIDE2LCAzMiwgNjQsIDEyOF0NCnN0cmF0ZWdpZXM6IFtyYW5kb20sIGtjZW50ZXJdDQphdWxjX25fbWF4OiAzMg0KcHJhY3RpY2FsX3RwcjogMC45NQ0KcHJhY3RpY2FsX2ZwcjogMC4wNQ0Kbl9zdGFyX21pbl9yZXBlYXRzOiAxNg0KZnByX2F1bGNfbWFyZ2luOiAwLjAxDQp0cHJfY29zdF9tYXJnaW46IDAuMDMNCnN0b3BwaW5nX2RlbHRhczogWzAuMDI1LCAwLjA1LCAwLjEwXQ0Kc3RvcHBpbmdfZGVsdGFfbWFpbjogMC4wNQ0KYm9vdHN0cmFwX3JlcGxpY2F0ZXM6IDEwMDAwDQpib290c3RyYXBfc2VlZDogNDINCmJ1ZGdldF9zZWVkX3RhZzogMjAyNjExMjANCmJ1ZGdldHM6IFs4MCwgMTIwLCAxNjAsIDIwMCwgMzAwXQ0KYnVkZ2V0X2dyaWQ6IFsxLCAyLCA0LCA4LCAxNiwgMzIsIDY0LCAxMjhdDQpidWRnZXRfbWluX2NhbGlicmF0aW9uOiAyMA0KcGFjX2Vwc2lsb246IDAuMDUNCnBhY19kZWx0YTogMC4xMA0KDQpiYWNrYm9uZTogd2lkZV9yZXNuZXQ1MF8yDQpsYXllcnM6IFtsYXllcjIsIGxheWVyM10NCnJlc2l6ZTogMjU2DQpjZW50ZXJfY3JvcDogMjI0DQp1cHNhbXBsZV9tb2RlOiBiaWxpbmVhcg0KaW1hZ2VuZXRfbWVhbjogWzAuNDg1LCAwLjQ1NiwgMC40MDZdDQppbWFnZW5ldF9zdGQ6IFswLjIyOSwgMC4yMjQsIDAuMjI1XQ0KZmVhdHVyZV9iYXRjaF9zaXplOiA4DQpkaXN0YW5jZV9xdWVyeV9iYXRjaDogOA0KZGlzdGFuY2VfdHJhaW5fYmF0Y2g6IDE2DQoNCmUwOg0KICBhdXJvY19tZWFuX3RvbGVyYW5jZTogMC4wMDUNCiAgc3BlYXJtYW5fbWluOiAwLjk5DQogIG51bV9uZWlnaGJvcnNfcmVmOiA5DQogIG51bV9uZWlnaGJvcnNfcnE6IDENCiAgY29yZXNldF9yYXRpbzogMC4xDQogICMgT3BlcmF0aW9uYWwgc2FuaXR5IGZsb29yIGZvciBBbm9tYWxpYiBQYXRjaENvcmUgdHJhaW5lZCBvbiB0aGUgMzAwLWltYWdlIHVuaXZlcnNlIFUuDQogIG1lYW5fYXVyb2NfZmxvb3I6IDAuOTANCiAgcmVmZXJlbmNlX21lYW5fYXVyb2M6IDAuOTUNCg0KcGF0aHM6DQogIGRhdGFfcm9vdDogZGF0YS92aXNhXzFjbHMNCiAgd29ya19yb290OiB2aXNhX3J1bnMNCiAgY2FjaGVfcm9vdDogdmlzYV9jYWNoZQ0KDQpjbGFzc2VzOg0KICAtIGNhbmRsZQ0KICAtIGNhcHN1bGVzDQogIC0gY2FzaGV3DQogIC0gY2hld2luZ2d1bQ0KICAtIGZyeXVtDQogIC0gbWFjYXJvbmkxDQogIC0gbWFjYXJvbmkyDQogIC0gcGNiMQ0KICAtIHBjYjINCiAgLSBwY2IzDQogIC0gcGNiNA0KICAtIHBpcGVfZnJ5dW0NCg==", "requirements.txt": "bnVtcHk+PTEuMjYNCnNjaXB5Pj0xLjExDQpwYW5kYXM+PTIuMQ0KcHlhcnJvdz49MTQNCnNjaWtpdC1sZWFybj49MS4zDQptYXRwbG90bGliPj0zLjgNClB5WUFNTD49Ng0KUGlsbG93Pj0xMA0KdHFkbT49NC42Ng0KdG9yY2g+PTIuMQ0KdG9yY2h2aXNpb24+PTAuMTYNCnRpbW0+PTAuOQ0KYW5vbWFsaWI+PTIuMA0KcHl0ZXN0Pj03LjQNCg==", "preregistration.md": "...", "preregistration_visa.md": "IyBQcmVyZWdpc3RyYXRpb24gdjI6IGNvbmZpcm1hdG9yeSByZXBsaWNhdGlvbiBvbiBWaXNBDQoNClN0YXR1czogKipGUk9aRU4qKiBvbiAyMCBOb3ZlbWJlciAyMDI2LCBiZWZvcmUgdGhlIGZpcnN0IFZpc0EgaW1hZ2UgaXMgc2NvcmVkIGJ5IHRoZSBwaXBlbGluZS4gQWZ0ZXIgdGhpcyBsaW5lLCB0aGlzIGZpbGUgaXMgbm90IGVkaXRlZC4gQW55dGhpbmcgYWRkZWQgbGF0ZXIgaXMgbWFya2VkIGV4cGxvcmF0b3J5Lg0KDQpSZWxhdGlvbiB0byB2MTogYHByZXJlZ2lzdHJhdGlvbi5tZGAgKE1WVGVjIEFELCBjb25maWcgaGFzaCBgZWM3MDk5MjMzNmI0YzRhNGAsIEUwIFBBU1MsIDYwMCB0cmFqZWN0b3JpZXMpIGFuZCBgZXhwbG9yYXRvcnlfYWRkZW5kdW0ubWRgIHN0YXkgZnJvemVuIGFuZCBhcmUgcmVwb3J0ZWQgZmlyc3QgYW5kIHVuY2hhbmdlZC4gVGhpcyBkb2N1bWVudCB0dXJucyB0aGUgbWFpbiBleHBsb3JhdG9yeSBmaW5kaW5ncyBvZiB0aGUgYWRkZW5kdW0gaW50byBjb25maXJtYXRvcnkgaHlwb3RoZXNlcyBvbiBhIHNlY29uZCBkYXRhc2V0IHRoYXQgdGhlIHYxIGFuYWx5c2lzIG5ldmVyIHRvdWNoZWQuDQoNClByaW9yIGV4cG9zdXJlLCBkZWNsYXJlZCBiZWZvcmUgZnJlZXppbmc6IHRoZSBhdXRob3JzIGhhdmUgbm90IGNvbXB1dGVkIFBhdGNoQ29yZSBzY29yZXMgb24gVmlzQSBiZWZvcmUuDQoNCkNvZGU6IGBweXRob24gLW0gY29tbWlzc2lvbmluZy52aXNhYC4gVGhlIE1WVGVjIHBhY2thZ2UgcGF0aCwgYGNvbmZpZy55YW1sYCwgYW5kIHRoZSB2MSBoYXNoIGFyZSBub3QgbW9kaWZpZWQ7IGEgdGVzdCBjaGVja3MgdGhhdCB0aGUgdjEgaGFzaCBpcyBzdGlsbCBgZWM3MDk5MjMzNmI0YzRhNGAuDQoNCiMjIFdoeSBhIHNlY29uZCBwcmVyZWdpc3RyYXRpb24NCg0KVGhlIE1WVGVjIHJ1biBwcm9kdWNlZCB0aHJlZSByZXN1bHRzIHRoYXQgYSBzaW5nbGUgZGF0YXNldCBjYW5ub3Qgc2V0dGxlOg0KDQoxLiBJbWFnZS1sZXZlbCBrLWNlbnRlciBzZWxlY3Rpb24gYmVhdCBhIHJhbmRvbSBuZXN0ZWQgYmFuayBpbiBBVUxDX1RQUiBvbiAxMyBvZiAxNSBjbGFzc2VzIChIb2xtIHAgPSAwLjAwMDUpIHdpdGhvdXQgYW4gRlBSIHBlbmFsdHkuIFRoaXMgaXMgdGhlIGNvbmZpcm1hdG9yeSByZXN1bHQgdG8gcmVwbGljYXRlLg0KMi4gVGhlIGNyaXRlcmlvbiBiZWhpbmQgTiogKDE2IG9mIDIwIHJlcGV0aXRpb25zIHdpdGggVFBSIOKJpSAwLjk1IGFuZCBGUFIg4omkIDAuMDUsIHRocmVzaG9sZCBtYXggb2YgMjApIHdhcyBpbmZlYXNpYmxlIGJ5IGNvbnN0cnVjdGlvbjogd2l0aCAxMuKAkzYwIGdvb2QgdGVzdCBpbWFnZXMgZXZlbiBhIHBlcmZlY3QgZGV0ZWN0b3IgaXMgZXhwZWN0ZWQgdG8gcGFzcyBpbiBvbmx5IDEw4oCTMTUgb2YgMjAgcmVwZXRpdGlvbnMgKGFkZGVuZHVtIEUxKS4gMTMgb2YgMTUgY2xhc3NlcyB3ZXJlIGNlbnNvcmVkLg0KMy4gV2hlcmUgZ29vZCB0cmFpbiBhbmQgZ29vZCB0ZXN0IHNjb3JlcyB3ZXJlIGV4Y2hhbmdlYWJsZSwgdGhlIHJlYWxpemVkIEZQUiBtYXRjaGVkIHRoZSBCZXRhKDEsIDIwKSB0aGVvcnkgKG1lYW4gMC4wNTHigJMwLjA1NyB2cyAxLzIxLCBmcmFjdGlvbiBhYm92ZSA1JSAwLjMx4oCTMC40MCB2cyAwLjMyKS4gV2hlcmUgdGhleSB3ZXJlIG5vdCAoY2FycGV0LCBsZWF0aGVyLCB3b29kLCBwYXJ0bHkgemlwcGVyKSwgdGhlIEZQUiB3YXMgMC4xOOKAkzAuMjggKGFkZGVuZHVtIEUzKS4NCg0KdjIgdGhlcmVmb3JlIChhKSByZXBsaWNhdGVzIHRoZSBrLWNlbnRlciBjb21wYXJpc29uLCAoYikgYWRkcyBhIGxhcmdlciBjYWxpYnJhdGlvbiBzZXQgd2hvc2UgZmVhc2liaWxpdHkgaXMgY2hlY2tlZCBhbmFseXRpY2FsbHkgYmVmb3JlIHRoZSBydW4gKHNlY3Rpb24gOCksIGFuZCAoYykgYWRkcyBhIGxhcmdlIGhlbGQtb3V0IHNldCBvZiBleGNoYW5nZWFibGUgZ29vZCBpbWFnZXMsIHNvIHRoYXQgdGhlIGNhbGlicmF0aW9uIGVycm9yIGFuZCB0aGUgdHJhaW4vdGVzdCBzaGlmdCBjYW4gYmUgbWVhc3VyZWQgc2VwYXJhdGVseS4NCg0KIyMgMS4gUXVlc3Rpb25zIGFuZCBoeXBvdGhlc2VzDQoNCkgxLCBwcmltYXJ5IChyZXBsaWNhdGlvbiBvZiB2MSBSUTMpLiBXaXRoIHRoZSB2MSB0aHJlc2hvbGQgUjIwLCBkb2VzIGZhcnRoZXN0LWZpcnN0IHNlbGVjdGlvbiBvbiBHQVAgZmVhdHVyZXMgZ2l2ZSBhIGhpZ2hlciBBVUxDX1RQUiB0aGFuIGEgcmFuZG9tIG5lc3RlZCBiYW5rPyBQcmVkaWN0aW9uOiBwb3NpdGl2ZSBtZWRpYW4gcGFpcmVkIGRpZmZlcmVuY2UuDQoNCkgyLCBwcmltYXJ5IChyZXBsaWNhdGlvbiBvZiB2MSBSUTMpLiBVbmRlciBSMjAsIGRvZXMgay1jZW50ZXIgY2hhbmdlIEFVTENfRlBSPyBQcmVkaWN0aW9uOiBubyBwcmFjdGljYWxseSByZWxldmFudCBpbmNyZWFzZSAodGhlIHYxIG1hcmdpbiBvZiArMC4wMSkuDQoNCkgzLCBwcmltYXJ5IGVzdGltYXRpb24sIG5vIHNpZ25pZmljYW5jZSB0ZXN0LiBBdCBOID0gfFB8LCB3aGF0IFRQUiBkb2VzIHRoZSBsYXJnZXIgY2FsaWJyYXRpb24gc2V0IGNvc3Q/IEJlY2F1c2UgbWF4KEM2MCkg4omlIG1heChDMjApIGJ5IGNvbnN0cnVjdGlvbiwgVFBSKFI2MCkg4omkIFRQUihSMjApIGluIGV2ZXJ5IHRyYWplY3RvcnksIHNvIGEgdGVzdCBvZiB0aGUgc2lnbiBpcyB1bmluZm9ybWF0aXZlLiBUaGUgcHJlLXNwZWNpZmllZCBwcmFjdGljYWwgcnVsZTogUjYwIGlzIGNhbGxlZCAqKnJlY29tbWVuZGVkKiogd2hlbiB0aGUgbWVkaWFuIG92ZXIgY2xhc3NlcyBvZiB0aGUgbWVhbiAob3ZlciAyMCByZXBldGl0aW9ucykgVFBSKFIyMCkg4oiSIFRQUihSNjApIGlzIGF0IG1vc3QgMC4wMy4NCg0KUHJlLWRlY2xhcmVkIHByZWRpY3Rpb25zLCBkZXNjcmlwdGl2ZSwgbm90IHRlc3RzOg0KDQpQMSAoZmVhc2liaWxpdHkpLiBXaXRoIFIyMCwgTiogKG9mZmljaWFsIHRlc3QpIGlzIGNlbnNvcmVkIGZvciBtb3N0IGNsYXNzZXMsIGFzIHNlY3Rpb24gOCBwcmVkaWN0cy4gV2l0aCBSNjAsIE4qIGlzIHJlYWNoZWQgZm9yIGV2ZXJ5IGNsYXNzIHdob3NlIFRQUiBhdCBOID0gfFB8IHJlYWNoZXMgMC45NSBpbiBhdCBsZWFzdCAxNiBvZiAyMCByZXBldGl0aW9ucy4NCg0KUDIgKGNhbGlicmF0aW9uIHVuZGVyIGV4Y2hhbmdlYWJpbGl0eSkuIE9uIHRoZSBoZWxkLW91dCBnb29kIHNldCBIOiB0aGUgbWVkaWFuIG92ZXIgY2xhc3NlcyBvZiB0aGUgbWVhbiByZWFsaXplZCBGUFIgKG1lYW4gb3ZlciBldmVyeSBncmlkIHBvaW50LCBib3RoIHN0cmF0ZWdpZXMsIGFuZCAyMCByZXBldGl0aW9uczsgZXZlcnkgYmFuayBpcyBkcmF3biBmcm9tIFAsIHNvIEMgYW5kIEggc3RheSBleGNoYW5nZWFibGUgZ2l2ZW4gdGhlIGJhbmspIGxpZXMgaW4gWzAuMDMsIDAuMDddIGZvciBSMjAgKHRoZW9yeSAxLzIxIOKJiCAwLjA0OCkgYW5kIGluIFswLjAwNSwgMC4wM10gZm9yIFI2MCAodGhlb3J5IDEvNjEg4omIIDAuMDE2KS4gVGhlIGZyYWN0aW9uIG9mIHJlcGV0aXRpb25zIHdpdGggRlBSX0ggPiAwLjA1IGlzIGNvbXBhcmVkIHdpdGggMC4zNTggKFIyMCkgYW5kIDAuMDQ2IChSNjApLg0KDQpQMyAoc2hpZnQpLiBBIGNsYXNzIHdob3NlIG9mZmljaWFsIGdvb2QgdGVzdCBzY29yZXMgZGlmZmVyIGZyb20gaXRzIGhlbGQtb3V0IGdvb2Qgc2NvcmVzIChzZWN0aW9uIDksIEhvbG0gcCA8IDAuMDUpIGhhcyBhIGhpZ2hlciBtZWFuIEZQUiBvbiB0aGUgb2ZmaWNpYWwgZ29vZCB0ZXN0IHRoYW4gb24gSCB1bmRlciBSMjAgKHNhbWUgYXZlcmFnaW5nIGFzIFAyKS4NCg0KU2Vjb25kYXJ5OiB0aGUgdjEgc3RvcHBpbmcgcnVsZSAoUlE0KSBhbmQgdGhlIGJ1ZGdldCBzcGxpdCAoc2VjdGlvbiAxMCkuDQoNCk91dCBvZiBzY29wZSwgYXMgaW4gdjE6IGNvcmVzZXQgaW5zaWRlIHRoZSBSUSBydW5zLCBvdGhlciBkZXRlY3RvcnMgb3IgYmFja2JvbmVzLCBDUFUgdGltaW5nLCBwaXhlbC1sZXZlbCBtZXRyaWNzLCBkZXBsb3ltZW50IG9uIGEgbGluZS4NCg0KIyMgMi4gRGF0YQ0KDQpWaXNBIChab3UgZXQgYWwuLCBFQ0NWIDIwMjIsIENDIEJZIDQuMCksIGFyY2hpdmUgYFZpc0FfMjAyMjA5MjIudGFyYCwgdGhlIG9mZmljaWFsIG9uZS1jbGFzcyBzcGxpdCBgc3BsaXRfY3N2LzFjbHMuY3N2YC4gQWxsIDEyIGNsYXNzZXMsIGluIHRoaXMgb3JkZXIgKHNlZWQgaW5kZXggMOKAkzExKTogY2FuZGxlLCBjYXBzdWxlcywgY2FzaGV3LCBjaGV3aW5nZ3VtLCBmcnl1bSwgbWFjYXJvbmkxLCBtYWNhcm9uaTIsIHBjYjEsIHBjYjIsIHBjYjMsIHBjYjQsIHBpcGVfZnJ5dW0uIEdyb3VwcyBmb3IgZmlndXJlcyAoZnJvbSB0aGUgVmlzQSBwYXBlcik6IG11bHRpcGxlIGluc3RhbmNlcyAoY2FuZGxlLCBjYXBzdWxlcywgbWFjYXJvbmkxLCBtYWNhcm9uaTIpLCBzaW5nbGUgaW5zdGFuY2UgKGNhc2hldywgY2hld2luZ2d1bSwgZnJ5dW0sIHBpcGVfZnJ5dW0pLCBjb21wbGV4IHN0cnVjdHVyZSAocGNiMeKAk3BjYjQpLg0KDQpPZmZpY2lhbCBjb3VudHMsIGhhcmQtY29kZWQ7IGEgbWlzbWF0Y2ggc3RvcHMgdGhlIHJ1bjoNCg0KfCBjbGFzcyB8IHRyYWluIGdvb2QgfCB0ZXN0IGdvb2QgfCB0ZXN0IGFub21hbHkgfCBvZmZpY2lhbCBGUiBidWRnZXQgfCB8SHwgfA0KfC0tLXwtLS18LS0tfC0tLXwtLS18LS0tfA0KfCBjYW5kbGUgfCA5MDAgfCAxMDAgfCAxMDAgfCA1IHwgNjAwIHwNCnwgY2Fwc3VsZXMgfCA1NDIgfCA2MCB8IDEwMCB8IDMgfCAyNDIgfA0KfCBjYXNoZXcgfCA0NTAgfCA1MCB8IDEwMCB8IDIgfCAxNTAgfA0KfCBjaGV3aW5nZ3VtIHwgNDUzIHwgNTAgfCAxMDAgfCAyIHwgMTUzIHwNCnwgZnJ5dW0gfCA0NTAgfCA1MCB8IDEwMCB8IDIgfCAxNTAgfA0KfCBtYWNhcm9uaTEgfCA5MDAgfCAxMDAgfCAxMDAgfCA1IHwgNjAwIHwNCnwgbWFjYXJvbmkyIHwgOTAwIHwgMTAwIHwgMTAwIHwgNSB8IDYwMCB8DQp8IHBjYjEgfCA5MDQgfCAxMDAgfCAxMDAgfCA1IHwgNjA0IHwNCnwgcGNiMiB8IDkwMSB8IDEwMCB8IDEwMCB8IDUgfCA2MDEgfA0KfCBwY2IzIHwgOTA1IHwgMTAxIHwgMTAwIHwgNSB8IDYwNSB8DQp8IHBjYjQgfCA5MDQgfCAxMDEgfCAxMDAgfCA1IHwgNjA0IHwNCnwgcGlwZV9mcnl1bSB8IDQ1MCB8IDUwIHwgMTAwIHwgMiB8IDE1MCB8DQoNClRoZSByYXcgYXJjaGl2ZSBpcyBjb252ZXJ0ZWQgb25jZSBpbnRvIHRoZSBNVlRlYyBsYXlvdXQgKGB7Y2xhc3N9L3RyYWluL2dvb2RgLCBge2NsYXNzfS90ZXN0L2dvb2RgLCBge2NsYXNzfS90ZXN0L2JhZGApIGJ5IGNvcHlpbmcgZmlsZXMgbGlzdGVkIGluIGAxY2xzLmNzdmA7IG5vdGhpbmcgaXMgcmVzaXplZCBvciByZS1lbmNvZGVkIGF0IHRoaXMgc3RlcC4NCg0KRHVwbGljYXRlcyAoU0hBLTEgb2YgZmlsZSBjb250ZW50KSwgZGVjaWRlZCBiZWZvcmUgYW55IHNjb3JlIGV4aXN0czogYSBkdXBsaWNhdGUgYWNyb3NzIHRyYWluIGFuZCB0ZXN0LCBvciBiZXR3ZWVuIHRlc3QgZ29vZCBhbmQgdGVzdCBhbm9tYWx5LCBzdG9wcyB0aGUgcnVuIGFuZCBpcyByZXBvcnRlZC4gRHVwbGljYXRlcyBpbnNpZGUgdHJhaW4vZ29vZCBrZWVwIHRoZSBmaXJzdCBmaWxlIGluIHNvcnRlZCBwYXRoIG9yZGVyIGFuZCBkcm9wIHRoZSByZXN0LiBEdXBsaWNhdGVzIGluc2lkZSB0ZXN0IGdvb2Qgb3IgaW5zaWRlIHRlc3QgYW5vbWFseSBhcmUgcmVwb3J0ZWQgYW5kIGtlcHQsIGJlY2F1c2UgdGhlIG9mZmljaWFsIHRlc3Qgc2V0IGlzIHVzZWQgdW5jaGFuZ2VkLiBUaGUgb2ZmaWNpYWwgY291bnRzIGFib3ZlIGFyZSBjaGVja2VkIGJlZm9yZSByZW1vdmFsOyB8SHwgaXMgdGhlbiB0aGUgZGVkdXBsaWNhdGVkIHRyYWluL2dvb2QgY291bnQgbWludXMgMzAwLiBFdmVyeXRoaW5nIGlzIGxpc3RlZCBpbiBgZGF0YV9jaGVjay5qc29uYC4gVGhlIGRhdGEgY2hlY2sgcnVucyBiZWZvcmUgZmVhdHVyZSBleHRyYWN0aW9uIGFuZCBkb2VzIG5vdCBjb21wdXRlIHNjb3Jlcy4NCg0KIyMgMy4gQ29tbWlzc2lvbmluZyB1bml2ZXJzZSwgc3BsaXRzLCBncmlkDQoNCkZvciBlYWNoIGNsYXNzLCBvbmNlOiB0aGUgY29tbWlzc2lvbmluZyB1bml2ZXJzZSBVIGlzIDMwMCBpbWFnZXMgZHJhd24gd2l0aG91dCByZXBsYWNlbWVudCBmcm9tIHRyYWluL2dvb2Qgd2l0aCBgU2VlZFNlcXVlbmNlKFs0MiwgMTAwMCArIGNsYXNzX2luZGV4XSlgLiBUaGUgaGVsZC1vdXQgZ29vZCBzZXQgSCBpcyB0cmFpbi9nb29kIG1pbnVzIFUgKDE1MOKAkzYwNSBpbWFnZXMpLiBIIG5ldmVyIGVudGVycyBhIGJhbmsgb3IgYSB0aHJlc2hvbGQuIFUgYW5kIEggYXJlIGZpeGVkIGFjcm9zcyByZXBldGl0aW9ucy4NCg0KRm9yIGVhY2ggY2xhc3MgYW5kIHJlcGV0aXRpb24gciA9IDAuLjE5LCBgU2VlZFNlcXVlbmNlKFs0MiwgMTAwMCArIGNsYXNzX2luZGV4LCByXSlgIGlzIHNwYXduZWQgaW50byBpbmRlcGVuZGVudCBzdHJlYW1zIGZvciB0aGUgc3BsaXQsIHRoZSByYW5kb20gcGVybXV0YXRpb24sIGFuZCB0aGUgZmlyc3Qgay1jZW50ZXIgaW1hZ2UuIEZyb20gdGhlIHNwbGl0IHN0cmVhbSwgQzYwIGlzIGFuIG9yZGVyZWQgcmFuZG9tIHNhbXBsZSBvZiA2MCBpbWFnZXMgb2YgVTsgQzIwIGlzIHRoZSBmaXJzdCAyMCBpbWFnZXMgb2YgQzYwLCBzbyBDMjAg4oqCIEM2MC4gVGhlIHBvb2wgaXMgUCA9IFUgXCBDNjAsIHxQfCA9IDI0MCBmb3IgZXZlcnkgY2xhc3MuDQoNCkFzc2VydGlvbnMsIGNoZWNrZWQgaW4gY29kZTogfFV8ID0gMzAwLCB8QzYwfCA9IDYwLCB8QzIwfCA9IDIwLCBDMjAg4oqCIEM2MCwgfFB8ID0gMjQwLCBhbmQgVSwgSCwgYW5kIHRoZSBvZmZpY2lhbCB0ZXN0IHNldCBhcmUgcGFpcndpc2UgZGlzam9pbnQuIFNwbGl0cyBhcmUgd3JpdHRlbiBvbmNlIHRvIGBzcGxpdHMve2NsYXNzfS9yZXBfWFguanNvbmAgYW5kIG5ldmVyIG92ZXJ3cml0dGVuLg0KDQpHcmlkOiBOIOKIiCB7MSwgMiwgNCwgOCwgMTYsIDMyLCA2NCwgMTI4LCAyNDB9LiAxMiBjbGFzc2VzIMOXIDIwIHJlcGV0aXRpb25zIMOXIDIgc3RyYXRlZ2llcyA9IDQ4MCBuZXN0ZWQgdHJhamVjdG9yaWVzLCA0MzIwIHJvd3MgcGVyIHRocmVzaG9sZCBydWxlLiBSYW5kb20gYW5kIGstY2VudGVyIGFyZSBkZWZpbmVkIGFzIGluIHYxIHNlY3Rpb24gMzsgay1jZW50ZXIgcnVucyBvbiBQIG9ubHkuDQoNCiMjIDQuIFBhdGNoQ29yZQ0KDQpJZGVudGljYWwgdG8gdjEgc2VjdGlvbiA0OiBXaWRlLVJlc05ldC01MC0yIChzYW1lIHRpbW0gd2VpZ2h0cyksIGBsYXllcjJgICsgYGxheWVyM2AsIDPDlzMgYXZlcmFnZSBwb29saW5nLCBiaWxpbmVhciB1cHNhbXBsaW5nLCA3ODQgw5cgMTUzNiBkZXNjcmlwdG9ycywgbm8gY29yZXNldCwgaW1hZ2Ugc2NvcmUgPSBtYXggb3ZlciBwYXRjaGVzIG9mIHRoZSBtaW4gZGlzdGFuY2UgdG8gdGhlIGJhbmssIHJlamVjdCB3aGVuIHN0cmljdGx5IGFib3ZlIHRoZSB0aHJlc2hvbGQuIFByZXByb2Nlc3NpbmcgYXMgdjE6IHJlc2l6ZSB0byAyNTYgw5cgMjU2IChhc3BlY3QgcmF0aW8gbm90IHByZXNlcnZlZDsgVmlzQSBpbWFnZXMgYXJlIGFib3V0IDEuNWsgw5cgMWspLCBjZW50ZXIgY3JvcCAyMjQsIEltYWdlTmV0IG5vcm1hbGl6YXRpb24uIFRoaXMgbWF0Y2hlcyB0aGUgQW5vbWFsaWIgZGVmYXVsdCBhbmQgaXMgYSBzdGF0ZWQgbGltaXRhdGlvbjogdGhlIGNyb3AgcmVtb3ZlcyBhYm91dCA2JSBvZiBlYWNoIGJvcmRlci4NCg0KVGhlIGRpc3RhbmNlIHRlbnNvciBoYXMgVSBhcyBjb2x1bW5zICgzMDApIGFuZCBVIOKIqiBIIOKIqiB0ZXN0IGFzIHJvd3MuDQoNCiMjIDUuIFRocmVzaG9sZCBydWxlcw0KDQpCb3RoIHJ1bGVzIGFyZSBldmFsdWF0ZWQgb24gdGhlIHNhbWUgYmFuayBpbiBldmVyeSB0cmFqZWN0b3J5LCBzbyB0aGUgY29tcGFyaXNvbiBpcyBwYWlyZWQuDQoNCi0gUjIwOiDPhCA9IG1heCBzKEMyMCB8IFMpLiBUaGlzIGlzIHRoZSB2MSBydWxlIChrID0gY2VpbCgyMSDDlyAwLjk1KSA9IDIwKS4NCi0gUjYwOiDPhCA9IG1heCBzKEM2MCB8IFMpLiBXaXRoIGNvbnRpbnVvdXMgc2NvcmVzIGFuZCBleGNoYW5nZWFiaWxpdHkgdGhlIGNvbmRpdGlvbmFsIEZQUiBpcyBCZXRhKDEsIDYwKTogbWVhbiAxLzYxLCBQKEZQUiA+IDAuMDUpID0gMC45NV42MCDiiYggMC4wNDYuIFI2MCBpcyBhbHNvIHRoZSBQQUMgcmFuayBvZiB0aGUgYWRkZW5kdW0gYXQgbSA9IDYwLCDOtSA9IDAuMDUsIM60ID0gMC4xMCAodGhlIHJhbmsgNTkgZ2l2ZXMgMC4xOSA+IM60KS4NCg0KIyMgNi4gTWV0cmljcw0KDQpGb3IgZXZlcnkgY2xhc3MsIHJlcCwgc3RyYXRlZ3ksIE4sIGFuZCBydWxlOg0KDQotIFRQUiBvbiBvZmZpY2lhbCB0ZXN0IGFub21hbGllcywgd2l0aCB0aGUgb25lLXNpZGVkIGxvd2VyIDk1JSBDbG9wcGVy4oCTUGVhcnNvbiBib3VuZDsNCi0gRlBSX29mZiBvbiBvZmZpY2lhbCBnb29kIHRlc3QgaW1hZ2VzIGFuZCBGUFJfSCBvbiBILCBlYWNoIHdpdGggdGhlIG9uZS1zaWRlZCB1cHBlciA5NSUgYm91bmQ7DQotIGltYWdlIEFVUk9DIGFuZCBvcmFjbGUgRlBSIGF0IFRQUiA9IDAuOTUgb24gdGhlIG9mZmljaWFsIHRlc3Qgc2V0IChydWxlLWluZGVwZW5kZW50KTsNCi0gY3JpdGVyaW9uX29mZjogVFBSIOKJpSAwLjk1IGFuZCBmYWxzZSByZWplY3RzIOKJpCBmbG9vcigwLjA1IMOXIG5fZ29vZF90ZXN0KSBvbiB0aGUgb2ZmaWNpYWwgZ29vZCB0ZXN0Ow0KLSBjcml0ZXJpb25fSDogVFBSIOKJpSAwLjk1IGFuZCBmYWxzZSByZWplY3RzIG9uIEgg4omkIGZsb29yKDAuMDUgw5cgfEh8KS4NCg0KQVVMQzogYXMgdjEsIHRyYXBlem9pZCBvbiBsb2cyIE4gb3ZlciB7MSwgMiwgNCwgOCwgMTYsIDMyfSwgZGl2aWRlZCBieSA1LCBtZWFuIG92ZXIgMjAgcmVwZXRpdGlvbnMuDQoNCk4qOiBhcyB2MSAoc21hbGxlc3QgZ3JpZCBOIGZyb20gd2hpY2ggYXQgbGVhc3QgMTYgb2YgMjAgcmVwZXRpdGlvbnMgbWVldCB0aGUgY3JpdGVyaW9uIGF0IGV2ZXJ5IGxhcmdlciBncmlkIHBvaW50OyBvdGhlcndpc2UgcmlnaHQtY2Vuc29yZWQgYW5kIHJlcG9ydGVkIGFzID4gfFB8KS4gUmVwb3J0ZWQgZm9yIGV2ZXJ5IHJ1bGUgw5cgY3JpdGVyaW9uIGNvbWJpbmF0aW9uLiBUaGUgaGVhZGxpbmUgTiogaXMgKipSNjAgd2l0aCBjcml0ZXJpb25fb2ZmKiosIHRoZSBjb21iaW5hdGlvbiB0aGF0IHNlY3Rpb24gOCBzaG93cyB0byBiZSBmZWFzaWJsZS4NCg0KIyMgNy4gU3RhdGlzdGljcw0KDQpIMSBhbmQgSDI6IHRoZSBleGFjdCB0d28tc2lkZWQgV2lsY294b24gc2lnbmVkLXJhbmsgdGVzdCBvbiB0aGUgMTIgcGVyLWNsYXNzIHBhaXJlZCBkaWZmZXJlbmNlcyAoay1jZW50ZXIgbWludXMgcmFuZG9tKSBvZiBBVUxDX1RQUiBhbmQgQVVMQ19GUFIgKFIyMCwgb2ZmaWNpYWwgdGVzdCkuIEhvbG0gb3ZlciB0aGVzZSB0d28gcC12YWx1ZXMuIFRoZSB2MSBwcmFjdGljYWwgcnVsZSBmb3Igay1jZW50ZXIgaXMgcmV1c2VkIHVuY2hhbmdlZDogZmF2b3JhYmxlIHdoZW4gdGhlIG1lZGlhbiDOlEFVTENfVFBSID4gMCBhbmQgdGhlIG1lZGlhbiDOlEFVTENfRlBSIOKJpCArMC4wMS4gU2lnbi1mbGlwIHRlc3QgKDJeMTIgYXNzaWdubWVudHMpIGFuZCB0aGUgYm9vdHN0cmFwIG9mIHRoZSBtZWRpYW4gKDEwIDAwMCwgc2VlZHMgNDIgYW5kIDQzKSBhcyBzZW5zaXRpdml0eSBkaXNwbGF5cy4gV2l0aCAxMiBjbGFzc2VzIHRoZSBzbWFsbGVzdCBhdHRhaW5hYmxlIHR3by1zaWRlZCBXaWxjb3hvbiBwIGlzIDIvNDA5NiDiiYggMC4wMDA1Lg0KDQpUaGUgc2FtZSB0d28gdGVzdHMgcmVwZWF0ZWQgd2l0aCBSNjAgYW5kIHdpdGggRlBSX0ggYXJlIHJlcG9ydGVkIGFzIHNlY29uZGFyeS4NCg0KSDM6IHRoZSBwZXItY2xhc3MgbWVhbiBvZiBUUFIoUjIwKSDiiJIgVFBSKFI2MCkgYXQgTiA9IHxQfCAocmFuZG9tIHN0cmF0ZWd5OyBhdCB8UHwgYm90aCBzdHJhdGVnaWVzIGdpdmUgdGhlIHNhbWUgYmFuayksIGl0cyBtZWRpYW4gb3ZlciBjbGFzc2VzIGFnYWluc3QgMC4wMywgYW5kIGEgY2xhc3MgYm9vdHN0cmFwIGludGVydmFsIG9mIHRoZSBtZWRpYW4gKDEwIDAwMCwgc2VlZCA0NCkuIFRoZSBzYW1lIHF1YW50aXR5IGF0IE4gPSAzMiBpcyBzZWNvbmRhcnkuDQoNCkEgbm9uLXNpZ25pZmljYW50IHRlc3QgaXMgbm90IGV2aWRlbmNlIG9mIGVxdWl2YWxlbmNlLg0KDQojIyA4LiBGZWFzaWJpbGl0eSBiZWZvcmUgdGhlIHJ1bg0KDQpGb3IgYSBkZXRlY3RvciB3aXRoIFRQUiA9IDEgYW5kIGV4Y2hhbmdlYWJsZSBnb29kIHNjb3JlcywgUChhIHJlcGV0aXRpb24gbWVldHMgdGhlIEZQUiBidWRnZXQpID0gQmV0YUJpbm9taWFsQ0RGKGJ1ZGdldDsgbiwgMSwgbSksIHdoZXJlIG0gaXMgdGhlIHNpemUgb2YgdGhlIGNhbGlicmF0aW9uIHNldC4gRXhwZWN0ZWQgcGFzc2luZyByZXBldGl0aW9ucyBvZiAyMCwgYW5kIFAoYXQgbGVhc3QgMTYgb2YgMjApIHVuZGVyIGFuIGluZGVwZW5kZW5jZSBhcHByb3hpbWF0aW9uIChyZXBldGl0aW9ucyBzaGFyZSB0aGUgdGVzdCBzZXQpOg0KDQp8IHRlc3Qgc2V0IHwgcnVsZSB8IGV4cGVjdGVkIG9mIDIwIHwgUCjiiaUgMTYpIHwNCnwtLS18LS0tfC0tLXwtLS18DQp8IG9mZmljaWFsIGdvb2QsIG4gPSA1MCB8IFIyMCB8IDEyLjggfCAwLjEwIHwNCnwgb2ZmaWNpYWwgZ29vZCwgbiA9IDEwMOKAkzEwMSB8IFIyMCB8IDEzLjTigJMxMy41IHwgMC4xNuKAkzAuMTcgfA0KfCBvZmZpY2lhbCBnb29kLCBuID0gNTDigJMxMDEgfCBSNjAgfCAxOC4y4oCTMTguOSB8IDAuOTfigJMxLjAwIHwNCnwgSCwgfEh8ID0gMTUw4oCTNjA1IHwgUjIwIHwgMTIuN+KAkzEzLjEgfCAwLjA54oCTMC4xMiB8DQp8IEgsIHxIfCA9IDE1MOKAkzYwNSB8IFI2MCB8IDE4LjfigJMxOS4wIHwgMC45OeKAkzEuMDAgfA0KDQpTbyBOKiB3aXRoIFIyMCBpcyBwcmVkaWN0ZWQgdG8gYmUgY2Vuc29yZWQgYnkgZGVzaWduIChQMSksIGFuZCBOKiB3aXRoIFI2MCBpcyBsaW1pdGVkIG9ubHkgYnkgdGhlIGRldGVjdG9yLg0KDQojIyA5LiBFeGNoYW5nZWFiaWxpdHkgY2hlY2sNCg0KQmFuayA9IFUgKGFsbCAzMDAgaW1hZ2VzKS4gU2NvcmVzIG9mIEggYWdhaW5zdCBVIGFuZCBvZiB0aGUgb2ZmaWNpYWwgZ29vZCB0ZXN0IGFnYWluc3QgVS4gUGVyIGNsYXNzOiBBVVJPQyBvZiBvZmZpY2lhbC1nb29kIHZlcnN1cyBIIHNjb3JlcyAoMC41ID0gbm8gc2hpZnQpLCB0d28tc2lkZWQgTWFubuKAk1doaXRuZXkgcCB3aXRoIEhvbG0gb3ZlciAxMiBjbGFzc2VzLCBtZWRpYW5zLCBhbmQgdGhlIGZyYWN0aW9uIG9mIG9mZmljaWFsIGdvb2QgaW1hZ2VzIHdpdGggY29uZm9ybWFsIHAg4omkIDAuMDUgYWdhaW5zdCBILiBUaGlzIHJlcGxhY2VzIHRoZSBsZWF2ZS1vbmUtb3V0IGNvbnN0cnVjdGlvbiBvZiBhZGRlbmR1bSBFMzsgSCBhbmQgVSBhcmUgZGlzam9pbnQsIHNvIG5vIGltYWdlIGlzIHNjb3JlZCBhZ2FpbnN0IGl0c2VsZi4NCg0KIyMgMTAuIFNlY29uZGFyeSBhbmFseXNlcw0KDQpTdG9wcGluZyBydWxlOiB2MSBzZWN0aW9uIDYgdW5jaGFuZ2VkLCBjb21wdXRlZCBvbiBDMjAsIM60IOKIiCB7MC4wMjUsIDAuMDUsIDAuMTB9LCBtYWluIDAuMDUsIGNvbXBhcmVkIHdpdGggTiogb2YgUjIwIGFuZCBvZiBSNjAuDQoNCkJ1ZGdldCBzcGxpdCAoZnJvbSBhZGRlbmR1bSBFMiwgbm93IHByZS1kZWNsYXJlZCk6IHBlciBjbGFzcyBhbmQgcmVwZXRpdGlvbiwgYSByYW5kb20gcGVybXV0YXRpb24gb2YgVSBmcm9tIGBTZWVkU2VxdWVuY2UoWzQyLCAxMDAwICsgY2xhc3NfaW5kZXgsIHIsIDIwMjYxMTIwXSlgOyBmb3IgQiDiiIggezgwLCAxMjAsIDE2MCwgMjAwLCAzMDB9LCBiYW5rID0gZmlyc3QgTiwgY2FsaWJyYXRpb24gPSBuZXh0IEIg4oiSIE4sIE4g4oiIIHsxLCAyLCA0LCA4LCAxNiwgMzIsIDY0LCAxMjh9IHdpdGggQiDiiJIgTiDiiaUgMjA7IHJ1bGVzOiBjb25mb3JtYWwgcmFuayBhbmQgUEFDIHJhbmsgKM61ID0gMC4wNSwgzrQgPSAwLjEwKTsgZXZhbHVhdGVkIG9uIHRoZSBvZmZpY2lhbCB0ZXN0IGFuZCBvbiBILiBIeXBvdGhlc2lzIGNhcnJpZWQgb3ZlciBmcm9tIE1WVGVjLCBkZXNjcmlwdGl2ZTogZm9yIEIg4omlIDEyMCB0aGUgc3BsaXQgd2l0aCB0aGUgbW9zdCBwYXNzaW5nIHJlcGV0aXRpb25zIHJlc2VydmVzIGF0IGxlYXN0IDQwIGltYWdlcyBmb3IgY2FsaWJyYXRpb24gaW4gbW9zdCBjbGFzc2VzLg0KDQojIyAxMS4gRTAgZ2F0ZQ0KDQpBcyB2MSBzZWN0aW9uIDgsIHdpdGggdGhlIGJhbmsgcmVzdHJpY3RlZCB0byBVOg0KDQotIEUwYTogQW5vbWFsaWIgUGF0Y2hDb3JlLCBjb3Jlc2V0IDAuMSwgYG51bV9uZWlnaGJvcnMgPSA5YCwgdHJhaW5lZCBvbiBVLCBvZmZpY2lhbCB0ZXN0LiBPcGVyYXRpb25hbCBmbG9vcjogbWVhbiBpbWFnZSBBVVJPQyDiiaUgMC45MC4gUHVibGlzaGVkIGZ1bGwtdHJhaW5pbmctc2V0IFBhdGNoQ29yZSByZXN1bHRzIG9uIFZpc0EgYXJlIGFyb3VuZCAwLjk1OyBhIHNtYWxsZXIgYmFuayBpcyBleHBlY3RlZCB0byBiZSBzbGlnaHRseSBsb3dlci4NCi0gRTBiOiBvd24gY2FjaGUgdmVyc3VzIEFub21hbGliLCBmdWxsIGJhbmsgVSwgYG51bV9uZWlnaGJvcnMgPSAxYDsgfM6UIG1lYW4gQVVST0N8IOKJpCAwLjAwNSBhbmQgU3BlYXJtYW4gz4Eg4omlIDAuOTkgaW4gZXZlcnkgY2xhc3MuDQoNCkludGVncml0eSBjaGVja3MgYXMgdjEsIHBsdXM6IHJvd3Mgb2YgSCBhbmQgb2YgdGhlIG9mZmljaWFsIHRlc3QgbmV2ZXIgYXBwZWFyIGFzIGNvbHVtbnMgb2YgdGhlIGRpc3RhbmNlIHRlbnNvci4NCg0KIyMgMTIuIENvbXB1dGUgYW5kIGFydGlmYWN0cw0KDQpGZWF0dXJlcyBhbmQgZGlzdGFuY2VzIG9uIGEgR1BVIChDb2xhYikuIFBlciBjbGFzcywgdGhlIGRpc3RhbmNlIHRlbnNvciBpcyBhdCBtb3N0IDExMDYgw5cgNzg0IMOXIDMwMCBmbG9hdDMyIOKJiCAxLjAgR0I7IGl0IGlzIGtlcHQgb24gdGhlIHJ1bnRpbWUncyBsb2NhbCBkaXNrIGFuZCBpcyBub3QgcmVxdWlyZWQgb24gRHJpdmUgYWZ0ZXIgdGhlIHRyYWplY3RvcmllcywgdGhlIGV4Y2hhbmdlYWJpbGl0eSBjaGVjaywgYW5kIHRoZSBidWRnZXQgc3BsaXQgYXJlIGZpbmlzaGVkLiBEcml2ZSBrZWVwcyBgc3BsaXRzL2AsIGBlMC9gLCBgcnVucy9wYXJ0cy9gLCBgcnVucy9zY29yZXMvYCwgYGFuYWx5c2lzL2AsIGByZXBvcnQvYCwgYW5kIGBlbnZpcm9ubWVudC5qc29uYC4gRXZlcnkgYXJ0aWZhY3Qgc3RvcmVzIHRoZSB2MiBjb25maWcgaGFzaC4NCg==", "pyproject.toml": "W2J1aWxkLXN5c3RlbV0NCnJlcXVpcmVzID0gWyJzZXR1cHRvb2xzPj02OCJdDQpidWlsZC1iYWNrZW5kID0gInNldHVwdG9vbHMuYnVpbGRfbWV0YSINCg0KW3Byb2plY3RdDQpuYW1lID0gImNvbW1pc3Npb25pbmciDQp2ZXJzaW9uID0gIjAuMS4wIg0KZGVzY3JpcHRpb24gPSAiRGVmZWN0LWZyZWUgY29tbWlzc2lvbmluZyBvZiBtZW1vcnktYmFuayBhbm9tYWx5IGRldGVjdGlvbiBvbiBNVlRlYyBBRCINCnJlcXVpcmVzLXB5dGhvbiA9ICI+PTMuMTAiDQpkZXBlbmRlbmNpZXMgPSBbDQogICAgIm51bXB5Pj0xLjI2IiwNCiAgICAic2NpcHk+PTEuMTEiLA0KICAgICJwYW5kYXM+PTIuMSIsDQogICAgInB5YXJyb3c+PTE0IiwNCiAgICAic2Npa2l0LWxlYXJuPj0xLjMiLA0KICAgICJtYXRwbG90bGliPj0zLjgiLA0KICAgICJQeVlBTUw+PTYiLA0KICAgICJQaWxsb3c+PTEwIiwNCiAgICAidHFkbT49NC42NiIsDQpdDQoNCltwcm9qZWN0Lm9wdGlvbmFsLWRlcGVuZGVuY2llc10NCmdwdSA9IFsidG9yY2g+PTIuMSIsICJ0b3JjaHZpc2lvbj49MC4xNiIsICJ0aW1tPj0wLjkiXQ0KYW5vbWFsaWIgPSBbImFub21hbGliPj0yLjAiXQ0KZGV2ID0gWyJweXRlc3Q+PTcuNCJdDQoNClt0b29sLnNldHVwdG9vbHMucGFja2FnZXMuZmluZF0NCmluY2x1ZGUgPSBbImNvbW1pc3Npb25pbmcqIl0NCg0KW3Rvb2wucHl0ZXN0LmluaV9vcHRpb25zXQ0KcHl0aG9ucGF0aCA9IFsiLiJdDQp0ZXN0cGF0aHMgPSBbInRlc3RzIl0NCg==", "commissioning/__init__.py": "IiIiRGVmZWN0LWZyZWUgY29tbWlzc2lvbmluZyBleHBlcmltZW50cyBmb3IgUGF0Y2hDb3JlIG9uIE1WVGVjIEFELiIiIg0KDQpfX3ZlcnNpb25fXyA9ICIwLjEuMCINCg==", "commissioning/__main__.py": "IiIiQ29tbWFuZCBsaW5lOiBweXRob24gLW0gY29tbWlzc2lvbmluZyB7cHJlcGFyZXxmZWF0dXJlc3xkaXN0YW5jZXN8ZTB8cnVufGFuYWx5emV8cmVwb3J0fGFsbHxleHBsb3JlfQ0KDQpleHBsb3JlIGlzIG5vdCBwYXJ0IG9mIGFsbC4gSXQgcnVucyB0aGUgcG9zdCBob2MgYW5hbHlzZXMgZnJvbSBleHBsb3JhdG9yeV9hZGRlbmR1bS5tZC4NCiIiIg0KDQpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zDQoNCmltcG9ydCBhcmdwYXJzZQ0KaW1wb3J0IGxvZ2dpbmcNCmltcG9ydCBwbGF0Zm9ybQ0KaW1wb3J0IHN5cw0KDQpmcm9tIGNvbW1pc3Npb25pbmcgaW1wb3J0IF9fdmVyc2lvbl9fDQpmcm9tIGNvbW1pc3Npb25pbmcuYW5hbHlzaXMgaW1wb3J0IGFuYWx5emUNCmZyb20gY29tbWlzc2lvbmluZy5jb25maWcgaW1wb3J0IGxvYWRfY29uZmlnDQpmcm9tIGNvbW1pc3Npb25pbmcuZGF0YSBpbXBvcnQgcHJlcGFyZQ0KZnJvbSBjb21taXNzaW9uaW5nLmRpc3RhbmNlcyBpbXBvcnQgY29tcHV0ZV9kaXN0YW5jZXMNCmZyb20gY29tbWlzc2lvbmluZy5lMCBpbXBvcnQgZ2F0ZV9wYXRoLCBydW5fZTANCmZyb20gY29tbWlzc2lvbmluZy5mZWF0dXJlcyBpbXBvcnQgZXh0cmFjdA0KZnJvbSBjb21taXNzaW9uaW5nLmlvX3V0aWxzIGltcG9ydCByZWFkX2pzb24sIHdyaXRlX2pzb24NCmZyb20gY29tbWlzc2lvbmluZy5yZXBvcnQgaW1wb3J0IGJ1aWxkX3JlcG9ydA0KZnJvbSBjb21taXNzaW9uaW5nLnJ1biBpbXBvcnQgcnVuDQoNCg0KZGVmIHdyaXRlX2Vudmlyb25tZW50KGNmZykgLT4gTm9uZToNCiAgICBpbmZvID0gew0KICAgICAgICAiY29tbWlzc2lvbmluZyI6IF9fdmVyc2lvbl9fLA0KICAgICAgICAicHl0aG9uIjogc3lzLnZlcnNpb24sDQogICAgICAgICJwbGF0Zm9ybSI6IHBsYXRmb3JtLnBsYXRmb3JtKCksDQogICAgICAgICJjb25maWdfaGFzaCI6IGNmZy5oYXNoLA0KICAgICAgICAiZGF0YV9yb290Ijogc3RyKGNmZy5kYXRhX3Jvb3QpLA0KICAgICAgICAid29ya19yb290Ijogc3RyKGNmZy53b3JrX3Jvb3QpLA0KICAgIH0NCiAgICBmb3IgbmFtZSBpbiAoIm51bXB5IiwgInNjaXB5IiwgInBhbmRhcyIsICJza2xlYXJuIiwgIm1hdHBsb3RsaWIiLCAiUElMIik6DQogICAgICAgIHRyeToNCiAgICAgICAgICAgIG1vZHVsZSA9IF9faW1wb3J0X18obmFtZSBpZiBuYW1lICE9ICJza2xlYXJuIiBlbHNlICJza2xlYXJuIikNCiAgICAgICAgICAgIGlmIG5hbWUgPT0gIlBJTCI6DQogICAgICAgICAgICAgICAgbW9kdWxlID0gX19pbXBvcnRfXygiUElMIikNCiAgICAgICAgICAgIGluZm9bbmFtZV0gPSBnZXRhdHRyKG1vZHVsZSwgIl9fdmVyc2lvbl9fIiwgInVua25vd24iKQ0KICAgICAgICBleGNlcHQgSW1wb3J0RXJyb3I6DQogICAgICAgICAgICBpbmZvW25hbWVdID0gTm9uZQ0KICAgIHRyeToNCiAgICAgICAgaW1wb3J0IHRvcmNoDQoNCiAgICAgICAgaW5mb1sidG9yY2giXSA9IHRvcmNoLl9fdmVyc2lvbl9fDQogICAgICAgIGluZm9bImN1ZGEiXSA9IGJvb2wodG9yY2guY3VkYS5pc19hdmFpbGFibGUoKSkNCiAgICAgICAgaW5mb1siZ3B1Il0gPSB0b3JjaC5jdWRhLmdldF9kZXZpY2VfbmFtZSgwKSBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpIGVsc2UgTm9uZQ0KICAgIGV4Y2VwdCBJbXBvcnRFcnJvcjoNCiAgICAgICAgaW5mb1sidG9yY2giXSA9IE5vbmUNCiAgICB0cnk6DQogICAgICAgIGltcG9ydCBhbm9tYWxpYg0KDQogICAgICAgIGluZm9bImFub21hbGliIl0gPSBnZXRhdHRyKGFub21hbGliLCAiX192ZXJzaW9uX18iLCAidW5rbm93biIpDQogICAgZXhjZXB0IEltcG9ydEVycm9yOg0KICAgICAgICBpbmZvWyJhbm9tYWxpYiJdID0gTm9uZQ0KICAgIHdyaXRlX2pzb24oY2ZnLndvcmtfcm9vdCAvICJlbnZpcm9ubWVudC5qc29uIiwgaW5mbykNCg0KDQpkZWYgX3BhcnNlcigpIC0+IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyOg0KICAgIHBhcnNlciA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKHByb2c9ImNvbW1pc3Npb25pbmciKQ0KICAgIHBhcnNlci5hZGRfYXJndW1lbnQoDQogICAgICAgICJzdGFnZSIsIGNob2ljZXM9WyJwcmVwYXJlIiwgImZlYXR1cmVzIiwgImRpc3RhbmNlcyIsICJlMCIsICJydW4iLCAiYW5hbHl6ZSIsICJyZXBvcnQiLCAiYWxsIiwgImV4cGxvcmUiXQ0KICAgICkNCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWNvbmZpZyIsIGRlZmF1bHQ9Tm9uZSkNCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWRhdGEtcm9vdCIsIGRlZmF1bHQ9Tm9uZSkNCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXdvcmstcm9vdCIsIGRlZmF1bHQ9Tm9uZSkNCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWNsYXNzZXMiLCBuYXJncz0iKiIsIGRlZmF1bHQ9Tm9uZSkNCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWRldmljZSIsIGRlZmF1bHQ9Tm9uZSkNCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWV4cGxvcmF0b3J5IiwgYWN0aW9uPSJzdG9yZV90cnVlIikNCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWUwLXBhcnQiLCBjaG9pY2VzPVsiYSIsICJiIiwgImFsbCJdLCBkZWZhdWx0PSJhbGwiKQ0KICAgIHJldHVybiBwYXJzZXINCg0KDQpkZWYgbWFpbihhcmd2OiBsaXN0W3N0cl0gfCBOb25lID0gTm9uZSkgLT4gTm9uZToNCiAgICBsb2dnaW5nLmJhc2ljQ29uZmlnKGxldmVsPWxvZ2dpbmcuSU5GTywgZm9ybWF0PSIlKGFzY3RpbWUpcyAlKGxldmVsbmFtZSlzICUobWVzc2FnZSlzIikNCiAgICBhcmdzID0gX3BhcnNlcigpLnBhcnNlX2FyZ3MoYXJndikNCiAgICBjZmcgPSBsb2FkX2NvbmZpZyhhcmdzLmNvbmZpZywgZGF0YV9yb290PWFyZ3MuZGF0YV9yb290LCB3b3JrX3Jvb3Q9YXJncy53b3JrX3Jvb3QpDQogICAgY2xhc3NlcyA9IGNmZy5zZWxlY3RfY2xhc3NlcyhhcmdzLmNsYXNzZXMpDQogICAgd3JpdGVfZW52aXJvbm1lbnQoY2ZnKQ0KICAgIGxvZ2dpbmcuaW5mbygiQ29uZmlnIGhhc2ggJXMiLCBjZmcuaGFzaCkNCiAgICBpZiBhcmdzLnN0YWdlIGluIHsicHJlcGFyZSIsICJhbGwifToNCiAgICAgICAgcHJlcGFyZShjZmcsIGNsYXNzZXMpDQogICAgaWYgYXJncy5zdGFnZSBpbiB7ImZlYXR1cmVzIiwgImFsbCJ9Og0KICAgICAgICBleHRyYWN0KGNmZywgY2xhc3NlcywgZGV2aWNlPWFyZ3MuZGV2aWNlKQ0KICAgIGlmIGFyZ3Muc3RhZ2UgaW4geyJkaXN0YW5jZXMiLCAiYWxsIn06DQogICAgICAgIGNvbXB1dGVfZGlzdGFuY2VzKGNmZywgY2xhc3NlcywgZGV2aWNlPWFyZ3MuZGV2aWNlKQ0KICAgIGlmIGFyZ3Muc3RhZ2UgaW4geyJlMCIsICJhbGwifToNCiAgICAgICAgZ2F0ZSA9IHJ1bl9lMChjZmcsIGNsYXNzZXMsIHBhcnQ9YXJncy5lMF9wYXJ0IGlmIGFyZ3Muc3RhZ2UgPT0gImUwIiBlbHNlICJhbGwiKQ0KICAgICAgICBpZiBhcmdzLnN0YWdlID09ICJhbGwiIGFuZCBnYXRlWyJzdGF0dXMiXSAhPSAiUEFTUyIgYW5kIG5vdCBhcmdzLmV4cGxvcmF0b3J5Og0KICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJFMCBkaWQgbm90IHBhc3MuIFRoZSByZW1haW5pbmcgc3RhZ2VzIHdlcmUgbm90IHN0YXJ0ZWQuIikNCiAgICBpZiBhcmdzLnN0YWdlIGluIHsicnVuIiwgImFsbCJ9Og0KICAgICAgICBydW4oY2ZnLCBjbGFzc2VzLCBleHBsb3JhdG9yeT1hcmdzLmV4cGxvcmF0b3J5KQ0KICAgIGlmIGFyZ3Muc3RhZ2UgaW4geyJhbmFseXplIiwgImFsbCJ9Og0KICAgICAgICBhbmFseXplKGNmZykNCiAgICBpZiBhcmdzLnN0YWdlIGluIHsicmVwb3J0IiwgImFsbCJ9Og0KICAgICAgICBmcm9tIGNvbW1pc3Npb25pbmcuYW5hbHlzaXMgaW1wb3J0IGxvYWRfYW5hbHlzaXMNCg0KICAgICAgICBmcmFtZXMgPSBsb2FkX2FuYWx5c2lzKGNmZy53b3JrX3Jvb3QgLyAiYW5hbHlzaXMiKQ0KICAgICAgICBnYXRlX2ZpbGUgPSBnYXRlX3BhdGgoY2ZnKQ0KICAgICAgICBnYXRlID0gcmVhZF9qc29uKGdhdGVfZmlsZSkgaWYgZ2F0ZV9maWxlLmlzX2ZpbGUoKSBlbHNlIE5vbmUNCiAgICAgICAgcGF0aCA9IGJ1aWxkX3JlcG9ydCgNCiAgICAgICAgICAgIGZyYW1lcywNCiAgICAgICAgICAgIGNmZy53b3JrX3Jvb3QgLyAicmVwb3J0IiwNCiAgICAgICAgICAgIGdhdGU9Z2F0ZSwNCiAgICAgICAgICAgIGRhdGFfcm9vdD1jZmcuZGF0YV9yb290LA0KICAgICAgICAgICAgd29ya19yb290PWNmZy53b3JrX3Jvb3QsDQogICAgICAgICAgICByZXNpemU9Y2ZnLnJlc2l6ZSwNCiAgICAgICAgICAgIGNyb3A9Y2ZnLmNlbnRlcl9jcm9wLA0KICAgICAgICApDQogICAgICAgIGxvZ2dpbmcuaW5mbygiUmVwb3J0IHdyaXR0ZW4gdG8gJXMiLCBwYXRoKQ0KICAgIGlmIGFyZ3Muc3RhZ2UgPT0gImV4cGxvcmUiOg0KICAgICAgICBmcm9tIGNvbW1pc3Npb25pbmcuZXhwbG9yYXRvcnkgaW1wb3J0IGV4cGxvcmUNCg0KICAgICAgICBleHBsb3JlKGNmZywgY2xhc3NlcykNCg0KDQppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOg0KICAgIG1haW4oKQ0K", "commissioning/analysis.py": "IiIiQWdncmVnYXRlIHRoZSB0cmFqZWN0b3J5IHRhYmxlIGludG8gQVVMQywgTiosIEZQUiBzdW1tYXJpZXMsIGFuZCBzdG9wcGluZyBvdXRjb21lcy4iIiINCg0KZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucw0KDQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgNCg0KaW1wb3J0IG51bXB5IGFzIG5wDQppbXBvcnQgcGFuZGFzIGFzIHBkDQoNCmZyb20gc2NpcHkuc3RhdHMgaW1wb3J0IGJldGFiaW5vbQ0KDQpmcm9tIGNvbW1pc3Npb25pbmcuY2FsaWJyYXRpb24gaW1wb3J0IGNsb3BwZXJfcGVhcnNvbl91cHBlciwgZmFsc2VfcmVqZWN0X2J1ZGdldA0KZnJvbSBjb21taXNzaW9uaW5nLmNvbmZpZyBpbXBvcnQgRVhQRUNURURfQ09VTlRTLCBDb25maWcNCmZyb20gY29tbWlzc2lvbmluZy5pb191dGlscyBpbXBvcnQgYXRvbWljX3BhcnF1ZXQsIHdyaXRlX2pzb24NCmZyb20gY29tbWlzc2lvbmluZy5zdGF0cyBpbXBvcnQgY29tcGFyZV9zdHJhdGVnaWVzDQpmcm9tIGNvbW1pc3Npb25pbmcuc3RvcHBpbmcgaW1wb3J0IG5fc3RvcA0KDQpNQUlOX0dSSUQgPSAoMSwgMiwgNCwgOCwgMTYsIDMyKQ0KDQoNCmRlZiBhdWxjKG5zLCB2YWx1ZXMsIG5fbWF4OiBpbnQgPSAzMikgLT4gZmxvYXQ6DQogICAgIiIiVHJhcGV6b2lkIG9uIGxvZzIoTiksIGRpdmlkZWQgYnkgbG9nMihuX21heCkgLSBsb2cyKDEpLiBQb2ludHMgYWJvdmUgbl9tYXggYXJlIGlnbm9yZWQuIiIiDQogICAgaWYgaW50KG5fbWF4KSAhPSAzMjoNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJUaGUgbWFpbiBBVUxDIG5vcm1hbGl6YXRpb24gaXMgZml4ZWQgYXQgbG9nMigzMikgLSBsb2cyKDEpLiIpDQogICAgcG9pbnRzID0ge2ludChuKTogZmxvYXQoeSkgZm9yIG4sIHkgaW4gemlwKG5zLCB2YWx1ZXMpfQ0KICAgIG1pc3NpbmcgPSBbbiBmb3IgbiBpbiBNQUlOX0dSSUQgaWYgbiBub3QgaW4gcG9pbnRzXQ0KICAgIGlmIG1pc3Npbmc6DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIkFVTEMgaXMgbWlzc2luZyBtYWluLWdyaWQgcG9pbnRzIHttaXNzaW5nfS4iKQ0KICAgIHhzID0gbnAubG9nMihucC5hc2FycmF5KE1BSU5fR1JJRCwgZHR5cGU9ZmxvYXQpKQ0KICAgIHlzID0gbnAuYXNhcnJheShbcG9pbnRzW25dIGZvciBuIGluIE1BSU5fR1JJRF0sIGR0eXBlPWZsb2F0KQ0KICAgIGFyZWEgPSBmbG9hdChucC5zdW0oKHlzWzotMV0gKyB5c1sxOl0pICogMC41ICogbnAuZGlmZih4cykpKQ0KICAgIHJldHVybiBhcmVhIC8gZmxvYXQobnAubG9nMihuX21heCkgLSBucC5sb2cyKDEuMCkpDQoNCg0KZGVmIG5fc3RhcihncmlkOiBsaXN0W2ludF0sIG5fcGFzc2VkOiBsaXN0W2ludF0sIG1pbl9wYXNzZXM6IGludCkgLT4gZGljdDoNCiAgICAiIiJTbWFsbGVzdCBOIGF0IHdoaWNoIGV2ZXJ5IGdyaWQgcG9pbnQgZnJvbSB0aGVyZSBvbiBoYXMgYXQgbGVhc3QgbWluX3Bhc3NlcyBzdWNjZXNzZXMuIiIiDQogICAgaWYgbGVuKGdyaWQpICE9IGxlbihuX3Bhc3NlZCk6DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiR3JpZCBhbmQgcGFzcyBjb3VudHMgbXVzdCBiZSBhbGlnbmVkLiIpDQogICAgb2sgPSBbaW50KGNvdW50KSA+PSBpbnQobWluX3Bhc3NlcykgZm9yIGNvdW50IGluIG5fcGFzc2VkXQ0KICAgIGZvciBpbmRleCwgbiBpbiBlbnVtZXJhdGUoZ3JpZCk6DQogICAgICAgIGlmIGFsbChva1tpbmRleDpdKToNCiAgICAgICAgICAgIHJldHVybiB7Im5fc3RhciI6IGludChuKSwgImNlbnNvcmVkIjogRmFsc2V9DQogICAgcmV0dXJuIHsibl9zdGFyIjogTm9uZSwgImNlbnNvcmVkIjogVHJ1ZX0NCg0KDQpkZWYgc3RvcF9jYXRlZ29yeShuX3N0b3BfdmFsdWUsIG5fc3Rhcl92YWx1ZSwgY2Vuc29yZWQ6IGJvb2wpIC0+IHN0cjoNCiAgICBpZiBjZW5zb3JlZDoNCiAgICAgICAgcmV0dXJuICJuX3N0YXJfbm90X3JlYWNoZWQiDQogICAgaWYgbl9zdG9wX3ZhbHVlIGlzIE5vbmUgb3IgKGlzaW5zdGFuY2Uobl9zdG9wX3ZhbHVlLCBmbG9hdCkgYW5kIG5wLmlzbmFuKG5fc3RvcF92YWx1ZSkpOg0KICAgICAgICByZXR1cm4gInJ1bGVfbm90X2ZpcmVkIg0KICAgIGlmIGludChuX3N0b3BfdmFsdWUpIDwgaW50KG5fc3Rhcl92YWx1ZSk6DQogICAgICAgIHJldHVybiAicHJlbWF0dXJlIg0KICAgIGlmIGludChuX3N0b3BfdmFsdWUpID09IGludChuX3N0YXJfdmFsdWUpOg0KICAgICAgICByZXR1cm4gImNvbmNvcmRhbnQiDQogICAgcmV0dXJuICJleGNlc3NpdmUiDQoNCg0KZGVmIGxvYWRfbWV0cmljcyhjZmc6IENvbmZpZykgLT4gcGQuRGF0YUZyYW1lOg0KICAgIHBhcnRzID0gc29ydGVkKChjZmcud29ya19yb290IC8gInJ1bnMiIC8gInBhcnRzIikuZ2xvYigiKi5wYXJxdWV0IikpDQogICAgaWYgbm90IHBhcnRzOg0KICAgICAgICByYWlzZSBGaWxlTm90Rm91bmRFcnJvcigiTm8gdHJhamVjdG9yeSB0YWJsZXMgaW4gcnVucy9wYXJ0cy4gUnVuIHRoZSB0cmFqZWN0b3JpZXMgZmlyc3QuIikNCiAgICBmcmFtZXMgPSBbcGQucmVhZF9wYXJxdWV0KHBhdGgpIGZvciBwYXRoIGluIHBhcnRzXQ0KICAgIG1ldHJpY3MgPSBwZC5jb25jYXQoZnJhbWVzLCBpZ25vcmVfaW5kZXg9VHJ1ZSkNCiAgICBoYXNoZXMgPSBzZXQobWV0cmljc1siY29uZmlnX2hhc2giXS5hc3R5cGUoc3RyKSkNCiAgICBpZiBoYXNoZXMgIT0ge2NmZy5oYXNofToNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYiVHJhamVjdG9yeSB0YWJsZXMgbWl4IGNvbmZpZyBoYXNoZXM6IHtoYXNoZXN9IikNCiAgICByZXR1cm4gbWV0cmljcw0KDQoNCmRlZiBzdW1tYXJpemVfY2xhc3NlcyhtZXRyaWNzOiBwZC5EYXRhRnJhbWUsIG5fbWF4OiBpbnQsIG1pbl9wYXNzZXM6IGludCkgLT4gcGQuRGF0YUZyYW1lOg0KICAgIHJvd3MgPSBbXQ0KICAgIGdyb3VwZWQgPSBtZXRyaWNzLmdyb3VwYnkoWyJjbGFzc19uYW1lIiwgImdyb3VwIiwgInN0cmF0ZWd5Il0sIHNvcnQ9VHJ1ZSkNCiAgICBmb3IgKGNsYXNzX25hbWUsIGdyb3VwLCBzdHJhdGVneSksIHN1YiBpbiBncm91cGVkOg0KICAgICAgICB0cHJfdmFsdWVzID0gW10NCiAgICAgICAgZnByX3ZhbHVlcyA9IFtdDQogICAgICAgIGZvciBfcmVwLCByZXBfZGYgaW4gc3ViLmdyb3VwYnkoInJlcCIpOg0KICAgICAgICAgICAgb3JkZXJlZCA9IHJlcF9kZi5zb3J0X3ZhbHVlcygiTiIpDQogICAgICAgICAgICB0cHJfdmFsdWVzLmFwcGVuZChhdWxjKG9yZGVyZWRbIk4iXSwgb3JkZXJlZFsidHByIl0sIG5fbWF4KSkNCiAgICAgICAgICAgIGZwcl92YWx1ZXMuYXBwZW5kKGF1bGMob3JkZXJlZFsiTiJdLCBvcmRlcmVkWyJmcHIiXSwgbl9tYXgpKQ0KICAgICAgICBncmlkID0gW2ludChuKSBmb3IgbiBpbiBzb3J0ZWQoc3ViWyJOIl0udW5pcXVlKCkpXQ0KICAgICAgICBwYXNzZWQgPSBbaW50KHN1Yi5sb2Nbc3ViWyJOIl0gPT0gbiwgImNyaXRlcmlvbiJdLnN1bSgpKSBmb3IgbiBpbiBncmlkXQ0KICAgICAgICBzdGFyID0gbl9zdGFyKGdyaWQsIHBhc3NlZCwgbWluX3Bhc3NlcykNCiAgICAgICAgcG9vbF9zaXplID0gaW50KHN1YlsicG9vbF9zaXplIl0uaWxvY1swXSkNCiAgICAgICAgcm93cy5hcHBlbmQoDQogICAgICAgICAgICB7DQogICAgICAgICAgICAgICAgImNsYXNzX25hbWUiOiBjbGFzc19uYW1lLA0KICAgICAgICAgICAgICAgICJncm91cCI6IGdyb3VwLA0KICAgICAgICAgICAgICAgICJzdHJhdGVneSI6IHN0cmF0ZWd5LA0KICAgICAgICAgICAgICAgICJhdWxjX3RwciI6IGZsb2F0KG5wLm1lYW4odHByX3ZhbHVlcykpLA0KICAgICAgICAgICAgICAgICJhdWxjX2ZwciI6IGZsb2F0KG5wLm1lYW4oZnByX3ZhbHVlcykpLA0KICAgICAgICAgICAgICAgICJhdWxjX3Rwcl9pcXIiOiBmbG9hdChucC5zdWJ0cmFjdCgqbnAucXVhbnRpbGUodHByX3ZhbHVlcywgWzAuNzUsIDAuMjVdKSkpLA0KICAgICAgICAgICAgICAgICJhdWxjX2Zwcl9pcXIiOiBmbG9hdChucC5zdWJ0cmFjdCgqbnAucXVhbnRpbGUoZnByX3ZhbHVlcywgWzAuNzUsIDAuMjVdKSkpLA0KICAgICAgICAgICAgICAgICJhdWxjX3Rwcl9yYW5nZSI6IGZsb2F0KG5wLm1heCh0cHJfdmFsdWVzKSAtIG5wLm1pbih0cHJfdmFsdWVzKSksDQogICAgICAgICAgICAgICAgImF1bGNfZnByX3JhbmdlIjogZmxvYXQobnAubWF4KGZwcl92YWx1ZXMpIC0gbnAubWluKGZwcl92YWx1ZXMpKSwNCiAgICAgICAgICAgICAgICAibl9zdGFyIjogKGZsb2F0KHN0YXJbIm5fc3RhciJdKSBpZiBub3Qgc3RhclsiY2Vuc29yZWQiXSBlbHNlIGZsb2F0KCJuYW4iKSksDQogICAgICAgICAgICAgICAgImNlbnNvcmVkIjogYm9vbChzdGFyWyJjZW5zb3JlZCJdKSwNCiAgICAgICAgICAgICAgICAicG9vbF9zaXplIjogcG9vbF9zaXplLA0KICAgICAgICAgICAgICAgICJuX3JlcHMiOiBpbnQoc3ViWyJyZXAiXS5udW5pcXVlKCkpLA0KICAgICAgICAgICAgfQ0KICAgICAgICApDQogICAgcmV0dXJuIHBkLkRhdGFGcmFtZShyb3dzKQ0KDQoNCmRlZiBmcHJfYWdncmVnYXRlKG1ldHJpY3M6IHBkLkRhdGFGcmFtZSkgLT4gcGQuRGF0YUZyYW1lOg0KICAgICIiIlNwcmVhZCBhY3Jvc3MgcmVwZXRpdGlvbnMuIFJlcGV0aXRpb25zIHNoYXJlIG9uZSB0ZXN0IHNldCwgc28gdGhlaXIgY291bnRzIGFyZSBub3QgcG9vbGVkLiIiIg0KICAgIHJvd3MgPSBbXQ0KICAgIGZvciAoY2xhc3NfbmFtZSwgc3RyYXRlZ3ksIG4pLCBzdWIgaW4gbWV0cmljcy5ncm91cGJ5KFsiY2xhc3NfbmFtZSIsICJzdHJhdGVneSIsICJOIl0sIHNvcnQ9VHJ1ZSk6DQogICAgICAgIGZwciA9IHN1YlsiZnByIl0udG9fbnVtcHkoZHR5cGU9ZmxvYXQpDQogICAgICAgIHExLCBxMyA9IG5wLnF1YW50aWxlKGZwciwgWzAuMjUsIDAuNzVdKQ0KICAgICAgICByb3dzLmFwcGVuZCgNCiAgICAgICAgICAgIHsNCiAgICAgICAgICAgICAgICAiY2xhc3NfbmFtZSI6IGNsYXNzX25hbWUsDQogICAgICAgICAgICAgICAgImdyb3VwIjogc3ViWyJncm91cCJdLmlsb2NbMF0sDQogICAgICAgICAgICAgICAgInN0cmF0ZWd5Ijogc3RyYXRlZ3ksDQogICAgICAgICAgICAgICAgIk4iOiBpbnQobiksDQogICAgICAgICAgICAgICAgIm1lYW5fZnByIjogZmxvYXQobnAubWVhbihmcHIpKSwNCiAgICAgICAgICAgICAgICAibWVkaWFuX2ZwciI6IGZsb2F0KG5wLm1lZGlhbihmcHIpKSwNCiAgICAgICAgICAgICAgICAiaXFyX2ZwciI6IGZsb2F0KHEzIC0gcTEpLA0KICAgICAgICAgICAgICAgICJtaW5fZnByIjogZmxvYXQobnAubWluKGZwcikpLA0KICAgICAgICAgICAgICAgICJtYXhfZnByIjogZmxvYXQobnAubWF4KGZwcikpLA0KICAgICAgICAgICAgICAgICJmcmFjdGlvbl9mcHJfYWJvdmVfMDUiOiBmbG9hdChucC5tZWFuKGZwciA+IDAuMDUpKSwNCiAgICAgICAgICAgICAgICAiZnJhY3Rpb25fbWVldGluZ19jcml0ZXJpb24iOiBmbG9hdChucC5tZWFuKHN1YlsiY3JpdGVyaW9uIl0uYXN0eXBlKGJvb2wpKSksDQogICAgICAgICAgICAgICAgIm5fcmVwcyI6IGludChsZW4oc3ViKSksDQogICAgICAgICAgICB9DQogICAgICAgICkNCiAgICByZXR1cm4gcGQuRGF0YUZyYW1lKHJvd3MpDQoNCg0KZGVmIHN0b3BwaW5nX3RhYmxlKG1ldHJpY3M6IHBkLkRhdGFGcmFtZSwgZGVsdGFzOiBsaXN0W2Zsb2F0XSkgLT4gcGQuRGF0YUZyYW1lOg0KICAgIHJvd3MgPSBbXQ0KICAgIGZvciAoY2xhc3NfbmFtZSwgc3RyYXRlZ3ksIHJlcCksIHN1YiBpbiBtZXRyaWNzLmdyb3VwYnkoWyJjbGFzc19uYW1lIiwgInN0cmF0ZWd5IiwgInJlcCJdLCBzb3J0PVRydWUpOg0KICAgICAgICBvcmRlcmVkID0gc3ViLnNvcnRfdmFsdWVzKCJOIikNCiAgICAgICAgZ3JpZCA9IFtpbnQobikgZm9yIG4gaW4gb3JkZXJlZFsiTiJdLnRvbGlzdCgpXQ0KICAgICAgICBtZWRpYW5zID0gb3JkZXJlZFsibWVkaWFuX2NhbGlicmF0aW9uIl0udG9fbnVtcHkoZHR5cGU9ZmxvYXQpDQogICAgICAgIGZvciBkZWx0YSBpbiBkZWx0YXM6DQogICAgICAgICAgICByb3dzLmFwcGVuZCgNCiAgICAgICAgICAgICAgICB7DQogICAgICAgICAgICAgICAgICAgICJjbGFzc19uYW1lIjogY2xhc3NfbmFtZSwNCiAgICAgICAgICAgICAgICAgICAgImdyb3VwIjogb3JkZXJlZFsiZ3JvdXAiXS5pbG9jWzBdLA0KICAgICAgICAgICAgICAgICAgICAic3RyYXRlZ3kiOiBzdHJhdGVneSwNCiAgICAgICAgICAgICAgICAgICAgInJlcCI6IGludChyZXApLA0KICAgICAgICAgICAgICAgICAgICAiZGVsdGEiOiBmbG9hdChkZWx0YSksDQogICAgICAgICAgICAgICAgICAgICJuX3N0b3AiOiBuX3N0b3AoZ3JpZCwgbWVkaWFucywgZmxvYXQoZGVsdGEpKSwNCiAgICAgICAgICAgICAgICAgICAgInBvb2xfc2l6ZSI6IGludChvcmRlcmVkWyJwb29sX3NpemUiXS5pbG9jWzBdKSwNCiAgICAgICAgICAgICAgICB9DQogICAgICAgICAgICApDQogICAgcmV0dXJuIHBkLkRhdGFGcmFtZShyb3dzKQ0KDQoNCmRlZiBidWRnZXRfdGFibGUoKSAtPiBwZC5EYXRhRnJhbWU6DQogICAgcm93cyA9IFtdDQogICAgZm9yIGNsYXNzX25hbWUsIGNvdW50cyBpbiBFWFBFQ1RFRF9DT1VOVFMuaXRlbXMoKToNCiAgICAgICAgbl9nb29kID0gaW50KGNvdW50c1sidGVzdF9nb29kIl0pDQogICAgICAgIGJ1ZGdldCA9IGZhbHNlX3JlamVjdF9idWRnZXQobl9nb29kKQ0KICAgICAgICBrX21pbiA9IGJ1ZGdldCArIDENCiAgICAgICAgZXhwbG9yYXRvcnlfcHJvYmFiaWxpdHkgPSBmbG9hdCgxLjAgLSBiZXRhYmlub20uY2RmKGtfbWluIC0gMSwgbl9nb29kLCAxLCAyMCkpDQogICAgICAgIHJvd3MuYXBwZW5kKA0KICAgICAgICAgICAgew0KICAgICAgICAgICAgICAgICJjbGFzc19uYW1lIjogY2xhc3NfbmFtZSwNCiAgICAgICAgICAgICAgICAibl90ZXN0X2dvb2QiOiBuX2dvb2QsDQogICAgICAgICAgICAgICAgIm5fdHJhaW4iOiBpbnQoY291bnRzWyJ0cmFpbiJdKSwNCiAgICAgICAgICAgICAgICAiZmFsc2VfcmVqZWN0X2J1ZGdldCI6IGJ1ZGdldCwNCiAgICAgICAgICAgICAgICAiY2xvcHBlcl91cHBlcl96ZXJvX2Vycm9ycyI6IGNsb3BwZXJfcGVhcnNvbl91cHBlcigwLCBuX2dvb2QpLA0KICAgICAgICAgICAgICAgICJleHBsb3JhdG9yeV9wX3NhbXBsZV9mcHJfYWJvdmVfMDUiOiBleHBsb3JhdG9yeV9wcm9iYWJpbGl0eSwNCiAgICAgICAgICAgIH0NCiAgICAgICAgKQ0KICAgIHJldHVybiBwZC5EYXRhRnJhbWUocm93cykNCg0KDQpkZWYgX21lZGlhbl9vcl9ub25lKHZhbHVlczogbGlzdFtpbnRdKSAtPiBmbG9hdCB8IE5vbmU6DQogICAgaWYgbm90IHZhbHVlczoNCiAgICAgICAgcmV0dXJuIE5vbmUNCiAgICByZXR1cm4gZmxvYXQobnAubWVkaWFuKG5wLmFzYXJyYXkodmFsdWVzLCBkdHlwZT1mbG9hdCkpKQ0KDQoNCmRlZiBhdHRhY2hfc3RvcHBpbmcoc3VtbWFyeTogcGQuRGF0YUZyYW1lLCBzdG9wcGluZzogcGQuRGF0YUZyYW1lLCBtYWluX2RlbHRhOiBmbG9hdCkgLT4gcGQuRGF0YUZyYW1lOg0KICAgIG1haW4gPSBzdG9wcGluZ1tucC5pc2Nsb3NlKHN0b3BwaW5nWyJkZWx0YSJdLCBtYWluX2RlbHRhKV0uY29weSgpDQogICAgcm93cyA9IFtdDQogICAgZm9yIHJvdyBpbiBzdW1tYXJ5Lml0ZXJ0dXBsZXMoaW5kZXg9RmFsc2UpOg0KICAgICAgICBzdWIgPSBtYWluWyhtYWluWyJjbGFzc19uYW1lIl0gPT0gcm93LmNsYXNzX25hbWUpICYgKG1haW5bInN0cmF0ZWd5Il0gPT0gcm93LnN0cmF0ZWd5KV0NCiAgICAgICAgZmlyZWQgPSBbaW50KHYpIGZvciB2IGluIHN1Ylsibl9zdG9wIl0uZHJvcG5hKCkudG9saXN0KCldDQogICAgICAgIGNhdGVnb3JpZXMgPSBbDQogICAgICAgICAgICBzdG9wX2NhdGVnb3J5KE5vbmUgaWYgcGQuaXNuYSh2KSBlbHNlIGludCh2KSwgTm9uZSBpZiByb3cuY2Vuc29yZWQgZWxzZSBpbnQocm93Lm5fc3RhciksIGJvb2wocm93LmNlbnNvcmVkKSkNCiAgICAgICAgICAgIGZvciB2IGluIHN1Ylsibl9zdG9wIl0udG9saXN0KCkNCiAgICAgICAgXQ0KICAgICAgICByYXRpbyA9IFtdDQogICAgICAgIGV4Y2VzcyA9IFtdDQogICAgICAgIGlmIG5vdCByb3cuY2Vuc29yZWQ6DQogICAgICAgICAgICByYXRpbyA9IFt2IC8gaW50KHJvdy5uX3N0YXIpIGZvciB2IGluIGZpcmVkXQ0KICAgICAgICAgICAgZXhjZXNzID0gW21heCgwLCB2IC0gaW50KHJvdy5uX3N0YXIpKSBmb3IgdiBpbiBmaXJlZF0NCiAgICAgICAgcmVjb3JkID0gcm93Ll9hc2RpY3QoKQ0KICAgICAgICByZWNvcmRbIm5fc3RvcF9tZWRpYW4iXSA9IF9tZWRpYW5fb3Jfbm9uZShmaXJlZCkNCiAgICAgICAgcmVjb3JkWyJuX3N0b3BfcmF0aW9fbWVkaWFuIl0gPSBmbG9hdChucC5tZWRpYW4ocmF0aW8pKSBpZiByYXRpbyBlbHNlIE5vbmUNCiAgICAgICAgcmVjb3JkWyJleGNlc3NfaW1hZ2VzX21lZGlhbiJdID0gZmxvYXQobnAubWVkaWFuKGV4Y2VzcykpIGlmIGV4Y2VzcyBlbHNlIE5vbmUNCiAgICAgICAgZm9yIG5hbWUgaW4gKCJwcmVtYXR1cmUiLCAiY29uY29yZGFudCIsICJleGNlc3NpdmUiLCAibl9zdGFyX25vdF9yZWFjaGVkIiwgInJ1bGVfbm90X2ZpcmVkIik6DQogICAgICAgICAgICByZWNvcmRbZiJmcmFjdGlvbl97bmFtZX0iXSA9IGZsb2F0KG5wLm1lYW4oW2MgPT0gbmFtZSBmb3IgYyBpbiBjYXRlZ29yaWVzXSkpIGlmIGNhdGVnb3JpZXMgZWxzZSBOb25lDQogICAgICAgIHJvd3MuYXBwZW5kKHJlY29yZCkNCiAgICByZXR1cm4gcGQuRGF0YUZyYW1lKHJvd3MpDQoNCg0KZGVmIGFuYWx5emUoY2ZnOiBDb25maWcpIC0+IGRpY3Q6DQogICAgbWV0cmljcyA9IGxvYWRfbWV0cmljcyhjZmcpDQogICAgc3VtbWFyeSA9IHN1bW1hcml6ZV9jbGFzc2VzKG1ldHJpY3MsIGNmZy5hdWxjX25fbWF4LCBjZmcubl9zdGFyX21pbl9yZXBlYXRzKQ0KICAgIHN0b3BwaW5nID0gc3RvcHBpbmdfdGFibGUobWV0cmljcywgY2ZnLnN0b3BwaW5nX2RlbHRhcykNCiAgICBzdW1tYXJ5ID0gYXR0YWNoX3N0b3BwaW5nKHN1bW1hcnksIHN0b3BwaW5nLCBjZmcuc3RvcHBpbmdfZGVsdGFfbWFpbikNCiAgICBhZ2dyZWdhdGUgPSBmcHJfYWdncmVnYXRlKG1ldHJpY3MpDQogICAgc3RhdGlzdGljcyA9IGNvbXBhcmVfc3RyYXRlZ2llcygNCiAgICAgICAgc3VtbWFyeSwNCiAgICAgICAgbWFyZ2luPWNmZy5mcHJfYXVsY19tYXJnaW4sDQogICAgICAgIG5fYm9vdHN0cmFwPWNmZy5ib290c3RyYXBfcmVwbGljYXRlcywNCiAgICAgICAgc2VlZD1jZmcuYm9vdHN0cmFwX3NlZWQsDQogICAgKQ0KICAgIG91dCA9IGNmZy53b3JrX3Jvb3QgLyAiYW5hbHlzaXMiDQogICAgb3V0Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkNCiAgICBhdG9taWNfcGFycXVldChvdXQgLyAibWV0cmljcy5wYXJxdWV0IiwgbWV0cmljcykNCiAgICBzdW1tYXJ5LnRvX2NzdihvdXQgLyAic3VtbWFyeS5jc3YiLCBpbmRleD1GYWxzZSkNCiAgICBhZ2dyZWdhdGUudG9fY3N2KG91dCAvICJmcHJfYWdncmVnYXRlLmNzdiIsIGluZGV4PUZhbHNlKQ0KICAgIHN0b3BwaW5nLnRvX2NzdihvdXQgLyAic3RvcHBpbmcuY3N2IiwgaW5kZXg9RmFsc2UpDQogICAgYnVkZ2V0X3RhYmxlKCkudG9fY3N2KG91dCAvICJmYWxzZV9yZWplY3RfYnVkZ2V0LmNzdiIsIGluZGV4PUZhbHNlKQ0KICAgIHdyaXRlX2pzb24ob3V0IC8gInN0YXRpc3RpY3MuanNvbiIsIHN0YXRpc3RpY3MpDQogICAgcmV0dXJuIHsNCiAgICAgICAgIm1ldHJpY3MiOiBtZXRyaWNzLA0KICAgICAgICAic3VtbWFyeSI6IHN1bW1hcnksDQogICAgICAgICJmcHJfYWdncmVnYXRlIjogYWdncmVnYXRlLA0KICAgICAgICAic3RvcHBpbmciOiBzdG9wcGluZywNCiAgICAgICAgInN0YXRpc3RpY3MiOiBzdGF0aXN0aWNzLA0KICAgICAgICAiYnVkZ2V0IjogYnVkZ2V0X3RhYmxlKCksDQogICAgICAgICJhbmFseXNpc19kaXIiOiBvdXQsDQogICAgfQ0KDQoNCmRlZiBsb2FkX2FuYWx5c2lzKGFuYWx5c2lzX2RpcjogUGF0aCkgLT4gZGljdDoNCiAgICByZXR1cm4gew0KICAgICAgICAibWV0cmljcyI6IHBkLnJlYWRfcGFycXVldChhbmFseXNpc19kaXIgLyAibWV0cmljcy5wYXJxdWV0IiksDQogICAgICAgICJzdW1tYXJ5IjogcGQucmVhZF9jc3YoYW5hbHlzaXNfZGlyIC8gInN1bW1hcnkuY3N2IiksDQogICAgICAgICJmcHJfYWdncmVnYXRlIjogcGQucmVhZF9jc3YoYW5hbHlzaXNfZGlyIC8gImZwcl9hZ2dyZWdhdGUuY3N2IiksDQogICAgICAgICJzdG9wcGluZyI6IHBkLnJlYWRfY3N2KGFuYWx5c2lzX2RpciAvICJzdG9wcGluZy5jc3YiKSwNCiAgICAgICAgImJ1ZGdldCI6IHBkLnJlYWRfY3N2KGFuYWx5c2lzX2RpciAvICJmYWxzZV9yZWplY3RfYnVkZ2V0LmNzdiIpLA0KICAgICAgICAic3RhdGlzdGljcyI6IF9faW1wb3J0X18oImpzb24iKS5sb2FkcygoYW5hbHlzaXNfZGlyIC8gInN0YXRpc3RpY3MuanNvbiIpLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKSksDQogICAgfQ0K", "commissioning/calibration.py": "IiIiVGhyZXNob2xkIGZyb20gZ29vZCBpbWFnZXMgb25seSwgYW5kIHRoZSBtZXRyaWNzIGNvbXB1dGVkIG9uIHRoZSBvZmZpY2lhbCB0ZXN0IHNldC4iIiINCg0KZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucw0KDQppbXBvcnQgbWF0aA0KDQppbXBvcnQgbnVtcHkgYXMgbnANCmZyb20gc2NpcHkuc3RhdHMgaW1wb3J0IGJldGENCmZyb20gc2tsZWFybi5tZXRyaWNzIGltcG9ydCByb2NfYXVjX3Njb3JlDQoNCg0KZGVmIG9yZGVyX3N0YXRpc3RpY19yYW5rKG46IGludCwgYWxwaGE6IGZsb2F0KSAtPiBpbnQ6DQogICAgIiIiMS1iYXNlZCByYW5rIGsgPSBjZWlsKChuICsgMSkgKiAoMSAtIGFscGhhKSksIGNsaXBwZWQgdG8gMS4ubi4iIiINCiAgICBpZiBuIDwgMToNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJDYWxpYnJhdGlvbiBzZXQgaXMgZW1wdHkuIikNCiAgICBpZiBub3QgMC4wIDwgYWxwaGEgPCAxLjA6DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmImFscGhhIG11c3QgYmUgaW4gKDAsIDEpLCBnb3Qge2FscGhhfSIpDQogICAgayA9IGludChtYXRoLmNlaWwoKG4gKyAxKSAqICgxLjAgLSBhbHBoYSkpKQ0KICAgIHJldHVybiBtaW4obWF4KGssIDEpLCBuKQ0KDQoNCmRlZiB0aHJlc2hvbGRfZnJvbV9zY29yZXMoc2NvcmVzOiBucC5uZGFycmF5LCBhbHBoYTogZmxvYXQpIC0+IHR1cGxlW2Zsb2F0LCBpbnRdOg0KICAgIHZhbHVlcyA9IG5wLmFzYXJyYXkoc2NvcmVzLCBkdHlwZT1ucC5mbG9hdDY0KQ0KICAgIGlmIHZhbHVlcy5uZGltICE9IDEgb3Igbm90IG5wLmlzZmluaXRlKHZhbHVlcykuYWxsKCk6DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiQ2FsaWJyYXRpb24gc2NvcmVzIG11c3QgYmUgYSBmaW5pdGUgMS1EIHZlY3Rvci4iKQ0KICAgIGsgPSBvcmRlcl9zdGF0aXN0aWNfcmFuayhsZW4odmFsdWVzKSwgYWxwaGEpDQogICAgb3JkZXJlZCA9IG5wLnNvcnQodmFsdWVzKQ0KICAgIHJldHVybiBmbG9hdChvcmRlcmVkW2sgLSAxXSksIGsNCg0KDQpkZWYgY2xvcHBlcl9wZWFyc29uX2xvd2VyKHN1Y2Nlc3NlczogaW50LCBuOiBpbnQsIGFscGhhOiBmbG9hdCA9IDAuMDUpIC0+IGZsb2F0Og0KICAgICIiIk9uZS1zaWRlZCBsb3dlciAxMDAqKDEtYWxwaGEpJSBDbG9wcGVyLVBlYXJzb24gYm91bmQuIiIiDQogICAgaWYgbiA8IDAgb3Igc3VjY2Vzc2VzIDwgMCBvciBzdWNjZXNzZXMgPiBuOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJJbnZhbGlkIGJpbm9taWFsIGNvdW50cyBrPXtzdWNjZXNzZXN9LCBuPXtufSIpDQogICAgaWYgbiA9PSAwOg0KICAgICAgICByZXR1cm4gZmxvYXQoIm5hbiIpDQogICAgaWYgc3VjY2Vzc2VzID09IDA6DQogICAgICAgIHJldHVybiAwLjANCiAgICByZXR1cm4gZmxvYXQoYmV0YS5wcGYoYWxwaGEsIHN1Y2Nlc3NlcywgbiAtIHN1Y2Nlc3NlcyArIDEpKQ0KDQoNCmRlZiBjbG9wcGVyX3BlYXJzb25fdXBwZXIoc3VjY2Vzc2VzOiBpbnQsIG46IGludCwgYWxwaGE6IGZsb2F0ID0gMC4wNSkgLT4gZmxvYXQ6DQogICAgIiIiT25lLXNpZGVkIHVwcGVyIDEwMCooMS1hbHBoYSklIENsb3BwZXItUGVhcnNvbiBib3VuZC4iIiINCiAgICBpZiBuIDwgMCBvciBzdWNjZXNzZXMgPCAwIG9yIHN1Y2Nlc3NlcyA+IG46DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIkludmFsaWQgYmlub21pYWwgY291bnRzIGs9e3N1Y2Nlc3Nlc30sIG49e259IikNCiAgICBpZiBuID09IDA6DQogICAgICAgIHJldHVybiBmbG9hdCgibmFuIikNCiAgICBpZiBzdWNjZXNzZXMgPT0gbjoNCiAgICAgICAgcmV0dXJuIDEuMA0KICAgIHJldHVybiBmbG9hdChiZXRhLnBwZigxLjAgLSBhbHBoYSwgc3VjY2Vzc2VzICsgMSwgbiAtIHN1Y2Nlc3NlcykpDQoNCg0KZGVmIGZhbHNlX3JlamVjdF9idWRnZXQobl9nb29kOiBpbnQpIC0+IGludDoNCiAgICAiIiJMYXJnZXN0IGsgd2l0aCBrIC8gbl9nb29kIDw9IDAuMDUuIEludGVnZXIgYXJpdGhtZXRpYyBhdm9pZHMgYSBiaW5hcnkgcm91bmRpbmcgZWRnZS4iIiINCiAgICBpZiBuX2dvb2QgPCAwOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJuX2dvb2QgbXVzdCBiZSBub24tbmVnYXRpdmUsIGdvdCB7bl9nb29kfSIpDQogICAgcmV0dXJuIChpbnQobl9nb29kKSAqIDUpIC8vIDEwMA0KDQoNCmRlZiBvcmFjbGVfZnByX2F0X3RwcihzY29yZXM6IG5wLm5kYXJyYXksIGxhYmVsczogbnAubmRhcnJheSwgdGFyZ2V0X3RwcjogZmxvYXQgPSAwLjk1KSAtPiBmbG9hdDoNCiAgICAiIiJMb3dlc3QgdGVzdC1zZXQgRlBSIGFtb25nIHRocmVzaG9sZHMgd2l0aCBUUFIgPj0gdGFyZ2V0LiBVc2VzIHRoZSBsYWJlbHM7IG5vdCBvcGVyYXRpb25hbC4iIiINCiAgICBzY29yZXMgPSBucC5hc2FycmF5KHNjb3JlcywgZHR5cGU9bnAuZmxvYXQ2NCkNCiAgICBsYWJlbHMgPSBucC5hc2FycmF5KGxhYmVscywgZHR5cGU9aW50KQ0KICAgIGlmIHNjb3Jlcy5zaGFwZSAhPSBsYWJlbHMuc2hhcGUgb3Igc2NvcmVzLm5kaW0gIT0gMToNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJTY29yZXMgYW5kIGxhYmVscyBtdXN0IGJlIDEtRCBhbmQgYWxpZ25lZC4iKQ0KICAgIGlmIG5vdCBucC5pc2Zpbml0ZShzY29yZXMpLmFsbCgpOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIk5vbi1maW5pdGUgdGVzdCBzY29yZXMuIikNCiAgICBkZWZlY3QgPSBzY29yZXNbbGFiZWxzID09IDFdDQogICAgZ29vZCA9IHNjb3Jlc1tsYWJlbHMgPT0gMF0NCiAgICBpZiBkZWZlY3Quc2l6ZSA9PSAwIG9yIGdvb2Quc2l6ZSA9PSAwOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIk9yYWNsZSBGUFIgbmVlZHMgYm90aCBnb29kIGFuZCBkZWZlY3QgdGVzdCBpbWFnZXMuIikNCiAgICBjYW5kaWRhdGVzID0gbnAudW5pcXVlKG5wLmNvbmNhdGVuYXRlKFtzY29yZXMsIFtzY29yZXMubWluKCkgLSAxLjBdXSkpDQogICAgYmVzdCA9IDEuMA0KICAgIGZvdW5kID0gRmFsc2UNCiAgICBmb3IgdGhyZXNob2xkIGluIGNhbmRpZGF0ZXM6DQogICAgICAgIHRwciA9IGZsb2F0KG5wLm1lYW4oZGVmZWN0ID4gdGhyZXNob2xkKSkNCiAgICAgICAgaWYgdHByICsgMWUtMTIgPj0gdGFyZ2V0X3RwcjoNCiAgICAgICAgICAgIGZwciA9IGZsb2F0KG5wLm1lYW4oZ29vZCA+IHRocmVzaG9sZCkpDQogICAgICAgICAgICBpZiBub3QgZm91bmQgb3IgZnByIDwgYmVzdDoNCiAgICAgICAgICAgICAgICBiZXN0ID0gZnByDQogICAgICAgICAgICAgICAgZm91bmQgPSBUcnVlDQogICAgaWYgbm90IGZvdW5kOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIk5vIHRocmVzaG9sZCByZWFjaGVzIHRoZSB0YXJnZXQgVFBSLiIpDQogICAgcmV0dXJuIGJlc3QNCg0KDQpkZWYgX21lZXRzKGNvdW50OiBpbnQsIHRvdGFsOiBpbnQsIGxldmVsOiBmbG9hdCwgYXRfbGVhc3Q6IGJvb2wpIC0+IGJvb2w6DQogICAgIiIiQ29tcGFyZSBjb3VudCAvIHRvdGFsIHdpdGggYSBsZXZlbCBpbiBpbnRlZ2VyIGFyaXRobWV0aWMsIHNvIDE5LzIwID49IDAuOTUgYW5kIDMvNjAgPD0gMC4wNSBob2xkIGV4YWN0bHkuIiIiDQogICAgZnJvbSBmcmFjdGlvbnMgaW1wb3J0IEZyYWN0aW9uDQoNCiAgICByYXRlID0gRnJhY3Rpb24oaW50KGNvdW50KSwgaW50KHRvdGFsKSkNCiAgICBib3VuZCA9IEZyYWN0aW9uKHN0cihsZXZlbCkpDQogICAgcmV0dXJuIGJvb2wocmF0ZSA+PSBib3VuZCkgaWYgYXRfbGVhc3QgZWxzZSBib29sKHJhdGUgPD0gYm91bmQpDQoNCg0KZGVmIGV2YWx1YXRlX3Njb3JlcygNCiAgICB0ZXN0X3Njb3JlczogbnAubmRhcnJheSwNCiAgICBsYWJlbHM6IG5wLm5kYXJyYXksDQogICAgY2FsaWJyYXRpb25fc2NvcmVzOiBucC5uZGFycmF5LA0KICAgIGFscGhhOiBmbG9hdCwNCiAgICBwcmFjdGljYWxfdHByOiBmbG9hdCwNCiAgICBwcmFjdGljYWxfZnByOiBmbG9hdCwNCikgLT4gZGljdDoNCiAgICAiIiJEZWNpc2lvbiBydWxlOiBhIHRlc3QgaW1hZ2UgaXMgcmVqZWN0ZWQgd2hlbiBpdHMgc2NvcmUgaXMgc3RyaWN0bHkgYWJvdmUgbWF4KEMpIChmb3IgYWxwaGE9MC4wNSwgbj0yMCkuIiIiDQogICAgdGVzdF9zY29yZXMgPSBucC5hc2FycmF5KHRlc3Rfc2NvcmVzLCBkdHlwZT1ucC5mbG9hdDY0KQ0KICAgIGxhYmVscyA9IG5wLmFzYXJyYXkobGFiZWxzLCBkdHlwZT1pbnQpDQogICAgY2FsaWJyYXRpb25fc2NvcmVzID0gbnAuYXNhcnJheShjYWxpYnJhdGlvbl9zY29yZXMsIGR0eXBlPW5wLmZsb2F0NjQpDQogICAgaWYgbm90IG5wLmlzZmluaXRlKHRlc3Rfc2NvcmVzKS5hbGwoKSBvciBub3QgbnAuaXNmaW5pdGUoY2FsaWJyYXRpb25fc2NvcmVzKS5hbGwoKToNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJOb24tZmluaXRlIHNjb3JlczsgcmVmdXNpbmcgdG8gcmVwbGFjZSB0aGVtIHdpdGggYSBzZW50aW5lbCB2YWx1ZS4iKQ0KICAgIHRocmVzaG9sZCwgcmFuayA9IHRocmVzaG9sZF9mcm9tX3Njb3JlcyhjYWxpYnJhdGlvbl9zY29yZXMsIGFscGhhKQ0KICAgIGRlZmVjdCA9IHRlc3Rfc2NvcmVzW2xhYmVscyA9PSAxXQ0KICAgIGdvb2QgPSB0ZXN0X3Njb3Jlc1tsYWJlbHMgPT0gMF0NCiAgICBpZiBkZWZlY3Quc2l6ZSA9PSAwIG9yIGdvb2Quc2l6ZSA9PSAwOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIlRoZSBvZmZpY2lhbCB0ZXN0IHNwbGl0IG11c3QgY29udGFpbiBnb29kIGFuZCBkZWZlY3QgaW1hZ2VzLiIpDQogICAgbl90cCA9IGludChucC5zdW0oZGVmZWN0ID4gdGhyZXNob2xkKSkNCiAgICBuX2ZwID0gaW50KG5wLnN1bShnb29kID4gdGhyZXNob2xkKSkNCiAgICBuX2RlZmVjdCA9IGludChkZWZlY3Quc2l6ZSkNCiAgICBuX2dvb2QgPSBpbnQoZ29vZC5zaXplKQ0KICAgIHRwciA9IG5fdHAgLyBuX2RlZmVjdA0KICAgIGZwciA9IG5fZnAgLyBuX2dvb2QNCiAgICByZXR1cm4gew0KICAgICAgICAidGhyZXNob2xkIjogdGhyZXNob2xkLA0KICAgICAgICAidGhyZXNob2xkX3JhbmsiOiByYW5rLA0KICAgICAgICAibWVkaWFuX2NhbGlicmF0aW9uIjogZmxvYXQobnAubWVkaWFuKGNhbGlicmF0aW9uX3Njb3JlcykpLA0KICAgICAgICAidHByIjogZmxvYXQodHByKSwNCiAgICAgICAgInRwcl9sb3dlciI6IGNsb3BwZXJfcGVhcnNvbl9sb3dlcihuX3RwLCBuX2RlZmVjdCksDQogICAgICAgICJmcHIiOiBmbG9hdChmcHIpLA0KICAgICAgICAiZnByX3VwcGVyIjogY2xvcHBlcl9wZWFyc29uX3VwcGVyKG5fZnAsIG5fZ29vZCksDQogICAgICAgICJuX3RwIjogbl90cCwNCiAgICAgICAgIm5fZnAiOiBuX2ZwLA0KICAgICAgICAibl90ZXN0X2RlZmVjdCI6IG5fZGVmZWN0LA0KICAgICAgICAibl90ZXN0X2dvb2QiOiBuX2dvb2QsDQogICAgICAgICJhdXJvYyI6IGZsb2F0KHJvY19hdWNfc2NvcmUobGFiZWxzLCB0ZXN0X3Njb3JlcykpLA0KICAgICAgICAib3JhY2xlX2ZwciI6IG9yYWNsZV9mcHJfYXRfdHByKHRlc3Rfc2NvcmVzLCBsYWJlbHMsIHByYWN0aWNhbF90cHIpLA0KICAgICAgICAiY3JpdGVyaW9uIjogX21lZXRzKG5fdHAsIG5fZGVmZWN0LCBwcmFjdGljYWxfdHByLCBhdF9sZWFzdD1UcnVlKQ0KICAgICAgICBhbmQgX21lZXRzKG5fZnAsIG5fZ29vZCwgcHJhY3RpY2FsX2ZwciwgYXRfbGVhc3Q9RmFsc2UpLA0KICAgIH0NCg==", "commissioning/config.py": "IiIiTG9hZCB0aGUgZnJvemVuIGV4cGVyaW1lbnQgY29uZmlndXJhdGlvbiBhbmQgcmVzb2x2ZSBwYXRocy4iIiINCg0KZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucw0KDQppbXBvcnQgaGFzaGxpYg0KaW1wb3J0IGpzb24NCmltcG9ydCBvcw0KZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzDQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgNCmZyb20gdHlwaW5nIGltcG9ydCBBbnkNCg0KaW1wb3J0IHlhbWwNCg0KIyBTZWVkIGluZGljZXMgZm9sbG93IHRoaXMgb3JkZXIuIEl0IGlzIHBhcnQgb2YgdGhlIGhhc2hlZCBwYXlsb2FkIHZpYSBjb25maWcueWFtbC4NCk9CSkVDVFMgPSBbDQogICAgImJvdHRsZSIsDQogICAgImNhYmxlIiwNCiAgICAiY2Fwc3VsZSIsDQogICAgImhhemVsbnV0IiwNCiAgICAibWV0YWxfbnV0IiwNCiAgICAicGlsbCIsDQogICAgInNjcmV3IiwNCiAgICAidG9vdGhicnVzaCIsDQogICAgInRyYW5zaXN0b3IiLA0KICAgICJ6aXBwZXIiLA0KXQ0KVEVYVFVSRVMgPSBbImNhcnBldCIsICJncmlkIiwgImxlYXRoZXIiLCAidGlsZSIsICJ3b29kIl0NCg0KIyBPZmZpY2lhbCBNVlRlYyBBRCBjb3VudHMuIEEgbWlzbWF0Y2ggc3RvcHMgdGhlIHJ1biBpbnN0ZWFkIG9mIHNpbGVudGx5IHVzaW5nIHRoZSB3cm9uZyBzcGxpdC4NCkVYUEVDVEVEX0NPVU5UUzogZGljdFtzdHIsIGRpY3Rbc3RyLCBpbnRdXSA9IHsNCiAgICAiYm90dGxlIjogeyJ0cmFpbiI6IDIwOSwgInRlc3RfZ29vZCI6IDIwfSwNCiAgICAiY2FibGUiOiB7InRyYWluIjogMjI0LCAidGVzdF9nb29kIjogNTh9LA0KICAgICJjYXBzdWxlIjogeyJ0cmFpbiI6IDIxOSwgInRlc3RfZ29vZCI6IDIzfSwNCiAgICAiY2FycGV0IjogeyJ0cmFpbiI6IDI4MCwgInRlc3RfZ29vZCI6IDI4fSwNCiAgICAiZ3JpZCI6IHsidHJhaW4iOiAyNjQsICJ0ZXN0X2dvb2QiOiAyMX0sDQogICAgImhhemVsbnV0IjogeyJ0cmFpbiI6IDM5MSwgInRlc3RfZ29vZCI6IDQwfSwNCiAgICAibGVhdGhlciI6IHsidHJhaW4iOiAyNDUsICJ0ZXN0X2dvb2QiOiAzMn0sDQogICAgIm1ldGFsX251dCI6IHsidHJhaW4iOiAyMjAsICJ0ZXN0X2dvb2QiOiAyMn0sDQogICAgInBpbGwiOiB7InRyYWluIjogMjY3LCAidGVzdF9nb29kIjogMjZ9LA0KICAgICJzY3JldyI6IHsidHJhaW4iOiAzMjAsICJ0ZXN0X2dvb2QiOiA0MX0sDQogICAgInRpbGUiOiB7InRyYWluIjogMjMwLCAidGVzdF9nb29kIjogMzN9LA0KICAgICJ0b290aGJydXNoIjogeyJ0cmFpbiI6IDYwLCAidGVzdF9nb29kIjogMTJ9LA0KICAgICJ0cmFuc2lzdG9yIjogeyJ0cmFpbiI6IDIxMywgInRlc3RfZ29vZCI6IDYwfSwNCiAgICAid29vZCI6IHsidHJhaW4iOiAyNDcsICJ0ZXN0X2dvb2QiOiAxOX0sDQogICAgInppcHBlciI6IHsidHJhaW4iOiAyNDAsICJ0ZXN0X2dvb2QiOiAzMn0sDQp9DQoNCiMgTGFyZ2VzdCBrIHdpdGggayAvIG5fZ29vZCA8PSAwLjA1IG9uIHRoZSBvZmZpY2lhbCBnb29kIHRlc3QgaW1hZ2VzLg0KRkFMU0VfUkVKRUNUX0JVREdFVDogZGljdFtzdHIsIGludF0gPSB7DQogICAgInRvb3RoYnJ1c2giOiAwLA0KICAgICJ3b29kIjogMCwNCiAgICAiYm90dGxlIjogMSwNCiAgICAiZ3JpZCI6IDEsDQogICAgIm1ldGFsX251dCI6IDEsDQogICAgImNhcHN1bGUiOiAxLA0KICAgICJwaWxsIjogMSwNCiAgICAiY2FycGV0IjogMSwNCiAgICAibGVhdGhlciI6IDEsDQogICAgInppcHBlciI6IDEsDQogICAgInRpbGUiOiAxLA0KICAgICJoYXplbG51dCI6IDIsDQogICAgInNjcmV3IjogMiwNCiAgICAiY2FibGUiOiAyLA0KICAgICJ0cmFuc2lzdG9yIjogMywNCn0NCg0KDQpkZWYgcHJvamVjdF9yb290KCkgLT4gUGF0aDoNCiAgICByZXR1cm4gUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudC5wYXJlbnQNCg0KDQpkZWYgX3Jlc29sdmVfcGF0aCh2YWx1ZTogc3RyIHwgUGF0aCwgYmFzZTogUGF0aCkgLT4gUGF0aDoNCiAgICBwYXRoID0gUGF0aCh2YWx1ZSkNCiAgICBpZiBwYXRoLmlzX2Fic29sdXRlKCk6DQogICAgICAgIHJldHVybiBwYXRoDQogICAgcmV0dXJuIChiYXNlIC8gcGF0aCkucmVzb2x2ZSgpDQoNCg0KQGRhdGFjbGFzcw0KY2xhc3MgQ29uZmlnOg0KICAgIHBheWxvYWQ6IGRpY3Rbc3RyLCBBbnldDQogICAgY29uZmlnX3BhdGg6IFBhdGgNCiAgICBkYXRhX3Jvb3Q6IFBhdGgNCiAgICB3b3JrX3Jvb3Q6IFBhdGgNCg0KICAgIEBwcm9wZXJ0eQ0KICAgIGRlZiBwcm90b2NvbF92ZXJzaW9uKHNlbGYpIC0+IGludDoNCiAgICAgICAgcmV0dXJuIGludChzZWxmLnBheWxvYWRbInByb3RvY29sX3ZlcnNpb24iXSkNCg0KICAgIEBwcm9wZXJ0eQ0KICAgIGRlZiBzZWVkKHNlbGYpIC0+IGludDoNCiAgICAgICAgcmV0dXJuIGludChzZWxmLnBheWxvYWRbInNlZWQiXSkNCg0KICAgIEBwcm9wZXJ0eQ0KICAgIGRlZiBuX2NhbGlicmF0aW9uKHNlbGYpIC0+IGludDoNCiAgICAgICAgcmV0dXJuIGludChzZWxmLnBheWxvYWRbIm5fY2FsaWJyYXRpb24iXSkNCg0KICAgIEBwcm9wZXJ0eQ0KICAgIGRlZiBhbHBoYShzZWxmKSAtPiBmbG9hdDoNCiAgICAgICAgcmV0dXJuIGZsb2F0KHNlbGYucGF5bG9hZFsiYWxwaGEiXSkNCg0KICAgIEBwcm9wZXJ0eQ0KICAgIGRlZiBuX3JlcGVhdHMoc2VsZikgLT4gaW50Og0KICAgICAgICByZXR1cm4gaW50KHNlbGYucGF5bG9hZFsibl9yZXBlYXRzIl0pDQoNCiAgICBAcHJvcGVydHkNCiAgICBkZWYgbl9ncmlkKHNlbGYpIC0+IGxpc3RbaW50XToNCiAgICAgICAgcmV0dXJuIFtpbnQobikgZm9yIG4gaW4gc2VsZi5wYXlsb2FkWyJuX2dyaWQiXV0NCg0KICAgIEBwcm9wZXJ0eQ0KICAgIGRlZiBzdHJhdGVnaWVzKHNlbGYpIC0+IGxpc3Rbc3RyXToNCiAgICAgICAgcmV0dXJuIFtzdHIocykgZm9yIHMgaW4gc2VsZi5wYXlsb2FkWyJzdHJhdGVnaWVzIl1dDQoNCiAgICBAcHJvcGVydHkNCiAgICBkZWYgYXVsY19uX21heChzZWxmKSAtPiBpbnQ6DQogICAgICAgIHJldHVybiBpbnQoc2VsZi5wYXlsb2FkWyJhdWxjX25fbWF4Il0pDQoNCiAgICBAcHJvcGVydHkNCiAgICBkZWYgcHJhY3RpY2FsX3RwcihzZWxmKSAtPiBmbG9hdDoNCiAgICAgICAgcmV0dXJuIGZsb2F0KHNlbGYucGF5bG9hZFsicHJhY3RpY2FsX3RwciJdKQ0KDQogICAgQHByb3BlcnR5DQogICAgZGVmIHByYWN0aWNhbF9mcHIoc2VsZikgLT4gZmxvYXQ6DQogICAgICAgIHJldHVybiBmbG9hdChzZWxmLnBheWxvYWRbInByYWN0aWNhbF9mcHIiXSkNCg0KICAgIEBwcm9wZXJ0eQ0KICAgIGRlZiBuX3N0YXJfbWluX3JlcGVhdHMoc2VsZikgLT4gaW50Og0KICAgICAgICByZXR1cm4gaW50KHNlbGYucGF5bG9hZFsibl9zdGFyX21pbl9yZXBlYXRzIl0pDQoNCiAgICBAcHJvcGVydHkNCiAgICBkZWYgZnByX2F1bGNfbWFyZ2luKHNlbGYpIC0+IGZsb2F0Og0KICAgICAgICByZXR1cm4gZmxvYXQoc2VsZi5wYXlsb2FkWyJmcHJfYXVsY19tYXJnaW4iXSkNCg0KICAgIEBwcm9wZXJ0eQ0KICAgIGRlZiBzdG9wcGluZ19kZWx0YXMoc2VsZikgLT4gbGlzdFtmbG9hdF06DQogICAgICAgIHJldHVybiBbZmxvYXQodikgZm9yIHYgaW4gc2VsZi5wYXlsb2FkWyJzdG9wcGluZ19kZWx0YXMiXV0NCg0KICAgIEBwcm9wZXJ0eQ0KICAgIGRlZiBzdG9wcGluZ19kZWx0YV9tYWluKHNlbGYpIC0+IGZsb2F0Og0KICAgICAgICByZXR1cm4gZmxvYXQoc2VsZi5wYXlsb2FkWyJzdG9wcGluZ19kZWx0YV9tYWluIl0pDQoNCiAgICBAcHJvcGVydHkNCiAgICBkZWYgYm9vdHN0cmFwX3JlcGxpY2F0ZXMoc2VsZikgLT4gaW50Og0KICAgICAgICByZXR1cm4gaW50KHNlbGYucGF5bG9hZFsiYm9vdHN0cmFwX3JlcGxpY2F0ZXMiXSkNCg0KICAgIEBwcm9wZXJ0eQ0KICAgIGRlZiBib290c3RyYXBfc2VlZChzZWxmKSAtPiBpbnQ6DQogICAgICAgIHJldHVybiBpbnQoc2VsZi5wYXlsb2FkWyJib290c3RyYXBfc2VlZCJdKQ0KDQogICAgQHByb3BlcnR5DQogICAgZGVmIGJhY2tib25lKHNlbGYpIC0+IHN0cjoNCiAgICAgICAgcmV0dXJuIHN0cihzZWxmLnBheWxvYWRbImJhY2tib25lIl0pDQoNCiAgICBAcHJvcGVydHkNCiAgICBkZWYgbGF5ZXJzKHNlbGYpIC0+IGxpc3Rbc3RyXToNCiAgICAgICAgcmV0dXJuIFtzdHIodikgZm9yIHYgaW4gc2VsZi5wYXlsb2FkWyJsYXllcnMiXV0NCg0KICAgIEBwcm9wZXJ0eQ0KICAgIGRlZiByZXNpemUoc2VsZikgLT4gaW50Og0KICAgICAgICByZXR1cm4gaW50KHNlbGYucGF5bG9hZFsicmVzaXplIl0pDQoNCiAgICBAcHJvcGVydHkNCiAgICBkZWYgY2VudGVyX2Nyb3Aoc2VsZikgLT4gaW50Og0KICAgICAgICByZXR1cm4gaW50KHNlbGYucGF5bG9hZFsiY2VudGVyX2Nyb3AiXSkNCg0KICAgIEBwcm9wZXJ0eQ0KICAgIGRlZiB1cHNhbXBsZV9tb2RlKHNlbGYpIC0+IHN0cjoNCiAgICAgICAgcmV0dXJuIHN0cihzZWxmLnBheWxvYWRbInVwc2FtcGxlX21vZGUiXSkNCg0KICAgIEBwcm9wZXJ0eQ0KICAgIGRlZiBpbWFnZW5ldF9tZWFuKHNlbGYpIC0+IGxpc3RbZmxvYXRdOg0KICAgICAgICByZXR1cm4gW2Zsb2F0KHYpIGZvciB2IGluIHNlbGYucGF5bG9hZFsiaW1hZ2VuZXRfbWVhbiJdXQ0KDQogICAgQHByb3BlcnR5DQogICAgZGVmIGltYWdlbmV0X3N0ZChzZWxmKSAtPiBsaXN0W2Zsb2F0XToNCiAgICAgICAgcmV0dXJuIFtmbG9hdCh2KSBmb3IgdiBpbiBzZWxmLnBheWxvYWRbImltYWdlbmV0X3N0ZCJdXQ0KDQogICAgQHByb3BlcnR5DQogICAgZGVmIGZlYXR1cmVfYmF0Y2hfc2l6ZShzZWxmKSAtPiBpbnQ6DQogICAgICAgIHJldHVybiBpbnQoc2VsZi5wYXlsb2FkWyJmZWF0dXJlX2JhdGNoX3NpemUiXSkNCg0KICAgIEBwcm9wZXJ0eQ0KICAgIGRlZiBkaXN0YW5jZV9xdWVyeV9iYXRjaChzZWxmKSAtPiBpbnQ6DQogICAgICAgIHJldHVybiBpbnQoc2VsZi5wYXlsb2FkWyJkaXN0YW5jZV9xdWVyeV9iYXRjaCJdKQ0KDQogICAgQHByb3BlcnR5DQogICAgZGVmIGRpc3RhbmNlX3RyYWluX2JhdGNoKHNlbGYpIC0+IGludDoNCiAgICAgICAgcmV0dXJuIGludChzZWxmLnBheWxvYWRbImRpc3RhbmNlX3RyYWluX2JhdGNoIl0pDQoNCiAgICBAcHJvcGVydHkNCiAgICBkZWYgZTAoc2VsZikgLT4gZGljdFtzdHIsIEFueV06DQogICAgICAgIHJldHVybiBkaWN0KHNlbGYucGF5bG9hZFsiZTAiXSkNCg0KICAgIEBwcm9wZXJ0eQ0KICAgIGRlZiBvYmplY3RzKHNlbGYpIC0+IGxpc3Rbc3RyXToNCiAgICAgICAgcmV0dXJuIFtzdHIodikgZm9yIHYgaW4gc2VsZi5wYXlsb2FkWyJvYmplY3RzIl1dDQoNCiAgICBAcHJvcGVydHkNCiAgICBkZWYgdGV4dHVyZXMoc2VsZikgLT4gbGlzdFtzdHJdOg0KICAgICAgICByZXR1cm4gW3N0cih2KSBmb3IgdiBpbiBzZWxmLnBheWxvYWRbInRleHR1cmVzIl1dDQoNCiAgICBAcHJvcGVydHkNCiAgICBkZWYgY2xhc3NlcyhzZWxmKSAtPiBsaXN0W3N0cl06DQogICAgICAgIHJldHVybiBzZWxmLm9iamVjdHMgKyBzZWxmLnRleHR1cmVzDQoNCiAgICBkZWYgY2xhc3NfaW5kZXgoc2VsZiwgY2xhc3NfbmFtZTogc3RyKSAtPiBpbnQ6DQogICAgICAgIHRyeToNCiAgICAgICAgICAgIHJldHVybiBzZWxmLmNsYXNzZXMuaW5kZXgoY2xhc3NfbmFtZSkNCiAgICAgICAgZXhjZXB0IFZhbHVlRXJyb3IgYXMgZXhjOg0KICAgICAgICAgICAgcmFpc2UgS2V5RXJyb3IoZiJVbmtub3duIGNsYXNzOiB7Y2xhc3NfbmFtZX0iKSBmcm9tIGV4Yw0KDQogICAgZGVmIGdyb3VwX29mKHNlbGYsIGNsYXNzX25hbWU6IHN0cikgLT4gc3RyOg0KICAgICAgICBpZiBjbGFzc19uYW1lIGluIHNlbGYub2JqZWN0czoNCiAgICAgICAgICAgIHJldHVybiAib2JqZWN0Ig0KICAgICAgICBpZiBjbGFzc19uYW1lIGluIHNlbGYudGV4dHVyZXM6DQogICAgICAgICAgICByZXR1cm4gInRleHR1cmUiDQogICAgICAgIHJhaXNlIEtleUVycm9yKGYiVW5rbm93biBjbGFzczoge2NsYXNzX25hbWV9IikNCg0KICAgIGRlZiBzZWxlY3RfY2xhc3NlcyhzZWxmLCByZXF1ZXN0ZWQ6IGxpc3Rbc3RyXSB8IE5vbmUpIC0+IGxpc3Rbc3RyXToNCiAgICAgICAgaWYgbm90IHJlcXVlc3RlZDoNCiAgICAgICAgICAgIHJldHVybiBsaXN0KHNlbGYuY2xhc3NlcykNCiAgICAgICAgdW5rbm93biA9IFtuYW1lIGZvciBuYW1lIGluIHJlcXVlc3RlZCBpZiBuYW1lIG5vdCBpbiBzZWxmLmNsYXNzZXNdDQogICAgICAgIGlmIHVua25vd246DQogICAgICAgICAgICByYWlzZSBLZXlFcnJvcihmIlVua25vd24gY2xhc3Nlczoge3Vua25vd259IikNCiAgICAgICAgcmV0dXJuIGxpc3QocmVxdWVzdGVkKQ0KDQogICAgQHByb3BlcnR5DQogICAgZGVmIGhhc2goc2VsZikgLT4gc3RyOg0KICAgICAgICBzY2llbnRpZmljID0ge2tleTogdmFsdWUgZm9yIGtleSwgdmFsdWUgaW4gc2VsZi5wYXlsb2FkLml0ZW1zKCkgaWYga2V5ICE9ICJwYXRocyJ9DQogICAgICAgIGJsb2IgPSB7DQogICAgICAgICAgICAicGF5bG9hZCI6IHNjaWVudGlmaWMsDQogICAgICAgICAgICAiZXhwZWN0ZWRfY291bnRzIjogRVhQRUNURURfQ09VTlRTLA0KICAgICAgICAgICAgImZhbHNlX3JlamVjdF9idWRnZXQiOiBGQUxTRV9SRUpFQ1RfQlVER0VULA0KICAgICAgICB9DQogICAgICAgIHJhdyA9IGpzb24uZHVtcHMoYmxvYiwgc29ydF9rZXlzPVRydWUsIHNlcGFyYXRvcnM9KCIsIiwgIjoiKSkNCiAgICAgICAgcmV0dXJuIGhhc2hsaWIuc2hhMjU2KHJhdy5lbmNvZGUoInV0Zi04IikpLmhleGRpZ2VzdCgpWzoxNl0NCg0KICAgIGRlZiB3b3JrKHNlbGYsICpwYXJ0czogc3RyKSAtPiBQYXRoOg0KICAgICAgICBwYXRoID0gc2VsZi53b3JrX3Jvb3Quam9pbnBhdGgoKnBhcnRzKQ0KICAgICAgICBwYXRoLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkNCiAgICAgICAgcmV0dXJuIHBhdGgNCg0KDQpkZWYgbG9hZF9jb25maWcoDQogICAgY29uZmlnX3BhdGg6IHN0ciB8IFBhdGggfCBOb25lID0gTm9uZSwNCiAgICBkYXRhX3Jvb3Q6IHN0ciB8IFBhdGggfCBOb25lID0gTm9uZSwNCiAgICB3b3JrX3Jvb3Q6IHN0ciB8IFBhdGggfCBOb25lID0gTm9uZSwNCikgLT4gQ29uZmlnOg0KICAgIHJvb3QgPSBwcm9qZWN0X3Jvb3QoKQ0KICAgIHBhdGggPSBQYXRoKGNvbmZpZ19wYXRoKSBpZiBjb25maWdfcGF0aCBlbHNlIHJvb3QgLyAiY29uZmlnLnlhbWwiDQogICAgaWYgbm90IHBhdGguaXNfZmlsZSgpOg0KICAgICAgICByYWlzZSBGaWxlTm90Rm91bmRFcnJvcihmIkNvbmZpZyBub3QgZm91bmQ6IHtwYXRofSIpDQogICAgcGF5bG9hZCA9IHlhbWwuc2FmZV9sb2FkKHBhdGgucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYtOCIpKQ0KICAgIGlmIG5vdCBpc2luc3RhbmNlKHBheWxvYWQsIGRpY3QpOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJDb25maWcgbXVzdCBiZSBhIG1hcHBpbmc6IHtwYXRofSIpDQoNCiAgICBfdmFsaWRhdGVfcGF5bG9hZChwYXlsb2FkKQ0KDQogICAgZGF0YV92YWx1ZSA9IGRhdGFfcm9vdCBvciBvcy5lbnZpcm9uLmdldCgiQ09NTUlTU0lPTklOR19EQVRBIikgb3IgcGF5bG9hZFsicGF0aHMiXVsiZGF0YV9yb290Il0NCiAgICB3b3JrX3ZhbHVlID0gd29ya19yb290IG9yIG9zLmVudmlyb24uZ2V0KCJDT01NSVNTSU9OSU5HX1dPUksiKSBvciBwYXlsb2FkWyJwYXRocyJdWyJ3b3JrX3Jvb3QiXQ0KICAgIHJldHVybiBDb25maWcoDQogICAgICAgIHBheWxvYWQ9cGF5bG9hZCwNCiAgICAgICAgY29uZmlnX3BhdGg9cGF0aC5yZXNvbHZlKCksDQogICAgICAgIGRhdGFfcm9vdD1fcmVzb2x2ZV9wYXRoKGRhdGFfdmFsdWUsIHJvb3QpLA0KICAgICAgICB3b3JrX3Jvb3Q9X3Jlc29sdmVfcGF0aCh3b3JrX3ZhbHVlLCByb290KSwNCiAgICApDQoNCg0KZGVmIF92YWxpZGF0ZV9wYXlsb2FkKHBheWxvYWQ6IGRpY3Rbc3RyLCBBbnldKSAtPiBOb25lOg0KICAgIGlmIGxpc3QocGF5bG9hZC5nZXQoIm9iamVjdHMiLCBbXSkpICE9IE9CSkVDVFMgb3IgbGlzdChwYXlsb2FkLmdldCgidGV4dHVyZXMiLCBbXSkpICE9IFRFWFRVUkVTOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIm9iamVjdHMvdGV4dHVyZXMgaW4gY29uZmlnLnlhbWwgbXVzdCBtYXRjaCB0aGUgY2Fub25pY2FsIGxpc3RzIGluIGNvbmZpZy5weS4iKQ0KICAgIGNsYXNzZXMgPSBPQkpFQ1RTICsgVEVYVFVSRVMNCiAgICBpZiBzZXQoRVhQRUNURURfQ09VTlRTKSAhPSBzZXQoY2xhc3Nlcykgb3Igc2V0KEZBTFNFX1JFSkVDVF9CVURHRVQpICE9IHNldChjbGFzc2VzKToNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJFWFBFQ1RFRF9DT1VOVFMgYW5kIEZBTFNFX1JFSkVDVF9CVURHRVQgbXVzdCBjb3ZlciBldmVyeSBNVlRlYyBjbGFzcy4iKQ0KICAgIGlmIGludChwYXlsb2FkWyJuX2NhbGlicmF0aW9uIl0pICE9IDIwIG9yIGZsb2F0KHBheWxvYWRbImFscGhhIl0pICE9IDAuMDU6DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiVGhpcyBwcm90b2NvbCBmaXhlcyBuX2NhbGlicmF0aW9uPTIwIGFuZCBhbHBoYT0wLjA1LCBzbyB0aGUgdGhyZXNob2xkIGlzIG1heChDKS4iKQ0KICAgIGlmICJyYW5kb20iIG5vdCBpbiBwYXlsb2FkWyJzdHJhdGVnaWVzIl0gb3IgImtjZW50ZXIiIG5vdCBpbiBwYXlsb2FkWyJzdHJhdGVnaWVzIl06DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigic3RyYXRlZ2llcyBtdXN0IGluY2x1ZGUgcmFuZG9tIGFuZCBrY2VudGVyLiIpDQogICAgaWYgaW50KHBheWxvYWRbImF1bGNfbl9tYXgiXSkgIT0gMzI6DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiTWFpbiBBVUxDIGlzIGZpeGVkIG9uIE4gPD0gMzIuIikNCiAgICBncmlkID0gW2ludChuKSBmb3IgbiBpbiBwYXlsb2FkWyJuX2dyaWQiXV0NCiAgICBpZiBncmlkICE9IHNvcnRlZChncmlkKSBvciBsZW4oZ3JpZCkgIT0gbGVuKHNldChncmlkKSk6DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIm5fZ3JpZCBtdXN0IGJlIHN0cmljdGx5IGluY3JlYXNpbmc6IHtncmlkfSIpDQogICAgaWYgMSBub3QgaW4gZ3JpZCBvciAzMiBub3QgaW4gZ3JpZDoNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJuX2dyaWQgbXVzdCBjb250YWluIDEgYW5kIDMyLiIpDQo=", "commissioning/data.py": "IiIiTVZUZWMgQUQgZGlzY292ZXJ5LCBvZmZpY2lhbC1jb3VudCBjaGVja3MsIGFuZCBwZXJtYW5lbnQgQy9QIHNwbGl0cy4iIiINCg0KZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucw0KDQppbXBvcnQgaGFzaGxpYg0KZnJvbSBwYXRobGliIGltcG9ydCBQYXRoDQoNCmltcG9ydCBudW1weSBhcyBucA0KDQpmcm9tIGNvbW1pc3Npb25pbmcuY29uZmlnIGltcG9ydCBFWFBFQ1RFRF9DT1VOVFMsIENvbmZpZw0KZnJvbSBjb21taXNzaW9uaW5nLmlvX3V0aWxzIGltcG9ydCByZWFkX2pzb24sIHdyaXRlX2pzb24NCg0KSU1BR0VfU1VGRklYRVMgPSB7Ii5wbmciLCAiLmpwZyIsICIuanBlZyIsICIuYm1wIiwgIi50aWYiLCAiLnRpZmYifQ0KDQoNCmRlZiBpdGVyX2ltYWdlcyhkaXJlY3Rvcnk6IFBhdGgpIC0+IGxpc3RbUGF0aF06DQogICAgaWYgbm90IGRpcmVjdG9yeS5pc19kaXIoKToNCiAgICAgICAgcmV0dXJuIFtdDQogICAgZmlsZXMgPSBbcGF0aCBmb3IgcGF0aCBpbiBkaXJlY3RvcnkuaXRlcmRpcigpIGlmIHBhdGguaXNfZmlsZSgpIGFuZCBwYXRoLnN1ZmZpeC5sb3dlcigpIGluIElNQUdFX1NVRkZJWEVTXQ0KICAgIHJldHVybiBzb3J0ZWQoZmlsZXMsIGtleT1sYW1iZGEgcGF0aDogcGF0aC5uYW1lLmxvd2VyKCkpDQoNCg0KZGVmIGNvbGxlY3RfY2xhc3MoZGF0YV9yb290OiBQYXRoLCBjbGFzc19uYW1lOiBzdHIpIC0+IGRpY3Rbc3RyLCBsaXN0XToNCiAgICByb290ID0gZGF0YV9yb290IC8gY2xhc3NfbmFtZQ0KICAgIGlmIG5vdCByb290LmlzX2RpcigpOg0KICAgICAgICByYWlzZSBGaWxlTm90Rm91bmRFcnJvcihmIntjbGFzc19uYW1lfTogZGF0YXNldCBkaXJlY3Rvcnkgbm90IGZvdW5kOiB7cm9vdH0iKQ0KICAgIHRyYWluID0gaXRlcl9pbWFnZXMocm9vdCAvICJ0cmFpbiIgLyAiZ29vZCIpDQogICAgdGVzdF9nb29kID0gaXRlcl9pbWFnZXMocm9vdCAvICJ0ZXN0IiAvICJnb29kIikNCiAgICB0ZXN0X3Jvb3QgPSByb290IC8gInRlc3QiDQogICAgZGVmZWN0czogbGlzdFt0dXBsZVtzdHIsIFBhdGhdXSA9IFtdDQogICAgaWYgdGVzdF9yb290LmlzX2RpcigpOg0KICAgICAgICBmb3Igc3ViIGluIHNvcnRlZCgocGF0aCBmb3IgcGF0aCBpbiB0ZXN0X3Jvb3QuaXRlcmRpcigpIGlmIHBhdGguaXNfZGlyKCkpLCBrZXk9bGFtYmRhIHBhdGg6IHBhdGgubmFtZS5sb3dlcigpKToNCiAgICAgICAgICAgIGlmIHN1Yi5uYW1lLmxvd2VyKCkgPT0gImdvb2QiOg0KICAgICAgICAgICAgICAgIGNvbnRpbnVlDQogICAgICAgICAgICBkZWZlY3RzLmV4dGVuZCgoc3ViLm5hbWUsIGltYWdlKSBmb3IgaW1hZ2UgaW4gaXRlcl9pbWFnZXMoc3ViKSkNCiAgICByZXR1cm4geyJ0cmFpbiI6IHRyYWluLCAidGVzdF9nb29kIjogdGVzdF9nb29kLCAidGVzdF9kZWZlY3QiOiBkZWZlY3RzfQ0KDQoNCmRlZiBhc3NlcnRfb2ZmaWNpYWxfY291bnRzKGNsYXNzX25hbWU6IHN0ciwgY29sbGVjdGVkOiBkaWN0W3N0ciwgbGlzdF0pIC0+IE5vbmU6DQogICAgZXhwZWN0ZWQgPSBFWFBFQ1RFRF9DT1VOVFNbY2xhc3NfbmFtZV0NCiAgICBmb3VuZCA9IHsidHJhaW4iOiBsZW4oY29sbGVjdGVkWyJ0cmFpbiJdKSwgInRlc3RfZ29vZCI6IGxlbihjb2xsZWN0ZWRbInRlc3RfZ29vZCJdKX0NCiAgICBpZiBmb3VuZCAhPSBleHBlY3RlZDoNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYie2NsYXNzX25hbWV9OiBleHBlY3RlZCBjb3VudHMge2V4cGVjdGVkfSwgZm91bmQge2ZvdW5kfSIpDQogICAgaWYgbGVuKGNvbGxlY3RlZFsidGVzdF9kZWZlY3QiXSkgPCAxOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJ7Y2xhc3NfbmFtZX06IG5vIGRlZmVjdCBpbWFnZXMgdW5kZXIgdGVzdC8iKQ0KICAgIF9hc3NlcnRfbm9fZHVwbGljYXRlX2ZpbGVzKGNsYXNzX25hbWUsIGNvbGxlY3RlZCkNCg0KDQpkZWYgX2Fzc2VydF9ub19kdXBsaWNhdGVfZmlsZXMoY2xhc3NfbmFtZTogc3RyLCBjb2xsZWN0ZWQ6IGRpY3Rbc3RyLCBsaXN0XSkgLT4gTm9uZToNCiAgICBzZWVuOiBkaWN0W3N0ciwgc3RyXSA9IHt9DQogICAgcGF0aHM6IGxpc3RbUGF0aF0gPSBsaXN0KGNvbGxlY3RlZFsidHJhaW4iXSkgKyBsaXN0KGNvbGxlY3RlZFsidGVzdF9nb29kIl0pDQogICAgcGF0aHMuZXh0ZW5kKHBhdGggZm9yIF8sIHBhdGggaW4gY29sbGVjdGVkWyJ0ZXN0X2RlZmVjdCJdKQ0KICAgIGZvciBwYXRoIGluIHBhdGhzOg0KICAgICAgICBkaWdlc3QgPSBzaGExX2ZpbGUocGF0aCkNCiAgICAgICAgcHJldmlvdXMgPSBzZWVuLmdldChkaWdlc3QpDQogICAgICAgIGlmIHByZXZpb3VzIGlzIG5vdCBOb25lOg0KICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYie2NsYXNzX25hbWV9OiBkdXBsaWNhdGUgaW1hZ2UgY29udGVudCB7cHJldmlvdXN9IGFuZCB7cGF0aH0iKQ0KICAgICAgICBzZWVuW2RpZ2VzdF0gPSBzdHIocGF0aCkNCg0KDQpkZWYgc2hhMV9maWxlKHBhdGg6IFBhdGgsIGNodW5rOiBpbnQgPSAxIDw8IDIwKSAtPiBzdHI6DQogICAgZGlnZXN0ID0gaGFzaGxpYi5zaGExKCkNCiAgICB3aXRoIFBhdGgocGF0aCkub3BlbigicmIiKSBhcyBoYW5kbGU6DQogICAgICAgIHdoaWxlIFRydWU6DQogICAgICAgICAgICBibG9jayA9IGhhbmRsZS5yZWFkKGNodW5rKQ0KICAgICAgICAgICAgaWYgbm90IGJsb2NrOg0KICAgICAgICAgICAgICAgIGJyZWFrDQogICAgICAgICAgICBkaWdlc3QudXBkYXRlKGJsb2NrKQ0KICAgIHJldHVybiBkaWdlc3QuaGV4ZGlnZXN0KCkNCg0KDQpkZWYgcmVsYXRpdmVfcGF0aChwYXRoOiBQYXRoLCBkYXRhX3Jvb3Q6IFBhdGgpIC0+IHN0cjoNCiAgICByZXR1cm4gcGF0aC5yZXNvbHZlKCkucmVsYXRpdmVfdG8oZGF0YV9yb290LnJlc29sdmUoKSkuYXNfcG9zaXgoKQ0KDQoNCmRlZiBhYnNvbHV0ZV9wYXRoKGRhdGFfcm9vdDogUGF0aCwgcmVsYXRpdmU6IHN0cikgLT4gUGF0aDoNCiAgICByZXR1cm4gZGF0YV9yb290IC8gUGF0aChyZWxhdGl2ZSkNCg0KDQpkZWYgc3BsaXRfc2VlZHMoY2ZnOiBDb25maWcsIGNsYXNzX25hbWU6IHN0ciwgcmVwOiBpbnQpIC0+IHR1cGxlW25wLnJhbmRvbS5HZW5lcmF0b3IsIG5wLnJhbmRvbS5HZW5lcmF0b3IsIG5wLnJhbmRvbS5HZW5lcmF0b3JdOg0KICAgICIiIkluZGVwZW5kZW50IHN0cmVhbXMgc28gQy9QIGRvZXMgbm90IGRlcGVuZCBvbiB3aGljaCBzdHJhdGVneSBydW5zIGZpcnN0LiIiIg0KICAgIHNlcXVlbmNlID0gbnAucmFuZG9tLlNlZWRTZXF1ZW5jZShbY2ZnLnNlZWQsIGNmZy5jbGFzc19pbmRleChjbGFzc19uYW1lKSwgaW50KHJlcCldKQ0KICAgIHNwbGl0X3NlZWQsIHJhbmRvbV9zZWVkLCBrY2VudGVyX3NlZWQgPSBzZXF1ZW5jZS5zcGF3bigzKQ0KICAgIHJldHVybiAoDQogICAgICAgIG5wLnJhbmRvbS5kZWZhdWx0X3JuZyhzcGxpdF9zZWVkKSwNCiAgICAgICAgbnAucmFuZG9tLmRlZmF1bHRfcm5nKHJhbmRvbV9zZWVkKSwNCiAgICAgICAgbnAucmFuZG9tLmRlZmF1bHRfcm5nKGtjZW50ZXJfc2VlZCksDQogICAgKQ0KDQoNCmRlZiBzcGxpdF90cmFpbl9pbmRpY2VzKG5fdHJhaW46IGludCwgbl9jYWxpYnJhdGlvbjogaW50LCBybmc6IG5wLnJhbmRvbS5HZW5lcmF0b3IpIC0+IHR1cGxlW25wLm5kYXJyYXksIG5wLm5kYXJyYXldOg0KICAgIGlmIG5fdHJhaW4gPD0gbl9jYWxpYnJhdGlvbjoNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYiTmVlZCBtb3JlIHRoYW4ge25fY2FsaWJyYXRpb259IHRyYWluaW5nIGltYWdlcywgZm91bmQge25fdHJhaW59LiIpDQogICAgY2FsaWJyYXRpb24gPSBucC5zb3J0KHJuZy5jaG9pY2Uobl90cmFpbiwgc2l6ZT1uX2NhbGlicmF0aW9uLCByZXBsYWNlPUZhbHNlKS5hc3R5cGUoaW50KSkNCiAgICBwb29sX21hc2sgPSBucC5vbmVzKG5fdHJhaW4sIGR0eXBlPWJvb2wpDQogICAgcG9vbF9tYXNrW2NhbGlicmF0aW9uXSA9IEZhbHNlDQogICAgcG9vbCA9IG5wLmZsYXRub256ZXJvKHBvb2xfbWFzaykuYXN0eXBlKGludCkNCiAgICBfYXNzZXJ0X3NwbGl0X2luZGljZXMoY2FsaWJyYXRpb24sIHBvb2wsIG5fdHJhaW4sIG5fY2FsaWJyYXRpb24pDQogICAgcmV0dXJuIGNhbGlicmF0aW9uLCBwb29sDQoNCg0KZGVmIF9hc3NlcnRfc3BsaXRfaW5kaWNlcyhjYWxpYnJhdGlvbjogbnAubmRhcnJheSwgcG9vbDogbnAubmRhcnJheSwgbl90cmFpbjogaW50LCBuX2NhbGlicmF0aW9uOiBpbnQpIC0+IE5vbmU6DQogICAgaWYgbGVuKGNhbGlicmF0aW9uKSAhPSBuX2NhbGlicmF0aW9uOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJ8Q3wgPSB7bGVuKGNhbGlicmF0aW9uKX0sIGV4cGVjdGVkIHtuX2NhbGlicmF0aW9ufSIpDQogICAgaWYgbGVuKGNhbGlicmF0aW9uKSArIGxlbihwb29sKSAhPSBuX3RyYWluOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoInxDfCArIHxQfCBtdXN0IGVxdWFsIHRoZSB0cmFpbmluZyBzZXQuIikNCiAgICBpZiBsZW4obnAuaW50ZXJzZWN0MWQoY2FsaWJyYXRpb24sIHBvb2wpKToNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJDIGFuZCBQIG92ZXJsYXAuIikNCiAgICBpZiBsZW4obnAudW5pcXVlKGNhbGlicmF0aW9uKSkgIT0gbGVuKGNhbGlicmF0aW9uKSBvciBsZW4obnAudW5pcXVlKHBvb2wpKSAhPSBsZW4ocG9vbCk6DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiRHVwbGljYXRlIGluZGljZXMgaW5zaWRlIEMgb3IgUC4iKQ0KDQoNCmRlZiBzcGxpdF9wYXRoKGNmZzogQ29uZmlnLCBjbGFzc19uYW1lOiBzdHIsIHJlcDogaW50KSAtPiBQYXRoOg0KICAgIHJldHVybiBjZmcud29ya19yb290IC8gInNwbGl0cyIgLyBjbGFzc19uYW1lIC8gZiJyZXBfe3JlcDowMmR9Lmpzb24iDQoNCg0KZGVmIGJ1aWxkX3NwbGl0X3BheWxvYWQoY2ZnOiBDb25maWcsIGNsYXNzX25hbWU6IHN0ciwgcmVwOiBpbnQsIGNvbGxlY3RlZDogZGljdFtzdHIsIGxpc3RdKSAtPiBkaWN0Og0KICAgIHJuZywgXywgXyA9IHNwbGl0X3NlZWRzKGNmZywgY2xhc3NfbmFtZSwgcmVwKQ0KICAgIGNhbGlicmF0aW9uX2lkeCwgcG9vbF9pZHggPSBzcGxpdF90cmFpbl9pbmRpY2VzKGxlbihjb2xsZWN0ZWRbInRyYWluIl0pLCBjZmcubl9jYWxpYnJhdGlvbiwgcm5nKQ0KICAgIHRyYWluX3JlbCA9IFtyZWxhdGl2ZV9wYXRoKHBhdGgsIGNmZy5kYXRhX3Jvb3QpIGZvciBwYXRoIGluIGNvbGxlY3RlZFsidHJhaW4iXV0NCiAgICB0ZXN0X2dvb2RfcmVsID0gW3JlbGF0aXZlX3BhdGgocGF0aCwgY2ZnLmRhdGFfcm9vdCkgZm9yIHBhdGggaW4gY29sbGVjdGVkWyJ0ZXN0X2dvb2QiXV0NCiAgICB0ZXN0X2RlZmVjdCA9IFsNCiAgICAgICAgeyJwYXRoIjogcmVsYXRpdmVfcGF0aChwYXRoLCBjZmcuZGF0YV9yb290KSwgImRlZmVjdF90eXBlIjogZGVmZWN0X3R5cGV9DQogICAgICAgIGZvciBkZWZlY3RfdHlwZSwgcGF0aCBpbiBjb2xsZWN0ZWRbInRlc3RfZGVmZWN0Il0NCiAgICBdDQogICAgY2FsaWJyYXRpb24gPSBbdHJhaW5fcmVsW2ludChpKV0gZm9yIGkgaW4gY2FsaWJyYXRpb25faWR4XQ0KICAgIHBvb2wgPSBbdHJhaW5fcmVsW2ludChpKV0gZm9yIGkgaW4gcG9vbF9pZHhdDQogICAgdGVzdF9wYXRocyA9IHRlc3RfZ29vZF9yZWwgKyBbaXRlbVsicGF0aCJdIGZvciBpdGVtIGluIHRlc3RfZGVmZWN0XQ0KICAgIF9hc3NlcnRfc3BsaXRfcGF0aHMoY2xhc3NfbmFtZSwgY2FsaWJyYXRpb24sIHBvb2wsIHRlc3RfcGF0aHMpDQogICAgcmV0dXJuIHsNCiAgICAgICAgImNvbmZpZ19oYXNoIjogY2ZnLmhhc2gsDQogICAgICAgICJjbGFzc19uYW1lIjogY2xhc3NfbmFtZSwNCiAgICAgICAgImNsYXNzX2luZGV4IjogY2ZnLmNsYXNzX2luZGV4KGNsYXNzX25hbWUpLA0KICAgICAgICAicmVwIjogaW50KHJlcCksDQogICAgICAgICJzZWVkIjogY2ZnLnNlZWQsDQogICAgICAgICJjYWxpYnJhdGlvbiI6IGNhbGlicmF0aW9uLA0KICAgICAgICAicG9vbCI6IHBvb2wsDQogICAgICAgICJ0ZXN0X2dvb2QiOiB0ZXN0X2dvb2RfcmVsLA0KICAgICAgICAidGVzdF9kZWZlY3QiOiB0ZXN0X2RlZmVjdCwNCiAgICAgICAgIm5fdHJhaW4iOiBsZW4odHJhaW5fcmVsKSwNCiAgICAgICAgIm5fdGVzdF9nb29kIjogbGVuKHRlc3RfZ29vZF9yZWwpLA0KICAgICAgICAibl90ZXN0X2RlZmVjdCI6IGxlbih0ZXN0X2RlZmVjdCksDQogICAgfQ0KDQoNCmRlZiBfYXNzZXJ0X3NwbGl0X3BhdGhzKGNsYXNzX25hbWU6IHN0ciwgY2FsaWJyYXRpb246IGxpc3Rbc3RyXSwgcG9vbDogbGlzdFtzdHJdLCB0ZXN0X3BhdGhzOiBsaXN0W3N0cl0pIC0+IE5vbmU6DQogICAgaWYgbGVuKGNhbGlicmF0aW9uKSAhPSBsZW4oc2V0KGNhbGlicmF0aW9uKSkgb3IgbGVuKHBvb2wpICE9IGxlbihzZXQocG9vbCkpOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJ7Y2xhc3NfbmFtZX06IGR1cGxpY2F0ZSBwYXRocyBpbnNpZGUgQyBvciBQLiIpDQogICAgaWYgc2V0KGNhbGlicmF0aW9uKSAmIHNldChwb29sKToNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYie2NsYXNzX25hbWV9OiBDIG92ZXJsYXBzIFAuIikNCiAgICBpZiBzZXQoY2FsaWJyYXRpb24pICYgc2V0KHRlc3RfcGF0aHMpIG9yIHNldChwb29sKSAmIHNldCh0ZXN0X3BhdGhzKToNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYie2NsYXNzX25hbWV9OiB0cmFpbiBzcGxpdCBvdmVybGFwcyB0aGUgb2ZmaWNpYWwgdGVzdCBzZXQuIikNCiAgICBpZiBsZW4odGVzdF9wYXRocykgIT0gbGVuKHNldCh0ZXN0X3BhdGhzKSk6DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIntjbGFzc19uYW1lfTogZHVwbGljYXRlIHRlc3QgcGF0aHMuIikNCg0KDQpkZWYgcHJlcGFyZV9jbGFzcyhjZmc6IENvbmZpZywgY2xhc3NfbmFtZTogc3RyLCByZXBzOiByYW5nZSB8IE5vbmUgPSBOb25lKSAtPiBOb25lOg0KICAgIGNvbGxlY3RlZCA9IGNvbGxlY3RfY2xhc3MoY2ZnLmRhdGFfcm9vdCwgY2xhc3NfbmFtZSkNCiAgICBhc3NlcnRfb2ZmaWNpYWxfY291bnRzKGNsYXNzX25hbWUsIGNvbGxlY3RlZCkNCiAgICByZXBfcmFuZ2UgPSByYW5nZShjZmcubl9yZXBlYXRzKSBpZiByZXBzIGlzIE5vbmUgZWxzZSByZXBzDQogICAgZm9yIHJlcCBpbiByZXBfcmFuZ2U6DQogICAgICAgIHBhdGggPSBzcGxpdF9wYXRoKGNmZywgY2xhc3NfbmFtZSwgcmVwKQ0KICAgICAgICBwYXlsb2FkID0gYnVpbGRfc3BsaXRfcGF5bG9hZChjZmcsIGNsYXNzX25hbWUsIHJlcCwgY29sbGVjdGVkKQ0KICAgICAgICBpZiBwYXRoLmlzX2ZpbGUoKToNCiAgICAgICAgICAgIGV4aXN0aW5nID0gcmVhZF9qc29uKHBhdGgpDQogICAgICAgICAgICBpZiBleGlzdGluZyAhPSBwYXlsb2FkOg0KICAgICAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigNCiAgICAgICAgICAgICAgICAgICAgZiJSZWZ1c2luZyB0byBvdmVyd3JpdGUgYW4gZXhpc3Rpbmcgc3BsaXQgdGhhdCBkb2VzIG5vdCBtYXRjaCB0aGlzIGNvbmZpZzoge3BhdGh9Ig0KICAgICAgICAgICAgICAgICkNCiAgICAgICAgICAgIGNvbnRpbnVlDQogICAgICAgIHdyaXRlX2pzb24ocGF0aCwgcGF5bG9hZCkNCg0KDQpkZWYgcHJlcGFyZShjZmc6IENvbmZpZywgY2xhc3NlczogbGlzdFtzdHJdKSAtPiBOb25lOg0KICAgIGZvciBjbGFzc19uYW1lIGluIGNsYXNzZXM6DQogICAgICAgIHByZXBhcmVfY2xhc3MoY2ZnLCBjbGFzc19uYW1lKQ0KDQoNCmRlZiBsb2FkX3NwbGl0KGNmZzogQ29uZmlnLCBjbGFzc19uYW1lOiBzdHIsIHJlcDogaW50KSAtPiBkaWN0Og0KICAgIHBhdGggPSBzcGxpdF9wYXRoKGNmZywgY2xhc3NfbmFtZSwgcmVwKQ0KICAgIGlmIG5vdCBwYXRoLmlzX2ZpbGUoKToNCiAgICAgICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoZiJNaXNzaW5nIHNwbGl0IHtwYXRofS4gUnVuIHByZXBhcmUgZmlyc3QuIikNCiAgICBwYXlsb2FkID0gcmVhZF9qc29uKHBhdGgpDQogICAgaWYgcGF5bG9hZC5nZXQoImNvbmZpZ19oYXNoIikgIT0gY2ZnLmhhc2g6DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIlNwbGl0IGhhc2ggbWlzbWF0Y2ggZm9yIHtjbGFzc19uYW1lfSByZXAge3JlcH0uIFRoZSBzcGxpdCB3YXMgYnVpbHQgZnJvbSBhbm90aGVyIGNvbmZpZy4iKQ0KICAgIF9hc3NlcnRfc3BsaXRfcGF0aHMoDQogICAgICAgIGNsYXNzX25hbWUsDQogICAgICAgIHBheWxvYWRbImNhbGlicmF0aW9uIl0sDQogICAgICAgIHBheWxvYWRbInBvb2wiXSwNCiAgICAgICAgbGlzdChwYXlsb2FkWyJ0ZXN0X2dvb2QiXSkgKyBbaXRlbVsicGF0aCJdIGZvciBpdGVtIGluIHBheWxvYWRbInRlc3RfZGVmZWN0Il1dLA0KICAgICkNCiAgICByZXR1cm4gcGF5bG9hZA0K", "commissioning/distances.py": "IiIiUGVyLWNsYXNzIGRpc3RhbmNlIHRlbnNvciBEW3F1ZXJ5LCBwYXRjaCwgdHJhaW4gaW1hZ2VdLg0KDQpEW3EsIHAsIGpdIGlzIHRoZSBFdWNsaWRlYW4gZGlzdGFuY2UgZnJvbSBwYXRjaCBwIG9mIHF1ZXJ5IHEgdG8gaXRzIG5lYXJlc3QNCnBhdGNoIGluIHRyYWluaW5nIGltYWdlIGouIFRoZSBpbWFnZSBzY29yZSBmb3IgYSBiYW5rIFMgaXMNCg0KICAgIG1heCBvdmVyIHAgb2YgbWluIG92ZXIgaiBpbiBTIG9mIERbcSwgcCwgal0NCg0Kd2hpY2ggaXMgdGhlIHNhbWUgMS1OTiBtYXhpbXVtIHVzZWQgYnkgUGF0Y2hDb3JlIHdpdGhvdXQgYSBjb3Jlc2V0Lg0KIiIiDQoNCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMNCg0KaW1wb3J0IGxvZ2dpbmcNCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aA0KDQppbXBvcnQgbnVtcHkgYXMgbnANCg0KZnJvbSBjb21taXNzaW9uaW5nLmNvbmZpZyBpbXBvcnQgQ29uZmlnDQpmcm9tIGNvbW1pc3Npb25pbmcuZmVhdHVyZXMgaW1wb3J0IGxvYWRfZmVhdHVyZV9jYWNoZQ0KZnJvbSBjb21taXNzaW9uaW5nLmlvX3V0aWxzIGltcG9ydCBhdG9taWNfbnB5LCByZWFkX2pzb24sIHdyaXRlX2pzb24NCg0KTE9HR0VSID0gbG9nZ2luZy5nZXRMb2dnZXIoX19uYW1lX18pDQoNCg0KZGVmIGRpc3RhbmNlX2NhY2hlX3BhdGgoY2ZnOiBDb25maWcsIGNsYXNzX25hbWU6IHN0cikgLT4gUGF0aDoNCiAgICByZXR1cm4gY2ZnLndvcmtfcm9vdCAvICJjYWNoZSIgLyAiZGlzdCIgLyBmIntjbGFzc19uYW1lfS5ucHkiDQoNCg0KZGVmIGRpc3RhbmNlX21ldGFfcGF0aChjZmc6IENvbmZpZywgY2xhc3NfbmFtZTogc3RyKSAtPiBQYXRoOg0KICAgIHJldHVybiBjZmcud29ya19yb290IC8gImNhY2hlIiAvICJkaXN0IiAvIGYie2NsYXNzX25hbWV9Lmpzb24iDQoNCg0KZGVmIG1pbl9wYXRjaF9kaXN0YW5jZV9ibG9jayhxdWVyeTogbnAubmRhcnJheSwgdHJhaW46IG5wLm5kYXJyYXkpIC0+IG5wLm5kYXJyYXk6DQogICAgIiIicXVlcnkgKEJxLCBQLCBDKSwgdHJhaW4gKEJ0LCBQLCBDKSAtPiAoQnEsIFAsIEJ0KSBtaW5pbXVtIG92ZXIgdHJhaW4gcGF0Y2hlcy4NCg0KICAgIE51bVB5IHJlZmVyZW5jZSB1c2VkIGJ5IHRlc3RzIGFuZCBieSB0aGUgQ1BVIGZhbGxiYWNrLiBQcm9kdWN0aW9uIHJ1bnMgdXNlIHRvcmNoLg0KICAgICIiIg0KICAgIHF1ZXJ5ID0gbnAuYXNjb250aWd1b3VzYXJyYXkocXVlcnksIGR0eXBlPW5wLmZsb2F0MzIpDQogICAgdHJhaW4gPSBucC5hc2NvbnRpZ3VvdXNhcnJheSh0cmFpbiwgZHR5cGU9bnAuZmxvYXQzMikNCiAgICBicSwgcGF0Y2hlcywgY2hhbm5lbHMgPSBxdWVyeS5zaGFwZQ0KICAgIGJ0ID0gdHJhaW4uc2hhcGVbMF0NCiAgICBpZiB0cmFpbi5zaGFwZVsxOl0gIT0gKHBhdGNoZXMsIGNoYW5uZWxzKToNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYiUGF0Y2ggc2hhcGUgbWlzbWF0Y2g6IHF1ZXJ5IHtxdWVyeS5zaGFwZX0sIHRyYWluIHt0cmFpbi5zaGFwZX0iKQ0KICAgIGRpc3RhbmNlcyA9IG5wLmVtcHR5KChicSwgcGF0Y2hlcywgYnQpLCBkdHlwZT1ucC5mbG9hdDMyKQ0KICAgIHF1ZXJ5X2ZsYXQgPSBxdWVyeS5yZXNoYXBlKGJxICogcGF0Y2hlcywgY2hhbm5lbHMpDQogICAgZm9yIHRyYWluX2luZGV4IGluIHJhbmdlKGJ0KToNCiAgICAgICAgZGVsdGEgPSBxdWVyeV9mbGF0WzosIE5vbmUsIDpdIC0gdHJhaW5bdHJhaW5faW5kZXhdW05vbmUsIDosIDpdDQogICAgICAgIG5lYXJlc3QgPSBucC5zcXJ0KG5wLnN1bShkZWx0YSAqIGRlbHRhLCBheGlzPS0xKSkubWluKGF4aXM9MSkNCiAgICAgICAgZGlzdGFuY2VzWzosIDosIHRyYWluX2luZGV4XSA9IG5lYXJlc3QucmVzaGFwZShicSwgcGF0Y2hlcykNCiAgICByZXR1cm4gZGlzdGFuY2VzDQoNCg0KZGVmIF9kaXN0YW5jZV90ZW5zb3JfdG9yY2goDQogICAgcGF0Y2hlczogbnAubmRhcnJheSwNCiAgICBuX3RyYWluOiBpbnQsDQogICAgcXVlcnlfYmF0Y2g6IGludCwNCiAgICB0cmFpbl9iYXRjaDogaW50LA0KICAgIGRldmljZTogc3RyLA0KKSAtPiBucC5uZGFycmF5Og0KICAgICIiIktlZXBzIHRoZSBmbG9hdDE2IGNhY2hlIG9uIHRoZSBkZXZpY2Ugb25jZTsgZWFjaCBibG9jayBpcyBjYXN0IHRvIGZsb2F0MzIgYmVmb3JlIGNkaXN0LiIiIg0KICAgIGltcG9ydCB0b3JjaA0KDQogICAgbl9pbWFnZXMsIG5fcGF0Y2hlcywgY2hhbm5lbHMgPSBwYXRjaGVzLnNoYXBlDQogICAgcmVzaWRlbnQgPSB0b3JjaC5mcm9tX251bXB5KG5wLmFzY29udGlndW91c2FycmF5KHBhdGNoZXMpKS50byhkZXZpY2UpDQogICAgb3V0cHV0ID0gbnAuZW1wdHkoKG5faW1hZ2VzLCBuX3BhdGNoZXMsIG5fdHJhaW4pLCBkdHlwZT1ucC5mbG9hdDMyKQ0KICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOg0KICAgICAgICBmb3IgcTAgaW4gcmFuZ2UoMCwgbl9pbWFnZXMsIHF1ZXJ5X2JhdGNoKToNCiAgICAgICAgICAgIHExID0gbWluKHEwICsgcXVlcnlfYmF0Y2gsIG5faW1hZ2VzKQ0KICAgICAgICAgICAgaWYgcTAgPT0gMCBvciBxMSA9PSBuX2ltYWdlcyBvciAocTAgLy8gcXVlcnlfYmF0Y2gpICUgMTAgPT0gMDoNCiAgICAgICAgICAgICAgICBMT0dHRVIuaW5mbygiRGlzdGFuY2UgcXVlcmllcyAlZC8lZCIsIHExLCBuX2ltYWdlcykNCiAgICAgICAgICAgIHF1ZXJ5ID0gcmVzaWRlbnRbcTA6cTFdLmZsb2F0KCkucmVzaGFwZSgtMSwgY2hhbm5lbHMpDQogICAgICAgICAgICByb3cgPSB0b3JjaC5lbXB0eSgocTEgLSBxMCwgbl9wYXRjaGVzLCBuX3RyYWluKSwgZHR5cGU9dG9yY2guZmxvYXQzMiwgZGV2aWNlPWRldmljZSkNCiAgICAgICAgICAgIGZvciB0MCBpbiByYW5nZSgwLCBuX3RyYWluLCB0cmFpbl9iYXRjaCk6DQogICAgICAgICAgICAgICAgdDEgPSBtaW4odDAgKyB0cmFpbl9iYXRjaCwgbl90cmFpbikNCiAgICAgICAgICAgICAgICB0cmFpbiA9IHJlc2lkZW50W3QwOnQxXS5mbG9hdCgpLnJlc2hhcGUoLTEsIGNoYW5uZWxzKQ0KICAgICAgICAgICAgICAgIGJsb2NrID0gdG9yY2guY2Rpc3QocXVlcnksIHRyYWluKS5yZXNoYXBlKHExIC0gcTAsIG5fcGF0Y2hlcywgdDEgLSB0MCwgbl9wYXRjaGVzKQ0KICAgICAgICAgICAgICAgIHJvd1s6LCA6LCB0MDp0MV0gPSBibG9jay5taW4oZGltPS0xKS52YWx1ZXMNCiAgICAgICAgICAgIGlmIG5vdCBib29sKHRvcmNoLmlzZmluaXRlKHJvdykuYWxsKCkpOg0KICAgICAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIk5vbi1maW5pdGUgZGlzdGFuY2VzIGZvciBxdWVyaWVzIHtxMH06e3ExfSIpDQogICAgICAgICAgICBvdXRwdXRbcTA6cTFdID0gcm93LmNwdSgpLm51bXB5KCkNCiAgICBkZWwgcmVzaWRlbnQNCiAgICBpZiBkZXZpY2Uuc3RhcnRzd2l0aCgiY3VkYSIpOg0KICAgICAgICB0b3JjaC5jdWRhLmVtcHR5X2NhY2hlKCkNCiAgICByZXR1cm4gb3V0cHV0DQoNCg0KZGVmIGRpc3RhbmNlX3RlbnNvcigNCiAgICBwYXRjaGVzOiBucC5uZGFycmF5LA0KICAgIG5fdHJhaW46IGludCwNCiAgICBxdWVyeV9iYXRjaDogaW50ID0gOCwNCiAgICB0cmFpbl9iYXRjaDogaW50ID0gMTYsDQogICAgZGV2aWNlOiBzdHIgfCBOb25lID0gTm9uZSwNCikgLT4gbnAubmRhcnJheToNCiAgICAiIiJwYXRjaGVzIHJvdyBvcmRlciBpcyB0cmFpbiBpbWFnZXMgZmlyc3QuIERpYWdvbmFsIHRyYWluLXRyYWluIGVudHJpZXMgYXJlICtpbmYuIiIiDQogICAgaWYgcGF0Y2hlcy5uZGltICE9IDM6DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIkV4cGVjdGVkIChOLCBQLCBDKSBwYXRjaGVzLCBnb3Qge3BhdGNoZXMuc2hhcGV9IikNCiAgICBuX2ltYWdlcywgbl9wYXRjaGVzLCBfY2hhbm5lbHMgPSBwYXRjaGVzLnNoYXBlDQogICAgaWYgbm90IDAgPCBuX3RyYWluIDw9IG5faW1hZ2VzOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJuX3RyYWluPXtuX3RyYWlufSBpcyBvdXRzaWRlIDEuLntuX2ltYWdlc30iKQ0KICAgIHRvcmNoX2RldmljZSA9IE5vbmUNCiAgICBpZiBkZXZpY2UgIT0gImNwdSI6DQogICAgICAgIHRyeToNCiAgICAgICAgICAgIGltcG9ydCB0b3JjaA0KDQogICAgICAgICAgICB0b3JjaF9kZXZpY2UgPSBkZXZpY2Ugb3IgKCJjdWRhIiBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpIGVsc2UgImNwdSIpDQogICAgICAgIGV4Y2VwdCBJbXBvcnRFcnJvcjoNCiAgICAgICAgICAgIHRvcmNoX2RldmljZSA9IE5vbmUNCiAgICBpZiB0b3JjaF9kZXZpY2UgaXMgbm90IE5vbmU6DQogICAgICAgIG91dHB1dCA9IF9kaXN0YW5jZV90ZW5zb3JfdG9yY2gocGF0Y2hlcywgbl90cmFpbiwgcXVlcnlfYmF0Y2gsIHRyYWluX2JhdGNoLCB0b3JjaF9kZXZpY2UpDQogICAgZWxzZToNCiAgICAgICAgb3V0cHV0ID0gbnAuZW1wdHkoKG5faW1hZ2VzLCBuX3BhdGNoZXMsIG5fdHJhaW4pLCBkdHlwZT1ucC5mbG9hdDMyKQ0KICAgICAgICBmb3IgcTAgaW4gcmFuZ2UoMCwgbl9pbWFnZXMsIHF1ZXJ5X2JhdGNoKToNCiAgICAgICAgICAgIHExID0gbWluKHEwICsgcXVlcnlfYmF0Y2gsIG5faW1hZ2VzKQ0KICAgICAgICAgICAgZm9yIHQwIGluIHJhbmdlKDAsIG5fdHJhaW4sIHRyYWluX2JhdGNoKToNCiAgICAgICAgICAgICAgICB0MSA9IG1pbih0MCArIHRyYWluX2JhdGNoLCBuX3RyYWluKQ0KICAgICAgICAgICAgICAgIGJsb2NrID0gbWluX3BhdGNoX2Rpc3RhbmNlX2Jsb2NrKHBhdGNoZXNbcTA6cTFdLCBwYXRjaGVzW3QwOnQxXSkNCiAgICAgICAgICAgICAgICBpZiBub3QgbnAuaXNmaW5pdGUoYmxvY2spLmFsbCgpOg0KICAgICAgICAgICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJOb24tZmluaXRlIGRpc3RhbmNlcyBmb3IgcXVlcmllcyB7cTB9OntxMX0sIHRyYWluIHt0MH06e3QxfSIpDQogICAgICAgICAgICAgICAgb3V0cHV0W3EwOnExLCA6LCB0MDp0MV0gPSBibG9jaw0KICAgIGZvciBpbmRleCBpbiByYW5nZShuX3RyYWluKToNCiAgICAgICAgb3V0cHV0W2luZGV4LCA6LCBpbmRleF0gPSBucC5pbmYNCiAgICByZXR1cm4gb3V0cHV0DQoNCg0KZGVmIGNvbXB1dGVfY2xhc3NfZGlzdGFuY2VzKGNmZzogQ29uZmlnLCBjbGFzc19uYW1lOiBzdHIsIGRldmljZTogc3RyIHwgTm9uZSA9IE5vbmUpIC0+IE5vbmU6DQogICAgb3V0X3BhdGggPSBkaXN0YW5jZV9jYWNoZV9wYXRoKGNmZywgY2xhc3NfbmFtZSkNCiAgICBtZXRhX3BhdGggPSBkaXN0YW5jZV9tZXRhX3BhdGgoY2ZnLCBjbGFzc19uYW1lKQ0KICAgIGlmIG91dF9wYXRoLmlzX2ZpbGUoKSBhbmQgbWV0YV9wYXRoLmlzX2ZpbGUoKToNCiAgICAgICAgbWV0YSA9IHJlYWRfanNvbihtZXRhX3BhdGgpDQogICAgICAgIGlmIG1ldGEuZ2V0KCJjb25maWdfaGFzaCIpID09IGNmZy5oYXNoOg0KICAgICAgICAgICAgTE9HR0VSLmluZm8oIkRpc3RhbmNlIGNhY2hlIGV4aXN0cyBmb3IgJXMiLCBjbGFzc19uYW1lKQ0KICAgICAgICAgICAgcmV0dXJuDQogICAgY2FjaGUgPSBsb2FkX2ZlYXR1cmVfY2FjaGUoY2ZnLCBjbGFzc19uYW1lKQ0KICAgIG5fdHJhaW4gPSBpbnQoY2FjaGVbIm1ldGEiXVsibl90cmFpbiJdKQ0KICAgIExPR0dFUi5pbmZvKCJDb21wdXRpbmcgZGlzdGFuY2UgdGVuc29yIGZvciAlcyAoJWQgdHJhaW4gaW1hZ2VzKSIsIGNsYXNzX25hbWUsIG5fdHJhaW4pDQogICAgdGVuc29yID0gZGlzdGFuY2VfdGVuc29yKA0KICAgICAgICBjYWNoZVsicGF0Y2hlcyJdLA0KICAgICAgICBuX3RyYWluPW5fdHJhaW4sDQogICAgICAgIHF1ZXJ5X2JhdGNoPWNmZy5kaXN0YW5jZV9xdWVyeV9iYXRjaCwNCiAgICAgICAgdHJhaW5fYmF0Y2g9Y2ZnLmRpc3RhbmNlX3RyYWluX2JhdGNoLA0KICAgICAgICBkZXZpY2U9ZGV2aWNlLA0KICAgICkNCiAgICBhdG9taWNfbnB5KG91dF9wYXRoLCB0ZW5zb3IpDQogICAgd3JpdGVfanNvbigNCiAgICAgICAgbWV0YV9wYXRoLA0KICAgICAgICB7DQogICAgICAgICAgICAiY29uZmlnX2hhc2giOiBjZmcuaGFzaCwNCiAgICAgICAgICAgICJjbGFzc19uYW1lIjogY2xhc3NfbmFtZSwNCiAgICAgICAgICAgICJzaGFwZSI6IGxpc3QodGVuc29yLnNoYXBlKSwNCiAgICAgICAgICAgICJuX3RyYWluIjogbl90cmFpbiwNCiAgICAgICAgICAgICJuX3Rlc3RfZ29vZCI6IGludChjYWNoZVsibWV0YSJdWyJuX3Rlc3RfZ29vZCJdKSwNCiAgICAgICAgICAgICJuX3Rlc3RfZGVmZWN0IjogaW50KGNhY2hlWyJtZXRhIl1bIm5fdGVzdF9kZWZlY3QiXSksDQogICAgICAgICAgICAicGF0aHMiOiBjYWNoZVsicGF0aHMiXSwNCiAgICAgICAgICAgICJkdHlwZSI6ICJmbG9hdDMyIiwNCiAgICAgICAgICAgICJieXRlcyI6IGludCh0ZW5zb3IubmJ5dGVzKSwNCiAgICAgICAgfSwNCiAgICApDQogICAgTE9HR0VSLmluZm8oIiVzIGRpc3RhbmNlczogc2hhcGUgJXMsICUuMmYgR0IiLCBjbGFzc19uYW1lLCB0ZW5zb3Iuc2hhcGUsIHRlbnNvci5uYnl0ZXMgLyAoMTAyNCoqMykpDQoNCg0KZGVmIGxvYWRfZGlzdGFuY2VfY2FjaGUoY2ZnOiBDb25maWcsIGNsYXNzX25hbWU6IHN0cikgLT4gZGljdDoNCiAgICBwYXRoID0gZGlzdGFuY2VfY2FjaGVfcGF0aChjZmcsIGNsYXNzX25hbWUpDQogICAgbWV0YV9wYXRoID0gZGlzdGFuY2VfbWV0YV9wYXRoKGNmZywgY2xhc3NfbmFtZSkNCiAgICBpZiBub3QgcGF0aC5pc19maWxlKCkgb3Igbm90IG1ldGFfcGF0aC5pc19maWxlKCk6DQogICAgICAgIHJhaXNlIEZpbGVOb3RGb3VuZEVycm9yKGYiTWlzc2luZyBkaXN0YW5jZSBjYWNoZSBmb3Ige2NsYXNzX25hbWV9LiBSdW4gZGlzdGFuY2VzIGZpcnN0LiIpDQogICAgbWV0YSA9IHJlYWRfanNvbihtZXRhX3BhdGgpDQogICAgaWYgbWV0YS5nZXQoImNvbmZpZ19oYXNoIikgIT0gY2ZnLmhhc2g6DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIkRpc3RhbmNlIGNhY2hlIGhhc2ggbWlzbWF0Y2ggZm9yIHtjbGFzc19uYW1lfS4iKQ0KICAgIHRlbnNvciA9IG5wLmxvYWQocGF0aCkNCiAgICBpZiB0dXBsZSh0ZW5zb3Iuc2hhcGUpICE9IHR1cGxlKG1ldGFbInNoYXBlIl0pOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJ7Y2xhc3NfbmFtZX06IGRpc3RhbmNlIHRlbnNvciBzaGFwZSB7dGVuc29yLnNoYXBlfSAhPSBtZXRhIHttZXRhWydzaGFwZSddfSIpDQogICAgcmV0dXJuIHsiRCI6IHRlbnNvciwgIm1ldGEiOiBtZXRhfQ0KDQoNCmRlZiBjb21wdXRlX2Rpc3RhbmNlcyhjZmc6IENvbmZpZywgY2xhc3NlczogbGlzdFtzdHJdLCBkZXZpY2U6IHN0ciB8IE5vbmUgPSBOb25lKSAtPiBOb25lOg0KICAgIGZvciBjbGFzc19uYW1lIGluIGNsYXNzZXM6DQogICAgICAgIGNvbXB1dGVfY2xhc3NfZGlzdGFuY2VzKGNmZywgY2xhc3NfbmFtZSwgZGV2aWNlPWRldmljZSkNCg==", "commissioning/e0.py": "IiIiRTBhIHJlZmVyZW5jZSBzYW5pdHkgY2hlY2sgYW5kIEUwYiBvd24tY2FjaGUgdmVyc3VzIEFub21hbGliLg0KDQpSUSB0cmFqZWN0b3JpZXMgcmVmdXNlIHRvIHN0YXJ0IHVubGVzcyBlMC9nYXRlLmpzb24gc2F5cyBQQVNTIGZvciB0aGlzIGNvbmZpZyBoYXNoLg0KIiIiDQoNCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMNCg0KaW1wb3J0IGluc3BlY3QNCmltcG9ydCBsb2dnaW5nDQoNCmltcG9ydCBudW1weSBhcyBucA0KZnJvbSBzY2lweS5zdGF0cyBpbXBvcnQgc3BlYXJtYW5yDQpmcm9tIHNrbGVhcm4ubWV0cmljcyBpbXBvcnQgcm9jX2F1Y19zY29yZQ0KDQpmcm9tIGNvbW1pc3Npb25pbmcuY29uZmlnIGltcG9ydCBDb25maWcNCmZyb20gY29tbWlzc2lvbmluZy5kYXRhIGltcG9ydCBhYnNvbHV0ZV9wYXRoLCBsb2FkX3NwbGl0LCBzcGxpdF9zZWVkcw0KZnJvbSBjb21taXNzaW9uaW5nLmRpc3RhbmNlcyBpbXBvcnQgbG9hZF9kaXN0YW5jZV9jYWNoZQ0KZnJvbSBjb21taXNzaW9uaW5nLmZlYXR1cmVzIGltcG9ydCBidWlsZF90cmFuc2Zvcm0sIGxvYWRfZmVhdHVyZV9jYWNoZSwgcmVhZF9yZ2INCmZyb20gY29tbWlzc2lvbmluZy5pbnRlZ3JpdHkgaW1wb3J0IGFzc2VydF9tZXRob2RzX2FncmVlLCBjaGVja19jb3JydXB0ZWRfaW5wdXQsIGNoZWNrX2ZlYXR1cmVfYWxpZ25tZW50DQpmcm9tIGNvbW1pc3Npb25pbmcuaW9fdXRpbHMgaW1wb3J0IHJlYWRfanNvbiwgd3JpdGVfanNvbg0KZnJvbSBjb21taXNzaW9uaW5nLnBhdGNoY29yZSBpbXBvcnQgc2NvcmVzX2Zvcl9iYW5rDQpmcm9tIGNvbW1pc3Npb25pbmcuc2VsZWN0aW9uIGltcG9ydCByYW5kb21fb3JkZXINCg0KTE9HR0VSID0gbG9nZ2luZy5nZXRMb2dnZXIoX19uYW1lX18pDQoNCg0KZGVmIGdhdGVfcGF0aChjZmc6IENvbmZpZyk6DQogICAgcmV0dXJuIGNmZy53b3JrX3Jvb3QgLyAiZTAiIC8gImdhdGUuanNvbiINCg0KDQpkZWYgX2NsYXNzX3BhdGgoY2ZnOiBDb25maWcsIGNsYXNzX25hbWU6IHN0cik6DQogICAgcmV0dXJuIGNmZy53b3JrX3Jvb3QgLyAiZTAiIC8gZiJ7Y2xhc3NfbmFtZX0uanNvbiINCg0KDQpkZWYgX2tjZW50ZXJfZ3JlZWR5X2NsYXNzKCk6DQogICAgdHJ5Og0KICAgICAgICBmcm9tIGFub21hbGliLm1vZGVscy5jb21wb25lbnRzIGltcG9ydCBLQ2VudGVyR3JlZWR5DQogICAgZXhjZXB0IEltcG9ydEVycm9yOg0KICAgICAgICBmcm9tIGFub21hbGliLm1vZGVscy5jb21wb25lbnRzLnNhbXBsaW5nIGltcG9ydCBLQ2VudGVyR3JlZWR5DQogICAgcmV0dXJuIEtDZW50ZXJHcmVlZHkNCg0KDQpkZWYgY2hlY2tfYW5vbWFsaWJfdXBzYW1wbGluZyhjZmc6IENvbmZpZykgLT4gc3RyOg0KICAgICIiIlRoZSBjYWNoZSBhbmQgQW5vbWFsaWIgbXVzdCByZXNpemUgbGF5ZXIzIHRoZSBzYW1lIHdheSwgb3RoZXJ3aXNlIEUwYiBjb21wYXJlcyB0d28gcGlwZWxpbmVzLiIiIg0KICAgIGZyb20gYW5vbWFsaWIubW9kZWxzLmltYWdlLnBhdGNoY29yZS50b3JjaF9tb2RlbCBpbXBvcnQgUGF0Y2hjb3JlTW9kZWwNCg0KICAgIHNvdXJjZSA9IGluc3BlY3QuZ2V0c291cmNlKFBhdGNoY29yZU1vZGVsLmdlbmVyYXRlX2VtYmVkZGluZykNCiAgICBtb2RlcyA9IFttb2RlIGZvciBtb2RlIGluICgiYmlsaW5lYXIiLCAibmVhcmVzdCIsICJiaWN1YmljIikgaWYgZidtb2RlPSJ7bW9kZX0iJyBpbiBzb3VyY2Ugb3IgZiJtb2RlPSd7bW9kZX0nIiBpbiBzb3VyY2VdDQogICAgaWYgbGVuKG1vZGVzKSAhPSAxOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJDYW5ub3QgcmVhZCB0aGUgaW50ZXJwb2xhdGlvbiBtb2RlIG9mIHRoZSBpbnN0YWxsZWQgQW5vbWFsaWIgUGF0Y2hDb3JlOiB7bW9kZXN9IikNCiAgICBpZiBtb2Rlc1swXSAhPSBjZmcudXBzYW1wbGVfbW9kZToNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKA0KICAgICAgICAgICAgZiJJbnN0YWxsZWQgQW5vbWFsaWIgdXBzYW1wbGVzIGxheWVyMyB3aXRoIHttb2Rlc1swXSFyfSwgY29uZmlnLnlhbWwgdXNlcyB7Y2ZnLnVwc2FtcGxlX21vZGUhcn0uICINCiAgICAgICAgICAgICJTZXQgdXBzYW1wbGVfbW9kZSBpbiBjb25maWcueWFtbCB0byBtYXRjaCBiZWZvcmUgY29tcHV0aW5nIGZlYXR1cmVzLiINCiAgICAgICAgKQ0KICAgIHJldHVybiBtb2Rlc1swXQ0KDQoNCmRlZiByZWNvcmRfc2FtcGxlcl9zb3VyY2UoY2ZnOiBDb25maWcpIC0+IHN0cjoNCiAgICBpbXBvcnQgYW5vbWFsaWINCg0KICAgIGNodW5rcyA9IFtmImFub21hbGliIHtnZXRhdHRyKGFub21hbGliLCAnX192ZXJzaW9uX18nLCAndW5rbm93bicpfSJdDQogICAgdHJ5Og0KICAgICAgICBmcm9tIGFub21hbGliLm1vZGVscy5pbWFnZS5wYXRjaGNvcmUudG9yY2hfbW9kZWwgaW1wb3J0IFBhdGNoY29yZU1vZGVsDQoNCiAgICAgICAgY2h1bmtzLmFwcGVuZChpbnNwZWN0LmdldHNvdXJjZShQYXRjaGNvcmVNb2RlbC5zdWJzYW1wbGVfZW1iZWRkaW5nKSkNCiAgICAgICAgY2h1bmtzLmFwcGVuZChpbnNwZWN0LmdldHNvdXJjZShQYXRjaGNvcmVNb2RlbC5nZW5lcmF0ZV9lbWJlZGRpbmcpKQ0KICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZXhjOg0KICAgICAgICBjaHVua3MuYXBwZW5kKGYiQ291bGQgbm90IHJlYWQgUGF0Y2hjb3JlTW9kZWwgc291cmNlOiB7ZXhjfSIpDQogICAgdHJ5Og0KICAgICAgICBmcm9tIGFub21hbGliLm1vZGVscy5jb21wb25lbnRzLnNhbXBsaW5nIGltcG9ydCBLQ2VudGVyR3JlZWR5DQogICAgZXhjZXB0IEltcG9ydEVycm9yOg0KICAgICAgICB0cnk6DQogICAgICAgICAgICBmcm9tIGFub21hbGliLm1vZGVscy5jb21wb25lbnRzIGltcG9ydCBLQ2VudGVyR3JlZWR5DQogICAgICAgIGV4Y2VwdCBJbXBvcnRFcnJvciBhcyBleGM6DQogICAgICAgICAgICBLQ2VudGVyR3JlZWR5ID0gTm9uZQ0KICAgICAgICAgICAgY2h1bmtzLmFwcGVuZChmIkNvdWxkIG5vdCBpbXBvcnQgS0NlbnRlckdyZWVkeToge2V4Y30iKQ0KICAgIGlmIEtDZW50ZXJHcmVlZHkgaXMgbm90IE5vbmU6DQogICAgICAgIGNodW5rcy5hcHBlbmQoaW5zcGVjdC5nZXRzb3VyY2UoS0NlbnRlckdyZWVkeS5zYW1wbGVfY29yZXNldCkpDQogICAgcGF0aCA9IGNmZy53b3JrX3Jvb3QgLyAiZTAiIC8gInNhbXBsZXJfc291cmNlLnR4dCINCiAgICBwYXRoLnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpDQogICAgcGF0aC53cml0ZV90ZXh0KCJcblxuIi5qb2luKGNodW5rcyksIGVuY29kaW5nPSJ1dGYtOCIpDQogICAgcmV0dXJuIHN0cihnZXRhdHRyKGFub21hbGliLCAiX192ZXJzaW9uX18iLCAidW5rbm93biIpKQ0KDQoNCmRlZiBfYmF0Y2hlcyhjZmc6IENvbmZpZywgY2xhc3NfbmFtZTogc3RyLCByZWxhdGl2ZV9wYXRoczogbGlzdFtzdHJdLCBiYXRjaF9zaXplOiBpbnQpOg0KICAgIGltcG9ydCB0b3JjaA0KDQogICAgdHJhbnNmb3JtID0gYnVpbGRfdHJhbnNmb3JtKGNmZykNCiAgICBmb3Igc3RhcnQgaW4gcmFuZ2UoMCwgbGVuKHJlbGF0aXZlX3BhdGhzKSwgYmF0Y2hfc2l6ZSk6DQogICAgICAgIGNodW5rID0gcmVsYXRpdmVfcGF0aHNbc3RhcnQgOiBzdGFydCArIGJhdGNoX3NpemVdDQogICAgICAgIGltYWdlcyA9IFt0cmFuc2Zvcm0ocmVhZF9yZ2IoYWJzb2x1dGVfcGF0aChjZmcuZGF0YV9yb290LCBwYXRoKSwgY2xhc3NfbmFtZSkpIGZvciBwYXRoIGluIGNodW5rXQ0KICAgICAgICB5aWVsZCB0b3JjaC5zdGFjayhpbWFnZXMsIGRpbT0wKQ0KDQoNCmRlZiBfcHJlZGljdGlvbl9zY29yZXMob3V0cHV0KSAtPiAibnAubmRhcnJheSI6DQogICAgaW1wb3J0IHRvcmNoDQoNCiAgICBpZiBoYXNhdHRyKG91dHB1dCwgInByZWRfc2NvcmUiKToNCiAgICAgICAgc2NvcmVzID0gb3V0cHV0LnByZWRfc2NvcmUNCiAgICBlbGlmIGlzaW5zdGFuY2Uob3V0cHV0LCB0dXBsZSkgYW5kIGxlbihvdXRwdXQpID49IDI6DQogICAgICAgIHNjb3JlcyA9IG91dHB1dFsxXQ0KICAgIGVsc2U6DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIkFub21hbGliIG91dHB1dCBoYXMgbm8gaW1hZ2Ugc2NvcmU6IHt0eXBlKG91dHB1dCl9IikNCiAgICBhcnJheSA9IHNjb3Jlcy5kZXRhY2goKS5jcHUoKS5udW1weSgpLnJlc2hhcGUoLTEpDQogICAgaWYgbm90IG5wLmlzZmluaXRlKGFycmF5KS5hbGwoKToNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJBbm9tYWxpYiByZXR1cm5lZCBhIG5vbi1maW5pdGUgaW1hZ2Ugc2NvcmUuIikNCiAgICByZXR1cm4gYXJyYXkNCg0KDQpkZWYgX2ZpdF9hbm9tYWxpYihjZmc6IENvbmZpZywgY2xhc3NfbmFtZTogc3RyLCB0cmFpbl9wYXRoczogbGlzdFtzdHJdLCBudW1fbmVpZ2hib3JzOiBpbnQsIGZ1bGxfYmFuazogYm9vbCk6DQogICAgIiIiVHJhaW5pbmctbW9kZSBmb3J3YXJkIHJldHVybnMgdGhlIHBhdGNoIGVtYmVkZGluZyBpbiBldmVyeSBBbm9tYWxpYiAyLnggcmVsZWFzZS4NCg0KICAgIEVtYmVkZGluZ3MgYXJlIGNvbGxlY3RlZCBoZXJlIGluc3RlYWQgb2YgdGhyb3VnaCBgYGVtYmVkZGluZ19zdG9yZWBgIG9yIGBgc3Vic2FtcGxlX2VtYmVkZGluZ2BgLA0KICAgIHdob3NlIHNpZ25hdHVyZXMgY2hhbmdlZCBiZXR3ZWVuIHJlbGVhc2VzLg0KICAgICIiIg0KICAgIGltcG9ydCB0b3JjaA0KICAgIGZyb20gYW5vbWFsaWIubW9kZWxzLmltYWdlLnBhdGNoY29yZS50b3JjaF9tb2RlbCBpbXBvcnQgUGF0Y2hjb3JlTW9kZWwNCg0KICAgIGRldmljZSA9IHRvcmNoLmRldmljZSgiY3VkYSIgaWYgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKSBlbHNlICJjcHUiKQ0KICAgIG1vZGVsID0gUGF0Y2hjb3JlTW9kZWwoDQogICAgICAgIGxheWVycz1saXN0KGNmZy5sYXllcnMpLA0KICAgICAgICBiYWNrYm9uZT1jZmcuYmFja2JvbmUsDQogICAgICAgIHByZV90cmFpbmVkPVRydWUsDQogICAgICAgIG51bV9uZWlnaGJvcnM9bnVtX25laWdoYm9ycywNCiAgICApLnRvKGRldmljZSkNCiAgICBtb2RlbC50cmFpbigpDQogICAgY29sbGVjdGVkID0gW10NCiAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKToNCiAgICAgICAgZm9yIGJhdGNoIGluIF9iYXRjaGVzKGNmZywgY2xhc3NfbmFtZSwgdHJhaW5fcGF0aHMsIGJhdGNoX3NpemU9NCk6DQogICAgICAgICAgICBlbWJlZGRpbmcgPSBtb2RlbChiYXRjaC50byhkZXZpY2UpKQ0KICAgICAgICAgICAgaWYgbm90IGlzaW5zdGFuY2UoZW1iZWRkaW5nLCB0b3JjaC5UZW5zb3IpIG9yIGVtYmVkZGluZy5uZGltICE9IDI6DQogICAgICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYie2NsYXNzX25hbWV9OiB1bmV4cGVjdGVkIEFub21hbGliIHRyYWluaW5nIG91dHB1dCB7dHlwZShlbWJlZGRpbmcpfSIpDQogICAgICAgICAgICBjb2xsZWN0ZWQuYXBwZW5kKGVtYmVkZGluZykNCiAgICAgICAgICAgIHN0b3JlID0gZ2V0YXR0cihtb2RlbCwgImVtYmVkZGluZ19zdG9yZSIsIE5vbmUpDQogICAgICAgICAgICBpZiBpc2luc3RhbmNlKHN0b3JlLCBsaXN0KToNCiAgICAgICAgICAgICAgICBzdG9yZS5jbGVhcigpDQogICAgbW9kZWwuZXZhbCgpDQogICAgc3RvcmVkID0gdG9yY2gudnN0YWNrKGNvbGxlY3RlZCkNCiAgICBkZWwgY29sbGVjdGVkDQogICAgZXhwZWN0ZWRfcm93cyA9IGxlbih0cmFpbl9wYXRocykgKiA3ODQNCiAgICBpZiBpbnQoc3RvcmVkLnNoYXBlWzBdKSAhPSBleHBlY3RlZF9yb3dzIG9yIGludChzdG9yZWQuc2hhcGVbMV0pICE9IDE1MzY6DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIntjbGFzc19uYW1lfTogQW5vbWFsaWIgZW1iZWRkaW5ncyB7dHVwbGUoc3RvcmVkLnNoYXBlKX0sIGV4cGVjdGVkICh7ZXhwZWN0ZWRfcm93c30sIDE1MzYpIikNCiAgICBpZiBmdWxsX2Jhbms6DQogICAgICAgICMgc2FtcGxpbmdfcmF0aW89MSB3b3VsZCBzdGlsbCBydW4gS0NlbnRlckdyZWVkeSBvdmVyIGV2ZXJ5IHBhdGNoLiBUaGUgc2FtcGxlciBpcyBza2lwcGVkLg0KICAgICAgICBtb2RlbC5tZW1vcnlfYmFuayA9IHN0b3JlZA0KICAgIGVsc2U6DQogICAgICAgIHNhbXBsZXIgPSBfa2NlbnRlcl9ncmVlZHlfY2xhc3MoKShlbWJlZGRpbmc9c3RvcmVkLCBzYW1wbGluZ19yYXRpbz1mbG9hdChjZmcuZTBbImNvcmVzZXRfcmF0aW8iXSkpDQogICAgICAgIG1vZGVsLm1lbW9yeV9iYW5rID0gc2FtcGxlci5zYW1wbGVfY29yZXNldCgpDQogICAgICAgIGRlbCBzdG9yZWQNCiAgICBpZiBpbnQobW9kZWwubWVtb3J5X2Jhbmsuc2hhcGVbMF0pID09IDA6DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIntjbGFzc19uYW1lfTogbWVtb3J5IGJhbmsgaXMgZW1wdHkuIikNCiAgICByZXR1cm4gbW9kZWwsIGRldmljZQ0KDQoNCmRlZiBfcmVsZWFzZShtb2RlbCkgLT4gTm9uZToNCiAgICBpbXBvcnQgZ2MNCg0KICAgIGltcG9ydCB0b3JjaA0KDQogICAgZGVsIG1vZGVsDQogICAgZ2MuY29sbGVjdCgpDQogICAgaWYgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKToNCiAgICAgICAgdG9yY2guY3VkYS5lbXB0eV9jYWNoZSgpDQoNCg0KZGVmIF9zY29yZV90ZXN0KGNmZzogQ29uZmlnLCBjbGFzc19uYW1lOiBzdHIsIG1vZGVsLCBkZXZpY2UpIC0+IHR1cGxlW25wLm5kYXJyYXksIG5wLm5kYXJyYXldOg0KICAgIGltcG9ydCB0b3JjaA0KDQogICAgc3BsaXQgPSBsb2FkX3NwbGl0KGNmZywgY2xhc3NfbmFtZSwgMCkNCiAgICB0ZXN0X3BhdGhzID0gbGlzdChzcGxpdFsidGVzdF9nb29kIl0pICsgW2l0ZW1bInBhdGgiXSBmb3IgaXRlbSBpbiBzcGxpdFsidGVzdF9kZWZlY3QiXV0NCiAgICBsYWJlbHMgPSBucC5hcnJheShbMF0gKiBsZW4oc3BsaXRbInRlc3RfZ29vZCJdKSArIFsxXSAqIGxlbihzcGxpdFsidGVzdF9kZWZlY3QiXSksIGR0eXBlPWludCkNCiAgICBzY29yZXMgPSBbXQ0KICAgIG1vZGVsLmV2YWwoKQ0KICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOg0KICAgICAgICBmb3IgYmF0Y2ggaW4gX2JhdGNoZXMoY2ZnLCBjbGFzc19uYW1lLCB0ZXN0X3BhdGhzLCBiYXRjaF9zaXplPTIpOg0KICAgICAgICAgICAgb3V0cHV0ID0gbW9kZWwoYmF0Y2gudG8oZGV2aWNlKSkNCiAgICAgICAgICAgIHNjb3Jlcy5hcHBlbmQoX3ByZWRpY3Rpb25fc2NvcmVzKG91dHB1dCkpDQogICAgc3RhY2tlZCA9IG5wLmNvbmNhdGVuYXRlKHNjb3JlcykNCiAgICBpZiBzdGFja2VkLnNoYXBlICE9IGxhYmVscy5zaGFwZToNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYie2NsYXNzX25hbWV9OiBzY29yZWQge3N0YWNrZWQuc2hhcGVbMF19IHRlc3QgaW1hZ2VzLCBleHBlY3RlZCB7bGFiZWxzLnNoYXBlWzBdfS4iKQ0KICAgIHJldHVybiBzdGFja2VkLCBsYWJlbHMNCg0KDQpkZWYgb3duX2Z1bGxfYmFua19zY29yZXMoc3BsaXQ6IGRpY3QsIGRpc3RhbmNlOiBucC5uZGFycmF5KSAtPiB0dXBsZVtucC5uZGFycmF5LCBucC5uZGFycmF5XToNCiAgICBuX3RyYWluID0gaW50KHNwbGl0WyJuX3RyYWluIl0pDQogICAgbl90ZXN0ID0gaW50KHNwbGl0WyJuX3Rlc3RfZ29vZCJdKSArIGludChzcGxpdFsibl90ZXN0X2RlZmVjdCJdKQ0KICAgIHF1ZXJpZXMgPSBucC5hcmFuZ2Uobl90cmFpbiwgbl90cmFpbiArIG5fdGVzdCkNCiAgICBiYW5rID0gbnAuYXJhbmdlKG5fdHJhaW4pDQogICAgc2NvcmVzID0gc2NvcmVzX2Zvcl9iYW5rKGRpc3RhbmNlLCBxdWVyaWVzLCBiYW5rKQ0KICAgIGxhYmVscyA9IG5wLmFycmF5KFswXSAqIGludChzcGxpdFsibl90ZXN0X2dvb2QiXSkgKyBbMV0gKiBpbnQoc3BsaXRbIm5fdGVzdF9kZWZlY3QiXSksIGR0eXBlPWludCkNCiAgICByZXR1cm4gc2NvcmVzLCBsYWJlbHMNCg0KDQpkZWYgX2F1cm9jKGxhYmVsczogbnAubmRhcnJheSwgc2NvcmVzOiBucC5uZGFycmF5KSAtPiBmbG9hdDoNCiAgICByZXR1cm4gZmxvYXQocm9jX2F1Y19zY29yZShsYWJlbHMsIHNjb3JlcykpDQoNCg0KZGVmIF9taW5tYXgodmFsdWVzOiBucC5uZGFycmF5KSAtPiBucC5uZGFycmF5Og0KICAgIHNwYW4gPSBmbG9hdCh2YWx1ZXMubWF4KCkgLSB2YWx1ZXMubWluKCkpDQogICAgaWYgc3BhbiA8PSAwLjA6DQogICAgICAgIHJldHVybiBucC56ZXJvc19saWtlKHZhbHVlcywgZHR5cGU9ZmxvYXQpDQogICAgcmV0dXJuICh2YWx1ZXMgLSB2YWx1ZXMubWluKCkpIC8gc3Bhbg0KDQoNCmRlZiBydW5fY2xhc3MoY2ZnOiBDb25maWcsIGNsYXNzX25hbWU6IHN0ciwgcGFydDogc3RyLCB0cmFpbl9wYXRoczogbGlzdFtzdHJdLCBkaXN0YW5jZTogbnAubmRhcnJheSkgLT4gZGljdDoNCiAgICBwYXRoID0gX2NsYXNzX3BhdGgoY2ZnLCBjbGFzc19uYW1lKQ0KICAgIGV4aXN0aW5nID0gcmVhZF9qc29uKHBhdGgpIGlmIHBhdGguaXNfZmlsZSgpIGVsc2Uge30NCiAgICBpZiBleGlzdGluZy5nZXQoImNvbmZpZ19oYXNoIikgIT0gY2ZnLmhhc2g6DQogICAgICAgIGV4aXN0aW5nID0ge30NCiAgICBzcGxpdCA9IGxvYWRfc3BsaXQoY2ZnLCBjbGFzc19uYW1lLCAwKQ0KICAgIGlmIHBhcnQgaW4geyJhIiwgImFsbCJ9IGFuZCAiZTBhIiBub3QgaW4gZXhpc3Rpbmc6DQogICAgICAgIExPR0dFUi5pbmZvKCJFMGEgJXMiLCBjbGFzc19uYW1lKQ0KICAgICAgICBtb2RlbCwgZGV2aWNlID0gX2ZpdF9hbm9tYWxpYihjZmcsIGNsYXNzX25hbWUsIHRyYWluX3BhdGhzLCBpbnQoY2ZnLmUwWyJudW1fbmVpZ2hib3JzX3JlZiJdKSwgZnVsbF9iYW5rPUZhbHNlKQ0KICAgICAgICBzY29yZXMsIGxhYmVscyA9IF9zY29yZV90ZXN0KGNmZywgY2xhc3NfbmFtZSwgbW9kZWwsIGRldmljZSkNCiAgICAgICAgZXhpc3RpbmdbImUwYSJdID0geyJhdXJvYyI6IF9hdXJvYyhsYWJlbHMsIHNjb3JlcyksICJuX3Rlc3QiOiBpbnQobGFiZWxzLnNpemUpfQ0KICAgICAgICBfcmVsZWFzZShtb2RlbCkNCiAgICAgICAgd3JpdGVfanNvbihwYXRoLCB7KipleGlzdGluZywgImNvbmZpZ19oYXNoIjogY2ZnLmhhc2gsICJjbGFzc19uYW1lIjogY2xhc3NfbmFtZX0pDQogICAgaWYgcGFydCBpbiB7ImIiLCAiYWxsIn0gYW5kICJlMGIiIG5vdCBpbiBleGlzdGluZzoNCiAgICAgICAgTE9HR0VSLmluZm8oIkUwYiAlcyIsIGNsYXNzX25hbWUpDQogICAgICAgIG1vZGVsLCBkZXZpY2UgPSBfZml0X2Fub21hbGliKGNmZywgY2xhc3NfbmFtZSwgdHJhaW5fcGF0aHMsIGludChjZmcuZTBbIm51bV9uZWlnaGJvcnNfcnEiXSksIGZ1bGxfYmFuaz1UcnVlKQ0KICAgICAgICBhbm9tYWxpYl9zY29yZXMsIGxhYmVscyA9IF9zY29yZV90ZXN0KGNmZywgY2xhc3NfbmFtZSwgbW9kZWwsIGRldmljZSkNCiAgICAgICAgX3JlbGVhc2UobW9kZWwpDQogICAgICAgIG93bl9zY29yZXMsIG93bl9sYWJlbHMgPSBvd25fZnVsbF9iYW5rX3Njb3JlcyhzcGxpdCwgZGlzdGFuY2UpDQogICAgICAgIGlmIG5vdCBucC5hcnJheV9lcXVhbChsYWJlbHMsIG93bl9sYWJlbHMpOg0KICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYie2NsYXNzX25hbWV9OiBFMGIgbGFiZWwgb3JkZXIgZGlmZmVycyBiZXR3ZWVuIHRoZSB0d28gaW1wbGVtZW50YXRpb25zLiIpDQogICAgICAgIHJobyA9IGZsb2F0KHNwZWFybWFucihvd25fc2NvcmVzLCBhbm9tYWxpYl9zY29yZXMpLnN0YXRpc3RpYykNCiAgICAgICAgZXhpc3RpbmdbImUwYiJdID0gew0KICAgICAgICAgICAgInNwZWFybWFuIjogcmhvLA0KICAgICAgICAgICAgImF1cm9jX293biI6IF9hdXJvYyhsYWJlbHMsIG93bl9zY29yZXMpLA0KICAgICAgICAgICAgImF1cm9jX2Fub21hbGliIjogX2F1cm9jKGxhYmVscywgYW5vbWFsaWJfc2NvcmVzKSwNCiAgICAgICAgICAgICJtYXhfYWJzX3Njb3JlX2dhcCI6IGZsb2F0KG5wLm1heChucC5hYnMob3duX3Njb3JlcyAtIGFub21hbGliX3Njb3JlcykpKSwNCiAgICAgICAgICAgICJtYXhfYWJzX2dhcF9hZnRlcl9jbGFzc19taW5tYXgiOiBmbG9hdChucC5tYXgobnAuYWJzKF9taW5tYXgob3duX3Njb3JlcykgLSBfbWlubWF4KGFub21hbGliX3Njb3JlcykpKSksDQogICAgICAgIH0NCiAgICBleGlzdGluZ1siY29uZmlnX2hhc2giXSA9IGNmZy5oYXNoDQogICAgZXhpc3RpbmdbImNsYXNzX25hbWUiXSA9IGNsYXNzX25hbWUNCiAgICB3cml0ZV9qc29uKHBhdGgsIGV4aXN0aW5nKQ0KICAgIHJldHVybiBleGlzdGluZw0KDQoNCmRlZiBkaXN0YW5jZV9hZ3JlZW1lbnQoY2ZnOiBDb25maWcsIGNsYXNzX25hbWU6IHN0ciwgZmVhdHVyZXM6IGRpY3QsIGRpc3RhbmNlOiBucC5uZGFycmF5LCBwYXRoczogbGlzdFtzdHJdKSAtPiBkaWN0W3N0ciwgZmxvYXRdOg0KICAgIHNwbGl0ID0gbG9hZF9zcGxpdChjZmcsIGNsYXNzX25hbWUsIDApDQogICAgaW5kZXggPSB7cGF0aDogcG9zaXRpb24gZm9yIHBvc2l0aW9uLCBwYXRoIGluIGVudW1lcmF0ZShwYXRocyl9DQogICAgcG9vbCA9IFtpbmRleFtwYXRoXSBmb3IgcGF0aCBpbiBzcGxpdFsicG9vbCJdXQ0KICAgIF9zcGxpdF9ybmcsIHJhbmRvbV9ybmcsIF9rY2VudGVyX3JuZyA9IHNwbGl0X3NlZWRzKGNmZywgY2xhc3NfbmFtZSwgMCkNCiAgICBvcmRlciA9IHJhbmRvbV9vcmRlcihwb29sLCByYW5kb21fcm5nKQ0KICAgIHF1ZXJpZXMgPSBbaW5kZXhbcGF0aF0gZm9yIHBhdGggaW4gbGlzdChzcGxpdFsiY2FsaWJyYXRpb24iXSlbOjJdICsgbGlzdChzcGxpdFsidGVzdF9nb29kIl0pWzoyXV0NCiAgICBnYXBzID0ge30NCiAgICBmb3IgbiBpbiAoMSwgOCwgbGVuKG9yZGVyKSk6DQogICAgICAgIGlmIG4gPiBsZW4ob3JkZXIpOg0KICAgICAgICAgICAgY29udGludWUNCiAgICAgICAgZ2Fwc1tzdHIobildID0gYXNzZXJ0X21ldGhvZHNfYWdyZWUoZmVhdHVyZXNbInBhdGNoZXMiXSwgZGlzdGFuY2UsIHF1ZXJpZXMsIG9yZGVyWzpuXSkNCiAgICByZXR1cm4gZ2Fwcw0KDQoNCmRlZiBfcGFzc2VkX2UwYihjZmc6IENvbmZpZywgcm93czogbGlzdFtkaWN0XSkgLT4gdHVwbGVbYm9vbCwgZGljdF06DQogICAgcmhvcyA9IFtmbG9hdChyb3dbImUwYiJdWyJzcGVhcm1hbiJdKSBmb3Igcm93IGluIHJvd3NdDQogICAgZGVsdGFzID0gW2FicyhmbG9hdChyb3dbImUwYiJdWyJhdXJvY19vd24iXSkgLSBmbG9hdChyb3dbImUwYiJdWyJhdXJvY19hbm9tYWxpYiJdKSkgZm9yIHJvdyBpbiByb3dzXQ0KICAgIG93bl9tZWFuID0gZmxvYXQobnAubWVhbihbZmxvYXQocm93WyJlMGIiXVsiYXVyb2Nfb3duIl0pIGZvciByb3cgaW4gcm93c10pKQ0KICAgIGFub21hbGliX21lYW4gPSBmbG9hdChucC5tZWFuKFtmbG9hdChyb3dbImUwYiJdWyJhdXJvY19hbm9tYWxpYiJdKSBmb3Igcm93IGluIHJvd3NdKSkNCiAgICBtZWFuX2RlbHRhID0gYWJzKG93bl9tZWFuIC0gYW5vbWFsaWJfbWVhbikNCiAgICBzdW1tYXJ5ID0gew0KICAgICAgICAic3BlYXJtYW5fbWluIjogZmxvYXQobnAubWluKHJob3MpKSwNCiAgICAgICAgInNwZWFybWFuX21lZGlhbiI6IGZsb2F0KG5wLm1lZGlhbihyaG9zKSksDQogICAgICAgICJtZWFuX2F1cm9jX293biI6IG93bl9tZWFuLA0KICAgICAgICAibWVhbl9hdXJvY19hbm9tYWxpYiI6IGFub21hbGliX21lYW4sDQogICAgICAgICJhYnNfZGVsdGFfb2ZfbWVhbl9hdXJvYyI6IG1lYW5fZGVsdGEsDQogICAgICAgICJtZWFuX29mX3Blcl9jbGFzc19hYnNfYXVyb2NfZGVsdGEiOiBmbG9hdChucC5tZWFuKGRlbHRhcykpLA0KICAgICAgICAiZmFpbGVkX2NsYXNzZXMiOiBbDQogICAgICAgICAgICByb3dbImNsYXNzX25hbWUiXQ0KICAgICAgICAgICAgZm9yIHJvdyBpbiByb3dzDQogICAgICAgICAgICBpZiBmbG9hdChyb3dbImUwYiJdWyJzcGVhcm1hbiJdKSA8IGZsb2F0KGNmZy5lMFsic3BlYXJtYW5fbWluIl0pDQogICAgICAgIF0sDQogICAgfQ0KICAgIG9rID0gc3VtbWFyeVsic3BlYXJtYW5fbWluIl0gPj0gZmxvYXQoY2ZnLmUwWyJzcGVhcm1hbl9taW4iXSkgYW5kIG1lYW5fZGVsdGEgPD0gZmxvYXQoY2ZnLmUwWyJhdXJvY19tZWFuX3RvbGVyYW5jZSJdKQ0KICAgIHJldHVybiBvaywgc3VtbWFyeQ0KDQoNCmRlZiBydW5fZTAoY2ZnOiBDb25maWcsIGNsYXNzZXM6IGxpc3Rbc3RyXSwgcGFydDogc3RyID0gImFsbCIpIC0+IGRpY3Q6DQogICAgdmVyc2lvbiA9IHJlY29yZF9zYW1wbGVyX3NvdXJjZShjZmcpDQogICAgdXBzYW1wbGluZyA9IGNoZWNrX2Fub21hbGliX3Vwc2FtcGxpbmcoY2ZnKQ0KICAgIGNoZWNrX2NvcnJ1cHRlZF9pbnB1dCgpDQogICAgcm93cyA9IFtdDQogICAgaW50ZWdyaXR5ID0geyJjb3JydXB0ZWRfaW5wdXQiOiAicGFzcyIsICJhbm9tYWxpYl91cHNhbXBsaW5nIjogdXBzYW1wbGluZywgImNsYXNzZXMiOiB7fX0NCiAgICBmb3IgY2xhc3NfbmFtZSBpbiBjbGFzc2VzOg0KICAgICAgICBmZWF0dXJlcyA9IGxvYWRfZmVhdHVyZV9jYWNoZShjZmcsIGNsYXNzX25hbWUpDQogICAgICAgIGNoZWNrX2ZlYXR1cmVfYWxpZ25tZW50KGNmZywgY2xhc3NfbmFtZSwgZmVhdHVyZXMpDQogICAgICAgIGRpc3RhbmNlX2NhY2hlID0gbG9hZF9kaXN0YW5jZV9jYWNoZShjZmcsIGNsYXNzX25hbWUpDQogICAgICAgIGlmIGxpc3QoZGlzdGFuY2VfY2FjaGVbIm1ldGEiXVsicGF0aHMiXSkgIT0gZmVhdHVyZXNbInBhdGhzIl06DQogICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJ7Y2xhc3NfbmFtZX06IGZlYXR1cmUgYW5kIGRpc3RhbmNlIGNhY2hlcyBhcmUgbm90IGFsaWduZWQuIikNCiAgICAgICAgYWdyZWVtZW50ID0gZGlzdGFuY2VfYWdyZWVtZW50KGNmZywgY2xhc3NfbmFtZSwgZmVhdHVyZXMsIGRpc3RhbmNlX2NhY2hlWyJEIl0sIGZlYXR1cmVzWyJwYXRocyJdKQ0KICAgICAgICBpbnRlZ3JpdHlbImNsYXNzZXMiXVtjbGFzc19uYW1lXSA9IHsiZGlzdGFuY2VfdnNfa25uIjogYWdyZWVtZW50fQ0KICAgICAgICB0cmFpbl9wYXRocyA9IGZlYXR1cmVzWyJwYXRocyJdWzogaW50KGZlYXR1cmVzWyJtZXRhIl1bIm5fdHJhaW4iXSldDQogICAgICAgIGRlbCBmZWF0dXJlcw0KICAgICAgICByb3dzLmFwcGVuZChydW5fY2xhc3MoY2ZnLCBjbGFzc19uYW1lLCBwYXJ0LCB0cmFpbl9wYXRocywgZGlzdGFuY2VfY2FjaGVbIkQiXSkpDQogICAgICAgIGRlbCBkaXN0YW5jZV9jYWNoZQ0KICAgIGUwYV9vayA9IFRydWUNCiAgICBlMGFfc3VtbWFyeSA9IE5vbmUNCiAgICBpZiBwYXJ0IGluIHsiYSIsICJhbGwifToNCiAgICAgICAgcGVyX2NsYXNzID0ge3Jvd1siY2xhc3NfbmFtZSJdOiBmbG9hdChyb3dbImUwYSJdWyJhdXJvYyJdKSBmb3Igcm93IGluIHJvd3N9DQogICAgICAgIGUwYV9zdW1tYXJ5ID0gew0KICAgICAgICAgICAgIm1lYW5fYXVyb2MiOiBmbG9hdChucC5tZWFuKGxpc3QocGVyX2NsYXNzLnZhbHVlcygpKSkpLA0KICAgICAgICAgICAgIm1pbl9hdXJvYyI6IGZsb2F0KG5wLm1pbihsaXN0KHBlcl9jbGFzcy52YWx1ZXMoKSkpKSwNCiAgICAgICAgICAgICJwZXJfY2xhc3MiOiBwZXJfY2xhc3MsDQogICAgICAgIH0NCiAgICAgICAgZTBhX29rID0gZTBhX3N1bW1hcnlbIm1lYW5fYXVyb2MiXSA+PSBmbG9hdChjZmcuZTBbIm1lYW5fYXVyb2NfZmxvb3IiXSkNCiAgICBlMGJfb2sgPSBUcnVlDQogICAgZTBiX3N1bW1hcnkgPSBOb25lDQogICAgaWYgcGFydCBpbiB7ImIiLCAiYWxsIn06DQogICAgICAgIGUwYl9vaywgZTBiX3N1bW1hcnkgPSBfcGFzc2VkX2UwYihjZmcsIHJvd3MpDQogICAgY29tcGxldGUgPSBwYXJ0ID09ICJhbGwiIGFuZCBzZXQoY2xhc3NlcykgPT0gc2V0KGNmZy5jbGFzc2VzKQ0KICAgIHN0YXR1cyA9ICJQQVNTIiBpZiBjb21wbGV0ZSBhbmQgZTBhX29rIGFuZCBlMGJfb2sgZWxzZSAiRkFJTCINCiAgICBpZiBlMGFfb2sgYW5kIGUwYl9vayBhbmQgbm90IGNvbXBsZXRlOg0KICAgICAgICBzdGF0dXMgPSAiUEFSVElBTCINCiAgICBwYXlsb2FkID0gew0KICAgICAgICAic3RhdHVzIjogc3RhdHVzLA0KICAgICAgICAiY29uZmlnX2hhc2giOiBjZmcuaGFzaCwNCiAgICAgICAgImFub21hbGliX3ZlcnNpb24iOiB2ZXJzaW9uLA0KICAgICAgICAicGFydCI6IHBhcnQsDQogICAgICAgICJjbGFzc2VzIjogY2xhc3NlcywNCiAgICAgICAgImUwYSI6IGUwYV9zdW1tYXJ5LA0KICAgICAgICAiZTBiIjogZTBiX3N1bW1hcnksDQogICAgICAgICJwZXJfY2xhc3MiOiByb3dzLA0KICAgICAgICAiaW50ZWdyaXR5IjogaW50ZWdyaXR5LA0KICAgICAgICAicmVmZXJlbmNlX21lYW5fYXVyb2MiOiBmbG9hdChjZmcuZTBbInJlZmVyZW5jZV9tZWFuX2F1cm9jIl0pLA0KICAgIH0NCiAgICB3cml0ZV9qc29uKGdhdGVfcGF0aChjZmcpLCBwYXlsb2FkKQ0KICAgIExPR0dFUi5pbmZvKCJFMCBnYXRlICVzIiwgc3RhdHVzKQ0KICAgIHJldHVybiBwYXlsb2FkDQo=", "commissioning/exploratory.py": "...", "commissioning/features.py": "IiIiV2lkZS1SZXNOZXQ1MC0yIHBhdGNoIGZlYXR1cmVzOiBsYXllcjIgKyBsYXllcjMsIDN4MyBwb29saW5nLCA3ODQgeCAxNTM2LiIiIg0KDQpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zDQoNCmltcG9ydCBsb2dnaW5nDQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgNCg0KaW1wb3J0IG51bXB5IGFzIG5wDQpmcm9tIFBJTCBpbXBvcnQgSW1hZ2UNCg0KZnJvbSBjb21taXNzaW9uaW5nLmNvbmZpZyBpbXBvcnQgQ29uZmlnDQpmcm9tIGNvbW1pc3Npb25pbmcuZGF0YSBpbXBvcnQgYWJzb2x1dGVfcGF0aCwgY29sbGVjdF9jbGFzcywgbG9hZF9zcGxpdCwgcmVsYXRpdmVfcGF0aA0KZnJvbSBjb21taXNzaW9uaW5nLmlvX3V0aWxzIGltcG9ydCBhdG9taWNfc2F2ZXosIHJlYWRfanNvbiwgd3JpdGVfanNvbg0KDQpMT0dHRVIgPSBsb2dnaW5nLmdldExvZ2dlcihfX25hbWVfXykNCg0KDQpkZWYgcmVhZF9yZ2IocGF0aDogUGF0aCwgY2xhc3NfbmFtZTogc3RyKSAtPiBJbWFnZS5JbWFnZToNCiAgICB0cnk6DQogICAgICAgIHdpdGggSW1hZ2Uub3BlbihwYXRoKSBhcyBpbWFnZToNCiAgICAgICAgICAgIHJnYiA9IGltYWdlLmNvbnZlcnQoIlJHQiIpDQogICAgICAgICAgICByZ2IubG9hZCgpDQogICAgICAgIGlmIHJnYi5zaXplWzBdIDwgMSBvciByZ2Iuc2l6ZVsxXSA8IDE6DQogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJlbXB0eSBpbWFnZSIpDQogICAgICAgIHJldHVybiByZ2INCiAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGV4YzoNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYie2NsYXNzX25hbWV9OiB7cGF0aH0iKSBmcm9tIGV4Yw0KDQoNCmRlZiBidWlsZF90cmFuc2Zvcm0oY2ZnOiBDb25maWcpOg0KICAgICIiIk1hdGNoIEFub21hbGliJ3MgUGF0Y2hjb3JlLmNvbmZpZ3VyZV9wcmVfcHJvY2Vzc29yKDI1NiwgY2VudGVyIGNyb3AgMjI0KS4iIiINCiAgICBmcm9tIHRvcmNodmlzaW9uLnRyYW5zZm9ybXMgaW1wb3J0IHYyDQoNCiAgICByZXR1cm4gdjIuQ29tcG9zZSgNCiAgICAgICAgWw0KICAgICAgICAgICAgdjIuVG9JbWFnZSgpLA0KICAgICAgICAgICAgdjIuVG9EdHlwZSh0b3JjaF9mbG9hdDMyKCksIHNjYWxlPVRydWUpLA0KICAgICAgICAgICAgdjIuUmVzaXplKChjZmcucmVzaXplLCBjZmcucmVzaXplKSwgYW50aWFsaWFzPVRydWUpLA0KICAgICAgICAgICAgdjIuQ2VudGVyQ3JvcCgoY2ZnLmNlbnRlcl9jcm9wLCBjZmcuY2VudGVyX2Nyb3ApKSwNCiAgICAgICAgICAgIHYyLk5vcm1hbGl6ZShtZWFuPWNmZy5pbWFnZW5ldF9tZWFuLCBzdGQ9Y2ZnLmltYWdlbmV0X3N0ZCksDQogICAgICAgIF0NCiAgICApDQoNCg0KZGVmIHRvcmNoX2Zsb2F0MzIoKToNCiAgICBpbXBvcnQgdG9yY2gNCg0KICAgIHJldHVybiB0b3JjaC5mbG9hdDMyDQoNCg0KZGVmIGJ1aWxkX2ZlYXR1cmVfZXh0cmFjdG9yKGNmZzogQ29uZmlnKToNCiAgICAiIiJQcmVmZXIgQW5vbWFsaWIncyBleHRyYWN0b3Igc28gdGhlIGNhY2hlIGFuZCBFMGIgc2hhcmUgd2VpZ2h0cyBhbmQgaG9va3MuIiIiDQogICAgaW1wb3J0IHRvcmNoLm5uIGFzIG5uDQoNCiAgICB0cnk6DQogICAgICAgIGZyb20gYW5vbWFsaWIubW9kZWxzLmNvbXBvbmVudHMgaW1wb3J0IFRpbW1GZWF0dXJlRXh0cmFjdG9yDQogICAgZXhjZXB0IEltcG9ydEVycm9yIGFzIGV4YzoNCiAgICAgICAgTE9HR0VSLndhcm5pbmcoImFub21hbGliIGlzIG5vdCBpbnN0YWxsZWQgKCVzKTsgdXNpbmcgYSB0aW1tIGhvb2sgZXh0cmFjdG9yLiIsIGV4YykNCiAgICAgICAgcmV0dXJuIF9Ib29rRXh0cmFjdG9yKGNmZy5iYWNrYm9uZSwgY2ZnLmxheWVycyksIGYidGltbS5ob29rcyAoe2V4Y30pIg0KDQogICAgbW9kZWwgPSBUaW1tRmVhdHVyZUV4dHJhY3RvcihiYWNrYm9uZT1jZmcuYmFja2JvbmUsIGxheWVycz1saXN0KGNmZy5sYXllcnMpLCBwcmVfdHJhaW5lZD1UcnVlKQ0KICAgIGlmIG5vdCBpc2luc3RhbmNlKG1vZGVsLCBubi5Nb2R1bGUpOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIlRpbW1GZWF0dXJlRXh0cmFjdG9yIGRpZCBub3QgcmV0dXJuIGEgdG9yY2ggbW9kdWxlLiIpDQogICAgcmV0dXJuIG1vZGVsLCBmImFub21hbGliLlRpbW1GZWF0dXJlRXh0cmFjdG9yIHdlaWdodHM9e193ZWlnaHRfdGFnKG1vZGVsKX0iDQoNCg0KZGVmIF93ZWlnaHRfdGFnKG1vZGVsKSAtPiBzdHI6DQogICAgIiIidGltbSBuYW1lcyBpdHMgd2VpZ2h0cywgZS5nLiB3aWRlX3Jlc25ldDUwXzIucmFjbV9pbjFrIHZzIC50dl9pbjFrLiBUaGUgcGFwZXIgbXVzdCBzdGF0ZSB3aGljaC4iIiINCiAgICBmb3IgbW9kdWxlIGluIG1vZGVsLm1vZHVsZXMoKToNCiAgICAgICAgY29uZmlnID0gZ2V0YXR0cihtb2R1bGUsICJwcmV0cmFpbmVkX2NmZyIsIE5vbmUpDQogICAgICAgIGlmIGlzaW5zdGFuY2UoY29uZmlnLCBkaWN0KSBhbmQgKGNvbmZpZy5nZXQoInRhZyIpIG9yIGNvbmZpZy5nZXQoImhmX2h1Yl9pZCIpKToNCiAgICAgICAgICAgIGFyY2hpdGVjdHVyZSA9IGNvbmZpZy5nZXQoImFyY2hpdGVjdHVyZSIsICIiKQ0KICAgICAgICAgICAgdGFnID0gY29uZmlnLmdldCgidGFnIiwgIiIpDQogICAgICAgICAgICByZXR1cm4gZiJ7YXJjaGl0ZWN0dXJlfS57dGFnfSIgaWYgdGFnIGVsc2Ugc3RyKGNvbmZpZy5nZXQoImhmX2h1Yl9pZCIpKQ0KICAgIHJldHVybiAidW5rbm93biINCg0KDQpjbGFzcyBfSG9va0V4dHJhY3RvcjoNCiAgICBkZWYgX19pbml0X18oc2VsZiwgYmFja2JvbmU6IHN0ciwgbGF5ZXJzOiBsaXN0W3N0cl0pIC0+IE5vbmU6DQogICAgICAgIGltcG9ydCB0aW1tDQogICAgICAgIGltcG9ydCB0b3JjaC5ubiBhcyBubg0KDQogICAgICAgIHNlbGYubGF5ZXJzID0gbGlzdChsYXllcnMpDQogICAgICAgIHNlbGYuYmFja2JvbmUgPSB0aW1tLmNyZWF0ZV9tb2RlbChiYWNrYm9uZSwgcHJldHJhaW5lZD1UcnVlKQ0KICAgICAgICBzZWxmLmJhY2tib25lLmV2YWwoKQ0KICAgICAgICBuYW1lZCA9IGRpY3Qoc2VsZi5iYWNrYm9uZS5uYW1lZF9tb2R1bGVzKCkpDQogICAgICAgIG1pc3NpbmcgPSBbbmFtZSBmb3IgbmFtZSBpbiBzZWxmLmxheWVycyBpZiBuYW1lIG5vdCBpbiBuYW1lZF0NCiAgICAgICAgaWYgbWlzc2luZzoNCiAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIkJhY2tib25lIHtiYWNrYm9uZX0gaGFzIG5vIGxheWVycyB7bWlzc2luZ30iKQ0KICAgICAgICBzZWxmLl9mZWF0dXJlczogZGljdCA9IHt9DQogICAgICAgIGZvciBuYW1lIGluIHNlbGYubGF5ZXJzOg0KICAgICAgICAgICAgbmFtZWRbbmFtZV0ucmVnaXN0ZXJfZm9yd2FyZF9ob29rKHNlbGYuX21ha2VfaG9vayhuYW1lKSkNCiAgICAgICAgc2VsZi5fbm4gPSBubg0KDQogICAgZGVmIF9tYWtlX2hvb2soc2VsZiwgbmFtZTogc3RyKToNCiAgICAgICAgZGVmIGhvb2soX21vZHVsZSwgX2lucHV0cywgb3V0cHV0KSAtPiBOb25lOg0KICAgICAgICAgICAgc2VsZi5fZmVhdHVyZXNbbmFtZV0gPSBvdXRwdXQNCg0KICAgICAgICByZXR1cm4gaG9vaw0KDQogICAgZGVmIGV2YWwoc2VsZik6DQogICAgICAgIHNlbGYuYmFja2JvbmUuZXZhbCgpDQogICAgICAgIHJldHVybiBzZWxmDQoNCiAgICBkZWYgdG8oc2VsZiwgZGV2aWNlKToNCiAgICAgICAgc2VsZi5iYWNrYm9uZS50byhkZXZpY2UpDQogICAgICAgIHJldHVybiBzZWxmDQoNCiAgICBkZWYgX19jYWxsX18oc2VsZiwgYmF0Y2gpOg0KICAgICAgICBzZWxmLl9mZWF0dXJlcyA9IHt9DQogICAgICAgIF8gPSBzZWxmLmJhY2tib25lKGJhdGNoKQ0KICAgICAgICBtaXNzaW5nID0gW25hbWUgZm9yIG5hbWUgaW4gc2VsZi5sYXllcnMgaWYgbmFtZSBub3QgaW4gc2VsZi5fZmVhdHVyZXNdDQogICAgICAgIGlmIG1pc3Npbmc6DQogICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJGZWF0dXJlIGhvb2tzIGRpZCBub3QgZmlyZSBmb3Ige21pc3Npbmd9IikNCiAgICAgICAgcmV0dXJuIHtuYW1lOiBzZWxmLl9mZWF0dXJlc1tuYW1lXSBmb3IgbmFtZSBpbiBzZWxmLmxheWVyc30NCg0KDQpkZWYgcG9vbGVkX2VtYmVkZGluZyhmZWF0dXJlczogZGljdCwgbGF5ZXJzOiBsaXN0W3N0cl0sIHBvb2wsIHVwc2FtcGxlX21vZGU6IHN0cik6DQogICAgIiIiQXZlcmFnZS1wb29sIGVhY2ggbWFwLCB1cHNhbXBsZSBsYXRlciBtYXBzIHRvIHRoZSBmaXJzdCBtYXAsIHRoZW4gY29uY2F0ZW5hdGUuIiIiDQogICAgaW1wb3J0IHRvcmNoDQogICAgaW1wb3J0IHRvcmNoLm5uLmZ1bmN0aW9uYWwgYXMgRg0KDQogICAgcG9vbGVkID0gW3Bvb2woZmVhdHVyZXNbbmFtZV0pIGZvciBuYW1lIGluIGxheWVyc10NCiAgICBlbWJlZGRpbmcgPSBwb29sZWRbMF0NCiAgICBmb3IgbGF5ZXIgaW4gcG9vbGVkWzE6XToNCiAgICAgICAga3dhcmdzID0geyJhbGlnbl9jb3JuZXJzIjogRmFsc2V9IGlmIHVwc2FtcGxlX21vZGUgaW4geyJiaWxpbmVhciIsICJiaWN1YmljIn0gZWxzZSB7fQ0KICAgICAgICBsYXllciA9IEYuaW50ZXJwb2xhdGUobGF5ZXIsIHNpemU9ZW1iZWRkaW5nLnNoYXBlWy0yOl0sIG1vZGU9dXBzYW1wbGVfbW9kZSwgKiprd2FyZ3MpDQogICAgICAgIGVtYmVkZGluZyA9IHRvcmNoLmNhdCgoZW1iZWRkaW5nLCBsYXllciksIGRpbT0xKQ0KICAgIHJldHVybiBlbWJlZGRpbmcNCg0KDQpkZWYgcGF0Y2hlc19hbmRfZ2FwKGVtYmVkZGluZyk6DQogICAgIiIiZW1iZWRkaW5nOiAoQiwgQywgSCwgVykgLT4gcGF0Y2hlcyAoQiwgSCpXLCBDKSwgZ2FwIChCLCBDKS4iIiINCiAgICBiYXRjaCwgY2hhbm5lbHMsIF9oZWlnaHQsIF93aWR0aCA9IGVtYmVkZGluZy5zaGFwZQ0KICAgIHBhdGNoZXMgPSBlbWJlZGRpbmcucGVybXV0ZSgwLCAyLCAzLCAxKS5yZXNoYXBlKGJhdGNoLCAtMSwgY2hhbm5lbHMpDQogICAgZ2FwID0gcGF0Y2hlcy5tZWFuKGRpbT0xKQ0KICAgIHJldHVybiBwYXRjaGVzLCBnYXANCg0KDQpkZWYgZmVhdHVyZV9jYWNoZV9wYXRoKGNmZzogQ29uZmlnLCBjbGFzc19uYW1lOiBzdHIpIC0+IFBhdGg6DQogICAgcmV0dXJuIGNmZy53b3JrX3Jvb3QgLyAiY2FjaGUiIC8gImZlYXR1cmVzIiAvIGYie2NsYXNzX25hbWV9Lm5weiINCg0KDQpkZWYgZmVhdHVyZV9tZXRhX3BhdGgoY2ZnOiBDb25maWcsIGNsYXNzX25hbWU6IHN0cikgLT4gUGF0aDoNCiAgICByZXR1cm4gY2ZnLndvcmtfcm9vdCAvICJjYWNoZSIgLyAiZmVhdHVyZXMiIC8gZiJ7Y2xhc3NfbmFtZX0uanNvbiINCg0KDQpkZWYgaW1hZ2Vfb3JkZXIoY2ZnOiBDb25maWcsIGNsYXNzX25hbWU6IHN0cikgLT4gbGlzdFtzdHJdOg0KICAgICIiIlRyYWluLCB0aGVuIG9mZmljaWFsIGdvb2QgdGVzdCwgdGhlbiBkZWZlY3RzLiBUaGlzIG9yZGVyIGlzIHRoZSByb3cgb3JkZXIgb2YgdGhlIGNhY2hlLiIiIg0KICAgIGNvbGxlY3RlZCA9IGNvbGxlY3RfY2xhc3MoY2ZnLmRhdGFfcm9vdCwgY2xhc3NfbmFtZSkNCiAgICBwYXRocyA9IFtyZWxhdGl2ZV9wYXRoKHBhdGgsIGNmZy5kYXRhX3Jvb3QpIGZvciBwYXRoIGluIGNvbGxlY3RlZFsidHJhaW4iXV0NCiAgICBwYXRocy5leHRlbmQocmVsYXRpdmVfcGF0aChwYXRoLCBjZmcuZGF0YV9yb290KSBmb3IgcGF0aCBpbiBjb2xsZWN0ZWRbInRlc3RfZ29vZCJdKQ0KICAgIHBhdGhzLmV4dGVuZChyZWxhdGl2ZV9wYXRoKHBhdGgsIGNmZy5kYXRhX3Jvb3QpIGZvciBfLCBwYXRoIGluIGNvbGxlY3RlZFsidGVzdF9kZWZlY3QiXSkNCiAgICByZXR1cm4gcGF0aHMNCg0KDQpkZWYgZXh0cmFjdF9jbGFzcyhjZmc6IENvbmZpZywgY2xhc3NfbmFtZTogc3RyLCBkZXZpY2U6IHN0ciB8IE5vbmUgPSBOb25lKSAtPiBOb25lOg0KICAgIGltcG9ydCB0b3JjaA0KICAgIGltcG9ydCB0b3JjaC5ubiBhcyBubg0KDQogICAgb3V0X3BhdGggPSBmZWF0dXJlX2NhY2hlX3BhdGgoY2ZnLCBjbGFzc19uYW1lKQ0KICAgIG1ldGFfcGF0aCA9IGZlYXR1cmVfbWV0YV9wYXRoKGNmZywgY2xhc3NfbmFtZSkNCiAgICBpZiBvdXRfcGF0aC5pc19maWxlKCkgYW5kIG1ldGFfcGF0aC5pc19maWxlKCk6DQogICAgICAgIG1ldGEgPSByZWFkX2pzb24obWV0YV9wYXRoKQ0KICAgICAgICBpZiBtZXRhLmdldCgiY29uZmlnX2hhc2giKSA9PSBjZmcuaGFzaCBhbmQgaW50KG1ldGEuZ2V0KCJuX2ltYWdlcyIsIDApKSA+IDA6DQogICAgICAgICAgICBMT0dHRVIuaW5mbygiRmVhdHVyZSBjYWNoZSBleGlzdHMgZm9yICVzIiwgY2xhc3NfbmFtZSkNCiAgICAgICAgICAgIHJldHVybg0KDQogICAgcGF0aHMgPSBpbWFnZV9vcmRlcihjZmcsIGNsYXNzX25hbWUpDQogICAgaWYgbm90IHBhdGhzOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJ7Y2xhc3NfbmFtZX06IG5vIGltYWdlcyB0byBlbmNvZGUiKQ0KICAgIGRldmljZV90ID0gdG9yY2guZGV2aWNlKGRldmljZSBvciAoImN1ZGEiIGlmIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCkgZWxzZSAiY3B1IikpDQogICAgZXh0cmFjdG9yLCBzb3VyY2UgPSBidWlsZF9mZWF0dXJlX2V4dHJhY3RvcihjZmcpDQogICAgZXh0cmFjdG9yID0gZXh0cmFjdG9yLmV2YWwoKS50byhkZXZpY2VfdCkNCiAgICB0cmFuc2Zvcm0gPSBidWlsZF90cmFuc2Zvcm0oY2ZnKQ0KICAgIHBvb2wgPSBubi5BdmdQb29sMmQoa2VybmVsX3NpemU9Mywgc3RyaWRlPTEsIHBhZGRpbmc9MSkudG8oZGV2aWNlX3QpDQoNCiAgICBwYXRjaF9yb3dzOiBsaXN0W25wLm5kYXJyYXldID0gW10NCiAgICBnYXBfcm93czogbGlzdFtucC5uZGFycmF5XSA9IFtdDQogICAgYmF0Y2hfc2l6ZSA9IGNmZy5mZWF0dXJlX2JhdGNoX3NpemUNCiAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKToNCiAgICAgICAgZm9yIHN0YXJ0IGluIHJhbmdlKDAsIGxlbihwYXRocyksIGJhdGNoX3NpemUpOg0KICAgICAgICAgICAgYmF0Y2hfcGF0aHMgPSBwYXRoc1tzdGFydCA6IHN0YXJ0ICsgYmF0Y2hfc2l6ZV0NCiAgICAgICAgICAgIGltYWdlcyA9IFtdDQogICAgICAgICAgICBmb3IgcmVsYXRpdmUgaW4gYmF0Y2hfcGF0aHM6DQogICAgICAgICAgICAgICAgcmdiID0gcmVhZF9yZ2IoYWJzb2x1dGVfcGF0aChjZmcuZGF0YV9yb290LCByZWxhdGl2ZSksIGNsYXNzX25hbWUpDQogICAgICAgICAgICAgICAgaW1hZ2VzLmFwcGVuZCh0cmFuc2Zvcm0ocmdiKSkNCiAgICAgICAgICAgIGJhdGNoID0gdG9yY2guc3RhY2soaW1hZ2VzLCBkaW09MCkudG8oZGV2aWNlX3QpDQogICAgICAgICAgICBmZWF0dXJlcyA9IGV4dHJhY3RvcihiYXRjaCkNCiAgICAgICAgICAgIGlmIG5vdCBpc2luc3RhbmNlKGZlYXR1cmVzLCBkaWN0KToNCiAgICAgICAgICAgICAgICBmZWF0dXJlcyA9IHtuYW1lOiB0ZW5zb3IgZm9yIG5hbWUsIHRlbnNvciBpbiB6aXAoY2ZnLmxheWVycywgZmVhdHVyZXMpfQ0KICAgICAgICAgICAgZW1iZWRkaW5nID0gcG9vbGVkX2VtYmVkZGluZyhmZWF0dXJlcywgY2ZnLmxheWVycywgcG9vbCwgY2ZnLnVwc2FtcGxlX21vZGUpDQogICAgICAgICAgICBpZiBlbWJlZGRpbmcuc2hhcGVbMV0gIT0gMTUzNiBvciBlbWJlZGRpbmcuc2hhcGVbLTI6XSAhPSAoMjgsIDI4KToNCiAgICAgICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJ7Y2xhc3NfbmFtZX06IHVuZXhwZWN0ZWQgZW1iZWRkaW5nIHNoYXBlIHt0dXBsZShlbWJlZGRpbmcuc2hhcGUpfSIpDQogICAgICAgICAgICBwYXRjaGVzLCBnYXAgPSBwYXRjaGVzX2FuZF9nYXAoZW1iZWRkaW5nKQ0KICAgICAgICAgICAgcGF0Y2hlc19ucCA9IHBhdGNoZXMuZGV0YWNoKCkuY3B1KCkubnVtcHkoKQ0KICAgICAgICAgICAgZ2FwX25wID0gZ2FwLmRldGFjaCgpLmNwdSgpLm51bXB5KCkNCiAgICAgICAgICAgIGlmIG5vdCBucC5pc2Zpbml0ZShwYXRjaGVzX25wKS5hbGwoKSBvciBub3QgbnAuaXNmaW5pdGUoZ2FwX25wKS5hbGwoKToNCiAgICAgICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJ7Y2xhc3NfbmFtZX06IG5vbi1maW5pdGUgZmVhdHVyZXMgaW4gYmF0Y2ggc3RhcnRpbmcgYXQge2JhdGNoX3BhdGhzWzBdfSIpDQogICAgICAgICAgICBwYXRjaF9yb3dzLmFwcGVuZChwYXRjaGVzX25wLmFzdHlwZShucC5mbG9hdDE2KSkNCiAgICAgICAgICAgIGdhcF9yb3dzLmFwcGVuZChnYXBfbnAuYXN0eXBlKG5wLmZsb2F0MzIpKQ0KDQogICAgcGF0Y2hlc19hbGwgPSBucC5jb25jYXRlbmF0ZShwYXRjaF9yb3dzLCBheGlzPTApDQogICAgZ2FwX2FsbCA9IG5wLmNvbmNhdGVuYXRlKGdhcF9yb3dzLCBheGlzPTApDQogICAgaWYgcGF0Y2hlc19hbGwuc2hhcGVbMTpdICE9ICg3ODQsIDE1MzYpOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJ7Y2xhc3NfbmFtZX06IGV4cGVjdGVkIHBhdGNoZXMgKE4sIDc4NCwgMTUzNiksIGdvdCB7cGF0Y2hlc19hbGwuc2hhcGV9IikNCiAgICBzcGxpdCA9IGxvYWRfc3BsaXQoY2ZnLCBjbGFzc19uYW1lLCAwKQ0KICAgIG5fdHJhaW4gPSBpbnQoc3BsaXRbIm5fdHJhaW4iXSkNCiAgICBuX3Rlc3RfZ29vZCA9IGludChzcGxpdFsibl90ZXN0X2dvb2QiXSkNCiAgICBuX3Rlc3RfZGVmZWN0ID0gaW50KHNwbGl0WyJuX3Rlc3RfZGVmZWN0Il0pDQogICAgaWYgcGF0Y2hlc19hbGwuc2hhcGVbMF0gIT0gbl90cmFpbiArIG5fdGVzdF9nb29kICsgbl90ZXN0X2RlZmVjdDoNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYie2NsYXNzX25hbWV9OiBmZWF0dXJlIHJvd3MgZG8gbm90IG1hdGNoIHNwbGl0IGNvdW50cyIpDQoNCiAgICBhdG9taWNfc2F2ZXooDQogICAgICAgIG91dF9wYXRoLA0KICAgICAgICBwYXRjaGVzPXBhdGNoZXNfYWxsLA0KICAgICAgICBnYXA9Z2FwX2FsbCwNCiAgICAgICAgcGF0aHM9bnAuYXJyYXkocGF0aHMpLA0KICAgICkNCiAgICB3cml0ZV9qc29uKA0KICAgICAgICBtZXRhX3BhdGgsDQogICAgICAgIHsNCiAgICAgICAgICAgICJjb25maWdfaGFzaCI6IGNmZy5oYXNoLA0KICAgICAgICAgICAgImNsYXNzX25hbWUiOiBjbGFzc19uYW1lLA0KICAgICAgICAgICAgIm5faW1hZ2VzIjogaW50KHBhdGNoZXNfYWxsLnNoYXBlWzBdKSwNCiAgICAgICAgICAgICJuX3RyYWluIjogbl90cmFpbiwNCiAgICAgICAgICAgICJuX3Rlc3RfZ29vZCI6IG5fdGVzdF9nb29kLA0KICAgICAgICAgICAgIm5fdGVzdF9kZWZlY3QiOiBuX3Rlc3RfZGVmZWN0LA0KICAgICAgICAgICAgInBhdGNoX3NoYXBlIjogWzc4NCwgMTUzNl0sDQogICAgICAgICAgICAiZXh0cmFjdG9yIjogc291cmNlLA0KICAgICAgICAgICAgInVwc2FtcGxlX21vZGUiOiBjZmcudXBzYW1wbGVfbW9kZSwNCiAgICAgICAgICAgICJyZXNpemUiOiBjZmcucmVzaXplLA0KICAgICAgICAgICAgImNlbnRlcl9jcm9wIjogY2ZnLmNlbnRlcl9jcm9wLA0KICAgICAgICAgICAgImJ5dGVzX3BhdGNoZXMiOiBpbnQocGF0Y2hlc19hbGwubmJ5dGVzKSwNCiAgICAgICAgfSwNCiAgICApDQogICAgTE9HR0VSLmluZm8oDQogICAgICAgICIlcyBmZWF0dXJlczogJXMgaW1hZ2VzLCAlLjJmIEdCIGZsb2F0MTYiLA0KICAgICAgICBjbGFzc19uYW1lLA0KICAgICAgICBwYXRjaGVzX2FsbC5zaGFwZVswXSwNCiAgICAgICAgcGF0Y2hlc19hbGwubmJ5dGVzIC8gKDEwMjQqKjMpLA0KICAgICkNCg0KDQpkZWYgbG9hZF9nYXBfdGFibGUoY2ZnOiBDb25maWcsIGNsYXNzX25hbWU6IHN0cikgLT4gZGljdDoNCiAgICAiIiJHQVAgdmVjdG9ycyBvbmx5LiBUcmFqZWN0b3J5IHNlbGVjdGlvbiBtdXN0IG5vdCBsb2FkIHRoZSBwYXRjaCB0ZW5zb3IuIiIiDQogICAgcGF0aCA9IGZlYXR1cmVfY2FjaGVfcGF0aChjZmcsIGNsYXNzX25hbWUpDQogICAgbWV0YV9wYXRoID0gZmVhdHVyZV9tZXRhX3BhdGgoY2ZnLCBjbGFzc19uYW1lKQ0KICAgIGlmIG5vdCBwYXRoLmlzX2ZpbGUoKSBvciBub3QgbWV0YV9wYXRoLmlzX2ZpbGUoKToNCiAgICAgICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoZiJNaXNzaW5nIGZlYXR1cmUgY2FjaGUgZm9yIHtjbGFzc19uYW1lfS4gUnVuIGZlYXR1cmVzIGZpcnN0LiIpDQogICAgbWV0YSA9IHJlYWRfanNvbihtZXRhX3BhdGgpDQogICAgaWYgbWV0YS5nZXQoImNvbmZpZ19oYXNoIikgIT0gY2ZnLmhhc2g6DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIkZlYXR1cmUgY2FjaGUgaGFzaCBtaXNtYXRjaCBmb3Ige2NsYXNzX25hbWV9LiIpDQogICAgd2l0aCBucC5sb2FkKHBhdGgsIGFsbG93X3BpY2tsZT1GYWxzZSkgYXMgZGF0YToNCiAgICAgICAgZ2FwID0gbnAuYXJyYXkoZGF0YVsiZ2FwIl0pDQogICAgICAgIHBhdGhzID0gW3N0cihpdGVtKSBmb3IgaXRlbSBpbiBkYXRhWyJwYXRocyJdLnRvbGlzdCgpXQ0KICAgIGlmIG5vdCBucC5pc2Zpbml0ZShnYXApLmFsbCgpOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJ7Y2xhc3NfbmFtZX06IG5vbi1maW5pdGUgR0FQIGZlYXR1cmVzIikNCiAgICByZXR1cm4geyJnYXAiOiBnYXAsICJwYXRocyI6IHBhdGhzLCAibWV0YSI6IG1ldGF9DQoNCg0KZGVmIGxvYWRfZmVhdHVyZV9jYWNoZShjZmc6IENvbmZpZywgY2xhc3NfbmFtZTogc3RyKSAtPiBkaWN0Og0KICAgIHBhdGggPSBmZWF0dXJlX2NhY2hlX3BhdGgoY2ZnLCBjbGFzc19uYW1lKQ0KICAgIG1ldGFfcGF0aCA9IGZlYXR1cmVfbWV0YV9wYXRoKGNmZywgY2xhc3NfbmFtZSkNCiAgICBpZiBub3QgcGF0aC5pc19maWxlKCkgb3Igbm90IG1ldGFfcGF0aC5pc19maWxlKCk6DQogICAgICAgIHJhaXNlIEZpbGVOb3RGb3VuZEVycm9yKGYiTWlzc2luZyBmZWF0dXJlIGNhY2hlIGZvciB7Y2xhc3NfbmFtZX0uIFJ1biBmZWF0dXJlcyBmaXJzdC4iKQ0KICAgIG1ldGEgPSByZWFkX2pzb24obWV0YV9wYXRoKQ0KICAgIGlmIG1ldGEuZ2V0KCJjb25maWdfaGFzaCIpICE9IGNmZy5oYXNoOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJGZWF0dXJlIGNhY2hlIGhhc2ggbWlzbWF0Y2ggZm9yIHtjbGFzc19uYW1lfS4iKQ0KICAgIHdpdGggbnAubG9hZChwYXRoLCBhbGxvd19waWNrbGU9RmFsc2UpIGFzIGRhdGE6DQogICAgICAgIHBhdGNoZXMgPSBucC5hcnJheShkYXRhWyJwYXRjaGVzIl0pDQogICAgICAgIGdhcCA9IG5wLmFycmF5KGRhdGFbImdhcCJdKQ0KICAgICAgICBwYXRocyA9IFtzdHIoaXRlbSkgZm9yIGl0ZW0gaW4gZGF0YVsicGF0aHMiXS50b2xpc3QoKV0NCiAgICBpZiBwYXRjaGVzLmR0eXBlICE9IG5wLmZsb2F0MTYgb3IgcGF0Y2hlcy5zaGFwZVsxOl0gIT0gKDc4NCwgMTUzNik6DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIntjbGFzc19uYW1lfTogYmFkIHBhdGNoIGNhY2hlIHNoYXBlIHtwYXRjaGVzLnNoYXBlfSB7cGF0Y2hlcy5kdHlwZX0iKQ0KICAgIGlmIG5vdCBucC5pc2Zpbml0ZShwYXRjaGVzLmFzdHlwZShucC5mbG9hdDMyKSkuYWxsKCkgb3Igbm90IG5wLmlzZmluaXRlKGdhcCkuYWxsKCk6DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIntjbGFzc19uYW1lfTogbm9uLWZpbml0ZSB2YWx1ZXMgaW4gdGhlIGZlYXR1cmUgY2FjaGUiKQ0KICAgIHJldHVybiB7InBhdGNoZXMiOiBwYXRjaGVzLCAiZ2FwIjogZ2FwLCAicGF0aHMiOiBwYXRocywgIm1ldGEiOiBtZXRhfQ0KDQoNCmRlZiBleHRyYWN0KGNmZzogQ29uZmlnLCBjbGFzc2VzOiBsaXN0W3N0cl0sIGRldmljZTogc3RyIHwgTm9uZSA9IE5vbmUpIC0+IE5vbmU6DQogICAgZm9yIGNsYXNzX25hbWUgaW4gY2xhc3NlczoNCiAgICAgICAgZXh0cmFjdF9jbGFzcyhjZmcsIGNsYXNzX25hbWUsIGRldmljZT1kZXZpY2UpDQo=", "commissioning/integrity.py": "IiIiQ2hlY2tzIHRoYXQgbXVzdCBwYXNzIGJlZm9yZSBhbnkgUlEgdHJhamVjdG9yeSBpcyB0cnVzdGVkLiIiIg0KDQpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zDQoNCmltcG9ydCBvcw0KaW1wb3J0IHRlbXBmaWxlDQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgNCg0KaW1wb3J0IG51bXB5IGFzIG5wDQoNCmZyb20gY29tbWlzc2lvbmluZy5jb25maWcgaW1wb3J0IENvbmZpZw0KZnJvbSBjb21taXNzaW9uaW5nLmRhdGEgaW1wb3J0IGxvYWRfc3BsaXQNCmZyb20gY29tbWlzc2lvbmluZy5mZWF0dXJlcyBpbXBvcnQgaW1hZ2Vfb3JkZXIsIGxvYWRfZmVhdHVyZV9jYWNoZSwgcmVhZF9yZ2INCmZyb20gY29tbWlzc2lvbmluZy5wYXRjaGNvcmUgaW1wb3J0IGtubl9zY29yZXMsIHNjb3Jlc19mb3JfYmFuaw0KDQoNCmRlZiBjaGVja19jb3JydXB0ZWRfaW5wdXQoY2xhc3NfbmFtZTogc3RyID0gImludGVncml0eSIpIC0+IE5vbmU6DQogICAgaGFuZGxlLCBuYW1lID0gdGVtcGZpbGUubWtzdGVtcChzdWZmaXg9Ii5wbmciKQ0KICAgIG9zLndyaXRlKGhhbmRsZSwgYiJ0aGlzIGlzIG5vdCBhIHBuZyIpDQogICAgb3MuY2xvc2UoaGFuZGxlKQ0KICAgIHBhdGggPSBQYXRoKG5hbWUpDQogICAgdHJ5Og0KICAgICAgICB0cnk6DQogICAgICAgICAgICByZWFkX3JnYihwYXRoLCBjbGFzc19uYW1lKQ0KICAgICAgICBleGNlcHQgUnVudGltZUVycm9yIGFzIGV4YzoNCiAgICAgICAgICAgIHRleHQgPSBzdHIoZXhjKQ0KICAgICAgICAgICAgaWYgY2xhc3NfbmFtZSBub3QgaW4gdGV4dCBvciBzdHIocGF0aCkgbm90IGluIHRleHQ6DQogICAgICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYiQ29ycnVwdGVkLWltYWdlIGVycm9yIG11c3QgbmFtZSB0aGUgY2xhc3MgYW5kIHBhdGgsIGdvdDoge3RleHR9IikgZnJvbSBleGMNCiAgICAgICAgICAgIHJldHVybg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIkEgY29ycnVwdGVkIGltYWdlIHdhcyByZWFkIHdpdGhvdXQgUnVudGltZUVycm9yLiIpDQogICAgZmluYWxseToNCiAgICAgICAgcGF0aC51bmxpbmsobWlzc2luZ19vaz1UcnVlKQ0KDQoNCmRlZiBjaGVja19mZWF0dXJlX2FsaWdubWVudChjZmc6IENvbmZpZywgY2xhc3NfbmFtZTogc3RyLCBjYWNoZTogZGljdCB8IE5vbmUgPSBOb25lKSAtPiBOb25lOg0KICAgIGlmIGNhY2hlIGlzIE5vbmU6DQogICAgICAgIGNhY2hlID0gbG9hZF9mZWF0dXJlX2NhY2hlKGNmZywgY2xhc3NfbmFtZSkNCiAgICBleHBlY3RlZCA9IGltYWdlX29yZGVyKGNmZywgY2xhc3NfbmFtZSkNCiAgICBpZiBjYWNoZVsicGF0aHMiXSAhPSBleHBlY3RlZDoNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYie2NsYXNzX25hbWV9OiBmZWF0dXJlLWNhY2hlIG9yZGVyIGRvZXMgbm90IG1hdGNoIHRoZSBzb3J0ZWQgZGF0YXNldCBvcmRlci4iKQ0KICAgIGlmIGNhY2hlWyJwYXRjaGVzIl0uc2hhcGVbMTpdICE9ICg3ODQsIDE1MzYpOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJ7Y2xhc3NfbmFtZX06IHBhdGNoIHRlbnNvciBzaGFwZSBpcyB7Y2FjaGVbJ3BhdGNoZXMnXS5zaGFwZX0iKQ0KICAgIGlmIGxlbihjYWNoZVsicGF0aHMiXSkgIT0gbGVuKHNldChjYWNoZVsicGF0aHMiXSkpOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJ7Y2xhc3NfbmFtZX06IGR1cGxpY2F0ZSBwYXRocyBpbiB0aGUgZmVhdHVyZSBjYWNoZS4iKQ0KICAgIHNwbGl0ID0gbG9hZF9zcGxpdChjZmcsIGNsYXNzX25hbWUsIDApDQogICAgbl90cmFpbiA9IGludChzcGxpdFsibl90cmFpbiJdKQ0KICAgIHRyYWluX3BhdGhzID0gY2FjaGVbInBhdGhzIl1bOm5fdHJhaW5dDQogICAgdGVzdF9wYXRocyA9IGNhY2hlWyJwYXRocyJdW25fdHJhaW46XQ0KICAgIGlmIHNldChzcGxpdFsiY2FsaWJyYXRpb24iXSkgfCBzZXQoc3BsaXRbInBvb2wiXSkgIT0gc2V0KHRyYWluX3BhdGhzKToNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYie2NsYXNzX25hbWV9OiBDIGFuZCBQIGFyZSBub3QgYSBwYXJ0aXRpb24gb2YgdGhlIGNhY2hlZCB0cmFpbmluZyBpbWFnZXMuIikNCiAgICBvZmZpY2lhbF90ZXN0ID0gbGlzdChzcGxpdFsidGVzdF9nb29kIl0pICsgW2l0ZW1bInBhdGgiXSBmb3IgaXRlbSBpbiBzcGxpdFsidGVzdF9kZWZlY3QiXV0NCiAgICBpZiB0ZXN0X3BhdGhzICE9IG9mZmljaWFsX3Rlc3Q6DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIntjbGFzc19uYW1lfTogY2FjaGVkIHRlc3Qgb3JkZXIgZG9lcyBub3QgbWF0Y2ggdGhlIHNwbGl0LiIpDQoNCg0KZGVmIGFzc2VydF9tZXRob2RzX2FncmVlKA0KICAgIHBhdGNoZXM6IG5wLm5kYXJyYXksDQogICAgZGlzdGFuY2U6IG5wLm5kYXJyYXksDQogICAgcXVlcnlfaW5kaWNlczogbnAubmRhcnJheSB8IGxpc3RbaW50XSwNCiAgICBiYW5rX2luZGljZXM6IG5wLm5kYXJyYXkgfCBsaXN0W2ludF0sDQogICAgdG9sZXJhbmNlOiBmbG9hdCA9IDFlLTQsDQopIC0+IGZsb2F0Og0KICAgICIiIlRoZSBkaXN0YW5jZS10ZW5zb3Igc2NvcmUgYW5kIGEgZGlyZWN0IDEtTk4gb3ZlciB0aGUgc2FtZSBwYXRjaGVzIG11c3QgYWdyZWUuDQoNCiAgICBCb3RoIGdvIHRocm91Z2ggZmxvYXQzMiBHRU1NLWJhc2VkIGRpc3RhbmNlcyB3aXRoIGRpZmZlcmVudCBibG9jayBzaGFwZXMsIHNvIHRoZSBjaGVjayBpcyByZWxhdGl2ZS4NCiAgICAiIiINCiAgICB2aWFfZGlzdGFuY2UgPSBzY29yZXNfZm9yX2JhbmsoZGlzdGFuY2UsIHF1ZXJ5X2luZGljZXMsIGJhbmtfaW5kaWNlcykNCiAgICB2aWFfa25uID0ga25uX3Njb3JlcyhwYXRjaGVzW25wLmFzYXJyYXkocXVlcnlfaW5kaWNlcyldLCBwYXRjaGVzW25wLmFzYXJyYXkoYmFua19pbmRpY2VzKV0pDQogICAgZ2FwID0gZmxvYXQobnAubWF4KG5wLmFicyh2aWFfZGlzdGFuY2UgLSB2aWFfa25uKSAvIG5wLm1heGltdW0oMS4wLCBucC5hYnModmlhX2tubikpKSkNCiAgICBpZiBnYXAgPiB0b2xlcmFuY2U6DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIkRpc3RhbmNlIHRlbnNvciBhbmQgZGlyZWN0IGtOTiBkaWZmZXIgYnkgYSByZWxhdGl2ZSB7Z2FwfSAodG9sZXJhbmNlIHt0b2xlcmFuY2V9KS4iKQ0KICAgIHJldHVybiBnYXANCg==", "commissioning/io_utils.py": "IiIiQXRvbWljIHdyaXRlcyBhbmQgc21hbGwgZmlsZSBoZWxwZXJzLiIiIg0KDQpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zDQoNCmltcG9ydCBqc29uDQppbXBvcnQgb3MNCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aA0KZnJvbSB0eXBpbmcgaW1wb3J0IEFueQ0KDQppbXBvcnQgbnVtcHkgYXMgbnANCg0KDQpkZWYgZW5zdXJlX2RpcihwYXRoOiBQYXRoKSAtPiBQYXRoOg0KICAgIHBhdGgubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQ0KICAgIHJldHVybiBwYXRoDQoNCg0KZGVmIHdyaXRlX2pzb24ocGF0aDogUGF0aCwgcGF5bG9hZDogZGljdFtzdHIsIEFueV0pIC0+IE5vbmU6DQogICAgcGF0aCA9IFBhdGgocGF0aCkNCiAgICBlbnN1cmVfZGlyKHBhdGgucGFyZW50KQ0KICAgIHRtcCA9IHBhdGgud2l0aF9zdWZmaXgocGF0aC5zdWZmaXggKyAiLnRtcCIpDQogICAgdG1wLndyaXRlX3RleHQoanNvbi5kdW1wcyhwYXlsb2FkLCBpbmRlbnQ9Miwgc29ydF9rZXlzPVRydWUpLCBlbmNvZGluZz0idXRmLTgiKQ0KICAgIG9zLnJlcGxhY2UodG1wLCBwYXRoKQ0KDQoNCmRlZiByZWFkX2pzb24ocGF0aDogUGF0aCkgLT4gZGljdFtzdHIsIEFueV06DQogICAgcmV0dXJuIGpzb24ubG9hZHMoUGF0aChwYXRoKS5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikpDQoNCg0KZGVmIGF0b21pY19zYXZleihwYXRoOiBQYXRoLCAqKmFycmF5czogQW55KSAtPiBOb25lOg0KICAgIHBhdGggPSBQYXRoKHBhdGgpDQogICAgZW5zdXJlX2RpcihwYXRoLnBhcmVudCkNCiAgICBwYXJ0aWFsID0gcGF0aC5wYXJlbnQgLyBmIntwYXRoLm5hbWV9LnBhcnRpYWwubnB6Ig0KICAgIG5wLnNhdmV6KHBhcnRpYWwsICoqYXJyYXlzKQ0KICAgIG9zLnJlcGxhY2UocGFydGlhbCwgcGF0aCkNCg0KDQpkZWYgYXRvbWljX25weShwYXRoOiBQYXRoLCBhcnJheTogbnAubmRhcnJheSkgLT4gTm9uZToNCiAgICBwYXRoID0gUGF0aChwYXRoKQ0KICAgIGVuc3VyZV9kaXIocGF0aC5wYXJlbnQpDQogICAgcGFydGlhbCA9IHBhdGgucGFyZW50IC8gZiJ7cGF0aC5uYW1lfS5wYXJ0aWFsLm5weSINCiAgICBucC5zYXZlKHBhcnRpYWwsIGFycmF5KQ0KICAgIG9zLnJlcGxhY2UocGFydGlhbCwgcGF0aCkNCg0KDQpkZWYgYXRvbWljX3BhcnF1ZXQocGF0aDogUGF0aCwgZnJhbWU6IEFueSkgLT4gTm9uZToNCiAgICBwYXRoID0gUGF0aChwYXRoKQ0KICAgIGVuc3VyZV9kaXIocGF0aC5wYXJlbnQpDQogICAgcGFydGlhbCA9IHBhdGgucGFyZW50IC8gZiJ7cGF0aC5uYW1lfS5wYXJ0aWFsLnBhcnF1ZXQiDQogICAgZnJhbWUudG9fcGFycXVldChwYXJ0aWFsLCBpbmRleD1GYWxzZSkNCiAgICBvcy5yZXBsYWNlKHBhcnRpYWwsIHBhdGgpDQo=", "commissioning/patchcore.py": "IiIiSW1hZ2Ugc2NvcmVzIGZyb20gYSBtZW1vcnkgYmFuay4NCg0KVGhlIG9wZXJhdGlvbmFsIHNjb3JlIGlzIHRoZSBtYXhpbXVtLCBvdmVyIHBhdGNoZXMgb2YgdGhlIHF1ZXJ5LCBvZiB0aGUgZGlzdGFuY2UNCnRvIHRoZSBuZWFyZXN0IHBhdGNoIGluIHRoZSBiYW5rLiBCYW5rcyBhcmUgcHJlZml4ZXMgb2Ygb25lIHRyYWplY3RvcnksIHNvIHRoZQ0Kc2NvcmUgbmV2ZXIgaW5jcmVhc2VzIGFzIE4gZ3Jvd3MuDQoiIiINCg0KZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucw0KDQppbXBvcnQgbnVtcHkgYXMgbnANCg0KDQpkZWYgX2FzX2luZGV4KGluZGljZXM6IG5wLm5kYXJyYXkgfCBsaXN0W2ludF0pIC0+IG5wLm5kYXJyYXk6DQogICAgYXJyYXkgPSBucC5hc2FycmF5KGluZGljZXMsIGR0eXBlPWludCkNCiAgICBpZiBhcnJheS5uZGltICE9IDE6DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIkV4cGVjdGVkIGEgMS1EIGluZGV4LCBnb3Qgc2hhcGUge2FycmF5LnNoYXBlfSIpDQogICAgcmV0dXJuIGFycmF5DQoNCg0KZGVmIHNjb3Jlc19mb3JfYmFuayhkaXN0YW5jZTogbnAubmRhcnJheSwgcXVlcnlfaW5kaWNlczogbnAubmRhcnJheSB8IGxpc3RbaW50XSwgYmFua19pbmRpY2VzOiBucC5uZGFycmF5IHwgbGlzdFtpbnRdKSAtPiBucC5uZGFycmF5Og0KICAgIHF1ZXJpZXMgPSBfYXNfaW5kZXgocXVlcnlfaW5kaWNlcykNCiAgICBiYW5rID0gX2FzX2luZGV4KGJhbmtfaW5kaWNlcykNCiAgICBpZiBiYW5rLnNpemUgPT0gMDoNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJFbXB0eSBtZW1vcnkgYmFuayIpDQogICAgaWYgbGVuKG5wLnVuaXF1ZShiYW5rKSkgIT0gbGVuKGJhbmspOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIk1lbW9yeSBiYW5rIGNvbnRhaW5zIGR1cGxpY2F0ZSBpbWFnZXMiKQ0KICAgIGJsb2NrID0gZGlzdGFuY2VbcXVlcmllc11bOiwgOiwgYmFua10NCiAgICBzY29yZXMgPSBibG9jay5taW4oYXhpcz0tMSkubWF4KGF4aXM9LTEpDQogICAgaWYgbm90IG5wLmlzZmluaXRlKHNjb3JlcykuYWxsKCk6DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiTm9uLWZpbml0ZSBpbWFnZSBzY29yZSIpDQogICAgcmV0dXJuIHNjb3Jlcy5hc3R5cGUobnAuZmxvYXQ2NCkNCg0KDQpkZWYgc2NvcmVzX2Fsb25nX29yZGVyKA0KICAgIGRpc3RhbmNlOiBucC5uZGFycmF5LA0KICAgIHF1ZXJ5X2luZGljZXM6IG5wLm5kYXJyYXkgfCBsaXN0W2ludF0sDQogICAgb3JkZXI6IG5wLm5kYXJyYXkgfCBsaXN0W2ludF0sDQogICAgZ3JpZDogbGlzdFtpbnRdLA0KKSAtPiBucC5uZGFycmF5Og0KICAgICIiIlNjb3JlcyBhdCBlYWNoIG5lc3RlZCBwcmVmaXggb2YgYGBvcmRlcmBgLiBSZXR1cm5zIChsZW4oZ3JpZCksIG5fcXVlcmllcykuIiIiDQogICAgcXVlcmllcyA9IF9hc19pbmRleChxdWVyeV9pbmRpY2VzKQ0KICAgIG9yZGVyID0gX2FzX2luZGV4KG9yZGVyKQ0KICAgIGlmIGxlbihucC51bmlxdWUob3JkZXIpKSAhPSBsZW4ob3JkZXIpOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIlRyYWplY3Rvcnkgb3JkZXIgY29udGFpbnMgZHVwbGljYXRlIGltYWdlcyIpDQogICAgZ3JpZCA9IFtpbnQobikgZm9yIG4gaW4gZ3JpZF0NCiAgICBpZiBncmlkICE9IHNvcnRlZChncmlkKSBvciBhbnkobiA8IDEgb3IgbiA+IGxlbihvcmRlcikgZm9yIG4gaW4gZ3JpZCk6DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIkdyaWQge2dyaWR9IGlzIG5vdCBhIHZhbGlkIGluY3JlYXNpbmcgc3Vic2V0IG9mIDEuLntsZW4ob3JkZXIpfSIpDQogICAgc2VsZWN0ZWQgPSBkaXN0YW5jZVtxdWVyaWVzXQ0KICAgIHJ1bm5pbmcgPSBOb25lDQogICAgcHJldmlvdXMgPSAwDQogICAgcm93cyA9IFtdDQogICAgZm9yIG4gaW4gZ3JpZDoNCiAgICAgICAgYWRkZWQgPSBvcmRlcltwcmV2aW91czpuXQ0KICAgICAgICBibG9ja19taW4gPSBzZWxlY3RlZFs6LCA6LCBhZGRlZF0ubWluKGF4aXM9LTEpDQogICAgICAgIHJ1bm5pbmcgPSBibG9ja19taW4gaWYgcnVubmluZyBpcyBOb25lIGVsc2UgbnAubWluaW11bShydW5uaW5nLCBibG9ja19taW4pDQogICAgICAgIHNjb3JlcyA9IHJ1bm5pbmcubWF4KGF4aXM9LTEpDQogICAgICAgIGlmIG5vdCBucC5pc2Zpbml0ZShzY29yZXMpLmFsbCgpOg0KICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJOb24tZmluaXRlIGltYWdlIHNjb3JlIikNCiAgICAgICAgcm93cy5hcHBlbmQoc2NvcmVzKQ0KICAgICAgICBwcmV2aW91cyA9IG4NCiAgICBzdGFja2VkID0gbnAuc3RhY2socm93cywgYXhpcz0wKS5hc3R5cGUobnAuZmxvYXQ2NCkNCiAgICBpZiBzdGFja2VkLnNoYXBlWzBdID4gMSBhbmQgbnAuYW55KG5wLmRpZmYoc3RhY2tlZCwgYXhpcz0wKSA+IDFlLTUpOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIkltYWdlIHNjb3JlIGluY3JlYXNlZCBhcyB0aGUgbmVzdGVkIGJhbmsgZ3JldyIpDQogICAgcmV0dXJuIHN0YWNrZWQNCg0KDQpkZWYga25uX3Njb3JlcyhxdWVyeV9wYXRjaGVzOiBucC5uZGFycmF5LCBiYW5rX3BhdGNoZXM6IG5wLm5kYXJyYXksIHF1ZXJ5X2NodW5rOiBpbnQgPSA4KSAtPiBucC5uZGFycmF5Og0KICAgICIiIlJlZmVyZW5jZSAxLU5OIG1heGltdW0uIGBgYmFua19wYXRjaGVzYGAgaXMgKE5fYmFuaywgUCwgQykgb3IgKE5fcGF0Y2hlcywgQykuIiIiDQogICAgaWYgYmFua19wYXRjaGVzLm5kaW0gPT0gMzoNCiAgICAgICAgYmFuayA9IG5wLmFzY29udGlndW91c2FycmF5KGJhbmtfcGF0Y2hlcy5yZXNoYXBlKC0xLCBiYW5rX3BhdGNoZXMuc2hhcGVbLTFdKSwgZHR5cGU9bnAuZmxvYXQzMikNCiAgICBlbGlmIGJhbmtfcGF0Y2hlcy5uZGltID09IDI6DQogICAgICAgIGJhbmsgPSBucC5hc2NvbnRpZ3VvdXNhcnJheShiYW5rX3BhdGNoZXMsIGR0eXBlPW5wLmZsb2F0MzIpDQogICAgZWxzZToNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYiQmFkIGJhbmsgc2hhcGUge2JhbmtfcGF0Y2hlcy5zaGFwZX0iKQ0KICAgIHRyeToNCiAgICAgICAgaW1wb3J0IHRvcmNoDQogICAgZXhjZXB0IEltcG9ydEVycm9yOg0KICAgICAgICB0b3JjaCA9IE5vbmUNCiAgICBpZiB0b3JjaCBpcyBOb25lIGFuZCBiYW5rLnNoYXBlWzBdID4gODE5MjoNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJEaXJlY3Qga05OIG9uIGEgYmFuayBvZiB0aGlzIHNpemUgbmVlZHMgdG9yY2guIikNCiAgICBuX3F1ZXJ5LCBuX3BhdGNoZXMsIGNoYW5uZWxzID0gcXVlcnlfcGF0Y2hlcy5zaGFwZQ0KICAgIHNjb3JlcyA9IG5wLmVtcHR5KG5fcXVlcnksIGR0eXBlPW5wLmZsb2F0NjQpDQogICAgZGV2aWNlID0gImNwdSINCiAgICBiYW5rX3QgPSBOb25lDQogICAgaWYgdG9yY2ggaXMgbm90IE5vbmU6DQogICAgICAgIGRldmljZSA9ICJjdWRhIiBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpIGVsc2UgImNwdSINCiAgICAgICAgYmFua190ID0gdG9yY2guZnJvbV9udW1weShiYW5rKS50byhkZXZpY2UpDQogICAgZm9yIHN0YXJ0IGluIHJhbmdlKDAsIG5fcXVlcnksIHF1ZXJ5X2NodW5rKToNCiAgICAgICAgc3RvcCA9IG1pbihzdGFydCArIHF1ZXJ5X2NodW5rLCBuX3F1ZXJ5KQ0KICAgICAgICBmbGF0ID0gbnAuYXNjb250aWd1b3VzYXJyYXkocXVlcnlfcGF0Y2hlc1tzdGFydDpzdG9wXS5yZXNoYXBlKC0xLCBjaGFubmVscyksIGR0eXBlPW5wLmZsb2F0MzIpDQogICAgICAgIGlmIGJhbmtfdCBpcyBub3QgTm9uZToNCiAgICAgICAgICAgIHF1ZXJ5X3QgPSB0b3JjaC5mcm9tX251bXB5KGZsYXQpLnRvKGRldmljZSkNCiAgICAgICAgICAgIG5lYXJlc3QgPSBbXQ0KICAgICAgICAgICAgc3RlcCA9IDI1Ng0KICAgICAgICAgICAgZm9yIHAwIGluIHJhbmdlKDAsIGZsYXQuc2hhcGVbMF0sIHN0ZXApOg0KICAgICAgICAgICAgICAgIGRpc3RhbmNlcyA9IHRvcmNoLmNkaXN0KHF1ZXJ5X3RbcDAgOiBwMCArIHN0ZXBdLCBiYW5rX3QpDQogICAgICAgICAgICAgICAgbmVhcmVzdC5hcHBlbmQoZGlzdGFuY2VzLm1pbihkaW09MSkudmFsdWVzLmRldGFjaCgpLmNwdSgpLm51bXB5KCkpDQogICAgICAgICAgICBwYXRjaF9zY29yZXMgPSBucC5jb25jYXRlbmF0ZShuZWFyZXN0KS5yZXNoYXBlKHN0b3AgLSBzdGFydCwgbl9wYXRjaGVzKQ0KICAgICAgICBlbHNlOg0KICAgICAgICAgICAgbmVhcmVzdCA9IG5wLmVtcHR5KGZsYXQuc2hhcGVbMF0sIGR0eXBlPW5wLmZsb2F0MzIpDQogICAgICAgICAgICBmb3IgcDAgaW4gcmFuZ2UoMCwgZmxhdC5zaGFwZVswXSwgMzIpOg0KICAgICAgICAgICAgICAgIHAxID0gbWluKHAwICsgMzIsIGZsYXQuc2hhcGVbMF0pDQogICAgICAgICAgICAgICAgZGVsdGEgPSBmbGF0W3AwOnAxLCBOb25lLCA6XSAtIGJhbmtbTm9uZSwgOiwgOl0NCiAgICAgICAgICAgICAgICBuZWFyZXN0W3AwOnAxXSA9IG5wLnNxcnQobnAuc3VtKGRlbHRhICogZGVsdGEsIGF4aXM9LTEpKS5taW4oYXhpcz0xKQ0KICAgICAgICAgICAgcGF0Y2hfc2NvcmVzID0gbmVhcmVzdC5yZXNoYXBlKHN0b3AgLSBzdGFydCwgbl9wYXRjaGVzKQ0KICAgICAgICBzY29yZXNbc3RhcnQ6c3RvcF0gPSBwYXRjaF9zY29yZXMubWF4KGF4aXM9MSkNCiAgICBpZiBub3QgbnAuaXNmaW5pdGUoc2NvcmVzKS5hbGwoKToNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJOb24tZmluaXRlIHJlZmVyZW5jZSBrTk4gc2NvcmUiKQ0KICAgIHJldHVybiBzY29yZXMNCg0KDQpkZWYgYW5vbWFseV9tYXAocGF0Y2hfc2NvcmVzOiBucC5uZGFycmF5LCBpbWFnZV9zaXplOiBpbnQgPSAyMjQsIHNpZ21hOiBmbG9hdCA9IDQuMCkgLT4gbnAubmRhcnJheToNCiAgICAiIiJVcHNhbXBsZSBhIDI4eDI4IHBhdGNoLXNjb3JlIGdyaWQgYW5kIGJsdXIgaXQgZm9yIGEgZmlndXJlIG92ZXJsYXkuIiIiDQogICAgZnJvbSBzY2lweS5uZGltYWdlIGltcG9ydCBnYXVzc2lhbl9maWx0ZXINCg0KICAgIGdyaWQgPSBucC5hc2FycmF5KHBhdGNoX3Njb3JlcywgZHR5cGU9bnAuZmxvYXQ2NCkucmVzaGFwZSgyOCwgMjgpDQogICAgem9vbSA9IGltYWdlX3NpemUgLy8gMjgNCiAgICBpZiBpbWFnZV9zaXplICUgMjggIT0gMDoNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJpbWFnZV9zaXplIG11c3QgYmUgYSBtdWx0aXBsZSBvZiAyOCIpDQogICAgdXBzYW1wbGVkID0gbnAucmVwZWF0KG5wLnJlcGVhdChncmlkLCB6b29tLCBheGlzPTApLCB6b29tLCBheGlzPTEpDQogICAgcmV0dXJuIGdhdXNzaWFuX2ZpbHRlcih1cHNhbXBsZWQsIHNpZ21hPXNpZ21hKQ0KDQoNCmRlZiBwYXRjaF9zY29yZXNfZm9yX2JhbmsoZGlzdGFuY2Vfcm93OiBucC5uZGFycmF5LCBiYW5rX2luZGljZXM6IG5wLm5kYXJyYXkgfCBsaXN0W2ludF0pIC0+IG5wLm5kYXJyYXk6DQogICAgIiIiZGlzdGFuY2Vfcm93OiAoNzg0LCBuX3RyYWluKSBmb3Igb25lIHF1ZXJ5LiIiIg0KICAgIGJhbmsgPSBfYXNfaW5kZXgoYmFua19pbmRpY2VzKQ0KICAgIHJldHVybiBkaXN0YW5jZV9yb3dbOiwgYmFua10ubWluKGF4aXM9MSkNCg==", "commissioning/report.py": "...", "commissioning/run.py": "IiIiUnVuIHRoZSAxNSB4IDIwIHggMiBuZXN0ZWQgdHJhamVjdG9yaWVzIGFuZCByZXN1bWUgZnJvbSBmaW5pc2hlZCBwYXJ0cy4iIiINCg0KZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucw0KDQppbXBvcnQgbG9nZ2luZw0KDQppbXBvcnQgbnVtcHkgYXMgbnANCmltcG9ydCBwYW5kYXMgYXMgcGQNCg0KZnJvbSBjb21taXNzaW9uaW5nLmNhbGlicmF0aW9uIGltcG9ydCBldmFsdWF0ZV9zY29yZXMNCmZyb20gY29tbWlzc2lvbmluZy5jb25maWcgaW1wb3J0IENvbmZpZw0KZnJvbSBjb21taXNzaW9uaW5nLmRhdGEgaW1wb3J0IGxvYWRfc3BsaXQsIHNwbGl0X3NlZWRzDQpmcm9tIGNvbW1pc3Npb25pbmcuZGlzdGFuY2VzIGltcG9ydCBsb2FkX2Rpc3RhbmNlX2NhY2hlDQpmcm9tIGNvbW1pc3Npb25pbmcuZTAgaW1wb3J0IGdhdGVfcGF0aA0KZnJvbSBjb21taXNzaW9uaW5nLmZlYXR1cmVzIGltcG9ydCBsb2FkX2dhcF90YWJsZQ0KZnJvbSBjb21taXNzaW9uaW5nLmlvX3V0aWxzIGltcG9ydCBhdG9taWNfcGFycXVldCwgYXRvbWljX3NhdmV6LCByZWFkX2pzb24NCmZyb20gY29tbWlzc2lvbmluZy5wYXRjaGNvcmUgaW1wb3J0IHNjb3Jlc19hbG9uZ19vcmRlcg0KZnJvbSBjb21taXNzaW9uaW5nLnNlbGVjdGlvbiBpbXBvcnQgZ3JpZF9mb3JfcG9vbCwga2NlbnRlcl9vcmRlciwgcmFuZG9tX29yZGVyDQoNCkxPR0dFUiA9IGxvZ2dpbmcuZ2V0TG9nZ2VyKF9fbmFtZV9fKQ0KDQoNCmRlZiB0cmFqZWN0b3J5X3BhdGhzKGNmZzogQ29uZmlnLCBjbGFzc19uYW1lOiBzdHIsIHJlcDogaW50LCBzdHJhdGVneTogc3RyKSAtPiB0dXBsZToNCiAgICBzdGVtID0gZiJ7Y2xhc3NfbmFtZX1fcmVwe3JlcDowMmR9X3tzdHJhdGVneX0iDQogICAgdGFibGUgPSBjZmcud29ya19yb290IC8gInJ1bnMiIC8gInBhcnRzIiAvIGYie3N0ZW19LnBhcnF1ZXQiDQogICAgc2NvcmVzID0gY2ZnLndvcmtfcm9vdCAvICJydW5zIiAvICJzY29yZXMiIC8gZiJ7c3RlbX0ubnB6Ig0KICAgIHJldHVybiB0YWJsZSwgc2NvcmVzDQoNCg0KZGVmIGdhdGVfaXNfZXhwbG9yYXRvcnkoY2ZnOiBDb25maWcsIGNsYXNzZXM6IGxpc3Rbc3RyXSwgZXhwbG9yYXRvcnk6IGJvb2wpIC0+IGJvb2w6DQogICAgcGF0aCA9IGdhdGVfcGF0aChjZmcpDQogICAgaWYgbm90IHBhdGguaXNfZmlsZSgpOg0KICAgICAgICBpZiBleHBsb3JhdG9yeToNCiAgICAgICAgICAgIExPR0dFUi53YXJuaW5nKCJFMCBnYXRlIGlzIG1pc3NpbmcuIFRoaXMgcnVuIGlzIGV4cGxvcmF0b3J5IGFuZCBjYW5ub3QgYmUgcmVwb3J0ZWQgYXMgY29uZmlybWF0b3J5LiIpDQogICAgICAgICAgICByZXR1cm4gVHJ1ZQ0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIkUwIGdhdGUgaXMgbWlzc2luZy4gUlEgdHJhamVjdG9yaWVzIHN0YXkgYmxvY2tlZCB1bnRpbCBlMC9nYXRlLmpzb24gaGFzIHN0YXR1cyBQQVNTLiIpDQogICAgZ2F0ZSA9IHJlYWRfanNvbihwYXRoKQ0KICAgIGlmIGdhdGUuZ2V0KCJjb25maWdfaGFzaCIpICE9IGNmZy5oYXNoOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIkUwIGdhdGUgd2FzIHdyaXR0ZW4gZm9yIGEgZGlmZmVyZW50IGNvbmZpZyBoYXNoLiIpDQogICAgY292ZXJlZCA9IHNldChnYXRlLmdldCgiY2xhc3NlcyIsIFtdKSkNCiAgICBtaXNzaW5nID0gW25hbWUgZm9yIG5hbWUgaW4gY2xhc3NlcyBpZiBuYW1lIG5vdCBpbiBjb3ZlcmVkXQ0KICAgIHBhc3NlZCA9IGdhdGUuZ2V0KCJzdGF0dXMiKSA9PSAiUEFTUyIgYW5kIGdhdGUuZ2V0KCJwYXJ0IikgPT0gImFsbCIgYW5kIG5vdCBtaXNzaW5nDQogICAgaWYgcGFzc2VkOg0KICAgICAgICByZXR1cm4gRmFsc2UNCiAgICBpZiBleHBsb3JhdG9yeToNCiAgICAgICAgTE9HR0VSLndhcm5pbmcoIkUwIGdhdGUgZG9lcyBub3QgY292ZXIgdGhpcyBydW4gKCVzKS4gTWFya2luZyBpdCBleHBsb3JhdG9yeS4iLCBnYXRlLmdldCgic3RhdHVzIikpDQogICAgICAgIHJldHVybiBUcnVlDQogICAgaWYgbWlzc2luZzoNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYiRTAgZ2F0ZSBkb2VzIG5vdCBjb3Zlcjoge21pc3Npbmd9LiBSdW4gZTAgb24gdGhlc2UgY2xhc3NlcyBiZWZvcmUgUlEuIikNCiAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJFMCBnYXRlIHN0YXR1cyBpcyB7Z2F0ZS5nZXQoJ3N0YXR1cycpfS4gUlEgdHJhamVjdG9yaWVzIHN0YXkgYmxvY2tlZC4iKQ0KDQoNCmRlZiBfZmluaXNoZWQoY2ZnOiBDb25maWcsIHRhYmxlX3BhdGgsIHNjb3Jlc19wYXRoKSAtPiBib29sOg0KICAgIGlmIG5vdCB0YWJsZV9wYXRoLmlzX2ZpbGUoKSBvciBub3Qgc2NvcmVzX3BhdGguaXNfZmlsZSgpOg0KICAgICAgICByZXR1cm4gRmFsc2UNCiAgICBmcmFtZSA9IHBkLnJlYWRfcGFycXVldCh0YWJsZV9wYXRoKQ0KICAgIGhhc2hlcyA9IHNldChmcmFtZVsiY29uZmlnX2hhc2giXS5hc3R5cGUoc3RyKSkNCiAgICBpZiBoYXNoZXMgIT0ge2NmZy5oYXNofToNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYiUmVmdXNpbmcgdG8gcmV1c2UgYSB0cmFqZWN0b3J5IGZyb20gYW5vdGhlciBjb25maWc6IHt0YWJsZV9wYXRofSIpDQogICAgcmV0dXJuIFRydWUNCg0KDQpkZWYgX2luZGljZXMocGF0aHM6IGxpc3Rbc3RyXSwgc2VsZWN0ZWQ6IGxpc3Rbc3RyXSwgY2xhc3NfbmFtZTogc3RyKSAtPiBsaXN0W2ludF06DQogICAgaW5kZXggPSB7cGF0aDogcG9zaXRpb24gZm9yIHBvc2l0aW9uLCBwYXRoIGluIGVudW1lcmF0ZShwYXRocyl9DQogICAgdHJ5Og0KICAgICAgICByZXR1cm4gW2luZGV4W3BhdGhdIGZvciBwYXRoIGluIHNlbGVjdGVkXQ0KICAgIGV4Y2VwdCBLZXlFcnJvciBhcyBleGM6DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIntjbGFzc19uYW1lfTogc3BsaXQgcGF0aCBpcyBub3QgaW4gdGhlIGZlYXR1cmUgY2FjaGU6IHtleGN9IikgZnJvbSBleGMNCg0KDQpkZWYgcnVuX29uZSgNCiAgICBjZmc6IENvbmZpZywNCiAgICBjbGFzc19uYW1lOiBzdHIsDQogICAgcmVwOiBpbnQsDQogICAgc3RyYXRlZ3k6IHN0ciwNCiAgICBleHBsb3JhdG9yeTogYm9vbCwNCiAgICBwYXRoczogbGlzdFtzdHJdLA0KICAgIGRpc3RhbmNlOiBucC5uZGFycmF5LA0KICAgIGdhcDogbnAubmRhcnJheSwNCikgLT4gTm9uZToNCiAgICB0YWJsZV9wYXRoLCBzY29yZXNfcGF0aCA9IHRyYWplY3RvcnlfcGF0aHMoY2ZnLCBjbGFzc19uYW1lLCByZXAsIHN0cmF0ZWd5KQ0KICAgIGlmIF9maW5pc2hlZChjZmcsIHRhYmxlX3BhdGgsIHNjb3Jlc19wYXRoKToNCiAgICAgICAgTE9HR0VSLmluZm8oIlNraXBwaW5nIGZpbmlzaGVkICVzIHJlcCAlMDJkICVzIiwgY2xhc3NfbmFtZSwgcmVwLCBzdHJhdGVneSkNCiAgICAgICAgcmV0dXJuDQogICAgc3BsaXQgPSBsb2FkX3NwbGl0KGNmZywgY2xhc3NfbmFtZSwgcmVwKQ0KICAgIG5fdHJhaW4gPSBpbnQoc3BsaXRbIm5fdHJhaW4iXSkNCiAgICBjYWxpYnJhdGlvbiA9IF9pbmRpY2VzKHBhdGhzLCBzcGxpdFsiY2FsaWJyYXRpb24iXSwgY2xhc3NfbmFtZSkNCiAgICBwb29sID0gX2luZGljZXMocGF0aHMsIHNwbGl0WyJwb29sIl0sIGNsYXNzX25hbWUpDQogICAgdGVzdF9wYXRocyA9IGxpc3Qoc3BsaXRbInRlc3RfZ29vZCJdKSArIFtpdGVtWyJwYXRoIl0gZm9yIGl0ZW0gaW4gc3BsaXRbInRlc3RfZGVmZWN0Il1dDQogICAgdGVzdCA9IF9pbmRpY2VzKHBhdGhzLCB0ZXN0X3BhdGhzLCBjbGFzc19uYW1lKQ0KICAgIGlmIHNldChjYWxpYnJhdGlvbikgJiBzZXQocG9vbCkgb3IgbWF4KGNhbGlicmF0aW9uICsgcG9vbCkgPj0gbl90cmFpbiBvciBtaW4odGVzdCkgPCBuX3RyYWluOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJ7Y2xhc3NfbmFtZX06IEMvUC90ZXN0IGluZGljZXMgYXJlIG5vdCBhIHZhbGlkIHBhcnRpdGlvbi4iKQ0KICAgIF9zcGxpdF9ybmcsIHJhbmRvbV9ybmcsIGtjZW50ZXJfcm5nID0gc3BsaXRfc2VlZHMoY2ZnLCBjbGFzc19uYW1lLCByZXApDQogICAgaWYgc3RyYXRlZ3kgPT0gInJhbmRvbSI6DQogICAgICAgIG9yZGVyID0gcmFuZG9tX29yZGVyKHBvb2wsIHJhbmRvbV9ybmcpDQogICAgZWxpZiBzdHJhdGVneSA9PSAia2NlbnRlciI6DQogICAgICAgIG9yZGVyID0ga2NlbnRlcl9vcmRlcihnYXAsIHBvb2wsIGtjZW50ZXJfcm5nKQ0KICAgIGVsc2U6DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIlVua25vd24gc3RyYXRlZ3k6IHtzdHJhdGVneX0iKQ0KICAgIGdyaWQgPSBncmlkX2Zvcl9wb29sKGxlbihwb29sKSwgY2ZnLm5fZ3JpZCkNCiAgICBxdWVyaWVzID0gbnAuYXNhcnJheShjYWxpYnJhdGlvbiArIHRlc3QsIGR0eXBlPWludCkNCiAgICBzdGFja2VkID0gc2NvcmVzX2Fsb25nX29yZGVyKGRpc3RhbmNlLCBxdWVyaWVzLCBvcmRlciwgZ3JpZCkNCiAgICBjYWxpYnJhdGlvbl9zY29yZXMgPSBzdGFja2VkWzosIDogbGVuKGNhbGlicmF0aW9uKV0NCiAgICB0ZXN0X3Njb3JlcyA9IHN0YWNrZWRbOiwgbGVuKGNhbGlicmF0aW9uKSA6XQ0KICAgIGxhYmVscyA9IG5wLmFycmF5KFswXSAqIGxlbihzcGxpdFsidGVzdF9nb29kIl0pICsgWzFdICogbGVuKHNwbGl0WyJ0ZXN0X2RlZmVjdCJdKSwgZHR5cGU9aW50KQ0KICAgIHJvd3MgPSBbXQ0KICAgIGZvciByb3dfaW5kZXgsIG4gaW4gZW51bWVyYXRlKGdyaWQpOg0KICAgICAgICBtZXRyaWNzID0gZXZhbHVhdGVfc2NvcmVzKA0KICAgICAgICAgICAgdGVzdF9zY29yZXNbcm93X2luZGV4XSwNCiAgICAgICAgICAgIGxhYmVscywNCiAgICAgICAgICAgIGNhbGlicmF0aW9uX3Njb3Jlc1tyb3dfaW5kZXhdLA0KICAgICAgICAgICAgY2ZnLmFscGhhLA0KICAgICAgICAgICAgY2ZnLnByYWN0aWNhbF90cHIsDQogICAgICAgICAgICBjZmcucHJhY3RpY2FsX2ZwciwNCiAgICAgICAgKQ0KICAgICAgICByb3dzLmFwcGVuZCgNCiAgICAgICAgICAgIHsNCiAgICAgICAgICAgICAgICAiY2xhc3NfbmFtZSI6IGNsYXNzX25hbWUsDQogICAgICAgICAgICAgICAgImdyb3VwIjogY2ZnLmdyb3VwX29mKGNsYXNzX25hbWUpLA0KICAgICAgICAgICAgICAgICJyZXAiOiBpbnQocmVwKSwNCiAgICAgICAgICAgICAgICAic3RyYXRlZ3kiOiBzdHJhdGVneSwNCiAgICAgICAgICAgICAgICAiTiI6IGludChuKSwNCiAgICAgICAgICAgICAgICAicG9vbF9zaXplIjogaW50KGxlbihwb29sKSksDQogICAgICAgICAgICAgICAgImNvbmZpZ19oYXNoIjogY2ZnLmhhc2gsDQogICAgICAgICAgICAgICAgImV4cGxvcmF0b3J5IjogYm9vbChleHBsb3JhdG9yeSksDQogICAgICAgICAgICAgICAgKiptZXRyaWNzLA0KICAgICAgICAgICAgfQ0KICAgICAgICApDQogICAgYXRvbWljX3BhcnF1ZXQodGFibGVfcGF0aCwgcGQuRGF0YUZyYW1lKHJvd3MpKQ0KICAgIGF0b21pY19zYXZleigNCiAgICAgICAgc2NvcmVzX3BhdGgsDQogICAgICAgIGdyaWQ9bnAuYXNhcnJheShncmlkLCBkdHlwZT1ucC5pbnQzMiksDQogICAgICAgIHNjb3Jlc190ZXN0PXRlc3Rfc2NvcmVzLmFzdHlwZShucC5mbG9hdDMyKSwNCiAgICAgICAgc2NvcmVzX2NhbGlicmF0aW9uPWNhbGlicmF0aW9uX3Njb3Jlcy5hc3R5cGUobnAuZmxvYXQzMiksDQogICAgICAgIHRlc3RfbGFiZWxzPWxhYmVscy5hc3R5cGUobnAuaW50OCksDQogICAgICAgIG9yZGVyPW9yZGVyLmFzdHlwZShucC5pbnQzMiksDQogICAgICAgIHRlc3RfcGF0aHM9bnAuYXNhcnJheSh0ZXN0X3BhdGhzKSwNCiAgICAgICAgY2FsaWJyYXRpb25fcGF0aHM9bnAuYXNhcnJheShzcGxpdFsiY2FsaWJyYXRpb24iXSksDQogICAgKQ0KICAgIExPR0dFUi5pbmZvKCJGaW5pc2hlZCAlcyByZXAgJTAyZCAlcyAoJWQgZ3JpZCBwb2ludHMpIiwgY2xhc3NfbmFtZSwgcmVwLCBzdHJhdGVneSwgbGVuKGdyaWQpKQ0KDQoNCmRlZiBydW4oY2ZnOiBDb25maWcsIGNsYXNzZXM6IGxpc3Rbc3RyXSwgZXhwbG9yYXRvcnk6IGJvb2wgPSBGYWxzZSkgLT4gTm9uZToNCiAgICBtYXJrID0gZ2F0ZV9pc19leHBsb3JhdG9yeShjZmcsIGNsYXNzZXMsIGV4cGxvcmF0b3J5KQ0KICAgIHRvdGFsID0gbGVuKGNsYXNzZXMpICogY2ZnLm5fcmVwZWF0cyAqIGxlbihjZmcuc3RyYXRlZ2llcykNCiAgICBkb25lID0gMA0KICAgIGZvciBjbGFzc19uYW1lIGluIGNsYXNzZXM6DQogICAgICAgIGRpc3RhbmNlX2NhY2hlID0gbG9hZF9kaXN0YW5jZV9jYWNoZShjZmcsIGNsYXNzX25hbWUpDQogICAgICAgIGdhcF90YWJsZSA9IGxvYWRfZ2FwX3RhYmxlKGNmZywgY2xhc3NfbmFtZSkNCiAgICAgICAgaWYgZ2FwX3RhYmxlWyJwYXRocyJdICE9IGxpc3QoZGlzdGFuY2VfY2FjaGVbIm1ldGEiXVsicGF0aHMiXSk6DQogICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJ7Y2xhc3NfbmFtZX06IGZlYXR1cmUgYW5kIGRpc3RhbmNlIGNhY2hlcyBhcmUgbm90IGFsaWduZWQuIikNCiAgICAgICAgZm9yIHJlcCBpbiByYW5nZShjZmcubl9yZXBlYXRzKToNCiAgICAgICAgICAgIGZvciBzdHJhdGVneSBpbiBjZmcuc3RyYXRlZ2llczoNCiAgICAgICAgICAgICAgICBydW5fb25lKA0KICAgICAgICAgICAgICAgICAgICBjZmcsDQogICAgICAgICAgICAgICAgICAgIGNsYXNzX25hbWUsDQogICAgICAgICAgICAgICAgICAgIHJlcCwNCiAgICAgICAgICAgICAgICAgICAgc3RyYXRlZ3ksDQogICAgICAgICAgICAgICAgICAgIG1hcmssDQogICAgICAgICAgICAgICAgICAgIGdhcF90YWJsZVsicGF0aHMiXSwNCiAgICAgICAgICAgICAgICAgICAgZGlzdGFuY2VfY2FjaGVbIkQiXSwNCiAgICAgICAgICAgICAgICAgICAgZ2FwX3RhYmxlWyJnYXAiXSwNCiAgICAgICAgICAgICAgICApDQogICAgICAgICAgICAgICAgZG9uZSArPSAxDQogICAgICAgICAgICAgICAgTE9HR0VSLmluZm8oIlByb2dyZXNzICVkLyVkIiwgZG9uZSwgdG90YWwpDQo=", "commissioning/selection.py": "IiIiTmVzdGVkIG1lbW9yeS1iYW5rIHRyYWplY3Rvcmllcy4gQm90aCBzdHJhdGVnaWVzIHJldHVybiBhIHBlcm11dGF0aW9uIG9mIFAuIiIiDQoNCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMNCg0KaW1wb3J0IG51bXB5IGFzIG5wDQoNCg0KZGVmIGdyaWRfZm9yX3Bvb2wocG9vbF9zaXplOiBpbnQsIGJhc2VfZ3JpZDogbGlzdFtpbnRdKSAtPiBsaXN0W2ludF06DQogICAgaWYgcG9vbF9zaXplIDwgMToNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJQb29sIGlzIGVtcHR5LiIpDQogICAgZ3JpZCA9IFtpbnQobikgZm9yIG4gaW4gYmFzZV9ncmlkIGlmIDEgPD0gaW50KG4pIDw9IHBvb2xfc2l6ZV0NCiAgICBpZiBwb29sX3NpemUgbm90IGluIGdyaWQ6DQogICAgICAgIGdyaWQuYXBwZW5kKGludChwb29sX3NpemUpKQ0KICAgIGlmIGdyaWQgIT0gc29ydGVkKHNldChncmlkKSk6DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIkdyaWQgaXMgbm90IHN0cmljdGx5IGluY3JlYXNpbmc6IHtncmlkfSIpDQogICAgcmV0dXJuIGdyaWQNCg0KDQpkZWYgcmFuZG9tX29yZGVyKHBvb2w6IG5wLm5kYXJyYXkgfCBsaXN0W2ludF0sIHJuZzogbnAucmFuZG9tLkdlbmVyYXRvcikgLT4gbnAubmRhcnJheToNCiAgICBvcmRlciA9IG5wLmFycmF5KHBvb2wsIGR0eXBlPWludCwgY29weT1UcnVlKQ0KICAgIGlmIG9yZGVyLnNpemUgPT0gMCBvciBsZW4obnAudW5pcXVlKG9yZGVyKSkgIT0gbGVuKG9yZGVyKToNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJSYW5kb20gdHJhamVjdG9yeSBleHBlY3RzIGEgbm9uLWVtcHR5IHBvb2wgd2l0aG91dCBkdXBsaWNhdGVzLiIpDQogICAgcm5nLnNodWZmbGUob3JkZXIpDQogICAgcmV0dXJuIG9yZGVyDQoNCg0KZGVmIGtjZW50ZXJfb3JkZXIoZmVhdHVyZXM6IG5wLm5kYXJyYXksIHBvb2w6IG5wLm5kYXJyYXkgfCBsaXN0W2ludF0sIHJuZzogbnAucmFuZG9tLkdlbmVyYXRvcikgLT4gbnAubmRhcnJheToNCiAgICAiIiJGYXJ0aGVzdC1maXJzdCB0cmF2ZXJzYWwgb24gR0FQIGZlYXR1cmVzLiBUaGUgZmlyc3QgaW1hZ2UgaXMgcmFuZG9tLg0KDQogICAgVGhpcyBpcyBpbWFnZS1sZXZlbCBkaXZlcnNpdHkgc2VsZWN0aW9uLiBJdCBpcyBub3QgY29yZXNldCBzZWxlY3Rpb24gb24gUGF0Y2hDb3JlIHBhdGNoZXMuDQogICAgIiIiDQogICAgcG9vbCA9IG5wLmFycmF5KHBvb2wsIGR0eXBlPWludCwgY29weT1UcnVlKQ0KICAgIGlmIHBvb2wuc2l6ZSA9PSAwIG9yIGxlbihucC51bmlxdWUocG9vbCkpICE9IGxlbihwb29sKToNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJrLWNlbnRlciB0cmFqZWN0b3J5IGV4cGVjdHMgYSBub24tZW1wdHkgcG9vbCB3aXRob3V0IGR1cGxpY2F0ZXMuIikNCiAgICBpZiBucC5hbnkocG9vbCA8IDApIG9yIG5wLmFueShwb29sID49IGxlbihmZWF0dXJlcykpOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIlBvb2wgaW5kZXggaXMgb3V0c2lkZSB0aGUgZmVhdHVyZSB0YWJsZS4iKQ0KICAgIGZpcnN0ID0gaW50KHJuZy5pbnRlZ2VycygwLCBsZW4ocG9vbCkpKQ0KICAgIHJlbWFpbmluZyA9IG5wLm9uZXMobGVuKHBvb2wpLCBkdHlwZT1ib29sKQ0KICAgIHJlbWFpbmluZ1tmaXJzdF0gPSBGYWxzZQ0KICAgIHNlbGVjdGVkID0gW2ZpcnN0XQ0KICAgIGNob3NlbiA9IGZlYXR1cmVzW3Bvb2xbZmlyc3RdXQ0KICAgIG1pbl9kaXN0ID0gbnAubGluYWxnLm5vcm0oZmVhdHVyZXNbcG9vbF0gLSBjaG9zZW4sIGF4aXM9MSkNCiAgICB3aGlsZSBib29sKHJlbWFpbmluZy5hbnkoKSk6DQogICAgICAgIG1hc2tlZCA9IG5wLndoZXJlKHJlbWFpbmluZywgbWluX2Rpc3QsIC1ucC5pbmYpDQogICAgICAgIG54dCA9IGludChucC5hcmdtYXgobWFza2VkKSkNCiAgICAgICAgcmVtYWluaW5nW254dF0gPSBGYWxzZQ0KICAgICAgICBzZWxlY3RlZC5hcHBlbmQobnh0KQ0KICAgICAgICB1cGRhdGVkID0gbnAubGluYWxnLm5vcm0oZmVhdHVyZXNbcG9vbF0gLSBmZWF0dXJlc1twb29sW254dF1dLCBheGlzPTEpDQogICAgICAgIG1pbl9kaXN0ID0gbnAubWluaW11bShtaW5fZGlzdCwgdXBkYXRlZCkNCiAgICBvcmRlciA9IHBvb2xbbnAuYXJyYXkoc2VsZWN0ZWQsIGR0eXBlPWludCldDQogICAgaWYgbGVuKG5wLnVuaXF1ZShvcmRlcikpICE9IGxlbihwb29sKSBvciBzZXQob3JkZXIudG9saXN0KCkpICE9IHNldChwb29sLnRvbGlzdCgpKToNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJrLWNlbnRlciBkaWQgbm90IHJldHVybiBhIHBlcm11dGF0aW9uIG9mIHRoZSBwb29sLiIpDQogICAgcmV0dXJuIG9yZGVyDQo=", "commissioning/stats.py": "IiIiUHJpbWFyeSBwYWlyZWQgY29tcGFyaXNvbnM6IGV4YWN0IFdpbGNveG9uLCBIb2xtLCBzaWduLWZsaXAsIGFuZCBhIGNsYXNzLWxldmVsIGJvb3RzdHJhcC4iIiINCg0KZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucw0KDQppbXBvcnQgbnVtcHkgYXMgbnANCmltcG9ydCBwYW5kYXMgYXMgcGQNCmZyb20gc2NpcHkuc3RhdHMgaW1wb3J0IHdpbGNveG9uDQoNCg0KZGVmIGhvbG0ocHZhbHVlczogbGlzdFtmbG9hdF0pIC0+IGxpc3RbZmxvYXRdOg0KICAgIHAgPSBucC5hc2FycmF5KHB2YWx1ZXMsIGR0eXBlPWZsb2F0KQ0KICAgIG9yZGVyID0gbnAuYXJnc29ydChwLCBraW5kPSJtZXJnZXNvcnQiKQ0KICAgIGFkanVzdGVkID0gbnAuZW1wdHkobGVuKHApLCBkdHlwZT1mbG9hdCkNCiAgICBydW5uaW5nID0gMC4wDQogICAgZm9yIHJhbmssIGluZGV4IGluIGVudW1lcmF0ZShvcmRlcik6DQogICAgICAgIHJ1bm5pbmcgPSBtYXgocnVubmluZywgbWluKDEuMCwgKGxlbihwKSAtIHJhbmspICogZmxvYXQocFtpbmRleF0pKSkNCiAgICAgICAgYWRqdXN0ZWRbaW5kZXhdID0gcnVubmluZw0KICAgIHJldHVybiBhZGp1c3RlZC50b2xpc3QoKQ0KDQoNCmRlZiBwYWlyZWRfZGlmZmVyZW5jZShzdW1tYXJ5OiBwZC5EYXRhRnJhbWUsIG1ldHJpYzogc3RyKSAtPiBwZC5TZXJpZXM6DQogICAgd2lkZSA9IHN1bW1hcnkucGl2b3QoaW5kZXg9ImNsYXNzX25hbWUiLCBjb2x1bW5zPSJzdHJhdGVneSIsIHZhbHVlcz1tZXRyaWMpDQogICAgaWYgImtjZW50ZXIiIG5vdCBpbiB3aWRlLmNvbHVtbnMgb3IgInJhbmRvbSIgbm90IGluIHdpZGUuY29sdW1uczoNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYiQm90aCBzdHJhdGVnaWVzIGFyZSByZXF1aXJlZCB0byBjb21wYXJlIHttZXRyaWN9LiIpDQogICAgZGlmZmVyZW5jZSA9ICh3aWRlWyJrY2VudGVyIl0gLSB3aWRlWyJyYW5kb20iXSkuYXN0eXBlKGZsb2F0KQ0KICAgIHJldHVybiBkaWZmZXJlbmNlLnNvcnRfaW5kZXgoKQ0KDQoNCmRlZiB3aWxjb3hvbl9wdmFsdWUoZGlmZmVyZW5jZTogbnAubmRhcnJheSkgLT4gZmxvYXQ6DQogICAgdmFsdWVzID0gbnAuYXNhcnJheShkaWZmZXJlbmNlLCBkdHlwZT1mbG9hdCkNCiAgICB2YWx1ZXMgPSB2YWx1ZXNbbnAuaXNmaW5pdGUodmFsdWVzKV0NCiAgICBpZiB2YWx1ZXMuc2l6ZSA9PSAwIG9yIG5wLmFsbGNsb3NlKHZhbHVlcywgMC4wKToNCiAgICAgICAgcmV0dXJuIDEuMA0KICAgIHRyeToNCiAgICAgICAgcmVzdWx0ID0gd2lsY294b24odmFsdWVzLCB6ZXJvX21ldGhvZD0id2lsY294IiwgYWx0ZXJuYXRpdmU9InR3by1zaWRlZCIsIG1ldGhvZD0iZXhhY3QiKQ0KICAgIGV4Y2VwdCBWYWx1ZUVycm9yOg0KICAgICAgICByZXN1bHQgPSB3aWxjb3hvbih2YWx1ZXMsIHplcm9fbWV0aG9kPSJ3aWxjb3giLCBhbHRlcm5hdGl2ZT0idHdvLXNpZGVkIiwgbWV0aG9kPSJhdXRvIikNCiAgICByZXR1cm4gZmxvYXQocmVzdWx0LnB2YWx1ZSkNCg0KDQpkZWYgc2lnbmZsaXBfcHZhbHVlKGRpZmZlcmVuY2U6IG5wLm5kYXJyYXkpIC0+IGZsb2F0Og0KICAgICIiIkV4YWN0IHR3by1zaWRlZCBwYWlyZWQgcGVybXV0YXRpb24gdGVzdCBvZiB0aGUgbWVhbiBkaWZmZXJlbmNlLiAyXm4gc2lnbiBmbGlwcy4iIiINCiAgICB2YWx1ZXMgPSBucC5hc2FycmF5KGRpZmZlcmVuY2UsIGR0eXBlPWZsb2F0KQ0KICAgIGlmIHZhbHVlcy5uZGltICE9IDEgb3Igbm90IG5wLmlzZmluaXRlKHZhbHVlcykuYWxsKCk6DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiU2lnbi1mbGlwIGV4cGVjdHMgYSBmaW5pdGUgMS1EIHZlY3RvciBvZiBwYWlyZWQgZGlmZmVyZW5jZXMuIikNCiAgICBpZiB2YWx1ZXMuc2l6ZSA+IDIwOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIkV4YWN0IHNpZ24tZmxpcCBpcyBsaW1pdGVkIHRvIDIwIG9yIGZld2VyIGNsYXNzZXMuIikNCiAgICBvYnNlcnZlZCA9IGFicyhmbG9hdChucC5tZWFuKHZhbHVlcykpKQ0KICAgIG4gPSBpbnQodmFsdWVzLnNpemUpDQogICAgdG90YWwgPSAxIDw8IG4NCiAgICBjb3VudCA9IDANCiAgICBjaHVuayA9IDQwOTYNCiAgICBmb3Igc3RhcnQgaW4gcmFuZ2UoMCwgdG90YWwsIGNodW5rKToNCiAgICAgICAgbWFza3MgPSBucC5hcmFuZ2Uoc3RhcnQsIG1pbihzdGFydCArIGNodW5rLCB0b3RhbCksIGR0eXBlPW5wLnVpbnQzMikNCiAgICAgICAgc2lnbnMgPSBucC5vbmVzKChtYXNrcy5zaXplLCBuKSwgZHR5cGU9bnAuZmxvYXQ2NCkNCiAgICAgICAgZm9yIGJpdCBpbiByYW5nZShuKToNCiAgICAgICAgICAgIGZsaXAgPSAoKG1hc2tzID4+IG5wLnVpbnQzMihiaXQpKSAmIG5wLnVpbnQzMigxKSkuYXN0eXBlKGJvb2wpDQogICAgICAgICAgICBzaWduc1tmbGlwLCBiaXRdID0gLTEuMA0KICAgICAgICBtZWFucyA9IG5wLmFicyhzaWducyBAIHZhbHVlcyAvIG4pDQogICAgICAgIGNvdW50ICs9IGludChucC5zdW0obWVhbnMgKyAxZS0xMiA+PSBvYnNlcnZlZCkpDQogICAgcmV0dXJuIGNvdW50IC8gdG90YWwNCg0KDQpkZWYgYm9vdHN0cmFwX21lZGlhbl9pbnRlcnZhbChkaWZmZXJlbmNlOiBucC5uZGFycmF5LCBuX2Jvb3RzdHJhcDogaW50LCBzZWVkOiBpbnQpIC0+IGRpY3Q6DQogICAgdmFsdWVzID0gbnAuYXNhcnJheShkaWZmZXJlbmNlLCBkdHlwZT1mbG9hdCkNCiAgICBybmcgPSBucC5yYW5kb20uZGVmYXVsdF9ybmcoc2VlZCkNCiAgICBkcmF3cyA9IHJuZy5pbnRlZ2VycygwLCB2YWx1ZXMuc2l6ZSwgc2l6ZT0oaW50KG5fYm9vdHN0cmFwKSwgdmFsdWVzLnNpemUpKQ0KICAgIG1lZGlhbnMgPSBucC5tZWRpYW4odmFsdWVzW2RyYXdzXSwgYXhpcz0xKQ0KICAgIGxvdywgaGlnaCA9IG5wLnF1YW50aWxlKG1lZGlhbnMsIFswLjAyNSwgMC45NzVdKQ0KICAgIHJldHVybiB7DQogICAgICAgICJtZWRpYW4iOiBmbG9hdChucC5tZWRpYW4odmFsdWVzKSksDQogICAgICAgICJjaTk1IjogW2Zsb2F0KGxvdyksIGZsb2F0KGhpZ2gpXSwNCiAgICAgICAgIm5fYm9vdHN0cmFwIjogaW50KG5fYm9vdHN0cmFwKSwNCiAgICAgICAgInNlZWQiOiBpbnQoc2VlZCksDQogICAgfQ0KDQoNCmRlZiBfZGlyZWN0aW9uX2NvdW50cyhkaWZmZXJlbmNlOiBwZC5TZXJpZXMsIGxvd2VyX2lzX2JldHRlcjogYm9vbCkgLT4gZGljdDoNCiAgICB2YWx1ZXMgPSBkaWZmZXJlbmNlLnRvX251bXB5KGR0eXBlPWZsb2F0KQ0KICAgIGJldHRlciA9IHZhbHVlcyA8IDAgaWYgbG93ZXJfaXNfYmV0dGVyIGVsc2UgdmFsdWVzID4gMA0KICAgIHdvcnNlID0gdmFsdWVzID4gMCBpZiBsb3dlcl9pc19iZXR0ZXIgZWxzZSB2YWx1ZXMgPCAwDQogICAgcmV0dXJuIHsNCiAgICAgICAgIm5fa2NlbnRlcl9iZXR0ZXIiOiBpbnQobnAuc3VtKGJldHRlcikpLA0KICAgICAgICAibl9yYW5kb21fYmV0dGVyIjogaW50KG5wLnN1bSh3b3JzZSkpLA0KICAgICAgICAibl90aWVzIjogaW50KG5wLnN1bShucC5pc2Nsb3NlKHZhbHVlcywgMC4wKSkpLA0KICAgICAgICAicGVyX2NsYXNzIjoge3N0cihuYW1lKTogZmxvYXQodmFsdWUpIGZvciBuYW1lLCB2YWx1ZSBpbiBkaWZmZXJlbmNlLml0ZW1zKCl9LA0KICAgIH0NCg0KDQpkZWYgY29tcGFyZV9zdHJhdGVnaWVzKHN1bW1hcnk6IHBkLkRhdGFGcmFtZSwgbWFyZ2luOiBmbG9hdCwgbl9ib290c3RyYXA6IGludCwgc2VlZDogaW50KSAtPiBkaWN0Og0KICAgIHRwciA9IHBhaXJlZF9kaWZmZXJlbmNlKHN1bW1hcnksICJhdWxjX3RwciIpDQogICAgZnByID0gcGFpcmVkX2RpZmZlcmVuY2Uoc3VtbWFyeSwgImF1bGNfZnByIikNCiAgICBwX3RwciA9IHdpbGNveG9uX3B2YWx1ZSh0cHIudG9fbnVtcHkoKSkNCiAgICBwX2ZwciA9IHdpbGNveG9uX3B2YWx1ZShmcHIudG9fbnVtcHkoKSkNCiAgICBhZGp1c3RlZCA9IGhvbG0oW3BfdHByLCBwX2Zwcl0pDQogICAgdHByX2Jvb3QgPSBib290c3RyYXBfbWVkaWFuX2ludGVydmFsKHRwci50b19udW1weSgpLCBuX2Jvb3RzdHJhcCwgc2VlZCkNCiAgICBmcHJfYm9vdCA9IGJvb3RzdHJhcF9tZWRpYW5faW50ZXJ2YWwoZnByLnRvX251bXB5KCksIG5fYm9vdHN0cmFwLCBzZWVkICsgMSkNCiAgICBwcmFjdGljYWwgPSBib29sKHRwcl9ib290WyJtZWRpYW4iXSA+IDAuMCBhbmQgZnByX2Jvb3RbIm1lZGlhbiJdIDw9IGZsb2F0KG1hcmdpbikpDQogICAgcmV0dXJuIHsNCiAgICAgICAgImRpZmZlcmVuY2UiOiAia2NlbnRlciAtIHJhbmRvbSIsDQogICAgICAgICJwcmFjdGljYWxfY3JpdGVyaW9uX3Bhc3NlZCI6IHByYWN0aWNhbCwNCiAgICAgICAgImZwcl9tYXJnaW4iOiBmbG9hdChtYXJnaW4pLA0KICAgICAgICAiYXVsY190cHIiOiB7DQogICAgICAgICAgICAid2lsY294b25fcCI6IHBfdHByLA0KICAgICAgICAgICAgImhvbG1fcCI6IGFkanVzdGVkWzBdLA0KICAgICAgICAgICAgInBlcm11dGF0aW9uX3AiOiBzaWduZmxpcF9wdmFsdWUodHByLnRvX251bXB5KCkpLA0KICAgICAgICAgICAgImJvb3RzdHJhcCI6IHRwcl9ib290LA0KICAgICAgICAgICAgKipfZGlyZWN0aW9uX2NvdW50cyh0cHIsIGxvd2VyX2lzX2JldHRlcj1GYWxzZSksDQogICAgICAgIH0sDQogICAgICAgICJhdWxjX2ZwciI6IHsNCiAgICAgICAgICAgICJ3aWxjb3hvbl9wIjogcF9mcHIsDQogICAgICAgICAgICAiaG9sbV9wIjogYWRqdXN0ZWRbMV0sDQogICAgICAgICAgICAicGVybXV0YXRpb25fcCI6IHNpZ25mbGlwX3B2YWx1ZShmcHIudG9fbnVtcHkoKSksDQogICAgICAgICAgICAiYm9vdHN0cmFwIjogZnByX2Jvb3QsDQogICAgICAgICAgICAqKl9kaXJlY3Rpb25fY291bnRzKGZwciwgbG93ZXJfaXNfYmV0dGVyPVRydWUpLA0KICAgICAgICB9LA0KICAgIH0NCg==", "commissioning/stopping.py": "IiIiTm9ybWFsLW9ubHkgc3RvcHBpbmcgcnVsZS4gRGVsdGEgaXMgbWVhc3VyZWQgYWdhaW5zdCB0aGUgcHJldmlvdXMgZ3JpZCBwb2ludCwgbm90IE4vMi4iIiINCg0KZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucw0KDQppbXBvcnQgbWF0aA0KDQppbXBvcnQgbnVtcHkgYXMgbnANCg0KDQpkZWYgcmVsYXRpdmVfZGVsdGFzKG1lZGlhbnM6IGxpc3RbZmxvYXRdIHwgbnAubmRhcnJheSkgLT4gbGlzdFtmbG9hdCB8IE5vbmVdOg0KICAgIHZhbHVlcyA9IFtmbG9hdCh2KSBmb3IgdiBpbiBtZWRpYW5zXQ0KICAgIGlmIGFueShub3QgbWF0aC5pc2Zpbml0ZSh2KSBmb3IgdiBpbiB2YWx1ZXMpOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIlN0b3BwaW5nIG1lZGlhbnMgbXVzdCBiZSBmaW5pdGUuIikNCiAgICBkZWx0YXM6IGxpc3RbZmxvYXQgfCBOb25lXSA9IFtOb25lXQ0KICAgIGZvciBwcmV2aW91cywgY3VycmVudCBpbiB6aXAodmFsdWVzLCB2YWx1ZXNbMTpdKToNCiAgICAgICAgaWYgcHJldmlvdXMgPT0gMC4wOg0KICAgICAgICAgICAgZGVsdGFzLmFwcGVuZCgwLjAgaWYgY3VycmVudCA9PSAwLjAgZWxzZSBtYXRoLmluZikNCiAgICAgICAgZWxzZToNCiAgICAgICAgICAgIGRlbHRhcy5hcHBlbmQoYWJzKGN1cnJlbnQgLSBwcmV2aW91cykgLyBhYnMocHJldmlvdXMpKQ0KICAgIHJldHVybiBkZWx0YXMNCg0KDQpkZWYgbl9zdG9wKGdyaWQ6IGxpc3RbaW50XSwgbWVkaWFuczogbGlzdFtmbG9hdF0gfCBucC5uZGFycmF5LCBkZWx0YV90aHJlc2hvbGQ6IGZsb2F0KSAtPiBpbnQgfCBOb25lOg0KICAgICIiIkVuZHBvaW50IG9mIHRoZSBzZWNvbmQgY29uc2VjdXRpdmUgZ3JpZCB0cmFuc2l0aW9uIHdob3NlIHJlbGF0aXZlIGNoYW5nZSBpcyBiZWxvdyB0aGUgdGhyZXNob2xkLg0KDQogICAgUmV0dXJucyBOb25lIHdoZW4gdGhlIHJ1bGUgbmV2ZXIgZmlyZXMuIFRoZXJlIGlzIG5vIGludGVycG9sYXRpb24gYmV0d2VlbiBncmlkIHBvaW50cy4NCiAgICAiIiINCiAgICBpZiBsZW4oZ3JpZCkgIT0gbGVuKG1lZGlhbnMpOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIkdyaWQgYW5kIGNhbGlicmF0aW9uIG1lZGlhbnMgbXVzdCBoYXZlIHRoZSBzYW1lIGxlbmd0aC4iKQ0KICAgIGlmIGxlbihncmlkKSA8IDI6DQogICAgICAgIHJldHVybiBOb25lDQogICAgZGVsdGFzID0gcmVsYXRpdmVfZGVsdGFzKG1lZGlhbnMpDQogICAgc3RyZWFrID0gMA0KICAgIGZvciBpbmRleCBpbiByYW5nZSgxLCBsZW4oZ3JpZCkpOg0KICAgICAgICBpZiBkZWx0YXNbaW5kZXhdIGlzIG5vdCBOb25lIGFuZCBkZWx0YXNbaW5kZXhdIDwgZGVsdGFfdGhyZXNob2xkOg0KICAgICAgICAgICAgc3RyZWFrICs9IDENCiAgICAgICAgICAgIGlmIHN0cmVhayA+PSAyOg0KICAgICAgICAgICAgICAgIHJldHVybiBpbnQoZ3JpZFtpbmRleF0pDQogICAgICAgIGVsc2U6DQogICAgICAgICAgICBzdHJlYWsgPSAwDQogICAgcmV0dXJuIE5vbmUNCg==", "commissioning/visa/__init__.py": "IiIiQ29uZmlybWF0b3J5IHJlcGxpY2F0aW9uIG9uIFZpc0EgKHByZXJlZ2lzdHJhdGlvbl92aXNhLm1kKS4gVGhlIE1WVGVjIG1vZHVsZXMgYXJlIHJldXNlZCwgbm90IG1vZGlmaWVkLiIiIg0K", "commissioning/visa/__main__.py": "IiIiQ29tbWFuZCBsaW5lOiBweXRob24gLW0gY29tbWlzc2lvbmluZy52aXNhIHtjb252ZXJ0fHByZXBhcmV8ZmVhdHVyZXN8ZGlzdGFuY2VzfGUwfHJ1bnxleGNoYW5nZXxidWRnZXR8YW5hbHl6ZXxyZXBvcnR8YWxsfSIiIg0KDQpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zDQoNCmltcG9ydCBhcmdwYXJzZQ0KaW1wb3J0IGxvZ2dpbmcNCmltcG9ydCBwbGF0Zm9ybQ0KaW1wb3J0IHN5cw0KDQpmcm9tIGNvbW1pc3Npb25pbmcgaW1wb3J0IF9fdmVyc2lvbl9fDQpmcm9tIGNvbW1pc3Npb25pbmcuaW9fdXRpbHMgaW1wb3J0IHdyaXRlX2pzb24NCmZyb20gY29tbWlzc2lvbmluZy52aXNhLmNvbmZpZyBpbXBvcnQgbG9hZF92aXNhX2NvbmZpZw0KDQpTVEFHRVMgPSBbImNvbnZlcnQiLCAicHJlcGFyZSIsICJmZWF0dXJlcyIsICJkaXN0YW5jZXMiLCAiZTAiLCAicnVuIiwgImV4Y2hhbmdlIiwgImJ1ZGdldCIsICJhbmFseXplIiwgInJlcG9ydCIsICJhbGwiXQ0KU0NPUklOR19TVEFHRVMgPSB7ImRpc3RhbmNlcyIsICJlMCIsICJydW4iLCAiZXhjaGFuZ2UiLCAiYnVkZ2V0IiwgImFsbCJ9DQpGUk9aRU5fTUFSSyA9ICJTdGF0dXM6ICoqRlJPWkVOKioiDQoNCg0KZGVmIHJlcXVpcmVfZnJvemVuX3ByZXJlZ2lzdHJhdGlvbihjZmcsIGV4cGxvcmF0b3J5OiBib29sKSAtPiBib29sOg0KICAgICIiIk5vIFZpc0EgaW1hZ2UgaXMgc2NvcmVkIGJlZm9yZSBwcmVyZWdpc3RyYXRpb25fdmlzYS5tZCBpcyBmcm96ZW47IGFmdGVyd2FyZHMgaXQgbXVzdCBub3QgY2hhbmdlLg0KDQogICAgUmV0dXJucyBUcnVlIHdoZW4gdGhlIHJ1biBtdXN0IGJlIG1hcmtlZCBleHBsb3JhdG9yeSBiZWNhdXNlIHRoZSBwcmVyZWdpc3RyYXRpb24gaXMgbm90IGZyb3plbi4NCiAgICAiIiINCiAgICBpbXBvcnQgaGFzaGxpYg0KDQogICAgZnJvbSBjb21taXNzaW9uaW5nLmNvbmZpZyBpbXBvcnQgcHJvamVjdF9yb290DQoNCiAgICBwYXRoID0gcHJvamVjdF9yb290KCkgLyAicHJlcmVnaXN0cmF0aW9uX3Zpc2EubWQiDQogICAgdGV4dCA9IHBhdGgucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYtOCIpDQogICAgaWYgRlJPWkVOX01BUksgbm90IGluIHRleHQ6DQogICAgICAgIGlmIGV4cGxvcmF0b3J5Og0KICAgICAgICAgICAgbG9nZ2luZy53YXJuaW5nKCJwcmVyZWdpc3RyYXRpb25fdmlzYS5tZCBpcyBub3QgZnJvemVuLiBUaGlzIHJ1biBpcyBleHBsb3JhdG9yeS4iKQ0KICAgICAgICAgICAgcmV0dXJuIFRydWUNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYie3BhdGgubmFtZX0gaXMgbm90IGZyb3plbi4gUmVwbGFjZSB0aGUgRFJBRlQgc3RhdHVzIGxpbmUgd2l0aCAne0ZST1pFTl9NQVJLfSBvbiA8ZGF0ZT4nLiIpDQogICAgZGlnZXN0ID0gaGFzaGxpYi5zaGEyNTYocGF0aC5yZWFkX2J5dGVzKCkpLmhleGRpZ2VzdCgpDQogICAgcmVjb3JkID0gY2ZnLndvcmtfcm9vdCAvICJwcmVyZWdpc3RyYXRpb25fdmlzYS5zaGEyNTYiDQogICAgaWYgcmVjb3JkLmlzX2ZpbGUoKToNCiAgICAgICAgaWYgcmVjb3JkLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKS5zdHJpcCgpICE9IGRpZ2VzdDoNCiAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigicHJlcmVnaXN0cmF0aW9uX3Zpc2EubWQgY2hhbmdlZCBhZnRlciB0aGUgZmlyc3Qgc2NvcmluZyBzdGFnZS4gSXQgbXVzdCBzdGF5IGZyb3plbi4iKQ0KICAgIGVsc2U6DQogICAgICAgIHJlY29yZC53cml0ZV90ZXh0KGRpZ2VzdCArICJcbiIsIGVuY29kaW5nPSJ1dGYtOCIpDQogICAgcmV0dXJuIEZhbHNlDQoNCg0KZGVmIHdyaXRlX2Vudmlyb25tZW50KGNmZykgLT4gTm9uZToNCiAgICBpbmZvID0gew0KICAgICAgICAiY29tbWlzc2lvbmluZyI6IF9fdmVyc2lvbl9fLA0KICAgICAgICAicHJvdG9jb2wiOiAidmlzYSB2MiIsDQogICAgICAgICJweXRob24iOiBzeXMudmVyc2lvbiwNCiAgICAgICAgInBsYXRmb3JtIjogcGxhdGZvcm0ucGxhdGZvcm0oKSwNCiAgICAgICAgImNvbmZpZ19oYXNoIjogY2ZnLmhhc2gsDQogICAgICAgICJkYXRhX3Jvb3QiOiBzdHIoY2ZnLmRhdGFfcm9vdCksDQogICAgICAgICJ3b3JrX3Jvb3QiOiBzdHIoY2ZnLndvcmtfcm9vdCksDQogICAgICAgICJjYWNoZV9yb290Ijogc3RyKGNmZy5jYWNoZV9yb290KSwNCiAgICB9DQogICAgZm9yIG5hbWUgaW4gKCJudW1weSIsICJzY2lweSIsICJwYW5kYXMiLCAic2tsZWFybiIsICJtYXRwbG90bGliIiwgIlBJTCIsICJhbm9tYWxpYiIsICJ0aW1tIik6DQogICAgICAgIHRyeToNCiAgICAgICAgICAgIGluZm9bbmFtZV0gPSBnZXRhdHRyKF9faW1wb3J0X18obmFtZSksICJfX3ZlcnNpb25fXyIsICJ1bmtub3duIikNCiAgICAgICAgZXhjZXB0IEltcG9ydEVycm9yOg0KICAgICAgICAgICAgaW5mb1tuYW1lXSA9IE5vbmUNCiAgICB0cnk6DQogICAgICAgIGltcG9ydCB0b3JjaA0KDQogICAgICAgIGluZm9bInRvcmNoIl0gPSB0b3JjaC5fX3ZlcnNpb25fXw0KICAgICAgICBpbmZvWyJjdWRhIl0gPSBib29sKHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCkpDQogICAgICAgIGluZm9bImdwdSJdID0gdG9yY2guY3VkYS5nZXRfZGV2aWNlX25hbWUoMCkgaWYgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKSBlbHNlIE5vbmUNCiAgICBleGNlcHQgSW1wb3J0RXJyb3I6DQogICAgICAgIGluZm9bInRvcmNoIl0gPSBOb25lDQogICAgd3JpdGVfanNvbihjZmcud29ya19yb290IC8gImVudmlyb25tZW50Lmpzb24iLCBpbmZvKQ0KDQoNCmRlZiBfcGFyc2VyKCkgLT4gYXJncGFyc2UuQXJndW1lbnRQYXJzZXI6DQogICAgcGFyc2VyID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIocHJvZz0iY29tbWlzc2lvbmluZy52aXNhIikNCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCJzdGFnZSIsIGNob2ljZXM9U1RBR0VTKQ0KICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tY29uZmlnIiwgZGVmYXVsdD1Ob25lKQ0KICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tZGF0YS1yb290IiwgZGVmYXVsdD1Ob25lKQ0KICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0td29yay1yb290IiwgZGVmYXVsdD1Ob25lKQ0KICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tY2FjaGUtcm9vdCIsIGRlZmF1bHQ9Tm9uZSkNCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXJhdy1yb290IiwgZGVmYXVsdD1Ob25lLCBoZWxwPSJFeHRyYWN0ZWQgVmlzQV8yMDIyMDkyMi50YXIgKGNvbnZlcnQgb25seSkuIikNCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWNsYXNzZXMiLCBuYXJncz0iKiIsIGRlZmF1bHQ9Tm9uZSkNCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWRldmljZSIsIGRlZmF1bHQ9Tm9uZSkNCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWV4cGxvcmF0b3J5IiwgYWN0aW9uPSJzdG9yZV90cnVlIikNCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWUwLXBhcnQiLCBjaG9pY2VzPVsiYSIsICJiIiwgImFsbCJdLCBkZWZhdWx0PSJhbGwiKQ0KICAgIHJldHVybiBwYXJzZXINCg0KDQpkZWYgbWFpbihhcmd2OiBsaXN0W3N0cl0gfCBOb25lID0gTm9uZSkgLT4gTm9uZToNCiAgICBsb2dnaW5nLmJhc2ljQ29uZmlnKGxldmVsPWxvZ2dpbmcuSU5GTywgZm9ybWF0PSIlKGFzY3RpbWUpcyAlKGxldmVsbmFtZSlzICUobWVzc2FnZSlzIikNCiAgICBhcmdzID0gX3BhcnNlcigpLnBhcnNlX2FyZ3MoYXJndikNCiAgICBjZmcgPSBsb2FkX3Zpc2FfY29uZmlnKGFyZ3MuY29uZmlnLCBhcmdzLmRhdGFfcm9vdCwgYXJncy53b3JrX3Jvb3QsIGFyZ3MuY2FjaGVfcm9vdCkNCiAgICBjbGFzc2VzID0gY2ZnLnNlbGVjdF9jbGFzc2VzKGFyZ3MuY2xhc3NlcykNCiAgICBjZmcud29ya19yb290Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkNCiAgICB3cml0ZV9lbnZpcm9ubWVudChjZmcpDQogICAgbG9nZ2luZy5pbmZvKCJWaXNBIGNvbmZpZyBoYXNoICVzIiwgY2ZnLmhhc2gpDQogICAgc3RhZ2UgPSBhcmdzLnN0YWdlDQogICAgbm90X2Zyb3plbiA9IHJlcXVpcmVfZnJvemVuX3ByZXJlZ2lzdHJhdGlvbihjZmcsIGFyZ3MuZXhwbG9yYXRvcnkpIGlmIHN0YWdlIGluIFNDT1JJTkdfU1RBR0VTIGVsc2UgRmFsc2UNCiAgICBpZiBzdGFnZSA9PSAiY29udmVydCI6DQogICAgICAgIGZyb20gY29tbWlzc2lvbmluZy52aXNhLmRhdGEgaW1wb3J0IGNvbnZlcnQNCg0KICAgICAgICBpZiBub3QgYXJncy5yYXdfcm9vdDoNCiAgICAgICAgICAgIHJhaXNlIFN5c3RlbUV4aXQoImNvbnZlcnQgbmVlZHMgLS1yYXctcm9vdCBwb2ludGluZyBhdCB0aGUgZXh0cmFjdGVkIFZpc0EgYXJjaGl2ZS4iKQ0KICAgICAgICBjb3VudHMgPSBjb252ZXJ0KGFyZ3MucmF3X3Jvb3QsIGNmZy5kYXRhX3Jvb3QpDQogICAgICAgIGxvZ2dpbmcuaW5mbygiQ29udmVydGVkIGNvdW50czogJXMiLCBjb3VudHMpDQogICAgICAgIHJldHVybg0KICAgIGlmIHN0YWdlIGluIHsicHJlcGFyZSIsICJhbGwifToNCiAgICAgICAgZnJvbSBjb21taXNzaW9uaW5nLnZpc2EuZGF0YSBpbXBvcnQgcHJlcGFyZQ0KDQogICAgICAgIHByZXBhcmUoY2ZnLCBjbGFzc2VzKQ0KICAgIGlmIHN0YWdlIGluIHsiZmVhdHVyZXMiLCAiYWxsIn06DQogICAgICAgIGZyb20gY29tbWlzc2lvbmluZy52aXNhLmNhY2hlcyBpbXBvcnQgZXh0cmFjdA0KDQogICAgICAgIGV4dHJhY3QoY2ZnLCBjbGFzc2VzLCBkZXZpY2U9YXJncy5kZXZpY2UpDQogICAgaWYgc3RhZ2UgaW4geyJkaXN0YW5jZXMiLCAiYWxsIn06DQogICAgICAgIGZyb20gY29tbWlzc2lvbmluZy52aXNhLmNhY2hlcyBpbXBvcnQgY29tcHV0ZV9kaXN0YW5jZXMNCg0KICAgICAgICBjb21wdXRlX2Rpc3RhbmNlcyhjZmcsIGNsYXNzZXMsIGRldmljZT1hcmdzLmRldmljZSkNCiAgICBpZiBzdGFnZSBpbiB7ImUwIiwgImFsbCJ9Og0KICAgICAgICBmcm9tIGNvbW1pc3Npb25pbmcudmlzYS5lMCBpbXBvcnQgcnVuX2UwDQoNCiAgICAgICAgZ2F0ZSA9IHJ1bl9lMChjZmcsIGNsYXNzZXMsIHBhcnQ9YXJncy5lMF9wYXJ0IGlmIHN0YWdlID09ICJlMCIgZWxzZSAiYWxsIikNCiAgICAgICAgaWYgc3RhZ2UgPT0gImFsbCIgYW5kIGdhdGVbInN0YXR1cyJdICE9ICJQQVNTIiBhbmQgbm90IGFyZ3MuZXhwbG9yYXRvcnk6DQogICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIkUwIGRpZCBub3QgcGFzcy4gVGhlIHJlbWFpbmluZyBzdGFnZXMgd2VyZSBub3Qgc3RhcnRlZC4iKQ0KICAgIGlmIHN0YWdlIGluIHsicnVuIiwgImFsbCJ9Og0KICAgICAgICBmcm9tIGNvbW1pc3Npb25pbmcudmlzYS5ydW4gaW1wb3J0IHJ1bg0KDQogICAgICAgIHJ1bihjZmcsIGNsYXNzZXMsIGV4cGxvcmF0b3J5PWFyZ3MuZXhwbG9yYXRvcnksIGZvcmNlX2V4cGxvcmF0b3J5PW5vdF9mcm96ZW4pDQogICAgaWYgc3RhZ2UgaW4geyJleGNoYW5nZSIsICJhbGwifToNCiAgICAgICAgZnJvbSBjb21taXNzaW9uaW5nLnZpc2EuZXhjaGFuZ2UgaW1wb3J0IHJ1bl9leGNoYW5nZQ0KDQogICAgICAgIHJ1bl9leGNoYW5nZShjZmcsIGNsYXNzZXMpDQogICAgaWYgc3RhZ2UgaW4geyJidWRnZXQiLCAiYWxsIn06DQogICAgICAgIGZyb20gY29tbWlzc2lvbmluZy52aXNhLmJ1ZGdldCBpbXBvcnQgcnVuX2J1ZGdldA0KDQogICAgICAgIHJ1bl9idWRnZXQoY2ZnLCBjbGFzc2VzKQ0KICAgIGlmIHN0YWdlIGluIHsiYW5hbHl6ZSIsICJhbGwifToNCiAgICAgICAgZnJvbSBjb21taXNzaW9uaW5nLnZpc2EuYW5hbHlzaXMgaW1wb3J0IGFuYWx5emUNCg0KICAgICAgICBhbmFseXplKGNmZykNCiAgICBpZiBzdGFnZSBpbiB7InJlcG9ydCIsICJhbGwifToNCiAgICAgICAgZnJvbSBjb21taXNzaW9uaW5nLnZpc2EuYW5hbHlzaXMgaW1wb3J0IGxvYWRfYW5hbHlzaXMNCiAgICAgICAgZnJvbSBjb21taXNzaW9uaW5nLnZpc2EucmVwb3J0IGltcG9ydCBidWlsZF9yZXBvcnQNCiAgICAgICAgZnJvbSBjb21taXNzaW9uaW5nLnZpc2EucnVuIGltcG9ydCByZWFkX2dhdGUNCg0KICAgICAgICBmcmFtZXMgPSBsb2FkX2FuYWx5c2lzKGNmZy53b3JrX3Jvb3QgLyAiYW5hbHlzaXMiKQ0KICAgICAgICBwYXRoID0gYnVpbGRfcmVwb3J0KGNmZywgZnJhbWVzLCByZWFkX2dhdGUoY2ZnKSkNCiAgICAgICAgbG9nZ2luZy5pbmZvKCJSZXBvcnQgd3JpdHRlbiB0byAlcyIsIHBhdGgpDQoNCg0KaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoNCiAgICBtYWluKCkNCg==", "commissioning/visa/analysis.py": "IiIiQ29uZmlybWF0b3J5IGFuYWx5c2lzIG9mIHByZXJlZ2lzdHJhdGlvbl92aXNhLm1kOiBIMS1IMywgUDEtUDMsIE4qLCBmZWFzaWJpbGl0eSwgc3RvcHBpbmcsIGJ1ZGdldCBzcGxpdC4iIiINCg0KZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucw0KDQppbXBvcnQganNvbg0KZnJvbSBwYXRobGliIGltcG9ydCBQYXRoDQoNCmltcG9ydCBudW1weSBhcyBucA0KaW1wb3J0IHBhbmRhcyBhcyBwZA0KZnJvbSBzY2lweS5zdGF0cyBpbXBvcnQgYmV0YWJpbm9tLCBiaW5vbQ0KDQpmcm9tIGNvbW1pc3Npb25pbmcuYW5hbHlzaXMgaW1wb3J0IGF1bGMsIG5fc3Rhciwgc3RvcF9jYXRlZ29yeQ0KZnJvbSBjb21taXNzaW9uaW5nLmNhbGlicmF0aW9uIGltcG9ydCBmYWxzZV9yZWplY3RfYnVkZ2V0DQpmcm9tIGNvbW1pc3Npb25pbmcuaW9fdXRpbHMgaW1wb3J0IGF0b21pY19wYXJxdWV0LCB3cml0ZV9qc29uDQpmcm9tIGNvbW1pc3Npb25pbmcuc3RhdHMgaW1wb3J0IGJvb3RzdHJhcF9tZWRpYW5faW50ZXJ2YWwsIGNvbXBhcmVfc3RyYXRlZ2llcw0KZnJvbSBjb21taXNzaW9uaW5nLnN0b3BwaW5nIGltcG9ydCBuX3N0b3ANCmZyb20gY29tbWlzc2lvbmluZy52aXNhLmJ1ZGdldCBpbXBvcnQgYnVkZ2V0X2h5cG90aGVzaXMsIHN1bW1hcml6ZV9idWRnZXQNCmZyb20gY29tbWlzc2lvbmluZy52aXNhLmNvbmZpZyBpbXBvcnQgRVhQRUNURURfQ09VTlRTLCBSVUxFUywgVmlzYUNvbmZpZw0KZnJvbSBjb21taXNzaW9uaW5nLnZpc2EuZXhjaGFuZ2UgaW1wb3J0IGxvYWRfZXhjaGFuZ2UNCmZyb20gY29tbWlzc2lvbmluZy52aXNhLnJ1biBpbXBvcnQgbG9hZF9tZXRyaWNzDQoNCkZQUl9DT0xVTU4gPSB7Im9mZiI6ICJmcHJfb2ZmIiwgIkgiOiAiZnByX2gifQ0KQ1JJVEVSSU9OX0NPTFVNTiA9IHsib2ZmIjogImNyaXRlcmlvbl9vZmYiLCAiSCI6ICJjcml0ZXJpb25faCJ9DQpUSEVPUllfTUVBTiA9IHsiUjIwIjogMSAvIDIxLCAiUjYwIjogMSAvIDYxfQ0KVEhFT1JZX0VYQ0VFRCA9IHsiUjIwIjogMC45NSoqMjAsICJSNjAiOiAwLjk1Kio2MH0NClAyX0JBTkQgPSB7IlIyMCI6ICgwLjAzLCAwLjA3KSwgIlI2MCI6ICgwLjAwNSwgMC4wMyl9DQoNCg0KZGVmIHN1bW1hcml6ZShtZXRyaWNzOiBwZC5EYXRhRnJhbWUsIGNmZzogVmlzYUNvbmZpZykgLT4gcGQuRGF0YUZyYW1lOg0KICAgIHJvd3MgPSBbXQ0KICAgIGZvciAoY2xhc3NfbmFtZSwgZ3JvdXAsIHN0cmF0ZWd5LCBydWxlKSwgc3ViIGluIG1ldHJpY3MuZ3JvdXBieShbImNsYXNzX25hbWUiLCAiZ3JvdXAiLCAic3RyYXRlZ3kiLCAicnVsZSJdLCBzb3J0PVRydWUpOg0KICAgICAgICByZWNvcmQgPSB7ImNsYXNzX25hbWUiOiBjbGFzc19uYW1lLCAiZ3JvdXAiOiBncm91cCwgInN0cmF0ZWd5Ijogc3RyYXRlZ3ksICJydWxlIjogcnVsZX0NCiAgICAgICAgcGVyX3JlcCA9IHsiYXVsY190cHIiOiBbXSwgImF1bGNfZnByX29mZiI6IFtdLCAiYXVsY19mcHJfaCI6IFtdfQ0KICAgICAgICBmb3IgX3JlcCwgcmVwX2RmIGluIHN1Yi5ncm91cGJ5KCJyZXAiKToNCiAgICAgICAgICAgIG9yZGVyZWQgPSByZXBfZGYuc29ydF92YWx1ZXMoIk4iKQ0KICAgICAgICAgICAgcGVyX3JlcFsiYXVsY190cHIiXS5hcHBlbmQoYXVsYyhvcmRlcmVkWyJOIl0sIG9yZGVyZWRbInRwciJdLCBjZmcuYXVsY19uX21heCkpDQogICAgICAgICAgICBwZXJfcmVwWyJhdWxjX2Zwcl9vZmYiXS5hcHBlbmQoYXVsYyhvcmRlcmVkWyJOIl0sIG9yZGVyZWRbImZwcl9vZmYiXSwgY2ZnLmF1bGNfbl9tYXgpKQ0KICAgICAgICAgICAgcGVyX3JlcFsiYXVsY19mcHJfaCJdLmFwcGVuZChhdWxjKG9yZGVyZWRbIk4iXSwgb3JkZXJlZFsiZnByX2giXSwgY2ZnLmF1bGNfbl9tYXgpKQ0KICAgICAgICBmb3Iga2V5LCB2YWx1ZXMgaW4gcGVyX3JlcC5pdGVtcygpOg0KICAgICAgICAgICAgcmVjb3JkW2tleV0gPSBmbG9hdChucC5tZWFuKHZhbHVlcykpDQogICAgICAgICAgICByZWNvcmRbZiJ7a2V5fV9pcXIiXSA9IGZsb2F0KG5wLnN1YnRyYWN0KCpucC5xdWFudGlsZSh2YWx1ZXMsIFswLjc1LCAwLjI1XSkpKQ0KICAgICAgICBncmlkID0gW2ludChuKSBmb3IgbiBpbiBzb3J0ZWQoc3ViWyJOIl0udW5pcXVlKCkpXQ0KICAgICAgICBmb3Igc291cmNlLCBjb2x1bW4gaW4gQ1JJVEVSSU9OX0NPTFVNTi5pdGVtcygpOg0KICAgICAgICAgICAgcGFzc2VkID0gW2ludChzdWIubG9jW3N1YlsiTiJdID09IG4sIGNvbHVtbl0uc3VtKCkpIGZvciBuIGluIGdyaWRdDQogICAgICAgICAgICBzdGFyID0gbl9zdGFyKGdyaWQsIHBhc3NlZCwgY2ZnLm5fc3Rhcl9taW5fcmVwZWF0cykNCiAgICAgICAgICAgIHJlY29yZFtmIm5fc3Rhcl97c291cmNlfSJdID0gZmxvYXQoIm5hbiIpIGlmIHN0YXJbImNlbnNvcmVkIl0gZWxzZSBmbG9hdChzdGFyWyJuX3N0YXIiXSkNCiAgICAgICAgICAgIHJlY29yZFtmImNlbnNvcmVkX3tzb3VyY2V9Il0gPSBib29sKHN0YXJbImNlbnNvcmVkIl0pDQogICAgICAgICAgICByZWNvcmRbZiJwYXNzaW5nX2Z1bGxfcG9vbF97c291cmNlfSJdID0gcGFzc2VkWy0xXQ0KICAgICAgICByZWNvcmRbIm1lYW5fZnByX29mZiJdID0gZmxvYXQoc3ViWyJmcHJfb2ZmIl0ubWVhbigpKQ0KICAgICAgICByZWNvcmRbIm1lYW5fZnByX2giXSA9IGZsb2F0KHN1YlsiZnByX2giXS5tZWFuKCkpDQogICAgICAgIHJlY29yZFsicG9vbF9zaXplIl0gPSBpbnQoc3ViWyJwb29sX3NpemUiXS5pbG9jWzBdKQ0KICAgICAgICByZWNvcmRbIm5fcmVwcyJdID0gaW50KHN1YlsicmVwIl0ubnVuaXF1ZSgpKQ0KICAgICAgICByb3dzLmFwcGVuZChyZWNvcmQpDQogICAgcmV0dXJuIHBkLkRhdGFGcmFtZShyb3dzKQ0KDQoNCmRlZiBfY29tcGFyaXNvbihzdW1tYXJ5OiBwZC5EYXRhRnJhbWUsIGNmZzogVmlzYUNvbmZpZywgcnVsZTogc3RyLCBzb3VyY2U6IHN0cikgLT4gZGljdDoNCiAgICBmcmFtZSA9IHN1bW1hcnlbc3VtbWFyeVsicnVsZSJdID09IHJ1bGVdW1siY2xhc3NfbmFtZSIsICJzdHJhdGVneSIsICJhdWxjX3RwciIsIGYiYXVsY19mcHJfe3NvdXJjZS5sb3dlcigpfSJdXQ0KICAgIGZyYW1lID0gZnJhbWUucmVuYW1lKGNvbHVtbnM9e2YiYXVsY19mcHJfe3NvdXJjZS5sb3dlcigpfSI6ICJhdWxjX2ZwciJ9KQ0KICAgIHJldHVybiBjb21wYXJlX3N0cmF0ZWdpZXMoZnJhbWUsIGNmZy5mcHJfYXVsY19tYXJnaW4sIGNmZy5ib290c3RyYXBfcmVwbGljYXRlcywgY2ZnLmJvb3RzdHJhcF9zZWVkKQ0KDQoNCmRlZiB0cHJfY29zdChtZXRyaWNzOiBwZC5EYXRhRnJhbWUsIGNmZzogVmlzYUNvbmZpZywgbjogaW50LCBzdHJhdGVneTogc3RyID0gInJhbmRvbSIpIC0+IHBkLkRhdGFGcmFtZToNCiAgICBzdWIgPSBtZXRyaWNzWyhtZXRyaWNzWyJOIl0gPT0gbikgJiAobWV0cmljc1sic3RyYXRlZ3kiXSA9PSBzdHJhdGVneSldDQogICAgd2lkZSA9IHN1Yi5waXZvdF90YWJsZShpbmRleD1bImNsYXNzX25hbWUiLCAicmVwIl0sIGNvbHVtbnM9InJ1bGUiLCB2YWx1ZXM9InRwciIpDQogICAgaWYgKHdpZGVbIlI2MCJdID4gd2lkZVsiUjIwIl0gKyAxZS0xMikuYW55KCk6DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiVFBSIHVuZGVyIFI2MCBleGNlZWRlZCBUUFIgdW5kZXIgUjIwOyBtYXgoQzYwKSA+PSBtYXgoQzIwKSBtdXN0IGhvbGQuIikNCiAgICBwZXJfY2xhc3MgPSAod2lkZVsiUjIwIl0gLSB3aWRlWyJSNjAiXSkuZ3JvdXBieSgiY2xhc3NfbmFtZSIpLm1lYW4oKQ0KICAgIHJldHVybiBwZXJfY2xhc3MucmVuYW1lKCJ0cHJfY29zdCIpLnJlc2V0X2luZGV4KCkNCg0KDQpkZWYgaDMobWV0cmljczogcGQuRGF0YUZyYW1lLCBjZmc6IFZpc2FDb25maWcpIC0+IGRpY3Q6DQogICAgZnVsbCA9IHRwcl9jb3N0KG1ldHJpY3MsIGNmZywgY2ZnLnBvb2xfc2l6ZSkNCiAgICBib290ID0gYm9vdHN0cmFwX21lZGlhbl9pbnRlcnZhbChmdWxsWyJ0cHJfY29zdCJdLnRvX251bXB5KCksIGNmZy5ib290c3RyYXBfcmVwbGljYXRlcywgY2ZnLmJvb3RzdHJhcF9zZWVkICsgMikNCiAgICBzZWNvbmRhcnkgPSB0cHJfY29zdChtZXRyaWNzLCBjZmcsIDMyKQ0KICAgIHJldHVybiB7DQogICAgICAgICJxdWFudGl0eSI6ICJtZWFuIG92ZXIgcmVwZXRpdGlvbnMgb2YgVFBSKFIyMCkgLSBUUFIoUjYwKSBhdCBOID0gfFB8LCByYW5kb20gc3RyYXRlZ3kiLA0KICAgICAgICAicGVyX2NsYXNzIjogZGljdCh6aXAoZnVsbFsiY2xhc3NfbmFtZSJdLCBmdWxsWyJ0cHJfY29zdCJdLmFzdHlwZShmbG9hdCkpKSwNCiAgICAgICAgIm1lZGlhbiI6IGJvb3RbIm1lZGlhbiJdLA0KICAgICAgICAiYm9vdHN0cmFwIjogYm9vdCwNCiAgICAgICAgIm1hcmdpbiI6IGNmZy50cHJfY29zdF9tYXJnaW4sDQogICAgICAgICJyNjBfcmVjb21tZW5kZWQiOiBib29sKGJvb3RbIm1lZGlhbiJdIDw9IGNmZy50cHJfY29zdF9tYXJnaW4pLA0KICAgICAgICAic2Vjb25kYXJ5X24zMiI6IHsNCiAgICAgICAgICAgICJwZXJfY2xhc3MiOiBkaWN0KHppcChzZWNvbmRhcnlbImNsYXNzX25hbWUiXSwgc2Vjb25kYXJ5WyJ0cHJfY29zdCJdLmFzdHlwZShmbG9hdCkpKSwNCiAgICAgICAgICAgICJtZWRpYW4iOiBmbG9hdChzZWNvbmRhcnlbInRwcl9jb3N0Il0ubWVkaWFuKCkpLA0KICAgICAgICB9LA0KICAgIH0NCg0KDQpkZWYgcDJfY2FsaWJyYXRpb24obWV0cmljczogcGQuRGF0YUZyYW1lKSAtPiB0dXBsZVtwZC5EYXRhRnJhbWUsIGRpY3RdOg0KICAgIHBlcl9jbGFzcyA9ICgNCiAgICAgICAgbWV0cmljcy5ncm91cGJ5KFsiY2xhc3NfbmFtZSIsICJydWxlIl0pDQogICAgICAgIC5hZ2cobWVhbl9mcHJfaD0oImZwcl9oIiwgIm1lYW4iKSwgZnJhY3Rpb25fZnByX2hfYWJvdmVfMDU9KCJmcHJfaCIsIGxhbWJkYSB4OiBmbG9hdChucC5tZWFuKHggPiAwLjA1KSkpKQ0KICAgICAgICAucmVzZXRfaW5kZXgoKQ0KICAgICkNCiAgICBvdXQgPSB7fQ0KICAgIGZvciBydWxlIGluIFJVTEVTOg0KICAgICAgICBzdWIgPSBwZXJfY2xhc3NbcGVyX2NsYXNzWyJydWxlIl0gPT0gcnVsZV0NCiAgICAgICAgbWVkaWFuID0gZmxvYXQoc3ViWyJtZWFuX2Zwcl9oIl0ubWVkaWFuKCkpDQogICAgICAgIGxvdywgaGlnaCA9IFAyX0JBTkRbcnVsZV0NCiAgICAgICAgb3V0W3J1bGVdID0gew0KICAgICAgICAgICAgIm1lZGlhbl9vdmVyX2NsYXNzZXNfb2ZfbWVhbl9mcHJfaCI6IG1lZGlhbiwNCiAgICAgICAgICAgICJiYW5kIjogW2xvdywgaGlnaF0sDQogICAgICAgICAgICAiaW5zaWRlX2JhbmQiOiBib29sKGxvdyA8PSBtZWRpYW4gPD0gaGlnaCksDQogICAgICAgICAgICAidGhlb3J5X21lYW4iOiBUSEVPUllfTUVBTltydWxlXSwNCiAgICAgICAgICAgICJmcmFjdGlvbl9mcHJfaF9hYm92ZV8wNV9tZWFuX292ZXJfY2xhc3NlcyI6IGZsb2F0KHN1YlsiZnJhY3Rpb25fZnByX2hfYWJvdmVfMDUiXS5tZWFuKCkpLA0KICAgICAgICAgICAgInRoZW9yeV9mcmFjdGlvbl9jb25kaXRpb25hbF9mcHJfYWJvdmVfMDUiOiBUSEVPUllfRVhDRUVEW3J1bGVdLA0KICAgICAgICB9DQogICAgYnlfbiA9ICgNCiAgICAgICAgbWV0cmljcy5ncm91cGJ5KFsicnVsZSIsICJOIl0pDQogICAgICAgIC5hZ2cobWVhbl9mcHJfaD0oImZwcl9oIiwgIm1lYW4iKSwgbWVhbl9mcHJfb2ZmPSgiZnByX29mZiIsICJtZWFuIiksDQogICAgICAgICAgICAgZnJhY3Rpb25fZnByX2hfYWJvdmVfMDU9KCJmcHJfaCIsIGxhbWJkYSB4OiBmbG9hdChucC5tZWFuKHggPiAwLjA1KSkpKQ0KICAgICAgICAucmVzZXRfaW5kZXgoKQ0KICAgICkNCiAgICBvdXRbImJ5X04iXSA9IGJ5X24udG9fZGljdChvcmllbnQ9InJlY29yZHMiKQ0KICAgIHJldHVybiBwZXJfY2xhc3MsIG91dA0KDQoNCmRlZiBwM19zaGlmdChtZXRyaWNzOiBwZC5EYXRhRnJhbWUsIGV4Y2hhbmdlOiBwZC5EYXRhRnJhbWUgfCBOb25lKSAtPiB0dXBsZVtwZC5EYXRhRnJhbWUgfCBOb25lLCBkaWN0XToNCiAgICBpZiBleGNoYW5nZSBpcyBOb25lOg0KICAgICAgICByZXR1cm4gTm9uZSwgeyJhdmFpbGFibGUiOiBGYWxzZX0NCiAgICByMjAgPSBtZXRyaWNzW21ldHJpY3NbInJ1bGUiXSA9PSAiUjIwIl0uZ3JvdXBieSgiY2xhc3NfbmFtZSIpLmFnZygNCiAgICAgICAgbWVhbl9mcHJfb2ZmPSgiZnByX29mZiIsICJtZWFuIiksIG1lYW5fZnByX2g9KCJmcHJfaCIsICJtZWFuIikNCiAgICApDQogICAgZnJhbWUgPSBleGNoYW5nZS5zZXRfaW5kZXgoImNsYXNzX25hbWUiKS5qb2luKHIyMCkucmVzZXRfaW5kZXgoKQ0KICAgIGZyYW1lWyJmcHJfZ2FwX29mZl9taW51c19oIl0gPSBmcmFtZVsibWVhbl9mcHJfb2ZmIl0gLSBmcmFtZVsibWVhbl9mcHJfaCJdDQogICAgc2hpZnRlZCA9IGZyYW1lW2ZyYW1lWyJzaGlmdF9kZXRlY3RlZCJdXQ0KICAgIHJldHVybiBmcmFtZSwgew0KICAgICAgICAiYXZhaWxhYmxlIjogVHJ1ZSwNCiAgICAgICAgImNsYXNzZXNfd2l0aF9zaGlmdCI6IHNoaWZ0ZWRbImNsYXNzX25hbWUiXS50b2xpc3QoKSwNCiAgICAgICAgInByZWRpY3Rpb25faG9sZHNfcGVyX3NoaWZ0ZWRfY2xhc3MiOiB7DQogICAgICAgICAgICBzdHIocm93LmNsYXNzX25hbWUpOiBib29sKHJvdy5mcHJfZ2FwX29mZl9taW51c19oID4gMCkgZm9yIHJvdyBpbiBzaGlmdGVkLml0ZXJ0dXBsZXMoKQ0KICAgICAgICB9LA0KICAgICAgICAibWVkaWFuX2dhcF9zaGlmdGVkIjogZmxvYXQoc2hpZnRlZFsiZnByX2dhcF9vZmZfbWludXNfaCJdLm1lZGlhbigpKSBpZiBsZW4oc2hpZnRlZCkgZWxzZSBOb25lLA0KICAgICAgICAibWVkaWFuX2dhcF9ub3Rfc2hpZnRlZCI6IGZsb2F0KGZyYW1lLmxvY1t+ZnJhbWVbInNoaWZ0X2RldGVjdGVkIl0sICJmcHJfZ2FwX29mZl9taW51c19oIl0ubWVkaWFuKCkpLA0KICAgIH0NCg0KDQpkZWYgbl9zdGFyX3RhYmxlKHN1bW1hcnk6IHBkLkRhdGFGcmFtZSkgLT4gcGQuRGF0YUZyYW1lOg0KICAgIGtlZXAgPSBbImNsYXNzX25hbWUiLCAiZ3JvdXAiLCAic3RyYXRlZ3kiLCAicnVsZSJdDQogICAgZm9yIHNvdXJjZSBpbiAoIm9mZiIsICJIIik6DQogICAgICAgIGtlZXAgKz0gW2Yibl9zdGFyX3tzb3VyY2V9IiwgZiJjZW5zb3JlZF97c291cmNlfSIsIGYicGFzc2luZ19mdWxsX3Bvb2xfe3NvdXJjZX0iXQ0KICAgIHJldHVybiBzdW1tYXJ5W2tlZXBdLnNvcnRfdmFsdWVzKFsicnVsZSIsICJzdHJhdGVneSIsICJjbGFzc19uYW1lIl0pLnJlc2V0X2luZGV4KGRyb3A9VHJ1ZSkNCg0KDQpkZWYgcDFfZmVhc2liaWxpdHkoc3VtbWFyeTogcGQuRGF0YUZyYW1lLCBtZXRyaWNzOiBwZC5EYXRhRnJhbWUsIGNmZzogVmlzYUNvbmZpZykgLT4gZGljdDoNCiAgICBvdXQgPSB7fQ0KICAgIGZvciAocnVsZSwgc3RyYXRlZ3kpLCBzdWIgaW4gc3VtbWFyeS5ncm91cGJ5KFsicnVsZSIsICJzdHJhdGVneSJdKToNCiAgICAgICAgb3V0W2Yie3J1bGV9X3tzdHJhdGVneX0iXSA9IHsNCiAgICAgICAgICAgICJjZW5zb3JlZF9vZmYiOiBpbnQoc3ViWyJjZW5zb3JlZF9vZmYiXS5zdW0oKSksDQogICAgICAgICAgICAiY2Vuc29yZWRfSCI6IGludChzdWJbImNlbnNvcmVkX0giXS5zdW0oKSksDQogICAgICAgICAgICAibl9jbGFzc2VzIjogaW50KGxlbihzdWIpKSwNCiAgICAgICAgfQ0KICAgIGZ1bGwgPSBtZXRyaWNzWyhtZXRyaWNzWyJOIl0gPT0gY2ZnLnBvb2xfc2l6ZSkgJiAobWV0cmljc1sic3RyYXRlZ3kiXSA9PSAicmFuZG9tIikgJiAobWV0cmljc1sicnVsZSJdID09ICJSNjAiKV0NCiAgICB0cHJfb2sgPSBmdWxsLmdyb3VwYnkoImNsYXNzX25hbWUiKVsidHByIl0uYXBwbHkobGFtYmRhIHg6IGludChucC5zdW0oeCA+PSBjZmcucHJhY3RpY2FsX3RwcikpKQ0KICAgIGVsaWdpYmxlID0gc29ydGVkKHRwcl9va1t0cHJfb2sgPj0gY2ZnLm5fc3Rhcl9taW5fcmVwZWF0c10uaW5kZXgpDQogICAgcjYwID0gc3VtbWFyeVsoc3VtbWFyeVsicnVsZSJdID09ICJSNjAiKSAmIChzdW1tYXJ5WyJzdHJhdGVneSJdID09ICJyYW5kb20iKV0uc2V0X2luZGV4KCJjbGFzc19uYW1lIikNCiAgICBvdXRbIlI2MF9lbGlnaWJsZV9ieV90cHIiXSA9IGVsaWdpYmxlDQogICAgb3V0WyJSNjBfZWxpZ2libGVfcmVhY2hlZF9uX3N0YXJfb2ZmIl0gPSB7bmFtZTogYm9vbChub3QgcjYwLmxvY1tuYW1lLCAiY2Vuc29yZWRfb2ZmIl0pIGZvciBuYW1lIGluIGVsaWdpYmxlfQ0KICAgIHJldHVybiBvdXQNCg0KDQpkZWYgY2VpbGluZ190YWJsZShjZmc6IFZpc2FDb25maWcsIG1ldHJpY3M6IHBkLkRhdGFGcmFtZSkgLT4gcGQuRGF0YUZyYW1lOg0KICAgIHNpemVzID0gbWV0cmljcy5ncm91cGJ5KCJjbGFzc19uYW1lIikuYWdnKG5fdGVzdF9nb29kPSgibl90ZXN0X2dvb2QiLCAiZmlyc3QiKSwgbl9ob2xkb3V0PSgibl9ob2xkb3V0IiwgImZpcnN0IikpDQogICAgcm93cyA9IFtdDQogICAgZm9yIGNsYXNzX25hbWUsIGNvdW50cyBpbiBzaXplcy5pdGVycm93cygpOg0KICAgICAgICBmb3Igc291cmNlLCBuIGluICgoIm9mZiIsIGludChjb3VudHNbIm5fdGVzdF9nb29kIl0pKSwgKCJIIiwgaW50KGNvdW50c1sibl9ob2xkb3V0Il0pKSk6DQogICAgICAgICAgICBmb3IgcnVsZSwgbSBpbiAoKCJSMjAiLCBjZmcubl9jYWxpYnJhdGlvbl9zbWFsbCksICgiUjYwIiwgY2ZnLm5fY2FsaWJyYXRpb25fbGFyZ2UpKToNCiAgICAgICAgICAgICAgICBwID0gZmxvYXQoYmV0YWJpbm9tLmNkZihmYWxzZV9yZWplY3RfYnVkZ2V0KG4pLCBuLCAxLCBtKSkNCiAgICAgICAgICAgICAgICByb3dzLmFwcGVuZCgNCiAgICAgICAgICAgICAgICAgICAgew0KICAgICAgICAgICAgICAgICAgICAgICAgImNsYXNzX25hbWUiOiBjbGFzc19uYW1lLA0KICAgICAgICAgICAgICAgICAgICAgICAgImZwcl9zb3VyY2UiOiBzb3VyY2UsDQogICAgICAgICAgICAgICAgICAgICAgICAibl9nb29kIjogbiwNCiAgICAgICAgICAgICAgICAgICAgICAgICJidWRnZXQiOiBmYWxzZV9yZWplY3RfYnVkZ2V0KG4pLA0KICAgICAgICAgICAgICAgICAgICAgICAgInJ1bGUiOiBydWxlLA0KICAgICAgICAgICAgICAgICAgICAgICAgInBfcmVwX2Zwcl9vayI6IHAsDQogICAgICAgICAgICAgICAgICAgICAgICAiZXhwZWN0ZWRfcGFzc2luZ19yZXBzIjogY2ZnLm5fcmVwZWF0cyAqIHAsDQogICAgICAgICAgICAgICAgICAgICAgICAicF9hdF9sZWFzdF9taW5fcmVwc19pZl9pbmRlcGVuZGVudCI6IGZsb2F0KA0KICAgICAgICAgICAgICAgICAgICAgICAgICAgIGJpbm9tLnNmKGNmZy5uX3N0YXJfbWluX3JlcGVhdHMgLSAxLCBjZmcubl9yZXBlYXRzLCBwKQ0KICAgICAgICAgICAgICAgICAgICAgICAgKSwNCiAgICAgICAgICAgICAgICAgICAgfQ0KICAgICAgICAgICAgICAgICkNCiAgICByZXR1cm4gcGQuRGF0YUZyYW1lKHJvd3MpDQoNCg0KZGVmIHN0b3BwaW5nKG1ldHJpY3M6IHBkLkRhdGFGcmFtZSwgc3VtbWFyeTogcGQuRGF0YUZyYW1lLCBjZmc6IFZpc2FDb25maWcpIC0+IHR1cGxlW3BkLkRhdGFGcmFtZSwgcGQuRGF0YUZyYW1lXToNCiAgICBiYXNlID0gbWV0cmljc1ttZXRyaWNzWyJydWxlIl0gPT0gIlIyMCJdDQogICAgcm93cyA9IFtdDQogICAgZm9yIChjbGFzc19uYW1lLCBzdHJhdGVneSwgcmVwKSwgc3ViIGluIGJhc2UuZ3JvdXBieShbImNsYXNzX25hbWUiLCAic3RyYXRlZ3kiLCAicmVwIl0sIHNvcnQ9VHJ1ZSk6DQogICAgICAgIG9yZGVyZWQgPSBzdWIuc29ydF92YWx1ZXMoIk4iKQ0KICAgICAgICBncmlkID0gW2ludChuKSBmb3IgbiBpbiBvcmRlcmVkWyJOIl1dDQogICAgICAgIG1lZGlhbnMgPSBvcmRlcmVkWyJtZWRpYW5fY2FsaWJyYXRpb25fYzIwIl0udG9fbnVtcHkoZHR5cGU9ZmxvYXQpDQogICAgICAgIGZvciBkZWx0YSBpbiBjZmcuc3RvcHBpbmdfZGVsdGFzOg0KICAgICAgICAgICAgcm93cy5hcHBlbmQoDQogICAgICAgICAgICAgICAgeyJjbGFzc19uYW1lIjogY2xhc3NfbmFtZSwgInN0cmF0ZWd5Ijogc3RyYXRlZ3ksICJyZXAiOiBpbnQocmVwKSwgImRlbHRhIjogZmxvYXQoZGVsdGEpLA0KICAgICAgICAgICAgICAgICAibl9zdG9wIjogbl9zdG9wKGdyaWQsIG1lZGlhbnMsIGZsb2F0KGRlbHRhKSl9DQogICAgICAgICAgICApDQogICAgdGFibGUgPSBwZC5EYXRhRnJhbWUocm93cykNCiAgICBtYWluID0gdGFibGVbbnAuaXNjbG9zZSh0YWJsZVsiZGVsdGEiXSwgY2ZnLnN0b3BwaW5nX2RlbHRhX21haW4pXQ0KICAgIGNhdGVnb3JpZXMgPSBbXQ0KICAgIGZvciBydWxlIGluIFJVTEVTOg0KICAgICAgICBsb29rdXAgPSBzdW1tYXJ5W3N1bW1hcnlbInJ1bGUiXSA9PSBydWxlXS5zZXRfaW5kZXgoWyJjbGFzc19uYW1lIiwgInN0cmF0ZWd5Il0pDQogICAgICAgIGZvciAoY2xhc3NfbmFtZSwgc3RyYXRlZ3kpLCBzdWIgaW4gbWFpbi5ncm91cGJ5KFsiY2xhc3NfbmFtZSIsICJzdHJhdGVneSJdKToNCiAgICAgICAgICAgIHN0YXIgPSBsb29rdXAubG9jWyhjbGFzc19uYW1lLCBzdHJhdGVneSldDQogICAgICAgICAgICBjZW5zb3JlZCA9IGJvb2woc3RhclsiY2Vuc29yZWRfb2ZmIl0pDQogICAgICAgICAgICBsYWJlbHMgPSBbDQogICAgICAgICAgICAgICAgc3RvcF9jYXRlZ29yeShOb25lIGlmIHBkLmlzbmEodikgZWxzZSBpbnQodiksIE5vbmUgaWYgY2Vuc29yZWQgZWxzZSBpbnQoc3Rhclsibl9zdGFyX29mZiJdKSwgY2Vuc29yZWQpDQogICAgICAgICAgICAgICAgZm9yIHYgaW4gc3ViWyJuX3N0b3AiXQ0KICAgICAgICAgICAgXQ0KICAgICAgICAgICAgcmVjb3JkID0geyJjbGFzc19uYW1lIjogY2xhc3NfbmFtZSwgInN0cmF0ZWd5Ijogc3RyYXRlZ3ksICJyZWZlcmVuY2UiOiBmIk4qX3tydWxlfV9vZmYifQ0KICAgICAgICAgICAgZm9yIG5hbWUgaW4gKCJwcmVtYXR1cmUiLCAiY29uY29yZGFudCIsICJleGNlc3NpdmUiLCAibl9zdGFyX25vdF9yZWFjaGVkIiwgInJ1bGVfbm90X2ZpcmVkIik6DQogICAgICAgICAgICAgICAgcmVjb3JkW2YiZnJhY3Rpb25fe25hbWV9Il0gPSBmbG9hdChucC5tZWFuKFtsYWJlbCA9PSBuYW1lIGZvciBsYWJlbCBpbiBsYWJlbHNdKSkNCiAgICAgICAgICAgIGNhdGVnb3JpZXMuYXBwZW5kKHJlY29yZCkNCiAgICByZXR1cm4gdGFibGUsIHBkLkRhdGFGcmFtZShjYXRlZ29yaWVzKQ0KDQoNCmRlZiBhbmFseXplKGNmZzogVmlzYUNvbmZpZykgLT4gZGljdDoNCiAgICBtZXRyaWNzID0gbG9hZF9tZXRyaWNzKGNmZykNCiAgICBleHBlY3RlZF9yb3dzID0gbGVuKGNmZy5jbGFzc2VzKSAqIGNmZy5uX3JlcGVhdHMgKiBsZW4oY2ZnLnN0cmF0ZWdpZXMpICogbGVuKGNmZy5uX2dyaWQpICogbGVuKFJVTEVTKQ0KICAgIGlmIGxlbihtZXRyaWNzKSAhPSBleHBlY3RlZF9yb3dzOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJFeHBlY3RlZCB7ZXhwZWN0ZWRfcm93c30gdHJhamVjdG9yeSByb3dzLCBmb3VuZCB7bGVuKG1ldHJpY3MpfS4gRmluaXNoIHRoZSBydW4gZmlyc3QuIikNCiAgICBzdW1tYXJ5ID0gc3VtbWFyaXplKG1ldHJpY3MsIGNmZykNCiAgICBzdGF0aXN0aWNzID0gew0KICAgICAgICAicHJpbWFyeSI6IHsiSDFfSDJfUjIwX29mZmljaWFsIjogX2NvbXBhcmlzb24oc3VtbWFyeSwgY2ZnLCAiUjIwIiwgIm9mZiIpfSwNCiAgICAgICAgInNlY29uZGFyeSI6IHsNCiAgICAgICAgICAgICJSNjBfb2ZmaWNpYWwiOiBfY29tcGFyaXNvbihzdW1tYXJ5LCBjZmcsICJSNjAiLCAib2ZmIiksDQogICAgICAgICAgICAiUjIwX2hvbGRvdXQiOiBfY29tcGFyaXNvbihzdW1tYXJ5LCBjZmcsICJSMjAiLCAiSCIpLA0KICAgICAgICAgICAgIlI2MF9ob2xkb3V0IjogX2NvbXBhcmlzb24oc3VtbWFyeSwgY2ZnLCAiUjYwIiwgIkgiKSwNCiAgICAgICAgfSwNCiAgICAgICAgIkgzIjogaDMobWV0cmljcywgY2ZnKSwNCiAgICB9DQogICAgcDJfZnJhbWUsIHN0YXRpc3RpY3NbIlAyIl0gPSBwMl9jYWxpYnJhdGlvbihtZXRyaWNzKQ0KICAgIGV4Y2hhbmdlID0gbG9hZF9leGNoYW5nZShjZmcpDQogICAgcDNfZnJhbWUsIHN0YXRpc3RpY3NbIlAzIl0gPSBwM19zaGlmdChtZXRyaWNzLCBleGNoYW5nZSkNCiAgICBzdGF0aXN0aWNzWyJQMSJdID0gcDFfZmVhc2liaWxpdHkoc3VtbWFyeSwgbWV0cmljcywgY2ZnKQ0KICAgIHN0b3BfdGFibGUsIHN0b3Bfc3VtbWFyeSA9IHN0b3BwaW5nKG1ldHJpY3MsIHN1bW1hcnksIGNmZykNCiAgICBvdXQgPSBjZmcud29ya19yb290IC8gImFuYWx5c2lzIg0KICAgIG91dC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpDQogICAgYnVkZ2V0X3BhcnRzID0gc29ydGVkKChjZmcud29ya19yb290IC8gImJ1ZGdldCIpLmdsb2IoIioucGFycXVldCIpKQ0KICAgIGlmIGxlbihidWRnZXRfcGFydHMpID09IGxlbihjZmcuY2xhc3Nlcyk6DQogICAgICAgIGJ1ZGdldF9mcmFtZSA9IHBkLmNvbmNhdChbcGQucmVhZF9wYXJxdWV0KHApIGZvciBwIGluIGJ1ZGdldF9wYXJ0c10sIGlnbm9yZV9pbmRleD1UcnVlKQ0KICAgICAgICBidWRnZXRfc3VtbWFyeSwgYnVkZ2V0X2Jlc3QgPSBzdW1tYXJpemVfYnVkZ2V0KGJ1ZGdldF9mcmFtZSwgY2ZnLm5fc3Rhcl9taW5fcmVwZWF0cykNCiAgICAgICAgc3RhdGlzdGljc1siYnVkZ2V0Il0gPSBidWRnZXRfaHlwb3RoZXNpcyhidWRnZXRfYmVzdCkNCiAgICAgICAgYnVkZ2V0X3N1bW1hcnkudG9fY3N2KG91dCAvICJidWRnZXRfc3VtbWFyeS5jc3YiLCBpbmRleD1GYWxzZSkNCiAgICAgICAgYnVkZ2V0X2Jlc3QudG9fY3N2KG91dCAvICJidWRnZXRfYmVzdC5jc3YiLCBpbmRleD1GYWxzZSkNCiAgICBhdG9taWNfcGFycXVldChvdXQgLyAibWV0cmljcy5wYXJxdWV0IiwgbWV0cmljcykNCiAgICBzdW1tYXJ5LnRvX2NzdihvdXQgLyAic3VtbWFyeS5jc3YiLCBpbmRleD1GYWxzZSkNCiAgICBuX3N0YXJfdGFibGUoc3VtbWFyeSkudG9fY3N2KG91dCAvICJuX3N0YXIuY3N2IiwgaW5kZXg9RmFsc2UpDQogICAgcDJfZnJhbWUudG9fY3N2KG91dCAvICJwMl9jYWxpYnJhdGlvbi5jc3YiLCBpbmRleD1GYWxzZSkNCiAgICBpZiBwM19mcmFtZSBpcyBub3QgTm9uZToNCiAgICAgICAgcDNfZnJhbWUudG9fY3N2KG91dCAvICJwM19zaGlmdC5jc3YiLCBpbmRleD1GYWxzZSkNCiAgICBjZWlsaW5nX3RhYmxlKGNmZywgbWV0cmljcykudG9fY3N2KG91dCAvICJjZWlsaW5nLmNzdiIsIGluZGV4PUZhbHNlKQ0KICAgIHN0b3BfdGFibGUudG9fY3N2KG91dCAvICJzdG9wcGluZy5jc3YiLCBpbmRleD1GYWxzZSkNCiAgICBzdG9wX3N1bW1hcnkudG9fY3N2KG91dCAvICJzdG9wcGluZ19zdW1tYXJ5LmNzdiIsIGluZGV4PUZhbHNlKQ0KICAgIHN0YXRpc3RpY3NbImNvbmZpZ19oYXNoIl0gPSBjZmcuaGFzaA0KICAgIHN0YXRpc3RpY3NbImV4cGxvcmF0b3J5X3Jvd3MiXSA9IGludChtZXRyaWNzWyJleHBsb3JhdG9yeSJdLnN1bSgpKQ0KICAgIHdyaXRlX2pzb24ob3V0IC8gInN0YXRpc3RpY3MuanNvbiIsIGpzb24ubG9hZHMoanNvbi5kdW1wcyhzdGF0aXN0aWNzLCBkZWZhdWx0PV9qc29uX2RlZmF1bHQpKSkNCiAgICByZXR1cm4geyJtZXRyaWNzIjogbWV0cmljcywgInN1bW1hcnkiOiBzdW1tYXJ5LCAic3RhdGlzdGljcyI6IHN0YXRpc3RpY3MsICJhbmFseXNpc19kaXIiOiBvdXR9DQoNCg0KZGVmIF9qc29uX2RlZmF1bHQodmFsdWUpOg0KICAgIGlmIGlzaW5zdGFuY2UodmFsdWUsIChucC5pbnRlZ2VyLCkpOg0KICAgICAgICByZXR1cm4gaW50KHZhbHVlKQ0KICAgIGlmIGlzaW5zdGFuY2UodmFsdWUsIChucC5mbG9hdGluZywpKToNCiAgICAgICAgcmV0dXJuIGZsb2F0KHZhbHVlKQ0KICAgIGlmIGlzaW5zdGFuY2UodmFsdWUsIChucC5ib29sXywpKToNCiAgICAgICAgcmV0dXJuIGJvb2wodmFsdWUpDQogICAgcmFpc2UgVHlwZUVycm9yKGYiTm90IEpTT04gc2VyaWFsaXphYmxlOiB7dHlwZSh2YWx1ZSl9IikNCg0KDQpkZWYgbG9hZF9hbmFseXNpcyhhbmFseXNpc19kaXI6IFBhdGgpIC0+IGRpY3Q6DQogICAgZnJhbWVzID0ge30NCiAgICBmb3IgbmFtZSBpbiAoInN1bW1hcnkiLCAibl9zdGFyIiwgInAyX2NhbGlicmF0aW9uIiwgInAzX3NoaWZ0IiwgImNlaWxpbmciLCAic3RvcHBpbmdfc3VtbWFyeSIsICJidWRnZXRfYmVzdCIpOg0KICAgICAgICBwYXRoID0gYW5hbHlzaXNfZGlyIC8gZiJ7bmFtZX0uY3N2Ig0KICAgICAgICBpZiBwYXRoLmlzX2ZpbGUoKToNCiAgICAgICAgICAgIGZyYW1lc1tuYW1lXSA9IHBkLnJlYWRfY3N2KHBhdGgpDQogICAgZnJhbWVzWyJtZXRyaWNzIl0gPSBwZC5yZWFkX3BhcnF1ZXQoYW5hbHlzaXNfZGlyIC8gIm1ldHJpY3MucGFycXVldCIpDQogICAgZnJhbWVzWyJzdGF0aXN0aWNzIl0gPSBqc29uLmxvYWRzKChhbmFseXNpc19kaXIgLyAic3RhdGlzdGljcy5qc29uIikucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYtOCIpKQ0KICAgIHJldHVybiBmcmFtZXMNCg==", "commissioning/visa/budget.py": "IiIiU2VjdGlvbiAxMDogYSBmaXhlZCBidWRnZXQgQiBvZiBnb29kIHBhcnRzIGZyb20gVSwgc3BsaXQgaW50byBhIHJhbmRvbSBiYW5rIGFuZCBhIGNhbGlicmF0aW9uIHNldC4iIiINCg0KZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucw0KDQppbXBvcnQgbG9nZ2luZw0KZnJvbSBwYXRobGliIGltcG9ydCBQYXRoDQoNCmltcG9ydCBudW1weSBhcyBucA0KaW1wb3J0IHBhbmRhcyBhcyBwZA0KDQpmcm9tIGNvbW1pc3Npb25pbmcuY2FsaWJyYXRpb24gaW1wb3J0IF9tZWV0cywgb3JkZXJfc3RhdGlzdGljX3JhbmsNCmZyb20gY29tbWlzc2lvbmluZy5leHBsb3JhdG9yeSBpbXBvcnQgcGFjX3JhbmsNCmZyb20gY29tbWlzc2lvbmluZy5pb191dGlscyBpbXBvcnQgYXRvbWljX3BhcnF1ZXQNCmZyb20gY29tbWlzc2lvbmluZy5wYXRjaGNvcmUgaW1wb3J0IHNjb3Jlc19hbG9uZ19vcmRlcg0KZnJvbSBjb21taXNzaW9uaW5nLnZpc2EuY2FjaGVzIGltcG9ydCBsb2FkX2Rpc3RhbmNlcw0KZnJvbSBjb21taXNzaW9uaW5nLnZpc2EuY29uZmlnIGltcG9ydCBWaXNhQ29uZmlnDQpmcm9tIGNvbW1pc3Npb25pbmcudmlzYS5kYXRhIGltcG9ydCBsb2FkX3NwbGl0LCByb3dfYmxvY2tzDQoNCkxPR0dFUiA9IGxvZ2dpbmcuZ2V0TG9nZ2VyKF9fbmFtZV9fKQ0KDQpCVURHRVRfUlVMRVMgPSAoImNvbmZvcm1hbCIsICJwYWMiKQ0KDQoNCmRlZiBwYXJ0X3BhdGgoY2ZnOiBWaXNhQ29uZmlnLCBjbGFzc19uYW1lOiBzdHIpIC0+IFBhdGg6DQogICAgcmV0dXJuIGNmZy53b3JrX3Jvb3QgLyAiYnVkZ2V0IiAvIGYie2NsYXNzX25hbWV9LnBhcnF1ZXQiDQoNCg0KZGVmIHJ1bGVfcmFuayhjZmc6IFZpc2FDb25maWcsIHJ1bGU6IHN0ciwgbTogaW50KSAtPiB0dXBsZVtpbnQsIGJvb2xdOg0KICAgIGlmIHJ1bGUgPT0gImNvbmZvcm1hbCI6DQogICAgICAgIHJldHVybiBvcmRlcl9zdGF0aXN0aWNfcmFuayhtLCBjZmcuYWxwaGEpLCBUcnVlDQogICAgayA9IHBhY19yYW5rKG0sIGNmZy5wYWNfZXBzaWxvbiwgY2ZnLnBhY19kZWx0YSkNCiAgICByZXR1cm4gKG0sIEZhbHNlKSBpZiBrIGlzIE5vbmUgZWxzZSAoaywgVHJ1ZSkNCg0KDQpkZWYgYnVkZ2V0X2NsYXNzKGNmZzogVmlzYUNvbmZpZywgY2xhc3NfbmFtZTogc3RyLCBkaXN0YW5jZTogbnAubmRhcnJheSwgc3BsaXQ6IGRpY3QpIC0+IHBkLkRhdGFGcmFtZToNCiAgICBibG9ja3MgPSByb3dfYmxvY2tzKHNwbGl0KQ0KICAgIHVuaXZlcnNlLCBob2xkb3V0ID0gYmxvY2tzWyJ1bml2ZXJzZSJdLCBibG9ja3NbImhvbGRvdXQiXQ0KICAgIGdvb2QsIGRlZmVjdCA9IGJsb2Nrc1sidGVzdF9nb29kIl0sIGJsb2Nrc1sidGVzdF9kZWZlY3QiXQ0KICAgIG5fZ29vZCwgbl9kZWZlY3QsIG5faCA9IGxlbihnb29kKSwgbGVuKGRlZmVjdCksIGxlbihob2xkb3V0KQ0KICAgIGdyaWQgPSBbbiBmb3IgbiBpbiBjZmcuYnVkZ2V0X2dyaWQgaWYgbiA8PSBtYXgoY2ZnLmJ1ZGdldHMpIC0gY2ZnLmJ1ZGdldF9taW5fY2FsaWJyYXRpb25dDQogICAgcm93cyA9IFtdDQogICAgZm9yIHJlcCBpbiByYW5nZShjZmcubl9yZXBlYXRzKToNCiAgICAgICAgc2VxdWVuY2UgPSBucC5yYW5kb20uU2VlZFNlcXVlbmNlKFtjZmcuc2VlZCwgY2ZnLnNlZWRfa2V5KGNsYXNzX25hbWUpLCByZXAsIGNmZy5idWRnZXRfc2VlZF90YWddKQ0KICAgICAgICBwZXJtdXRhdGlvbiA9IHVuaXZlcnNlW25wLnJhbmRvbS5kZWZhdWx0X3JuZyhzZXF1ZW5jZSkucGVybXV0YXRpb24obGVuKHVuaXZlcnNlKSldDQogICAgICAgICMgVGhlIGZpcnN0IGJhbmsgaW1hZ2UgaXMgbmV2ZXIgYSBxdWVyeTogaXRzIG9ubHkgYmFuayBtZW1iZXIgd291bGQgYmUgaXRzZWxmICgraW5mKS4NCiAgICAgICAgcXVlcmllcyA9IG5wLmNvbmNhdGVuYXRlKFtwZXJtdXRhdGlvblsxOl0sIGhvbGRvdXQsIGdvb2QsIGRlZmVjdF0pDQogICAgICAgIHN0YWNrZWQgPSBzY29yZXNfYWxvbmdfb3JkZXIoZGlzdGFuY2UsIHF1ZXJpZXMsIHBlcm11dGF0aW9uWzogZ3JpZFstMV1dLCBncmlkKQ0KICAgICAgICBuX3UgPSBsZW4ocGVybXV0YXRpb24pIC0gMQ0KICAgICAgICBmb3Igcm93X2luZGV4LCBuIGluIGVudW1lcmF0ZShncmlkKToNCiAgICAgICAgICAgIHNjb3JlcyA9IHN0YWNrZWRbcm93X2luZGV4XQ0KICAgICAgICAgICAgaF9zY29yZXMgPSBzY29yZXNbbl91IDogbl91ICsgbl9oXQ0KICAgICAgICAgICAgZ29vZF9zY29yZXMgPSBzY29yZXNbbl91ICsgbl9oIDogbl91ICsgbl9oICsgbl9nb29kXQ0KICAgICAgICAgICAgZGVmZWN0X3Njb3JlcyA9IHNjb3Jlc1tuX3UgKyBuX2ggKyBuX2dvb2QgOl0NCiAgICAgICAgICAgIGZvciBidWRnZXQgaW4gY2ZnLmJ1ZGdldHM6DQogICAgICAgICAgICAgICAgbSA9IGJ1ZGdldCAtIG4NCiAgICAgICAgICAgICAgICBpZiBtIDwgY2ZnLmJ1ZGdldF9taW5fY2FsaWJyYXRpb246DQogICAgICAgICAgICAgICAgICAgIGNvbnRpbnVlDQogICAgICAgICAgICAgICAgb3JkZXJlZCA9IG5wLnNvcnQoc2NvcmVzW24gLSAxIDogYnVkZ2V0IC0gMV0pDQogICAgICAgICAgICAgICAgZm9yIHJ1bGUgaW4gQlVER0VUX1JVTEVTOg0KICAgICAgICAgICAgICAgICAgICBrLCBmZWFzaWJsZSA9IHJ1bGVfcmFuayhjZmcsIHJ1bGUsIG0pDQogICAgICAgICAgICAgICAgICAgIHRocmVzaG9sZCA9IGZsb2F0KG9yZGVyZWRbayAtIDFdKQ0KICAgICAgICAgICAgICAgICAgICBuX3RwID0gaW50KG5wLnN1bShkZWZlY3Rfc2NvcmVzID4gdGhyZXNob2xkKSkNCiAgICAgICAgICAgICAgICAgICAgbl9mcF9vZmYgPSBpbnQobnAuc3VtKGdvb2Rfc2NvcmVzID4gdGhyZXNob2xkKSkNCiAgICAgICAgICAgICAgICAgICAgbl9mcF9oID0gaW50KG5wLnN1bShoX3Njb3JlcyA+IHRocmVzaG9sZCkpDQogICAgICAgICAgICAgICAgICAgIHRwcl9vayA9IF9tZWV0cyhuX3RwLCBuX2RlZmVjdCwgY2ZnLnByYWN0aWNhbF90cHIsIGF0X2xlYXN0PVRydWUpDQogICAgICAgICAgICAgICAgICAgIHJvd3MuYXBwZW5kKA0KICAgICAgICAgICAgICAgICAgICAgICAgew0KICAgICAgICAgICAgICAgICAgICAgICAgICAgICJjbGFzc19uYW1lIjogY2xhc3NfbmFtZSwNCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAiZ3JvdXAiOiBjZmcuZ3JvdXBfb2YoY2xhc3NfbmFtZSksDQogICAgICAgICAgICAgICAgICAgICAgICAgICAgInJlcCI6IHJlcCwNCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAiYnVkZ2V0IjogYnVkZ2V0LA0KICAgICAgICAgICAgICAgICAgICAgICAgICAgICJOIjogbiwNCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAibl9jYWxpYnJhdGlvbiI6IG0sDQogICAgICAgICAgICAgICAgICAgICAgICAgICAgInJ1bGUiOiBydWxlLA0KICAgICAgICAgICAgICAgICAgICAgICAgICAgICJyYW5rIjogaywNCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAicnVsZV9mZWFzaWJsZSI6IGZlYXNpYmxlLA0KICAgICAgICAgICAgICAgICAgICAgICAgICAgICJ0cHIiOiBuX3RwIC8gbl9kZWZlY3QsDQogICAgICAgICAgICAgICAgICAgICAgICAgICAgImZwcl9vZmYiOiBuX2ZwX29mZiAvIG5fZ29vZCwNCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAiZnByX2giOiBuX2ZwX2ggLyBuX2gsDQogICAgICAgICAgICAgICAgICAgICAgICAgICAgIm5fdHAiOiBuX3RwLA0KICAgICAgICAgICAgICAgICAgICAgICAgICAgICJuX2ZwX29mZiI6IG5fZnBfb2ZmLA0KICAgICAgICAgICAgICAgICAgICAgICAgICAgICJuX2ZwX2giOiBuX2ZwX2gsDQogICAgICAgICAgICAgICAgICAgICAgICAgICAgImNyaXRlcmlvbl9vZmYiOiBib29sKHRwcl9vayBhbmQgX21lZXRzKG5fZnBfb2ZmLCBuX2dvb2QsIGNmZy5wcmFjdGljYWxfZnByLCBhdF9sZWFzdD1GYWxzZSkpLA0KICAgICAgICAgICAgICAgICAgICAgICAgICAgICJjcml0ZXJpb25faCI6IGJvb2wodHByX29rIGFuZCBfbWVldHMobl9mcF9oLCBuX2gsIGNmZy5wcmFjdGljYWxfZnByLCBhdF9sZWFzdD1GYWxzZSkpLA0KICAgICAgICAgICAgICAgICAgICAgICAgICAgICJjb25maWdfaGFzaCI6IGNmZy5oYXNoLA0KICAgICAgICAgICAgICAgICAgICAgICAgfQ0KICAgICAgICAgICAgICAgICAgICApDQogICAgcmV0dXJuIHBkLkRhdGFGcmFtZShyb3dzKQ0KDQoNCmRlZiBydW5fYnVkZ2V0KGNmZzogVmlzYUNvbmZpZywgY2xhc3NlczogbGlzdFtzdHJdKSAtPiBwZC5EYXRhRnJhbWU6DQogICAgZnJhbWVzID0gW10NCiAgICBmb3IgY2xhc3NfbmFtZSBpbiBjbGFzc2VzOg0KICAgICAgICBwYXRoID0gcGFydF9wYXRoKGNmZywgY2xhc3NfbmFtZSkNCiAgICAgICAgaWYgcGF0aC5pc19maWxlKCk6DQogICAgICAgICAgICBmcmFtZSA9IHBkLnJlYWRfcGFycXVldChwYXRoKQ0KICAgICAgICAgICAgaWYgc2V0KGZyYW1lWyJjb25maWdfaGFzaCJdLmFzdHlwZShzdHIpKSAhPSB7Y2ZnLmhhc2h9Og0KICAgICAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIlJlZnVzaW5nIHRvIHJldXNlIGEgYnVkZ2V0IHBhcnQgZnJvbSBhbm90aGVyIGNvbmZpZzoge3BhdGh9IikNCiAgICAgICAgICAgIGZyYW1lcy5hcHBlbmQoZnJhbWUpDQogICAgICAgICAgICBjb250aW51ZQ0KICAgICAgICBMT0dHRVIuaW5mbygiQnVkZ2V0IHNwbGl0ICVzIiwgY2xhc3NfbmFtZSkNCiAgICAgICAgc3BsaXQgPSBsb2FkX3NwbGl0KGNmZywgY2xhc3NfbmFtZSwgMCkNCiAgICAgICAgZGlzdGFuY2UgPSBsb2FkX2Rpc3RhbmNlcyhjZmcsIGNsYXNzX25hbWUpWyJEIl0NCiAgICAgICAgZnJhbWUgPSBidWRnZXRfY2xhc3MoY2ZnLCBjbGFzc19uYW1lLCBkaXN0YW5jZSwgc3BsaXQpDQogICAgICAgIGRlbCBkaXN0YW5jZQ0KICAgICAgICBhdG9taWNfcGFycXVldChwYXRoLCBmcmFtZSkNCiAgICAgICAgZnJhbWVzLmFwcGVuZChmcmFtZSkNCiAgICByZXR1cm4gcGQuY29uY2F0KGZyYW1lcywgaWdub3JlX2luZGV4PVRydWUpDQoNCg0KZGVmIHN1bW1hcml6ZV9idWRnZXQoZnJhbWU6IHBkLkRhdGFGcmFtZSwgbWluX3Bhc3NlczogaW50KSAtPiB0dXBsZVtwZC5EYXRhRnJhbWUsIHBkLkRhdGFGcmFtZV06DQogICAga2V5cyA9IFsiY2xhc3NfbmFtZSIsICJncm91cCIsICJidWRnZXQiLCAiTiIsICJuX2NhbGlicmF0aW9uIiwgInJ1bGUiXQ0KICAgIHN1bW1hcnkgPSAoDQogICAgICAgIGZyYW1lLmdyb3VwYnkoa2V5cywgc29ydD1UcnVlKQ0KICAgICAgICAuYWdnKA0KICAgICAgICAgICAgcGFzc2luZ19vZmY9KCJjcml0ZXJpb25fb2ZmIiwgInN1bSIpLA0KICAgICAgICAgICAgcGFzc2luZ19oPSgiY3JpdGVyaW9uX2giLCAic3VtIiksDQogICAgICAgICAgICBtZWRpYW5fdHByPSgidHByIiwgIm1lZGlhbiIpLA0KICAgICAgICAgICAgbWVhbl9mcHJfb2ZmPSgiZnByX29mZiIsICJtZWFuIiksDQogICAgICAgICAgICBtZWFuX2Zwcl9oPSgiZnByX2giLCAibWVhbiIpLA0KICAgICAgICAgICAgcnVsZV9mZWFzaWJsZT0oInJ1bGVfZmVhc2libGUiLCAiZmlyc3QiKSwNCiAgICAgICAgKQ0KICAgICAgICAucmVzZXRfaW5kZXgoKQ0KICAgICkNCiAgICBiZXN0ID0gW10NCiAgICBmb3Igc291cmNlIGluICgib2ZmIiwgImgiKToNCiAgICAgICAgY29sdW1uID0gZiJwYXNzaW5nX3tzb3VyY2V9Ig0KICAgICAgICBmb3IgKGNsYXNzX25hbWUsIGdyb3VwLCBidWRnZXQsIHJ1bGUpLCBzdWIgaW4gc3VtbWFyeS5ncm91cGJ5KFsiY2xhc3NfbmFtZSIsICJncm91cCIsICJidWRnZXQiLCAicnVsZSJdKToNCiAgICAgICAgICAgIHRvcCA9IHN1Yi5zb3J0X3ZhbHVlcyhbY29sdW1uLCAibWVkaWFuX3RwciIsICJOIl0sIGFzY2VuZGluZz1bRmFsc2UsIEZhbHNlLCBUcnVlXSkuaWxvY1swXQ0KICAgICAgICAgICAgYmVzdC5hcHBlbmQoDQogICAgICAgICAgICAgICAgew0KICAgICAgICAgICAgICAgICAgICAiY2xhc3NfbmFtZSI6IGNsYXNzX25hbWUsDQogICAgICAgICAgICAgICAgICAgICJncm91cCI6IGdyb3VwLA0KICAgICAgICAgICAgICAgICAgICAiYnVkZ2V0IjogaW50KGJ1ZGdldCksDQogICAgICAgICAgICAgICAgICAgICJydWxlIjogcnVsZSwNCiAgICAgICAgICAgICAgICAgICAgImZwcl9zb3VyY2UiOiBzb3VyY2UsDQogICAgICAgICAgICAgICAgICAgICJiZXN0X04iOiBpbnQodG9wWyJOIl0pLA0KICAgICAgICAgICAgICAgICAgICAiYmVzdF9uX2NhbGlicmF0aW9uIjogaW50KHRvcFsibl9jYWxpYnJhdGlvbiJdKSwNCiAgICAgICAgICAgICAgICAgICAgInBhc3NpbmdfcmVwcyI6IGludCh0b3BbY29sdW1uXSksDQogICAgICAgICAgICAgICAgICAgICJtZWV0c19taW5fcmVwcyI6IGJvb2wodG9wW2NvbHVtbl0gPj0gbWluX3Bhc3NlcyksDQogICAgICAgICAgICAgICAgICAgICJtZWRpYW5fdHByIjogZmxvYXQodG9wWyJtZWRpYW5fdHByIl0pLA0KICAgICAgICAgICAgICAgIH0NCiAgICAgICAgICAgICkNCiAgICByZXR1cm4gc3VtbWFyeSwgcGQuRGF0YUZyYW1lKGJlc3QpDQoNCg0KZGVmIGJ1ZGdldF9oeXBvdGhlc2lzKGJlc3Q6IHBkLkRhdGFGcmFtZSkgLT4gZGljdDoNCiAgICAiIiJTZWN0aW9uIDEwOiBmb3IgQiA+PSAxMjAsIGRvZXMgdGhlIGJlc3Qgc3BsaXQgcmVzZXJ2ZSBhdCBsZWFzdCA0MCBpbWFnZXMgZm9yIGNhbGlicmF0aW9uIGluIG1vc3QgY2xhc3Nlcz8iIiINCiAgICBvdXQgPSB7fQ0KICAgIGZvciAocnVsZSwgc291cmNlKSwgc3ViIGluIGJlc3RbYmVzdFsiYnVkZ2V0Il0gPj0gMTIwXS5ncm91cGJ5KFsicnVsZSIsICJmcHJfc291cmNlIl0pOg0KICAgICAgICBwZXJfYnVkZ2V0ID0gew0KICAgICAgICAgICAgc3RyKGludChiKSk6IGZsb2F0KG5wLm1lYW4ocGFydFsiYmVzdF9uX2NhbGlicmF0aW9uIl0gPj0gNDApKSBmb3IgYiwgcGFydCBpbiBzdWIuZ3JvdXBieSgiYnVkZ2V0IikNCiAgICAgICAgfQ0KICAgICAgICBvdXRbZiJ7cnVsZX1fe3NvdXJjZX0iXSA9IHsNCiAgICAgICAgICAgICJmcmFjdGlvbl9vZl9jbGFzc19idWRnZXRfcGFpcnNfd2l0aF9jYWxpYnJhdGlvbl9nZV80MCI6IGZsb2F0KG5wLm1lYW4oc3ViWyJiZXN0X25fY2FsaWJyYXRpb24iXSA+PSA0MCkpLA0KICAgICAgICAgICAgInBlcl9idWRnZXQiOiBwZXJfYnVkZ2V0LA0KICAgICAgICB9DQogICAgcmV0dXJuIG91dA0K", "commissioning/visa/caches.py": "IiIiUGF0Y2ggZmVhdHVyZXMgYW5kIHRoZSBkaXN0YW5jZSB0ZW5zb3IgZm9yIFZpc0EuDQoNCkxhcmdlIHRlbnNvcnMgbGl2ZSB1bmRlciBjYWNoZV9yb290IChsb2NhbCBkaXNrKS4gVGhlIEdBUCB0YWJsZSwgd2hpY2ggZml4ZXMgdGhlIGstY2VudGVyIG9yZGVyLA0KaXMgd3JpdHRlbiBvbmNlIHVuZGVyIHdvcmtfcm9vdCBzbyB0aGF0IGEgcmVjb21wdXRlZCBjYWNoZSBjYW5ub3QgY2hhbmdlIGEgZmluaXNoZWQgdHJhamVjdG9yeS4NCiIiIg0KDQpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zDQoNCmltcG9ydCBsb2dnaW5nDQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgNCg0KaW1wb3J0IG51bXB5IGFzIG5wDQoNCmZyb20gY29tbWlzc2lvbmluZy5kaXN0YW5jZXMgaW1wb3J0IGRpc3RhbmNlX3RlbnNvcg0KZnJvbSBjb21taXNzaW9uaW5nLmZlYXR1cmVzIGltcG9ydCAoDQogICAgYnVpbGRfZmVhdHVyZV9leHRyYWN0b3IsDQogICAgYnVpbGRfdHJhbnNmb3JtLA0KICAgIHBhdGNoZXNfYW5kX2dhcCwNCiAgICBwb29sZWRfZW1iZWRkaW5nLA0KICAgIHJlYWRfcmdiLA0KKQ0KZnJvbSBjb21taXNzaW9uaW5nLmlvX3V0aWxzIGltcG9ydCBhdG9taWNfbnB5LCBhdG9taWNfc2F2ZXosIHJlYWRfanNvbiwgd3JpdGVfanNvbg0KZnJvbSBjb21taXNzaW9uaW5nLnZpc2EuY29uZmlnIGltcG9ydCBWaXNhQ29uZmlnDQpmcm9tIGNvbW1pc3Npb25pbmcudmlzYS5kYXRhIGltcG9ydCBpbWFnZV9vcmRlciwgbG9hZF9zcGxpdA0KDQpMT0dHRVIgPSBsb2dnaW5nLmdldExvZ2dlcihfX25hbWVfXykNCg0KDQpkZWYgZmVhdHVyZV9wYXRoKGNmZzogVmlzYUNvbmZpZywgY2xhc3NfbmFtZTogc3RyKSAtPiBQYXRoOg0KICAgIHJldHVybiBjZmcuY2FjaGVfcm9vdCAvICJmZWF0dXJlcyIgLyBmIntjbGFzc19uYW1lfS5ucHoiDQoNCg0KZGVmIGRpc3RhbmNlX3BhdGgoY2ZnOiBWaXNhQ29uZmlnLCBjbGFzc19uYW1lOiBzdHIpIC0+IFBhdGg6DQogICAgcmV0dXJuIGNmZy5jYWNoZV9yb290IC8gImRpc3QiIC8gZiJ7Y2xhc3NfbmFtZX0ubnB5Ig0KDQoNCmRlZiBnYXBfcGF0aChjZmc6IFZpc2FDb25maWcsIGNsYXNzX25hbWU6IHN0cikgLT4gUGF0aDoNCiAgICByZXR1cm4gY2ZnLndvcmtfcm9vdCAvICJnYXAiIC8gZiJ7Y2xhc3NfbmFtZX0ubnB6Ig0KDQoNCmRlZiBfbWV0YV9wYXRoKHBhdGg6IFBhdGgpIC0+IFBhdGg6DQogICAgcmV0dXJuIHBhdGgud2l0aF9zdWZmaXgoIi5qc29uIikNCg0KDQpkZWYgX2N1cnJlbnQoY2ZnOiBWaXNhQ29uZmlnLCBwYXRoOiBQYXRoKSAtPiBib29sOg0KICAgIG1ldGEgPSBfbWV0YV9wYXRoKHBhdGgpDQogICAgcmV0dXJuIHBhdGguaXNfZmlsZSgpIGFuZCBtZXRhLmlzX2ZpbGUoKSBhbmQgcmVhZF9qc29uKG1ldGEpLmdldCgiY29uZmlnX2hhc2giKSA9PSBjZmcuaGFzaA0KDQoNCmRlZiBleHRyYWN0X2NsYXNzKGNmZzogVmlzYUNvbmZpZywgY2xhc3NfbmFtZTogc3RyLCBkZXZpY2U6IHN0ciB8IE5vbmUgPSBOb25lKSAtPiBOb25lOg0KICAgIGltcG9ydCB0b3JjaA0KICAgIGltcG9ydCB0b3JjaC5ubiBhcyBubg0KDQogICAgb3V0ID0gZmVhdHVyZV9wYXRoKGNmZywgY2xhc3NfbmFtZSkNCiAgICBpZiBfY3VycmVudChjZmcsIG91dCk6DQogICAgICAgIExPR0dFUi5pbmZvKCJGZWF0dXJlIGNhY2hlIGV4aXN0cyBmb3IgJXMiLCBjbGFzc19uYW1lKQ0KICAgICAgICByZXR1cm4NCiAgICBzcGxpdCA9IGxvYWRfc3BsaXQoY2ZnLCBjbGFzc19uYW1lLCAwKQ0KICAgIHBhdGhzID0gaW1hZ2Vfb3JkZXIoc3BsaXQpDQogICAgZGV2aWNlX3QgPSB0b3JjaC5kZXZpY2UoZGV2aWNlIG9yICgiY3VkYSIgaWYgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKSBlbHNlICJjcHUiKSkNCiAgICBleHRyYWN0b3IsIHNvdXJjZSA9IGJ1aWxkX2ZlYXR1cmVfZXh0cmFjdG9yKGNmZykNCiAgICBleHRyYWN0b3IgPSBleHRyYWN0b3IuZXZhbCgpLnRvKGRldmljZV90KQ0KICAgIHRyYW5zZm9ybSA9IGJ1aWxkX3RyYW5zZm9ybShjZmcpDQogICAgcG9vbCA9IG5uLkF2Z1Bvb2wyZChrZXJuZWxfc2l6ZT0zLCBzdHJpZGU9MSwgcGFkZGluZz0xKS50byhkZXZpY2VfdCkNCiAgICBwYXRjaGVzID0gbnAuZW1wdHkoKGxlbihwYXRocyksIDc4NCwgMTUzNiksIGR0eXBlPW5wLmZsb2F0MTYpDQogICAgZ2FwID0gbnAuZW1wdHkoKGxlbihwYXRocyksIDE1MzYpLCBkdHlwZT1ucC5mbG9hdDMyKQ0KICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOg0KICAgICAgICBmb3Igc3RhcnQgaW4gcmFuZ2UoMCwgbGVuKHBhdGhzKSwgY2ZnLmZlYXR1cmVfYmF0Y2hfc2l6ZSk6DQogICAgICAgICAgICBjaHVuayA9IHBhdGhzW3N0YXJ0IDogc3RhcnQgKyBjZmcuZmVhdHVyZV9iYXRjaF9zaXplXQ0KICAgICAgICAgICAgYmF0Y2ggPSB0b3JjaC5zdGFjayhbdHJhbnNmb3JtKHJlYWRfcmdiKGNmZy5kYXRhX3Jvb3QgLyBwLCBjbGFzc19uYW1lKSkgZm9yIHAgaW4gY2h1bmtdKS50byhkZXZpY2VfdCkNCiAgICAgICAgICAgIGZlYXR1cmVzID0gZXh0cmFjdG9yKGJhdGNoKQ0KICAgICAgICAgICAgaWYgbm90IGlzaW5zdGFuY2UoZmVhdHVyZXMsIGRpY3QpOg0KICAgICAgICAgICAgICAgIGZlYXR1cmVzID0gZGljdCh6aXAoY2ZnLmxheWVycywgZmVhdHVyZXMpKQ0KICAgICAgICAgICAgZW1iZWRkaW5nID0gcG9vbGVkX2VtYmVkZGluZyhmZWF0dXJlcywgY2ZnLmxheWVycywgcG9vbCwgY2ZnLnVwc2FtcGxlX21vZGUpDQogICAgICAgICAgICBpZiBlbWJlZGRpbmcuc2hhcGVbMV0gIT0gMTUzNiBvciBlbWJlZGRpbmcuc2hhcGVbLTI6XSAhPSAoMjgsIDI4KToNCiAgICAgICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJ7Y2xhc3NfbmFtZX06IHVuZXhwZWN0ZWQgZW1iZWRkaW5nIHNoYXBlIHt0dXBsZShlbWJlZGRpbmcuc2hhcGUpfSIpDQogICAgICAgICAgICBibG9ja19wYXRjaGVzLCBibG9ja19nYXAgPSBwYXRjaGVzX2FuZF9nYXAoZW1iZWRkaW5nKQ0KICAgICAgICAgICAgYmxvY2tfcGF0Y2hlcyA9IGJsb2NrX3BhdGNoZXMuY3B1KCkubnVtcHkoKQ0KICAgICAgICAgICAgYmxvY2tfZ2FwID0gYmxvY2tfZ2FwLmNwdSgpLm51bXB5KCkNCiAgICAgICAgICAgIGlmIG5vdCBucC5pc2Zpbml0ZShibG9ja19wYXRjaGVzKS5hbGwoKSBvciBub3QgbnAuaXNmaW5pdGUoYmxvY2tfZ2FwKS5hbGwoKToNCiAgICAgICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJ7Y2xhc3NfbmFtZX06IG5vbi1maW5pdGUgZmVhdHVyZXMgaW4gYmF0Y2ggc3RhcnRpbmcgYXQge2NodW5rWzBdfSIpDQogICAgICAgICAgICBwYXRjaGVzW3N0YXJ0IDogc3RhcnQgKyBsZW4oY2h1bmspXSA9IGJsb2NrX3BhdGNoZXMuYXN0eXBlKG5wLmZsb2F0MTYpDQogICAgICAgICAgICBnYXBbc3RhcnQgOiBzdGFydCArIGxlbihjaHVuayldID0gYmxvY2tfZ2FwDQogICAgYXRvbWljX3NhdmV6KG91dCwgcGF0Y2hlcz1wYXRjaGVzLCBwYXRocz1ucC5hcnJheShwYXRocykpDQogICAgd3JpdGVfanNvbigNCiAgICAgICAgX21ldGFfcGF0aChvdXQpLA0KICAgICAgICB7DQogICAgICAgICAgICAiY29uZmlnX2hhc2giOiBjZmcuaGFzaCwNCiAgICAgICAgICAgICJjbGFzc19uYW1lIjogY2xhc3NfbmFtZSwNCiAgICAgICAgICAgICJuX2ltYWdlcyI6IGxlbihwYXRocyksDQogICAgICAgICAgICAibl91bml2ZXJzZSI6IGxlbihzcGxpdFsidW5pdmVyc2UiXSksDQogICAgICAgICAgICAibl9ob2xkb3V0IjogbGVuKHNwbGl0WyJob2xkb3V0Il0pLA0KICAgICAgICAgICAgIm5fdGVzdF9nb29kIjogbGVuKHNwbGl0WyJ0ZXN0X2dvb2QiXSksDQogICAgICAgICAgICAibl90ZXN0X2RlZmVjdCI6IGxlbihzcGxpdFsidGVzdF9kZWZlY3QiXSksDQogICAgICAgICAgICAiZXh0cmFjdG9yIjogc291cmNlLA0KICAgICAgICB9LA0KICAgICkNCiAgICBnYXBfZmlsZSA9IGdhcF9wYXRoKGNmZywgY2xhc3NfbmFtZSkNCiAgICBpZiBub3QgX2N1cnJlbnQoY2ZnLCBnYXBfZmlsZSk6DQogICAgICAgIGF0b21pY19zYXZleihnYXBfZmlsZSwgZ2FwPWdhcCwgcGF0aHM9bnAuYXJyYXkocGF0aHMpKQ0KICAgICAgICB3cml0ZV9qc29uKF9tZXRhX3BhdGgoZ2FwX2ZpbGUpLCB7ImNvbmZpZ19oYXNoIjogY2ZnLmhhc2gsICJjbGFzc19uYW1lIjogY2xhc3NfbmFtZSwgImV4dHJhY3RvciI6IHNvdXJjZX0pDQogICAgTE9HR0VSLmluZm8oIiVzIGZlYXR1cmVzOiAlZCBpbWFnZXMsICUuMmYgR0IgZmxvYXQxNiIsIGNsYXNzX25hbWUsIGxlbihwYXRocyksIHBhdGNoZXMubmJ5dGVzIC8gMTAyNCoqMykNCg0KDQpkZWYgbG9hZF9mZWF0dXJlcyhjZmc6IFZpc2FDb25maWcsIGNsYXNzX25hbWU6IHN0cikgLT4gZGljdDoNCiAgICBwYXRoID0gZmVhdHVyZV9wYXRoKGNmZywgY2xhc3NfbmFtZSkNCiAgICBpZiBub3QgX2N1cnJlbnQoY2ZnLCBwYXRoKToNCiAgICAgICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoZiJNaXNzaW5nIG9yIHN0YWxlIGZlYXR1cmUgY2FjaGUgZm9yIHtjbGFzc19uYW1lfS4gUnVuIGZlYXR1cmVzIGZpcnN0LiIpDQogICAgd2l0aCBucC5sb2FkKHBhdGgsIGFsbG93X3BpY2tsZT1GYWxzZSkgYXMgZGF0YToNCiAgICAgICAgcGF0Y2hlcyA9IG5wLmFycmF5KGRhdGFbInBhdGNoZXMiXSkNCiAgICAgICAgcGF0aHMgPSBbc3RyKHApIGZvciBwIGluIGRhdGFbInBhdGhzIl0udG9saXN0KCldDQogICAgaWYgcGF0Y2hlcy5kdHlwZSAhPSBucC5mbG9hdDE2IG9yIHBhdGNoZXMuc2hhcGVbMTpdICE9ICg3ODQsIDE1MzYpOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJ7Y2xhc3NfbmFtZX06IGJhZCBwYXRjaCBjYWNoZSB7cGF0Y2hlcy5zaGFwZX0ge3BhdGNoZXMuZHR5cGV9IikNCiAgICByZXR1cm4geyJwYXRjaGVzIjogcGF0Y2hlcywgInBhdGhzIjogcGF0aHMsICJtZXRhIjogcmVhZF9qc29uKF9tZXRhX3BhdGgocGF0aCkpfQ0KDQoNCmRlZiBsb2FkX2dhcChjZmc6IFZpc2FDb25maWcsIGNsYXNzX25hbWU6IHN0cikgLT4gZGljdDoNCiAgICBwYXRoID0gZ2FwX3BhdGgoY2ZnLCBjbGFzc19uYW1lKQ0KICAgIGlmIG5vdCBfY3VycmVudChjZmcsIHBhdGgpOg0KICAgICAgICByYWlzZSBGaWxlTm90Rm91bmRFcnJvcihmIk1pc3NpbmcgR0FQIHRhYmxlIGZvciB7Y2xhc3NfbmFtZX0uIFJ1biBmZWF0dXJlcyBmaXJzdC4iKQ0KICAgIHdpdGggbnAubG9hZChwYXRoLCBhbGxvd19waWNrbGU9RmFsc2UpIGFzIGRhdGE6DQogICAgICAgIGdhcCA9IG5wLmFycmF5KGRhdGFbImdhcCJdKQ0KICAgICAgICBwYXRocyA9IFtzdHIocCkgZm9yIHAgaW4gZGF0YVsicGF0aHMiXS50b2xpc3QoKV0NCiAgICBpZiBub3QgbnAuaXNmaW5pdGUoZ2FwKS5hbGwoKToNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYie2NsYXNzX25hbWV9OiBub24tZmluaXRlIEdBUCBmZWF0dXJlcyIpDQogICAgcmV0dXJuIHsiZ2FwIjogZ2FwLCAicGF0aHMiOiBwYXRoc30NCg0KDQpkZWYgY29tcHV0ZV9jbGFzc19kaXN0YW5jZXMoY2ZnOiBWaXNhQ29uZmlnLCBjbGFzc19uYW1lOiBzdHIsIGRldmljZTogc3RyIHwgTm9uZSA9IE5vbmUpIC0+IE5vbmU6DQogICAgb3V0ID0gZGlzdGFuY2VfcGF0aChjZmcsIGNsYXNzX25hbWUpDQogICAgaWYgX2N1cnJlbnQoY2ZnLCBvdXQpOg0KICAgICAgICBMT0dHRVIuaW5mbygiRGlzdGFuY2UgY2FjaGUgZXhpc3RzIGZvciAlcyIsIGNsYXNzX25hbWUpDQogICAgICAgIHJldHVybg0KICAgIGNhY2hlID0gbG9hZF9mZWF0dXJlcyhjZmcsIGNsYXNzX25hbWUpDQogICAgc3BsaXQgPSBsb2FkX3NwbGl0KGNmZywgY2xhc3NfbmFtZSwgMCkNCiAgICBpZiBjYWNoZVsicGF0aHMiXSAhPSBpbWFnZV9vcmRlcihzcGxpdCk6DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIntjbGFzc19uYW1lfTogZmVhdHVyZSByb3dzIGRvIG5vdCBmb2xsb3cgVSwgSCwgdGVzdCBnb29kLCB0ZXN0IGFub21hbHkuIikNCiAgICBuX3VuaXZlcnNlID0gbGVuKHNwbGl0WyJ1bml2ZXJzZSJdKQ0KICAgIExPR0dFUi5pbmZvKCJEaXN0YW5jZSB0ZW5zb3IgJXM6ICVkIHJvd3MgeCAlZCB1bml2ZXJzZSBjb2x1bW5zIiwgY2xhc3NfbmFtZSwgbGVuKGNhY2hlWyJwYXRocyJdKSwgbl91bml2ZXJzZSkNCiAgICB0ZW5zb3IgPSBkaXN0YW5jZV90ZW5zb3IoDQogICAgICAgIGNhY2hlWyJwYXRjaGVzIl0sDQogICAgICAgIG5fdHJhaW49bl91bml2ZXJzZSwNCiAgICAgICAgcXVlcnlfYmF0Y2g9Y2ZnLmRpc3RhbmNlX3F1ZXJ5X2JhdGNoLA0KICAgICAgICB0cmFpbl9iYXRjaD1jZmcuZGlzdGFuY2VfdHJhaW5fYmF0Y2gsDQogICAgICAgIGRldmljZT1kZXZpY2UsDQogICAgKQ0KICAgIGF0b21pY19ucHkob3V0LCB0ZW5zb3IpDQogICAgd3JpdGVfanNvbigNCiAgICAgICAgX21ldGFfcGF0aChvdXQpLA0KICAgICAgICB7DQogICAgICAgICAgICAiY29uZmlnX2hhc2giOiBjZmcuaGFzaCwNCiAgICAgICAgICAgICJjbGFzc19uYW1lIjogY2xhc3NfbmFtZSwNCiAgICAgICAgICAgICJzaGFwZSI6IGxpc3QodGVuc29yLnNoYXBlKSwNCiAgICAgICAgICAgICJuX3VuaXZlcnNlIjogbl91bml2ZXJzZSwNCiAgICAgICAgICAgICJwYXRocyI6IGNhY2hlWyJwYXRocyJdLA0KICAgICAgICAgICAgImR0eXBlIjogImZsb2F0MzIiLA0KICAgICAgICB9LA0KICAgICkNCiAgICBMT0dHRVIuaW5mbygiJXMgZGlzdGFuY2VzOiAlcywgJS4yZiBHQiIsIGNsYXNzX25hbWUsIHRlbnNvci5zaGFwZSwgdGVuc29yLm5ieXRlcyAvIDEwMjQqKjMpDQoNCg0KZGVmIGxvYWRfZGlzdGFuY2VzKGNmZzogVmlzYUNvbmZpZywgY2xhc3NfbmFtZTogc3RyKSAtPiBkaWN0Og0KICAgIHBhdGggPSBkaXN0YW5jZV9wYXRoKGNmZywgY2xhc3NfbmFtZSkNCiAgICBpZiBub3QgX2N1cnJlbnQoY2ZnLCBwYXRoKToNCiAgICAgICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoZiJNaXNzaW5nIG9yIHN0YWxlIGRpc3RhbmNlIGNhY2hlIGZvciB7Y2xhc3NfbmFtZX0uIFJ1biBkaXN0YW5jZXMgZmlyc3QuIikNCiAgICBtZXRhID0gcmVhZF9qc29uKF9tZXRhX3BhdGgocGF0aCkpDQogICAgdGVuc29yID0gbnAubG9hZChwYXRoKQ0KICAgIGlmIGxpc3QodGVuc29yLnNoYXBlKSAhPSBsaXN0KG1ldGFbInNoYXBlIl0pOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJ7Y2xhc3NfbmFtZX06IGRpc3RhbmNlIHRlbnNvciB7dGVuc29yLnNoYXBlfSAhPSBtZXRhIHttZXRhWydzaGFwZSddfSIpDQogICAgcmV0dXJuIHsiRCI6IHRlbnNvciwgIm1ldGEiOiBtZXRhfQ0KDQoNCmRlZiBleHRyYWN0KGNmZzogVmlzYUNvbmZpZywgY2xhc3NlczogbGlzdFtzdHJdLCBkZXZpY2U6IHN0ciB8IE5vbmUgPSBOb25lKSAtPiBOb25lOg0KICAgIGZvciBjbGFzc19uYW1lIGluIGNsYXNzZXM6DQogICAgICAgIGV4dHJhY3RfY2xhc3MoY2ZnLCBjbGFzc19uYW1lLCBkZXZpY2U9ZGV2aWNlKQ0KDQoNCmRlZiBjb21wdXRlX2Rpc3RhbmNlcyhjZmc6IFZpc2FDb25maWcsIGNsYXNzZXM6IGxpc3Rbc3RyXSwgZGV2aWNlOiBzdHIgfCBOb25lID0gTm9uZSkgLT4gTm9uZToNCiAgICBmb3IgY2xhc3NfbmFtZSBpbiBjbGFzc2VzOg0KICAgICAgICBjb21wdXRlX2NsYXNzX2Rpc3RhbmNlcyhjZmcsIGNsYXNzX25hbWUsIGRldmljZT1kZXZpY2UpDQo=", "commissioning/visa/config.py": "IiIiRnJvemVuIFZpc0EgY29uZmlndXJhdGlvbiwgb2ZmaWNpYWwgY291bnRzLCBhbmQgcGF0aHMuIiIiDQoNCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMNCg0KaW1wb3J0IGhhc2hsaWINCmltcG9ydCBqc29uDQppbXBvcnQgb3MNCmZyb20gZGF0YWNsYXNzZXMgaW1wb3J0IGRhdGFjbGFzcw0KZnJvbSBwYXRobGliIGltcG9ydCBQYXRoDQpmcm9tIHR5cGluZyBpbXBvcnQgQW55DQoNCmltcG9ydCB5YW1sDQoNCmZyb20gY29tbWlzc2lvbmluZy5jb25maWcgaW1wb3J0IHByb2plY3Rfcm9vdA0KDQojIFNlZWQgaW5kaWNlcyBmb2xsb3cgdGhpcyBvcmRlciAoY2xhc3Nfc2VlZF9vZmZzZXQgKyBpbmRleCkuDQpDTEFTU0VTID0gWw0KICAgICJjYW5kbGUiLA0KICAgICJjYXBzdWxlcyIsDQogICAgImNhc2hldyIsDQogICAgImNoZXdpbmdndW0iLA0KICAgICJmcnl1bSIsDQogICAgIm1hY2Fyb25pMSIsDQogICAgIm1hY2Fyb25pMiIsDQogICAgInBjYjEiLA0KICAgICJwY2IyIiwNCiAgICAicGNiMyIsDQogICAgInBjYjQiLA0KICAgICJwaXBlX2ZyeXVtIiwNCl0NCg0KR1JPVVBTOiBkaWN0W3N0ciwgc3RyXSA9IHsNCiAgICAiY2FuZGxlIjogIm11bHRpcGxlIiwNCiAgICAiY2Fwc3VsZXMiOiAibXVsdGlwbGUiLA0KICAgICJtYWNhcm9uaTEiOiAibXVsdGlwbGUiLA0KICAgICJtYWNhcm9uaTIiOiAibXVsdGlwbGUiLA0KICAgICJjYXNoZXciOiAic2luZ2xlIiwNCiAgICAiY2hld2luZ2d1bSI6ICJzaW5nbGUiLA0KICAgICJmcnl1bSI6ICJzaW5nbGUiLA0KICAgICJwaXBlX2ZyeXVtIjogInNpbmdsZSIsDQogICAgInBjYjEiOiAiY29tcGxleCIsDQogICAgInBjYjIiOiAiY29tcGxleCIsDQogICAgInBjYjMiOiAiY29tcGxleCIsDQogICAgInBjYjQiOiAiY29tcGxleCIsDQp9DQoNCiMgT2ZmaWNpYWwgb25lLWNsYXNzIHNwbGl0LCBzcGxpdF9jc3YvMWNscy5jc3YuIENoZWNrZWQgYmVmb3JlIGR1cGxpY2F0ZSByZW1vdmFsLg0KRVhQRUNURURfQ09VTlRTOiBkaWN0W3N0ciwgZGljdFtzdHIsIGludF1dID0gew0KICAgICJjYW5kbGUiOiB7InRyYWluIjogOTAwLCAidGVzdF9nb29kIjogMTAwLCAidGVzdF9kZWZlY3QiOiAxMDB9LA0KICAgICJjYXBzdWxlcyI6IHsidHJhaW4iOiA1NDIsICJ0ZXN0X2dvb2QiOiA2MCwgInRlc3RfZGVmZWN0IjogMTAwfSwNCiAgICAiY2FzaGV3IjogeyJ0cmFpbiI6IDQ1MCwgInRlc3RfZ29vZCI6IDUwLCAidGVzdF9kZWZlY3QiOiAxMDB9LA0KICAgICJjaGV3aW5nZ3VtIjogeyJ0cmFpbiI6IDQ1MywgInRlc3RfZ29vZCI6IDUwLCAidGVzdF9kZWZlY3QiOiAxMDB9LA0KICAgICJmcnl1bSI6IHsidHJhaW4iOiA0NTAsICJ0ZXN0X2dvb2QiOiA1MCwgInRlc3RfZGVmZWN0IjogMTAwfSwNCiAgICAibWFjYXJvbmkxIjogeyJ0cmFpbiI6IDkwMCwgInRlc3RfZ29vZCI6IDEwMCwgInRlc3RfZGVmZWN0IjogMTAwfSwNCiAgICAibWFjYXJvbmkyIjogeyJ0cmFpbiI6IDkwMCwgInRlc3RfZ29vZCI6IDEwMCwgInRlc3RfZGVmZWN0IjogMTAwfSwNCiAgICAicGNiMSI6IHsidHJhaW4iOiA5MDQsICJ0ZXN0X2dvb2QiOiAxMDAsICJ0ZXN0X2RlZmVjdCI6IDEwMH0sDQogICAgInBjYjIiOiB7InRyYWluIjogOTAxLCAidGVzdF9nb29kIjogMTAwLCAidGVzdF9kZWZlY3QiOiAxMDB9LA0KICAgICJwY2IzIjogeyJ0cmFpbiI6IDkwNSwgInRlc3RfZ29vZCI6IDEwMSwgInRlc3RfZGVmZWN0IjogMTAwfSwNCiAgICAicGNiNCI6IHsidHJhaW4iOiA5MDQsICJ0ZXN0X2dvb2QiOiAxMDEsICJ0ZXN0X2RlZmVjdCI6IDEwMH0sDQogICAgInBpcGVfZnJ5dW0iOiB7InRyYWluIjogNDUwLCAidGVzdF9nb29kIjogNTAsICJ0ZXN0X2RlZmVjdCI6IDEwMH0sDQp9DQoNClJVTEVTID0gKCJSMjAiLCAiUjYwIikNCkZQUl9TT1VSQ0VTID0gKCJvZmYiLCAiSCIpDQoNCg0KZGVmIF9yZXNvbHZlKHZhbHVlOiBzdHIgfCBQYXRoLCBiYXNlOiBQYXRoKSAtPiBQYXRoOg0KICAgIHBhdGggPSBQYXRoKHZhbHVlKQ0KICAgIHJldHVybiBwYXRoIGlmIHBhdGguaXNfYWJzb2x1dGUoKSBlbHNlIChiYXNlIC8gcGF0aCkucmVzb2x2ZSgpDQoNCg0KQGRhdGFjbGFzcw0KY2xhc3MgVmlzYUNvbmZpZzoNCiAgICBwYXlsb2FkOiBkaWN0W3N0ciwgQW55XQ0KICAgIGNvbmZpZ19wYXRoOiBQYXRoDQogICAgZGF0YV9yb290OiBQYXRoDQogICAgd29ya19yb290OiBQYXRoDQogICAgY2FjaGVfcm9vdDogUGF0aA0KDQogICAgZGVmIF9nZXQoc2VsZiwga2V5OiBzdHIpIC0+IEFueToNCiAgICAgICAgcmV0dXJuIHNlbGYucGF5bG9hZFtrZXldDQoNCiAgICBzZWVkID0gcHJvcGVydHkobGFtYmRhIHNlbGY6IGludChzZWxmLl9nZXQoInNlZWQiKSkpDQogICAgY2xhc3Nfc2VlZF9vZmZzZXQgPSBwcm9wZXJ0eShsYW1iZGEgc2VsZjogaW50KHNlbGYuX2dldCgiY2xhc3Nfc2VlZF9vZmZzZXQiKSkpDQogICAgdW5pdmVyc2Vfc2l6ZSA9IHByb3BlcnR5KGxhbWJkYSBzZWxmOiBpbnQoc2VsZi5fZ2V0KCJ1bml2ZXJzZV9zaXplIikpKQ0KICAgIG5fY2FsaWJyYXRpb25fc21hbGwgPSBwcm9wZXJ0eShsYW1iZGEgc2VsZjogaW50KHNlbGYuX2dldCgibl9jYWxpYnJhdGlvbl9zbWFsbCIpKSkNCiAgICBuX2NhbGlicmF0aW9uX2xhcmdlID0gcHJvcGVydHkobGFtYmRhIHNlbGY6IGludChzZWxmLl9nZXQoIm5fY2FsaWJyYXRpb25fbGFyZ2UiKSkpDQogICAgYWxwaGEgPSBwcm9wZXJ0eShsYW1iZGEgc2VsZjogZmxvYXQoc2VsZi5fZ2V0KCJhbHBoYSIpKSkNCiAgICBuX3JlcGVhdHMgPSBwcm9wZXJ0eShsYW1iZGEgc2VsZjogaW50KHNlbGYuX2dldCgibl9yZXBlYXRzIikpKQ0KICAgIHN0cmF0ZWdpZXMgPSBwcm9wZXJ0eShsYW1iZGEgc2VsZjogW3N0cihzKSBmb3IgcyBpbiBzZWxmLl9nZXQoInN0cmF0ZWdpZXMiKV0pDQogICAgYXVsY19uX21heCA9IHByb3BlcnR5KGxhbWJkYSBzZWxmOiBpbnQoc2VsZi5fZ2V0KCJhdWxjX25fbWF4IikpKQ0KICAgIHByYWN0aWNhbF90cHIgPSBwcm9wZXJ0eShsYW1iZGEgc2VsZjogZmxvYXQoc2VsZi5fZ2V0KCJwcmFjdGljYWxfdHByIikpKQ0KICAgIHByYWN0aWNhbF9mcHIgPSBwcm9wZXJ0eShsYW1iZGEgc2VsZjogZmxvYXQoc2VsZi5fZ2V0KCJwcmFjdGljYWxfZnByIikpKQ0KICAgIG5fc3Rhcl9taW5fcmVwZWF0cyA9IHByb3BlcnR5KGxhbWJkYSBzZWxmOiBpbnQoc2VsZi5fZ2V0KCJuX3N0YXJfbWluX3JlcGVhdHMiKSkpDQogICAgZnByX2F1bGNfbWFyZ2luID0gcHJvcGVydHkobGFtYmRhIHNlbGY6IGZsb2F0KHNlbGYuX2dldCgiZnByX2F1bGNfbWFyZ2luIikpKQ0KICAgIHRwcl9jb3N0X21hcmdpbiA9IHByb3BlcnR5KGxhbWJkYSBzZWxmOiBmbG9hdChzZWxmLl9nZXQoInRwcl9jb3N0X21hcmdpbiIpKSkNCiAgICBzdG9wcGluZ19kZWx0YXMgPSBwcm9wZXJ0eShsYW1iZGEgc2VsZjogW2Zsb2F0KHYpIGZvciB2IGluIHNlbGYuX2dldCgic3RvcHBpbmdfZGVsdGFzIildKQ0KICAgIHN0b3BwaW5nX2RlbHRhX21haW4gPSBwcm9wZXJ0eShsYW1iZGEgc2VsZjogZmxvYXQoc2VsZi5fZ2V0KCJzdG9wcGluZ19kZWx0YV9tYWluIikpKQ0KICAgIGJvb3RzdHJhcF9yZXBsaWNhdGVzID0gcHJvcGVydHkobGFtYmRhIHNlbGY6IGludChzZWxmLl9nZXQoImJvb3RzdHJhcF9yZXBsaWNhdGVzIikpKQ0KICAgIGJvb3RzdHJhcF9zZWVkID0gcHJvcGVydHkobGFtYmRhIHNlbGY6IGludChzZWxmLl9nZXQoImJvb3RzdHJhcF9zZWVkIikpKQ0KICAgIGJ1ZGdldF9zZWVkX3RhZyA9IHByb3BlcnR5KGxhbWJkYSBzZWxmOiBpbnQoc2VsZi5fZ2V0KCJidWRnZXRfc2VlZF90YWciKSkpDQogICAgYnVkZ2V0cyA9IHByb3BlcnR5KGxhbWJkYSBzZWxmOiBbaW50KHYpIGZvciB2IGluIHNlbGYuX2dldCgiYnVkZ2V0cyIpXSkNCiAgICBidWRnZXRfZ3JpZCA9IHByb3BlcnR5KGxhbWJkYSBzZWxmOiBbaW50KHYpIGZvciB2IGluIHNlbGYuX2dldCgiYnVkZ2V0X2dyaWQiKV0pDQogICAgYnVkZ2V0X21pbl9jYWxpYnJhdGlvbiA9IHByb3BlcnR5KGxhbWJkYSBzZWxmOiBpbnQoc2VsZi5fZ2V0KCJidWRnZXRfbWluX2NhbGlicmF0aW9uIikpKQ0KICAgIHBhY19lcHNpbG9uID0gcHJvcGVydHkobGFtYmRhIHNlbGY6IGZsb2F0KHNlbGYuX2dldCgicGFjX2Vwc2lsb24iKSkpDQogICAgcGFjX2RlbHRhID0gcHJvcGVydHkobGFtYmRhIHNlbGY6IGZsb2F0KHNlbGYuX2dldCgicGFjX2RlbHRhIikpKQ0KICAgIGJhY2tib25lID0gcHJvcGVydHkobGFtYmRhIHNlbGY6IHN0cihzZWxmLl9nZXQoImJhY2tib25lIikpKQ0KICAgIGxheWVycyA9IHByb3BlcnR5KGxhbWJkYSBzZWxmOiBbc3RyKHYpIGZvciB2IGluIHNlbGYuX2dldCgibGF5ZXJzIildKQ0KICAgIHJlc2l6ZSA9IHByb3BlcnR5KGxhbWJkYSBzZWxmOiBpbnQoc2VsZi5fZ2V0KCJyZXNpemUiKSkpDQogICAgY2VudGVyX2Nyb3AgPSBwcm9wZXJ0eShsYW1iZGEgc2VsZjogaW50KHNlbGYuX2dldCgiY2VudGVyX2Nyb3AiKSkpDQogICAgdXBzYW1wbGVfbW9kZSA9IHByb3BlcnR5KGxhbWJkYSBzZWxmOiBzdHIoc2VsZi5fZ2V0KCJ1cHNhbXBsZV9tb2RlIikpKQ0KICAgIGltYWdlbmV0X21lYW4gPSBwcm9wZXJ0eShsYW1iZGEgc2VsZjogW2Zsb2F0KHYpIGZvciB2IGluIHNlbGYuX2dldCgiaW1hZ2VuZXRfbWVhbiIpXSkNCiAgICBpbWFnZW5ldF9zdGQgPSBwcm9wZXJ0eShsYW1iZGEgc2VsZjogW2Zsb2F0KHYpIGZvciB2IGluIHNlbGYuX2dldCgiaW1hZ2VuZXRfc3RkIildKQ0KICAgIGZlYXR1cmVfYmF0Y2hfc2l6ZSA9IHByb3BlcnR5KGxhbWJkYSBzZWxmOiBpbnQoc2VsZi5fZ2V0KCJmZWF0dXJlX2JhdGNoX3NpemUiKSkpDQogICAgZGlzdGFuY2VfcXVlcnlfYmF0Y2ggPSBwcm9wZXJ0eShsYW1iZGEgc2VsZjogaW50KHNlbGYuX2dldCgiZGlzdGFuY2VfcXVlcnlfYmF0Y2giKSkpDQogICAgZGlzdGFuY2VfdHJhaW5fYmF0Y2ggPSBwcm9wZXJ0eShsYW1iZGEgc2VsZjogaW50KHNlbGYuX2dldCgiZGlzdGFuY2VfdHJhaW5fYmF0Y2giKSkpDQogICAgZTAgPSBwcm9wZXJ0eShsYW1iZGEgc2VsZjogZGljdChzZWxmLl9nZXQoImUwIikpKQ0KICAgIGNsYXNzZXMgPSBwcm9wZXJ0eShsYW1iZGEgc2VsZjogW3N0cih2KSBmb3IgdiBpbiBzZWxmLl9nZXQoImNsYXNzZXMiKV0pDQoNCiAgICBAcHJvcGVydHkNCiAgICBkZWYgcG9vbF9zaXplKHNlbGYpIC0+IGludDoNCiAgICAgICAgcmV0dXJuIHNlbGYudW5pdmVyc2Vfc2l6ZSAtIHNlbGYubl9jYWxpYnJhdGlvbl9sYXJnZQ0KDQogICAgQHByb3BlcnR5DQogICAgZGVmIG5fZ3JpZChzZWxmKSAtPiBsaXN0W2ludF06DQogICAgICAgICIiIkJhc2UgZ3JpZCBwbHVzIHRoZSBmdWxsIHBvb2wgfFB8LiIiIg0KICAgICAgICBncmlkID0gW2ludChuKSBmb3IgbiBpbiBzZWxmLl9nZXQoIm5fZ3JpZCIpIGlmIGludChuKSA8IHNlbGYucG9vbF9zaXplXQ0KICAgICAgICByZXR1cm4gZ3JpZCArIFtzZWxmLnBvb2xfc2l6ZV0NCg0KICAgIGRlZiBjbGFzc19pbmRleChzZWxmLCBjbGFzc19uYW1lOiBzdHIpIC0+IGludDoNCiAgICAgICAgdHJ5Og0KICAgICAgICAgICAgcmV0dXJuIHNlbGYuY2xhc3Nlcy5pbmRleChjbGFzc19uYW1lKQ0KICAgICAgICBleGNlcHQgVmFsdWVFcnJvciBhcyBleGM6DQogICAgICAgICAgICByYWlzZSBLZXlFcnJvcihmIlVua25vd24gVmlzQSBjbGFzczoge2NsYXNzX25hbWV9IikgZnJvbSBleGMNCg0KICAgIGRlZiBzZWVkX2tleShzZWxmLCBjbGFzc19uYW1lOiBzdHIpIC0+IGludDoNCiAgICAgICAgcmV0dXJuIHNlbGYuY2xhc3Nfc2VlZF9vZmZzZXQgKyBzZWxmLmNsYXNzX2luZGV4KGNsYXNzX25hbWUpDQoNCiAgICBkZWYgZ3JvdXBfb2Yoc2VsZiwgY2xhc3NfbmFtZTogc3RyKSAtPiBzdHI6DQogICAgICAgIHJldHVybiBHUk9VUFNbY2xhc3NfbmFtZV0NCg0KICAgIGRlZiBzZWxlY3RfY2xhc3NlcyhzZWxmLCByZXF1ZXN0ZWQ6IGxpc3Rbc3RyXSB8IE5vbmUpIC0+IGxpc3Rbc3RyXToNCiAgICAgICAgaWYgbm90IHJlcXVlc3RlZDoNCiAgICAgICAgICAgIHJldHVybiBsaXN0KHNlbGYuY2xhc3NlcykNCiAgICAgICAgdW5rbm93biA9IFtuYW1lIGZvciBuYW1lIGluIHJlcXVlc3RlZCBpZiBuYW1lIG5vdCBpbiBzZWxmLmNsYXNzZXNdDQogICAgICAgIGlmIHVua25vd246DQogICAgICAgICAgICByYWlzZSBLZXlFcnJvcihmIlVua25vd24gVmlzQSBjbGFzc2VzOiB7dW5rbm93bn0iKQ0KICAgICAgICByZXR1cm4gbGlzdChyZXF1ZXN0ZWQpDQoNCiAgICBAcHJvcGVydHkNCiAgICBkZWYgaGFzaChzZWxmKSAtPiBzdHI6DQogICAgICAgIHNjaWVudGlmaWMgPSB7a2V5OiB2YWx1ZSBmb3Iga2V5LCB2YWx1ZSBpbiBzZWxmLnBheWxvYWQuaXRlbXMoKSBpZiBrZXkgIT0gInBhdGhzIn0NCiAgICAgICAgYmxvYiA9IHsicGF5bG9hZCI6IHNjaWVudGlmaWMsICJleHBlY3RlZF9jb3VudHMiOiBFWFBFQ1RFRF9DT1VOVFMsICJncm91cHMiOiBHUk9VUFN9DQogICAgICAgIHJhdyA9IGpzb24uZHVtcHMoYmxvYiwgc29ydF9rZXlzPVRydWUsIHNlcGFyYXRvcnM9KCIsIiwgIjoiKSkNCiAgICAgICAgcmV0dXJuIGhhc2hsaWIuc2hhMjU2KHJhdy5lbmNvZGUoInV0Zi04IikpLmhleGRpZ2VzdCgpWzoxNl0NCg0KDQpkZWYgbG9hZF92aXNhX2NvbmZpZygNCiAgICBjb25maWdfcGF0aDogc3RyIHwgUGF0aCB8IE5vbmUgPSBOb25lLA0KICAgIGRhdGFfcm9vdDogc3RyIHwgUGF0aCB8IE5vbmUgPSBOb25lLA0KICAgIHdvcmtfcm9vdDogc3RyIHwgUGF0aCB8IE5vbmUgPSBOb25lLA0KICAgIGNhY2hlX3Jvb3Q6IHN0ciB8IFBhdGggfCBOb25lID0gTm9uZSwNCikgLT4gVmlzYUNvbmZpZzoNCiAgICByb290ID0gcHJvamVjdF9yb290KCkNCiAgICBwYXRoID0gUGF0aChjb25maWdfcGF0aCkgaWYgY29uZmlnX3BhdGggZWxzZSByb290IC8gImNvbmZpZ192aXNhLnlhbWwiDQogICAgaWYgbm90IHBhdGguaXNfZmlsZSgpOg0KICAgICAgICByYWlzZSBGaWxlTm90Rm91bmRFcnJvcihmIkNvbmZpZyBub3QgZm91bmQ6IHtwYXRofSIpDQogICAgcGF5bG9hZCA9IHlhbWwuc2FmZV9sb2FkKHBhdGgucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYtOCIpKQ0KICAgIGlmIG5vdCBpc2luc3RhbmNlKHBheWxvYWQsIGRpY3QpOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJDb25maWcgbXVzdCBiZSBhIG1hcHBpbmc6IHtwYXRofSIpDQogICAgX3ZhbGlkYXRlKHBheWxvYWQpDQogICAgcGF0aHMgPSBwYXlsb2FkWyJwYXRocyJdDQogICAgcmV0dXJuIFZpc2FDb25maWcoDQogICAgICAgIHBheWxvYWQ9cGF5bG9hZCwNCiAgICAgICAgY29uZmlnX3BhdGg9cGF0aC5yZXNvbHZlKCksDQogICAgICAgIGRhdGFfcm9vdD1fcmVzb2x2ZShkYXRhX3Jvb3Qgb3Igb3MuZW52aXJvbi5nZXQoIlZJU0FfREFUQSIpIG9yIHBhdGhzWyJkYXRhX3Jvb3QiXSwgcm9vdCksDQogICAgICAgIHdvcmtfcm9vdD1fcmVzb2x2ZSh3b3JrX3Jvb3Qgb3Igb3MuZW52aXJvbi5nZXQoIlZJU0FfV09SSyIpIG9yIHBhdGhzWyJ3b3JrX3Jvb3QiXSwgcm9vdCksDQogICAgICAgIGNhY2hlX3Jvb3Q9X3Jlc29sdmUoY2FjaGVfcm9vdCBvciBvcy5lbnZpcm9uLmdldCgiVklTQV9DQUNIRSIpIG9yIHBhdGhzWyJjYWNoZV9yb290Il0sIHJvb3QpLA0KICAgICkNCg0KDQpkZWYgX3ZhbGlkYXRlKHBheWxvYWQ6IGRpY3Rbc3RyLCBBbnldKSAtPiBOb25lOg0KICAgIGlmIHBheWxvYWQuZ2V0KCJkYXRhc2V0IikgIT0gInZpc2EiIG9yIGludChwYXlsb2FkLmdldCgicHJvdG9jb2xfdmVyc2lvbiIsIDApKSAhPSAyOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoImNvbmZpZ192aXNhLnlhbWwgbXVzdCBkZWNsYXJlIGRhdGFzZXQ6IHZpc2EgYW5kIHByb3RvY29sX3ZlcnNpb246IDIuIikNCiAgICBpZiBsaXN0KHBheWxvYWQuZ2V0KCJjbGFzc2VzIiwgW10pKSAhPSBDTEFTU0VTOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoImNsYXNzZXMgaW4gY29uZmlnX3Zpc2EueWFtbCBtdXN0IG1hdGNoIHRoZSBjYW5vbmljYWwgVmlzQSBvcmRlci4iKQ0KICAgIGlmIGludChwYXlsb2FkWyJuX2NhbGlicmF0aW9uX3NtYWxsIl0pICE9IDIwIG9yIGludChwYXlsb2FkWyJuX2NhbGlicmF0aW9uX2xhcmdlIl0pICE9IDYwOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIlRoZSBwcm90b2NvbCBmaXhlcyB8QzIwfCA9IDIwIGFuZCB8QzYwfCA9IDYwLiIpDQogICAgaWYgaW50KHBheWxvYWRbInVuaXZlcnNlX3NpemUiXSkgIT0gMzAwIG9yIGZsb2F0KHBheWxvYWRbImFscGhhIl0pICE9IDAuMDU6DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiVGhlIHByb3RvY29sIGZpeGVzIHxVfCA9IDMwMCBhbmQgYWxwaGEgPSAwLjA1LiIpDQogICAgaWYgaW50KHBheWxvYWRbImF1bGNfbl9tYXgiXSkgIT0gMzI6DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiTWFpbiBBVUxDIGlzIGZpeGVkIG9uIE4gPD0gMzIuIikNCiAgICBncmlkID0gW2ludChuKSBmb3IgbiBpbiBwYXlsb2FkWyJuX2dyaWQiXV0NCiAgICBpZiBncmlkICE9IHNvcnRlZChzZXQoZ3JpZCkpIG9yIDEgbm90IGluIGdyaWQgb3IgMzIgbm90IGluIGdyaWQ6DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIm5fZ3JpZCBtdXN0IGJlIHN0cmljdGx5IGluY3JlYXNpbmcgYW5kIGNvbnRhaW4gMSBhbmQgMzI6IHtncmlkfSIpDQogICAgaWYgInJhbmRvbSIgbm90IGluIHBheWxvYWRbInN0cmF0ZWdpZXMiXSBvciAia2NlbnRlciIgbm90IGluIHBheWxvYWRbInN0cmF0ZWdpZXMiXToNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJzdHJhdGVnaWVzIG11c3QgaW5jbHVkZSByYW5kb20gYW5kIGtjZW50ZXIuIikNCiAgICBzbWFsbGVzdF90cmFpbiA9IG1pbihjb3VudHNbInRyYWluIl0gZm9yIGNvdW50cyBpbiBFWFBFQ1RFRF9DT1VOVFMudmFsdWVzKCkpDQogICAgaWYgaW50KHBheWxvYWRbInVuaXZlcnNlX3NpemUiXSkgPj0gc21hbGxlc3RfdHJhaW46DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiVGhlIHVuaXZlcnNlIG11c3QgbGVhdmUgYSBub24tZW1wdHkgaGVsZC1vdXQgc2V0IEggaW4gZXZlcnkgY2xhc3MuIikNCg==", "commissioning/visa/data.py": "IiIiVmlzQSBjb252ZXJzaW9uIHRvIHRoZSBNVlRlYyBsYXlvdXQsIHRoZSBkYXRhIGNoZWNrLCB0aGUgdW5pdmVyc2UgVSwgdGhlIGhlbGQtb3V0IHNldCBILCBhbmQgQy9QIHNwbGl0cy4iIiINCg0KZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucw0KDQppbXBvcnQgY3N2DQppbXBvcnQgbG9nZ2luZw0KaW1wb3J0IHNodXRpbA0KZnJvbSBwYXRobGliIGltcG9ydCBQYXRoDQoNCmltcG9ydCBudW1weSBhcyBucA0KDQpmcm9tIGNvbW1pc3Npb25pbmcuZGF0YSBpbXBvcnQgaXRlcl9pbWFnZXMsIHJlbGF0aXZlX3BhdGgsIHNoYTFfZmlsZQ0KZnJvbSBjb21taXNzaW9uaW5nLmlvX3V0aWxzIGltcG9ydCByZWFkX2pzb24sIHdyaXRlX2pzb24NCmZyb20gY29tbWlzc2lvbmluZy52aXNhLmNvbmZpZyBpbXBvcnQgRVhQRUNURURfQ09VTlRTLCBWaXNhQ29uZmlnDQoNCkxPR0dFUiA9IGxvZ2dpbmcuZ2V0TG9nZ2VyKF9fbmFtZV9fKQ0KDQpTUExJVF9GT0xERVJTID0geygidHJhaW4iLCAibm9ybWFsIik6ICgidHJhaW4iLCAiZ29vZCIpLCAoInRlc3QiLCAibm9ybWFsIik6ICgidGVzdCIsICJnb29kIiksICgidGVzdCIsICJhbm9tYWx5Iik6ICgidGVzdCIsICJiYWQiKX0NCg0KDQojIENvbnZlcnNpb24gLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tDQoNCg0KZGVmIGZpbmRfc3BsaXRfY3N2KHJhd19yb290OiBQYXRoKSAtPiBQYXRoOg0KICAgIGNhbmRpZGF0ZXMgPSBbcmF3X3Jvb3QgLyAic3BsaXRfY3N2IiAvICIxY2xzLmNzdiIsIHJhd19yb290LnBhcmVudCAvICJzcGxpdF9jc3YiIC8gIjFjbHMuY3N2Il0NCiAgICBmb3IgY2FuZGlkYXRlIGluIGNhbmRpZGF0ZXM6DQogICAgICAgIGlmIGNhbmRpZGF0ZS5pc19maWxlKCk6DQogICAgICAgICAgICByZXR1cm4gY2FuZGlkYXRlDQogICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoDQogICAgICAgIGYic3BsaXRfY3N2LzFjbHMuY3N2IG5vdCBmb3VuZCBuZXh0IHRvIHtyYXdfcm9vdH0uIEl0IHNoaXBzIGluc2lkZSBWaXNBXzIwMjIwOTIyLnRhciBhbmQgaW4gIg0KICAgICAgICAiZ2l0aHViLmNvbS9hbWF6b24tc2NpZW5jZS9zcG90LWRpZmYvc3BsaXRfY3N2LiINCiAgICApDQoNCg0KZGVmIGNvbnZlcnQocmF3X3Jvb3Q6IFBhdGgsIG91dF9yb290OiBQYXRoLCBzcGxpdF9jc3Y6IFBhdGggfCBOb25lID0gTm9uZSkgLT4gZGljdFtzdHIsIGRpY3Rbc3RyLCBpbnRdXToNCiAgICAiIiJDb3B5IHRoZSBmaWxlcyBsaXN0ZWQgaW4gMWNscy5jc3YgaW50byB7Y2xhc3N9L3RyYWluL2dvb2QsIHRlc3QvZ29vZCwgdGVzdC9iYWQuIE5vIHJlLWVuY29kaW5nLiIiIg0KICAgIHJhd19yb290ID0gUGF0aChyYXdfcm9vdCkNCiAgICBvdXRfcm9vdCA9IFBhdGgob3V0X3Jvb3QpDQogICAgc3BsaXRfY3N2ID0gUGF0aChzcGxpdF9jc3YpIGlmIHNwbGl0X2NzdiBlbHNlIGZpbmRfc3BsaXRfY3N2KHJhd19yb290KQ0KICAgIGNvdW50czogZGljdFtzdHIsIGRpY3Rbc3RyLCBpbnRdXSA9IHt9DQogICAgdGFyZ2V0czogZGljdFtQYXRoLCBQYXRoXSA9IHt9DQogICAgd2l0aCBzcGxpdF9jc3Yub3BlbihuZXdsaW5lPSIiLCBlbmNvZGluZz0idXRmLTgiKSBhcyBoYW5kbGU6DQogICAgICAgIGZvciByb3cgaW4gY3N2LkRpY3RSZWFkZXIoaGFuZGxlKToNCiAgICAgICAgICAgIGtleSA9IChyb3dbInNwbGl0Il0uc3RyaXAoKSwgcm93WyJsYWJlbCJdLnN0cmlwKCkpDQogICAgICAgICAgICBpZiBrZXkgbm90IGluIFNQTElUX0ZPTERFUlM6DQogICAgICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYiVW5leHBlY3RlZCBzcGxpdC9sYWJlbCBpbiB7c3BsaXRfY3N2fToge2tleX0iKQ0KICAgICAgICAgICAgc291cmNlID0gcmF3X3Jvb3QgLyByb3dbImltYWdlIl0uc3RyaXAoKQ0KICAgICAgICAgICAgaWYgbm90IHNvdXJjZS5pc19maWxlKCk6DQogICAgICAgICAgICAgICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoZiJ7cm93WydvYmplY3QnXX06IGxpc3RlZCBpbWFnZSBpcyBtaXNzaW5nOiB7c291cmNlfSIpDQogICAgICAgICAgICBmaXJzdCwgc2Vjb25kID0gU1BMSVRfRk9MREVSU1trZXldDQogICAgICAgICAgICBkZXN0aW5hdGlvbiA9IG91dF9yb290IC8gcm93WyJvYmplY3QiXS5zdHJpcCgpIC8gZmlyc3QgLyBzZWNvbmQgLyBzb3VyY2UubmFtZQ0KICAgICAgICAgICAgaWYgZGVzdGluYXRpb24gaW4gdGFyZ2V0cyBhbmQgdGFyZ2V0c1tkZXN0aW5hdGlvbl0gIT0gc291cmNlOg0KICAgICAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIlR3byBWaXNBIGZpbGVzIG1hcCB0byB7ZGVzdGluYXRpb259OiB7dGFyZ2V0c1tkZXN0aW5hdGlvbl19IGFuZCB7c291cmNlfSIpDQogICAgICAgICAgICB0YXJnZXRzW2Rlc3RpbmF0aW9uXSA9IHNvdXJjZQ0KICAgICAgICAgICAgbmFtZSA9ICJ0ZXN0X2RlZmVjdCIgaWYgc2Vjb25kID09ICJiYWQiIGVsc2UgKCJ0cmFpbiIgaWYgZmlyc3QgPT0gInRyYWluIiBlbHNlICJ0ZXN0X2dvb2QiKQ0KICAgICAgICAgICAgY291bnRzLnNldGRlZmF1bHQocm93WyJvYmplY3QiXS5zdHJpcCgpLCB7InRyYWluIjogMCwgInRlc3RfZ29vZCI6IDAsICJ0ZXN0X2RlZmVjdCI6IDB9KVtuYW1lXSArPSAxDQogICAgZm9yIGRlc3RpbmF0aW9uLCBzb3VyY2UgaW4gdGFyZ2V0cy5pdGVtcygpOg0KICAgICAgICBpZiBkZXN0aW5hdGlvbi5pc19maWxlKCkgYW5kIGRlc3RpbmF0aW9uLnN0YXQoKS5zdF9zaXplID09IHNvdXJjZS5zdGF0KCkuc3Rfc2l6ZToNCiAgICAgICAgICAgIGNvbnRpbnVlDQogICAgICAgIGRlc3RpbmF0aW9uLnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpDQogICAgICAgIHNodXRpbC5jb3B5ZmlsZShzb3VyY2UsIGRlc3RpbmF0aW9uKQ0KICAgIExPR0dFUi5pbmZvKCJDb252ZXJ0ZWQgJWQgVmlzQSBmaWxlcyBpbnRvICVzIiwgbGVuKHRhcmdldHMpLCBvdXRfcm9vdCkNCiAgICByZXR1cm4gY291bnRzDQoNCg0KIyBEYXRhIGNoZWNrIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQ0KDQoNCmRlZiBjb2xsZWN0X2NsYXNzKGRhdGFfcm9vdDogUGF0aCwgY2xhc3NfbmFtZTogc3RyKSAtPiBkaWN0W3N0ciwgbGlzdFtQYXRoXV06DQogICAgcm9vdCA9IFBhdGgoZGF0YV9yb290KSAvIGNsYXNzX25hbWUNCiAgICBpZiBub3Qgcm9vdC5pc19kaXIoKToNCiAgICAgICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoZiJ7Y2xhc3NfbmFtZX06IGNvbnZlcnRlZCBWaXNBIGRpcmVjdG9yeSBub3QgZm91bmQ6IHtyb290fSIpDQogICAgcmV0dXJuIHsNCiAgICAgICAgInRyYWluIjogaXRlcl9pbWFnZXMocm9vdCAvICJ0cmFpbiIgLyAiZ29vZCIpLA0KICAgICAgICAidGVzdF9nb29kIjogaXRlcl9pbWFnZXMocm9vdCAvICJ0ZXN0IiAvICJnb29kIiksDQogICAgICAgICJ0ZXN0X2RlZmVjdCI6IGl0ZXJfaW1hZ2VzKHJvb3QgLyAidGVzdCIgLyAiYmFkIiksDQogICAgfQ0KDQoNCmRlZiBjaGVja19jbGFzcyhjZmc6IFZpc2FDb25maWcsIGNsYXNzX25hbWU6IHN0cikgLT4gZGljdDoNCiAgICAiIiJPZmZpY2lhbCBjb3VudHMgYmVmb3JlIGRlZHVwbGljYXRpb24sIHRoZW4gdGhlIGR1cGxpY2F0ZSBwb2xpY3kgb2YgcHJlcmVnaXN0cmF0aW9uX3Zpc2EubWQgc2VjdGlvbiAyLiIiIg0KICAgIGNvbGxlY3RlZCA9IGNvbGxlY3RfY2xhc3MoY2ZnLmRhdGFfcm9vdCwgY2xhc3NfbmFtZSkNCiAgICBmb3VuZCA9IHtuYW1lOiBsZW4ocGF0aHMpIGZvciBuYW1lLCBwYXRocyBpbiBjb2xsZWN0ZWQuaXRlbXMoKX0NCiAgICBpZiBmb3VuZCAhPSBFWFBFQ1RFRF9DT1VOVFNbY2xhc3NfbmFtZV06DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIntjbGFzc19uYW1lfTogZXhwZWN0ZWQgb2ZmaWNpYWwgY291bnRzIHtFWFBFQ1RFRF9DT1VOVFNbY2xhc3NfbmFtZV19LCBmb3VuZCB7Zm91bmR9IikNCiAgICBkaWdlc3RzID0ge25hbWU6IFtzaGExX2ZpbGUocGF0aCkgZm9yIHBhdGggaW4gcGF0aHNdIGZvciBuYW1lLCBwYXRocyBpbiBjb2xsZWN0ZWQuaXRlbXMoKX0NCiAgICB0cmFpbl9zZXQgPSBzZXQoZGlnZXN0c1sidHJhaW4iXSkNCiAgICBnb29kX3NldCA9IHNldChkaWdlc3RzWyJ0ZXN0X2dvb2QiXSkNCiAgICBkZWZlY3Rfc2V0ID0gc2V0KGRpZ2VzdHNbInRlc3RfZGVmZWN0Il0pDQogICAgbGVha3MgPSBzb3J0ZWQoKHRyYWluX3NldCAmIChnb29kX3NldCB8IGRlZmVjdF9zZXQpKSB8IChnb29kX3NldCAmIGRlZmVjdF9zZXQpKQ0KICAgIGlmIGxlYWtzOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJ7Y2xhc3NfbmFtZX06IGlkZW50aWNhbCBmaWxlcyBhY3Jvc3MgdHJhaW4vdGVzdCBvciB0ZXN0IGdvb2QvYW5vbWFseToge2xlYWtzWzo1XX0iKQ0KICAgIGtlcHQ6IGxpc3Rbc3RyXSA9IFtdDQogICAgZHJvcHBlZDogbGlzdFtkaWN0W3N0ciwgc3RyXV0gPSBbXQ0KICAgIGZpcnN0X3NlZW46IGRpY3Rbc3RyLCBzdHJdID0ge30NCiAgICBmb3IgcGF0aCwgZGlnZXN0IGluIHppcChjb2xsZWN0ZWRbInRyYWluIl0sIGRpZ2VzdHNbInRyYWluIl0pOg0KICAgICAgICByZWxhdGl2ZSA9IHJlbGF0aXZlX3BhdGgocGF0aCwgY2ZnLmRhdGFfcm9vdCkNCiAgICAgICAgaWYgZGlnZXN0IGluIGZpcnN0X3NlZW46DQogICAgICAgICAgICBkcm9wcGVkLmFwcGVuZCh7InBhdGgiOiByZWxhdGl2ZSwgImR1cGxpY2F0ZV9vZiI6IGZpcnN0X3NlZW5bZGlnZXN0XX0pDQogICAgICAgICAgICBjb250aW51ZQ0KICAgICAgICBmaXJzdF9zZWVuW2RpZ2VzdF0gPSByZWxhdGl2ZQ0KICAgICAgICBrZXB0LmFwcGVuZChyZWxhdGl2ZSkNCiAgICB0ZXN0X2R1cGxpY2F0ZXMgPSB7DQogICAgICAgIG5hbWU6IGxlbihkaWdlc3RzW25hbWVdKSAtIGxlbihzZXQoZGlnZXN0c1tuYW1lXSkpIGZvciBuYW1lIGluICgidGVzdF9nb29kIiwgInRlc3RfZGVmZWN0IikNCiAgICB9DQogICAgaWYgbGVuKGtlcHQpIDw9IGNmZy51bml2ZXJzZV9zaXplOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJ7Y2xhc3NfbmFtZX06IHtsZW4oa2VwdCl9IHVuaXF1ZSB0cmFpbiBpbWFnZXMgbGVhdmUgbm8gaGVsZC1vdXQgc2V0IEguIikNCiAgICByZXR1cm4gew0KICAgICAgICAiY2xhc3NfbmFtZSI6IGNsYXNzX25hbWUsDQogICAgICAgICJvZmZpY2lhbF9jb3VudHMiOiBmb3VuZCwNCiAgICAgICAgInRyYWluX3VuaXF1ZSI6IGxlbihrZXB0KSwNCiAgICAgICAgInRyYWluX2Ryb3BwZWQiOiBkcm9wcGVkLA0KICAgICAgICAidGVzdF9kdXBsaWNhdGVzX2tlcHQiOiB0ZXN0X2R1cGxpY2F0ZXMsDQogICAgICAgICJob2xkb3V0X3NpemUiOiBsZW4oa2VwdCkgLSBjZmcudW5pdmVyc2Vfc2l6ZSwNCiAgICAgICAgInRyYWluIjoga2VwdCwNCiAgICAgICAgInRlc3RfZ29vZCI6IFtyZWxhdGl2ZV9wYXRoKHBhdGgsIGNmZy5kYXRhX3Jvb3QpIGZvciBwYXRoIGluIGNvbGxlY3RlZFsidGVzdF9nb29kIl1dLA0KICAgICAgICAidGVzdF9kZWZlY3QiOiBbcmVsYXRpdmVfcGF0aChwYXRoLCBjZmcuZGF0YV9yb290KSBmb3IgcGF0aCBpbiBjb2xsZWN0ZWRbInRlc3RfZGVmZWN0Il1dLA0KICAgIH0NCg0KDQpkZWYgZGF0YV9jaGVja19wYXRoKGNmZzogVmlzYUNvbmZpZykgLT4gUGF0aDoNCiAgICByZXR1cm4gY2ZnLndvcmtfcm9vdCAvICJkYXRhX2NoZWNrLmpzb24iDQoNCg0KZGVmIHJ1bl9kYXRhX2NoZWNrKGNmZzogVmlzYUNvbmZpZywgY2xhc3NlczogbGlzdFtzdHJdKSAtPiBkaWN0Og0KICAgIHBhdGggPSBkYXRhX2NoZWNrX3BhdGgoY2ZnKQ0KICAgIGV4aXN0aW5nID0gcmVhZF9qc29uKHBhdGgpIGlmIHBhdGguaXNfZmlsZSgpIGVsc2Uge30NCiAgICBpZiBleGlzdGluZyBhbmQgZXhpc3RpbmcuZ2V0KCJjb25maWdfaGFzaCIpICE9IGNmZy5oYXNoOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJ7cGF0aH0gd2FzIHdyaXR0ZW4gZm9yIGFub3RoZXIgY29uZmlnIGhhc2guIikNCiAgICBwZXJfY2xhc3MgPSBkaWN0KGV4aXN0aW5nLmdldCgiY2xhc3NlcyIsIHt9KSkNCiAgICBmb3IgY2xhc3NfbmFtZSBpbiBjbGFzc2VzOg0KICAgICAgICBpZiBjbGFzc19uYW1lIGluIHBlcl9jbGFzczoNCiAgICAgICAgICAgIGNvbnRpbnVlDQogICAgICAgIExPR0dFUi5pbmZvKCJEYXRhIGNoZWNrICVzIiwgY2xhc3NfbmFtZSkNCiAgICAgICAgcGVyX2NsYXNzW2NsYXNzX25hbWVdID0gY2hlY2tfY2xhc3MoY2ZnLCBjbGFzc19uYW1lKQ0KICAgIHBheWxvYWQgPSB7ImNvbmZpZ19oYXNoIjogY2ZnLmhhc2gsICJjbGFzc2VzIjogcGVyX2NsYXNzfQ0KICAgIHdyaXRlX2pzb24ocGF0aCwgcGF5bG9hZCkNCiAgICByZXR1cm4gcGF5bG9hZA0KDQoNCmRlZiBjaGVja2VkX2NsYXNzKGNmZzogVmlzYUNvbmZpZywgY2xhc3NfbmFtZTogc3RyKSAtPiBkaWN0Og0KICAgIHBhdGggPSBkYXRhX2NoZWNrX3BhdGgoY2ZnKQ0KICAgIGlmIG5vdCBwYXRoLmlzX2ZpbGUoKToNCiAgICAgICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoImRhdGFfY2hlY2suanNvbiBpcyBtaXNzaW5nLiBSdW4gdGhlIHByZXBhcmUgc3RhZ2UgZmlyc3QuIikNCiAgICBwYXlsb2FkID0gcmVhZF9qc29uKHBhdGgpDQogICAgaWYgcGF5bG9hZC5nZXQoImNvbmZpZ19oYXNoIikgIT0gY2ZnLmhhc2ggb3IgY2xhc3NfbmFtZSBub3QgaW4gcGF5bG9hZC5nZXQoImNsYXNzZXMiLCB7fSk6DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmImRhdGFfY2hlY2suanNvbiBkb2VzIG5vdCBjb3ZlciB7Y2xhc3NfbmFtZX0gZm9yIHRoaXMgY29uZmlnIGhhc2guIikNCiAgICByZXR1cm4gcGF5bG9hZFsiY2xhc3NlcyJdW2NsYXNzX25hbWVdDQoNCg0KIyBVbml2ZXJzZSBhbmQgc3BsaXRzIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0NCg0KDQpkZWYgdW5pdmVyc2VfaW5kaWNlcyhjZmc6IFZpc2FDb25maWcsIGNsYXNzX25hbWU6IHN0ciwgbl90cmFpbjogaW50KSAtPiBucC5uZGFycmF5Og0KICAgIHJuZyA9IG5wLnJhbmRvbS5kZWZhdWx0X3JuZyhucC5yYW5kb20uU2VlZFNlcXVlbmNlKFtjZmcuc2VlZCwgY2ZnLnNlZWRfa2V5KGNsYXNzX25hbWUpXSkpDQogICAgcmV0dXJuIG5wLnNvcnQocm5nLmNob2ljZShuX3RyYWluLCBzaXplPWNmZy51bml2ZXJzZV9zaXplLCByZXBsYWNlPUZhbHNlKS5hc3R5cGUoaW50KSkNCg0KDQpkZWYgc3BsaXRfc2VlZHMoY2ZnOiBWaXNhQ29uZmlnLCBjbGFzc19uYW1lOiBzdHIsIHJlcDogaW50KSAtPiB0dXBsZVtucC5yYW5kb20uR2VuZXJhdG9yLCAuLi5dOg0KICAgIHNlcXVlbmNlID0gbnAucmFuZG9tLlNlZWRTZXF1ZW5jZShbY2ZnLnNlZWQsIGNmZy5zZWVkX2tleShjbGFzc19uYW1lKSwgaW50KHJlcCldKQ0KICAgIHJldHVybiB0dXBsZShucC5yYW5kb20uZGVmYXVsdF9ybmcoY2hpbGQpIGZvciBjaGlsZCBpbiBzZXF1ZW5jZS5zcGF3bigzKSkNCg0KDQpkZWYgYnVpbGRfc3BsaXQoY2ZnOiBWaXNhQ29uZmlnLCBjbGFzc19uYW1lOiBzdHIsIHJlcDogaW50LCBjaGVja2VkOiBkaWN0KSAtPiBkaWN0Og0KICAgIHRyYWluID0gbGlzdChjaGVja2VkWyJ0cmFpbiJdKQ0KICAgIGNob3NlbiA9IHVuaXZlcnNlX2luZGljZXMoY2ZnLCBjbGFzc19uYW1lLCBsZW4odHJhaW4pKQ0KICAgIHVuaXZlcnNlID0gW3RyYWluW2ludChpKV0gZm9yIGkgaW4gY2hvc2VuXQ0KICAgIGluX3VuaXZlcnNlID0gc2V0KHVuaXZlcnNlKQ0KICAgIGhvbGRvdXQgPSBbcGF0aCBmb3IgcGF0aCBpbiB0cmFpbiBpZiBwYXRoIG5vdCBpbiBpbl91bml2ZXJzZV0NCiAgICBzcGxpdF9ybmcsIF9yYW5kb21fcm5nLCBfa2NlbnRlcl9ybmcgPSBzcGxpdF9zZWVkcyhjZmcsIGNsYXNzX25hbWUsIHJlcCkNCiAgICBwb3NpdGlvbnMgPSBzcGxpdF9ybmcuY2hvaWNlKGxlbih1bml2ZXJzZSksIHNpemU9Y2ZnLm5fY2FsaWJyYXRpb25fbGFyZ2UsIHJlcGxhY2U9RmFsc2UpDQogICAgY2FsaWJyYXRpb24gPSBbdW5pdmVyc2VbaW50KGkpXSBmb3IgaSBpbiBwb3NpdGlvbnNdDQogICAgdGFrZW4gPSBzZXQoY2FsaWJyYXRpb24pDQogICAgcG9vbCA9IFtwYXRoIGZvciBwYXRoIGluIHVuaXZlcnNlIGlmIHBhdGggbm90IGluIHRha2VuXQ0KICAgIHBheWxvYWQgPSB7DQogICAgICAgICJjb25maWdfaGFzaCI6IGNmZy5oYXNoLA0KICAgICAgICAiY2xhc3NfbmFtZSI6IGNsYXNzX25hbWUsDQogICAgICAgICJjbGFzc19pbmRleCI6IGNmZy5jbGFzc19pbmRleChjbGFzc19uYW1lKSwNCiAgICAgICAgInJlcCI6IGludChyZXApLA0KICAgICAgICAic2VlZCI6IGNmZy5zZWVkLA0KICAgICAgICAidW5pdmVyc2UiOiB1bml2ZXJzZSwNCiAgICAgICAgImhvbGRvdXQiOiBob2xkb3V0LA0KICAgICAgICAiY2FsaWJyYXRpb24iOiBjYWxpYnJhdGlvbiwNCiAgICAgICAgInBvb2wiOiBwb29sLA0KICAgICAgICAidGVzdF9nb29kIjogbGlzdChjaGVja2VkWyJ0ZXN0X2dvb2QiXSksDQogICAgICAgICJ0ZXN0X2RlZmVjdCI6IGxpc3QoY2hlY2tlZFsidGVzdF9kZWZlY3QiXSksDQogICAgfQ0KICAgIGFzc2VydF9zcGxpdChjZmcsIHBheWxvYWQpDQogICAgcmV0dXJuIHBheWxvYWQNCg0KDQpkZWYgYXNzZXJ0X3NwbGl0KGNmZzogVmlzYUNvbmZpZywgc3BsaXQ6IGRpY3QpIC0+IE5vbmU6DQogICAgbmFtZSA9IHNwbGl0WyJjbGFzc19uYW1lIl0NCiAgICB1bml2ZXJzZSwgaG9sZG91dCA9IHNwbGl0WyJ1bml2ZXJzZSJdLCBzcGxpdFsiaG9sZG91dCJdDQogICAgY2FsaWJyYXRpb24sIHBvb2wgPSBzcGxpdFsiY2FsaWJyYXRpb24iXSwgc3BsaXRbInBvb2wiXQ0KICAgIHRlc3QgPSBsaXN0KHNwbGl0WyJ0ZXN0X2dvb2QiXSkgKyBsaXN0KHNwbGl0WyJ0ZXN0X2RlZmVjdCJdKQ0KICAgIGlmIGxlbih1bml2ZXJzZSkgIT0gY2ZnLnVuaXZlcnNlX3NpemUgb3IgbGVuKHNldCh1bml2ZXJzZSkpICE9IGxlbih1bml2ZXJzZSk6DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIntuYW1lfTogfFV8IG11c3QgYmUge2NmZy51bml2ZXJzZV9zaXplfSB3aXRob3V0IGR1cGxpY2F0ZXMuIikNCiAgICBpZiBsZW4oY2FsaWJyYXRpb24pICE9IGNmZy5uX2NhbGlicmF0aW9uX2xhcmdlIG9yIGxlbihzZXQoY2FsaWJyYXRpb24pKSAhPSBsZW4oY2FsaWJyYXRpb24pOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJ7bmFtZX06IHxDNjB8IG11c3QgYmUge2NmZy5uX2NhbGlicmF0aW9uX2xhcmdlfSB3aXRob3V0IGR1cGxpY2F0ZXMuIikNCiAgICBpZiBsZW4ocG9vbCkgIT0gY2ZnLnBvb2xfc2l6ZSBvciBzZXQocG9vbCkgfCBzZXQoY2FsaWJyYXRpb24pICE9IHNldCh1bml2ZXJzZSkgb3Igc2V0KHBvb2wpICYgc2V0KGNhbGlicmF0aW9uKToNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYie25hbWV9OiBDNjAgYW5kIFAgbXVzdCBwYXJ0aXRpb24gVSB3aXRoIHxQfCA9IHtjZmcucG9vbF9zaXplfS4iKQ0KICAgIGlmIG5vdCBzZXQoY2FsaWJyYXRpb25bOiBjZmcubl9jYWxpYnJhdGlvbl9zbWFsbF0pIDw9IHNldChjYWxpYnJhdGlvbik6DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIntuYW1lfTogQzIwIG11c3QgYmUgYSBzdWJzZXQgb2YgQzYwLiIpDQogICAgaWYgbm90IGhvbGRvdXQgb3Igc2V0KGhvbGRvdXQpICYgc2V0KHVuaXZlcnNlKSBvciBzZXQoaG9sZG91dCkgJiBzZXQodGVzdCkgb3Igc2V0KHVuaXZlcnNlKSAmIHNldCh0ZXN0KToNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYie25hbWV9OiBVLCBILCBhbmQgdGhlIG9mZmljaWFsIHRlc3Qgc2V0IG11c3QgYmUgbm9uLWVtcHR5IGFuZCBwYWlyd2lzZSBkaXNqb2ludC4iKQ0KICAgIGlmIGxlbih0ZXN0KSAhPSBsZW4oc2V0KHRlc3QpKToNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYie25hbWV9OiBkdXBsaWNhdGUgdGVzdCBwYXRocy4iKQ0KDQoNCmRlZiBzcGxpdF9wYXRoKGNmZzogVmlzYUNvbmZpZywgY2xhc3NfbmFtZTogc3RyLCByZXA6IGludCkgLT4gUGF0aDoNCiAgICByZXR1cm4gY2ZnLndvcmtfcm9vdCAvICJzcGxpdHMiIC8gY2xhc3NfbmFtZSAvIGYicmVwX3tyZXA6MDJkfS5qc29uIg0KDQoNCmRlZiBwcmVwYXJlKGNmZzogVmlzYUNvbmZpZywgY2xhc3NlczogbGlzdFtzdHJdKSAtPiBOb25lOg0KICAgIHJlcG9ydCA9IHJ1bl9kYXRhX2NoZWNrKGNmZywgY2xhc3NlcykNCiAgICBmb3IgY2xhc3NfbmFtZSBpbiBjbGFzc2VzOg0KICAgICAgICBjaGVja2VkID0gcmVwb3J0WyJjbGFzc2VzIl1bY2xhc3NfbmFtZV0NCiAgICAgICAgZm9yIHJlcCBpbiByYW5nZShjZmcubl9yZXBlYXRzKToNCiAgICAgICAgICAgIHBhdGggPSBzcGxpdF9wYXRoKGNmZywgY2xhc3NfbmFtZSwgcmVwKQ0KICAgICAgICAgICAgcGF5bG9hZCA9IGJ1aWxkX3NwbGl0KGNmZywgY2xhc3NfbmFtZSwgcmVwLCBjaGVja2VkKQ0KICAgICAgICAgICAgaWYgcGF0aC5pc19maWxlKCk6DQogICAgICAgICAgICAgICAgaWYgcmVhZF9qc29uKHBhdGgpICE9IHBheWxvYWQ6DQogICAgICAgICAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIlJlZnVzaW5nIHRvIG92ZXJ3cml0ZSBhbiBleGlzdGluZyBzcGxpdCB0aGF0IGRvZXMgbm90IG1hdGNoOiB7cGF0aH0iKQ0KICAgICAgICAgICAgICAgIGNvbnRpbnVlDQogICAgICAgICAgICB3cml0ZV9qc29uKHBhdGgsIHBheWxvYWQpDQoNCg0KZGVmIGxvYWRfc3BsaXQoY2ZnOiBWaXNhQ29uZmlnLCBjbGFzc19uYW1lOiBzdHIsIHJlcDogaW50KSAtPiBkaWN0Og0KICAgIHBhdGggPSBzcGxpdF9wYXRoKGNmZywgY2xhc3NfbmFtZSwgcmVwKQ0KICAgIGlmIG5vdCBwYXRoLmlzX2ZpbGUoKToNCiAgICAgICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoZiJNaXNzaW5nIHNwbGl0IHtwYXRofS4gUnVuIHByZXBhcmUgZmlyc3QuIikNCiAgICBwYXlsb2FkID0gcmVhZF9qc29uKHBhdGgpDQogICAgaWYgcGF5bG9hZC5nZXQoImNvbmZpZ19oYXNoIikgIT0gY2ZnLmhhc2g6DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIlNwbGl0IGhhc2ggbWlzbWF0Y2ggZm9yIHtjbGFzc19uYW1lfSByZXAge3JlcH0uIikNCiAgICBhc3NlcnRfc3BsaXQoY2ZnLCBwYXlsb2FkKQ0KICAgIHJldHVybiBwYXlsb2FkDQoNCg0KZGVmIGltYWdlX29yZGVyKHNwbGl0OiBkaWN0KSAtPiBsaXN0W3N0cl06DQogICAgIiIiUm93IG9yZGVyIG9mIGV2ZXJ5IGNhY2hlOiBVLCBILCBvZmZpY2lhbCBnb29kIHRlc3QsIG9mZmljaWFsIGFub21hbGllcy4gQ29sdW1ucyBhcmUgVS4iIiINCiAgICByZXR1cm4gbGlzdChzcGxpdFsidW5pdmVyc2UiXSkgKyBsaXN0KHNwbGl0WyJob2xkb3V0Il0pICsgbGlzdChzcGxpdFsidGVzdF9nb29kIl0pICsgbGlzdChzcGxpdFsidGVzdF9kZWZlY3QiXSkNCg0KDQpkZWYgcm93X2Jsb2NrcyhzcGxpdDogZGljdCkgLT4gZGljdFtzdHIsIG5wLm5kYXJyYXldOg0KICAgIG5fdSwgbl9oID0gbGVuKHNwbGl0WyJ1bml2ZXJzZSJdKSwgbGVuKHNwbGl0WyJob2xkb3V0Il0pDQogICAgbl9nLCBuX2QgPSBsZW4oc3BsaXRbInRlc3RfZ29vZCJdKSwgbGVuKHNwbGl0WyJ0ZXN0X2RlZmVjdCJdKQ0KICAgIHJldHVybiB7DQogICAgICAgICJ1bml2ZXJzZSI6IG5wLmFyYW5nZSgwLCBuX3UpLA0KICAgICAgICAiaG9sZG91dCI6IG5wLmFyYW5nZShuX3UsIG5fdSArIG5faCksDQogICAgICAgICJ0ZXN0X2dvb2QiOiBucC5hcmFuZ2Uobl91ICsgbl9oLCBuX3UgKyBuX2ggKyBuX2cpLA0KICAgICAgICAidGVzdF9kZWZlY3QiOiBucC5hcmFuZ2Uobl91ICsgbl9oICsgbl9nLCBuX3UgKyBuX2ggKyBuX2cgKyBuX2QpLA0KICAgIH0NCg==", "commissioning/visa/e0.py": "IiIiRTAgZ2F0ZSBmb3IgVmlzQTogQW5vbWFsaWIgcmVmZXJlbmNlIG9uIFUgKEUwYSksIG93biBjYWNoZSB2ZXJzdXMgQW5vbWFsaWIgd2l0aCBiYW5rIFUgKEUwYiksIGludGVncml0eS4iIiINCg0KZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucw0KDQppbXBvcnQgbG9nZ2luZw0KZnJvbSBwYXRobGliIGltcG9ydCBQYXRoDQoNCmltcG9ydCBudW1weSBhcyBucA0KZnJvbSBzY2lweS5zdGF0cyBpbXBvcnQgc3BlYXJtYW5yDQoNCmZyb20gY29tbWlzc2lvbmluZy5lMCBpbXBvcnQgKA0KICAgIF9hdXJvYywNCiAgICBfYmF0Y2hlcywNCiAgICBfZml0X2Fub21hbGliLA0KICAgIF9taW5tYXgsDQogICAgX3Bhc3NlZF9lMGIsDQogICAgX3ByZWRpY3Rpb25fc2NvcmVzLA0KICAgIF9yZWxlYXNlLA0KICAgIGNoZWNrX2Fub21hbGliX3Vwc2FtcGxpbmcsDQogICAgcmVjb3JkX3NhbXBsZXJfc291cmNlLA0KKQ0KZnJvbSBjb21taXNzaW9uaW5nLmludGVncml0eSBpbXBvcnQgYXNzZXJ0X21ldGhvZHNfYWdyZWUsIGNoZWNrX2NvcnJ1cHRlZF9pbnB1dA0KZnJvbSBjb21taXNzaW9uaW5nLmlvX3V0aWxzIGltcG9ydCByZWFkX2pzb24sIHdyaXRlX2pzb24NCmZyb20gY29tbWlzc2lvbmluZy5wYXRjaGNvcmUgaW1wb3J0IHNjb3Jlc19mb3JfYmFuaw0KZnJvbSBjb21taXNzaW9uaW5nLnNlbGVjdGlvbiBpbXBvcnQgcmFuZG9tX29yZGVyDQpmcm9tIGNvbW1pc3Npb25pbmcudmlzYS5jYWNoZXMgaW1wb3J0IGxvYWRfZGlzdGFuY2VzLCBsb2FkX2ZlYXR1cmVzDQpmcm9tIGNvbW1pc3Npb25pbmcudmlzYS5jb25maWcgaW1wb3J0IFZpc2FDb25maWcNCmZyb20gY29tbWlzc2lvbmluZy52aXNhLmRhdGEgaW1wb3J0IGltYWdlX29yZGVyLCBsb2FkX3NwbGl0LCByb3dfYmxvY2tzLCBzcGxpdF9zZWVkcw0KDQpMT0dHRVIgPSBsb2dnaW5nLmdldExvZ2dlcihfX25hbWVfXykNCg0KDQpkZWYgZ2F0ZV9wYXRoKGNmZzogVmlzYUNvbmZpZykgLT4gUGF0aDoNCiAgICByZXR1cm4gY2ZnLndvcmtfcm9vdCAvICJlMCIgLyAiZ2F0ZS5qc29uIg0KDQoNCmRlZiBfY2xhc3NfcGF0aChjZmc6IFZpc2FDb25maWcsIGNsYXNzX25hbWU6IHN0cikgLT4gUGF0aDoNCiAgICByZXR1cm4gY2ZnLndvcmtfcm9vdCAvICJlMCIgLyBmIntjbGFzc19uYW1lfS5qc29uIg0KDQoNCmRlZiBfdGVzdChzcGxpdDogZGljdCkgLT4gdHVwbGVbbGlzdFtzdHJdLCBucC5uZGFycmF5XToNCiAgICBwYXRocyA9IGxpc3Qoc3BsaXRbInRlc3RfZ29vZCJdKSArIGxpc3Qoc3BsaXRbInRlc3RfZGVmZWN0Il0pDQogICAgbGFiZWxzID0gbnAuYXJyYXkoWzBdICogbGVuKHNwbGl0WyJ0ZXN0X2dvb2QiXSkgKyBbMV0gKiBsZW4oc3BsaXRbInRlc3RfZGVmZWN0Il0pLCBkdHlwZT1pbnQpDQogICAgcmV0dXJuIHBhdGhzLCBsYWJlbHMNCg0KDQpkZWYgX3Njb3JlX3dpdGhfYW5vbWFsaWIoY2ZnOiBWaXNhQ29uZmlnLCBjbGFzc19uYW1lOiBzdHIsIG1vZGVsLCBkZXZpY2UsIHBhdGhzOiBsaXN0W3N0cl0pIC0+IG5wLm5kYXJyYXk6DQogICAgaW1wb3J0IHRvcmNoDQoNCiAgICBzY29yZXMgPSBbXQ0KICAgIG1vZGVsLmV2YWwoKQ0KICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOg0KICAgICAgICBmb3IgYmF0Y2ggaW4gX2JhdGNoZXMoY2ZnLCBjbGFzc19uYW1lLCBwYXRocywgYmF0Y2hfc2l6ZT0yKToNCiAgICAgICAgICAgIHNjb3Jlcy5hcHBlbmQoX3ByZWRpY3Rpb25fc2NvcmVzKG1vZGVsKGJhdGNoLnRvKGRldmljZSkpKSkNCiAgICBzdGFja2VkID0gbnAuY29uY2F0ZW5hdGUoc2NvcmVzKQ0KICAgIGlmIHN0YWNrZWQuc2hhcGVbMF0gIT0gbGVuKHBhdGhzKToNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYie2NsYXNzX25hbWV9OiBBbm9tYWxpYiBzY29yZWQge3N0YWNrZWQuc2hhcGVbMF19IG9mIHtsZW4ocGF0aHMpfSB0ZXN0IGltYWdlcy4iKQ0KICAgIHJldHVybiBzdGFja2VkDQoNCg0KZGVmIGludGVncml0eV9jbGFzcyhjZmc6IFZpc2FDb25maWcsIGNsYXNzX25hbWU6IHN0ciwgZmVhdHVyZXM6IGRpY3QsIGRpc3RhbmNlOiBucC5uZGFycmF5LCBzcGxpdDogZGljdCkgLT4gZGljdDoNCiAgICBpZiBmZWF0dXJlc1sicGF0aHMiXSAhPSBpbWFnZV9vcmRlcihzcGxpdCk6DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIntjbGFzc19uYW1lfTogZmVhdHVyZSByb3dzIGRvIG5vdCBmb2xsb3cgVSwgSCwgdGVzdCBnb29kLCB0ZXN0IGFub21hbHkuIikNCiAgICBpZiBsZW4oZmVhdHVyZXNbInBhdGhzIl0pICE9IGxlbihzZXQoZmVhdHVyZXNbInBhdGhzIl0pKToNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYie2NsYXNzX25hbWV9OiBkdXBsaWNhdGUgcGF0aHMgaW4gdGhlIGZlYXR1cmUgY2FjaGUuIikNCiAgICBibG9ja3MgPSByb3dfYmxvY2tzKHNwbGl0KQ0KICAgIGlmIGRpc3RhbmNlLnNoYXBlICE9IChsZW4oZmVhdHVyZXNbInBhdGhzIl0pLCA3ODQsIGxlbihzcGxpdFsidW5pdmVyc2UiXSkpOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJ7Y2xhc3NfbmFtZX06IGRpc3RhbmNlIHRlbnNvciBjb2x1bW5zIG11c3QgYmUgZXhhY3RseSBVLCBnb3Qge2Rpc3RhbmNlLnNoYXBlfS4iKQ0KICAgIGlmIG5vdCBucC5pc2luZihkaXN0YW5jZVtibG9ja3NbInVuaXZlcnNlIl0sIDosIGJsb2Nrc1sidW5pdmVyc2UiXV0pLmFsbCgpOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJ7Y2xhc3NfbmFtZX06IHVuaXZlcnNlIHNlbGYtZGlzdGFuY2VzIG11c3QgYmUgK2luZi4iKQ0KICAgIGZvciBuYW1lIGluICgiaG9sZG91dCIsICJ0ZXN0X2dvb2QiLCAidGVzdF9kZWZlY3QiKToNCiAgICAgICAgaWYgbm90IG5wLmlzZmluaXRlKGRpc3RhbmNlW2Jsb2Nrc1tuYW1lXV0pLmFsbCgpOg0KICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYie2NsYXNzX25hbWV9OiBub24tZmluaXRlIGRpc3RhbmNlcyBpbiB0aGUge25hbWV9IHJvd3MuIikNCiAgICBpbmRleCA9IHtwYXRoOiBwb3NpdGlvbiBmb3IgcG9zaXRpb24sIHBhdGggaW4gZW51bWVyYXRlKGZlYXR1cmVzWyJwYXRocyJdKX0NCiAgICBwb29sID0gW2luZGV4W3BhdGhdIGZvciBwYXRoIGluIHNwbGl0WyJwb29sIl1dDQogICAgX3NwbGl0X3JuZywgcmFuZG9tX3JuZywgX2tjZW50ZXJfcm5nID0gc3BsaXRfc2VlZHMoY2ZnLCBjbGFzc19uYW1lLCAwKQ0KICAgIG9yZGVyID0gcmFuZG9tX29yZGVyKHBvb2wsIHJhbmRvbV9ybmcpDQogICAgcXVlcmllcyA9ICgNCiAgICAgICAgW2luZGV4W3BdIGZvciBwIGluIHNwbGl0WyJjYWxpYnJhdGlvbiJdWzoyXV0NCiAgICAgICAgKyBbaW5kZXhbcF0gZm9yIHAgaW4gc3BsaXRbImhvbGRvdXQiXVs6Ml1dDQogICAgICAgICsgW2luZGV4W3BdIGZvciBwIGluIHNwbGl0WyJ0ZXN0X2dvb2QiXVs6Ml1dDQogICAgKQ0KICAgIGdhcHMgPSB7fQ0KICAgIGZvciBuIGluICgxLCA4LCBsZW4ob3JkZXIpKToNCiAgICAgICAgZ2Fwc1tzdHIobildID0gYXNzZXJ0X21ldGhvZHNfYWdyZWUoZmVhdHVyZXNbInBhdGNoZXMiXSwgZGlzdGFuY2UsIHF1ZXJpZXMsIG9yZGVyWzpuXSkNCiAgICByZXR1cm4geyJkaXN0YW5jZV92c19rbm4iOiBnYXBzfQ0KDQoNCmRlZiBydW5fY2xhc3MoY2ZnOiBWaXNhQ29uZmlnLCBjbGFzc19uYW1lOiBzdHIsIHBhcnQ6IHN0cikgLT4gZGljdDoNCiAgICBwYXRoID0gX2NsYXNzX3BhdGgoY2ZnLCBjbGFzc19uYW1lKQ0KICAgIGV4aXN0aW5nID0gcmVhZF9qc29uKHBhdGgpIGlmIHBhdGguaXNfZmlsZSgpIGVsc2Uge30NCiAgICBpZiBleGlzdGluZy5nZXQoImNvbmZpZ19oYXNoIikgIT0gY2ZnLmhhc2g6DQogICAgICAgIGV4aXN0aW5nID0ge30NCiAgICBzcGxpdCA9IGxvYWRfc3BsaXQoY2ZnLCBjbGFzc19uYW1lLCAwKQ0KICAgIHRlc3RfcGF0aHMsIGxhYmVscyA9IF90ZXN0KHNwbGl0KQ0KICAgIHVuaXZlcnNlID0gbGlzdChzcGxpdFsidW5pdmVyc2UiXSkNCiAgICBpZiAiaW50ZWdyaXR5IiBub3QgaW4gZXhpc3Rpbmc6DQogICAgICAgIGZlYXR1cmVzID0gbG9hZF9mZWF0dXJlcyhjZmcsIGNsYXNzX25hbWUpDQogICAgICAgIGRpc3RhbmNlID0gbG9hZF9kaXN0YW5jZXMoY2ZnLCBjbGFzc19uYW1lKVsiRCJdDQogICAgICAgIGV4aXN0aW5nWyJpbnRlZ3JpdHkiXSA9IGludGVncml0eV9jbGFzcyhjZmcsIGNsYXNzX25hbWUsIGZlYXR1cmVzLCBkaXN0YW5jZSwgc3BsaXQpDQogICAgICAgIGRlbCBmZWF0dXJlcywgZGlzdGFuY2UNCiAgICAgICAgd3JpdGVfanNvbihwYXRoLCB7KipleGlzdGluZywgImNvbmZpZ19oYXNoIjogY2ZnLmhhc2gsICJjbGFzc19uYW1lIjogY2xhc3NfbmFtZX0pDQogICAgaWYgcGFydCBpbiB7ImEiLCAiYWxsIn0gYW5kICJlMGEiIG5vdCBpbiBleGlzdGluZzoNCiAgICAgICAgTE9HR0VSLmluZm8oIkUwYSAlcyIsIGNsYXNzX25hbWUpDQogICAgICAgIG1vZGVsLCBkZXZpY2UgPSBfZml0X2Fub21hbGliKGNmZywgY2xhc3NfbmFtZSwgdW5pdmVyc2UsIGludChjZmcuZTBbIm51bV9uZWlnaGJvcnNfcmVmIl0pLCBmdWxsX2Jhbms9RmFsc2UpDQogICAgICAgIHNjb3JlcyA9IF9zY29yZV93aXRoX2Fub21hbGliKGNmZywgY2xhc3NfbmFtZSwgbW9kZWwsIGRldmljZSwgdGVzdF9wYXRocykNCiAgICAgICAgX3JlbGVhc2UobW9kZWwpDQogICAgICAgIGV4aXN0aW5nWyJlMGEiXSA9IHsiYXVyb2MiOiBfYXVyb2MobGFiZWxzLCBzY29yZXMpLCAibl90ZXN0IjogaW50KGxhYmVscy5zaXplKSwgIm5fYmFua19pbWFnZXMiOiBsZW4odW5pdmVyc2UpfQ0KICAgICAgICB3cml0ZV9qc29uKHBhdGgsIHsqKmV4aXN0aW5nLCAiY29uZmlnX2hhc2giOiBjZmcuaGFzaCwgImNsYXNzX25hbWUiOiBjbGFzc19uYW1lfSkNCiAgICBpZiBwYXJ0IGluIHsiYiIsICJhbGwifSBhbmQgImUwYiIgbm90IGluIGV4aXN0aW5nOg0KICAgICAgICBMT0dHRVIuaW5mbygiRTBiICVzIiwgY2xhc3NfbmFtZSkNCiAgICAgICAgbW9kZWwsIGRldmljZSA9IF9maXRfYW5vbWFsaWIoY2ZnLCBjbGFzc19uYW1lLCB1bml2ZXJzZSwgaW50KGNmZy5lMFsibnVtX25laWdoYm9yc19ycSJdKSwgZnVsbF9iYW5rPVRydWUpDQogICAgICAgIGFub21hbGliX3Njb3JlcyA9IF9zY29yZV93aXRoX2Fub21hbGliKGNmZywgY2xhc3NfbmFtZSwgbW9kZWwsIGRldmljZSwgdGVzdF9wYXRocykNCiAgICAgICAgX3JlbGVhc2UobW9kZWwpDQogICAgICAgIGRpc3RhbmNlID0gbG9hZF9kaXN0YW5jZXMoY2ZnLCBjbGFzc19uYW1lKVsiRCJdDQogICAgICAgIGJsb2NrcyA9IHJvd19ibG9ja3Moc3BsaXQpDQogICAgICAgIHJvd3MgPSBucC5jb25jYXRlbmF0ZShbYmxvY2tzWyJ0ZXN0X2dvb2QiXSwgYmxvY2tzWyJ0ZXN0X2RlZmVjdCJdXSkNCiAgICAgICAgb3duX3Njb3JlcyA9IHNjb3Jlc19mb3JfYmFuayhkaXN0YW5jZSwgcm93cywgYmxvY2tzWyJ1bml2ZXJzZSJdKQ0KICAgICAgICBkZWwgZGlzdGFuY2UNCiAgICAgICAgZXhpc3RpbmdbImUwYiJdID0gew0KICAgICAgICAgICAgInNwZWFybWFuIjogZmxvYXQoc3BlYXJtYW5yKG93bl9zY29yZXMsIGFub21hbGliX3Njb3Jlcykuc3RhdGlzdGljKSwNCiAgICAgICAgICAgICJhdXJvY19vd24iOiBfYXVyb2MobGFiZWxzLCBvd25fc2NvcmVzKSwNCiAgICAgICAgICAgICJhdXJvY19hbm9tYWxpYiI6IF9hdXJvYyhsYWJlbHMsIGFub21hbGliX3Njb3JlcyksDQogICAgICAgICAgICAibWF4X2Fic19zY29yZV9nYXAiOiBmbG9hdChucC5tYXgobnAuYWJzKG93bl9zY29yZXMgLSBhbm9tYWxpYl9zY29yZXMpKSksDQogICAgICAgICAgICAibWF4X2Fic19nYXBfYWZ0ZXJfY2xhc3NfbWlubWF4IjogZmxvYXQobnAubWF4KG5wLmFicyhfbWlubWF4KG93bl9zY29yZXMpIC0gX21pbm1heChhbm9tYWxpYl9zY29yZXMpKSkpLA0KICAgICAgICB9DQogICAgZXhpc3RpbmdbImNvbmZpZ19oYXNoIl0gPSBjZmcuaGFzaA0KICAgIGV4aXN0aW5nWyJjbGFzc19uYW1lIl0gPSBjbGFzc19uYW1lDQogICAgd3JpdGVfanNvbihwYXRoLCBleGlzdGluZykNCiAgICByZXR1cm4gZXhpc3RpbmcNCg0KDQpkZWYgcnVuX2UwKGNmZzogVmlzYUNvbmZpZywgY2xhc3NlczogbGlzdFtzdHJdLCBwYXJ0OiBzdHIgPSAiYWxsIikgLT4gZGljdDoNCiAgICB2ZXJzaW9uID0gcmVjb3JkX3NhbXBsZXJfc291cmNlKGNmZykNCiAgICB1cHNhbXBsaW5nID0gY2hlY2tfYW5vbWFsaWJfdXBzYW1wbGluZyhjZmcpDQogICAgY2hlY2tfY29ycnVwdGVkX2lucHV0KCkNCiAgICByb3dzID0gW3J1bl9jbGFzcyhjZmcsIGNsYXNzX25hbWUsIHBhcnQpIGZvciBjbGFzc19uYW1lIGluIGNsYXNzZXNdDQogICAgZTBhX29rLCBlMGFfc3VtbWFyeSA9IFRydWUsIE5vbmUNCiAgICBpZiBwYXJ0IGluIHsiYSIsICJhbGwifToNCiAgICAgICAgcGVyX2NsYXNzID0ge3Jvd1siY2xhc3NfbmFtZSJdOiBmbG9hdChyb3dbImUwYSJdWyJhdXJvYyJdKSBmb3Igcm93IGluIHJvd3N9DQogICAgICAgIGUwYV9zdW1tYXJ5ID0gew0KICAgICAgICAgICAgIm1lYW5fYXVyb2MiOiBmbG9hdChucC5tZWFuKGxpc3QocGVyX2NsYXNzLnZhbHVlcygpKSkpLA0KICAgICAgICAgICAgIm1pbl9hdXJvYyI6IGZsb2F0KG5wLm1pbihsaXN0KHBlcl9jbGFzcy52YWx1ZXMoKSkpKSwNCiAgICAgICAgICAgICJwZXJfY2xhc3MiOiBwZXJfY2xhc3MsDQogICAgICAgIH0NCiAgICAgICAgZTBhX29rID0gZTBhX3N1bW1hcnlbIm1lYW5fYXVyb2MiXSA+PSBmbG9hdChjZmcuZTBbIm1lYW5fYXVyb2NfZmxvb3IiXSkNCiAgICBlMGJfb2ssIGUwYl9zdW1tYXJ5ID0gVHJ1ZSwgTm9uZQ0KICAgIGlmIHBhcnQgaW4geyJiIiwgImFsbCJ9Og0KICAgICAgICBlMGJfb2ssIGUwYl9zdW1tYXJ5ID0gX3Bhc3NlZF9lMGIoY2ZnLCByb3dzKQ0KICAgIGNvbXBsZXRlID0gcGFydCA9PSAiYWxsIiBhbmQgc2V0KGNsYXNzZXMpID09IHNldChjZmcuY2xhc3NlcykNCiAgICBzdGF0dXMgPSAiUEFTUyIgaWYgY29tcGxldGUgYW5kIGUwYV9vayBhbmQgZTBiX29rIGVsc2UgKCJQQVJUSUFMIiBpZiBlMGFfb2sgYW5kIGUwYl9vayBlbHNlICJGQUlMIikNCiAgICBwYXlsb2FkID0gew0KICAgICAgICAic3RhdHVzIjogc3RhdHVzLA0KICAgICAgICAiY29uZmlnX2hhc2giOiBjZmcuaGFzaCwNCiAgICAgICAgImFub21hbGliX3ZlcnNpb24iOiB2ZXJzaW9uLA0KICAgICAgICAiYW5vbWFsaWJfdXBzYW1wbGluZyI6IHVwc2FtcGxpbmcsDQogICAgICAgICJwYXJ0IjogcGFydCwNCiAgICAgICAgImNsYXNzZXMiOiBjbGFzc2VzLA0KICAgICAgICAiZTBhIjogZTBhX3N1bW1hcnksDQogICAgICAgICJlMGIiOiBlMGJfc3VtbWFyeSwNCiAgICAgICAgInBlcl9jbGFzcyI6IHJvd3MsDQogICAgICAgICJyZWZlcmVuY2VfbWVhbl9hdXJvYyI6IGZsb2F0KGNmZy5lMFsicmVmZXJlbmNlX21lYW5fYXVyb2MiXSksDQogICAgICAgICJtZWFuX2F1cm9jX2Zsb29yIjogZmxvYXQoY2ZnLmUwWyJtZWFuX2F1cm9jX2Zsb29yIl0pLA0KICAgIH0NCiAgICB3cml0ZV9qc29uKGdhdGVfcGF0aChjZmcpLCBwYXlsb2FkKQ0KICAgIExPR0dFUi5pbmZvKCJFMCBnYXRlICVzIiwgc3RhdHVzKQ0KICAgIHJldHVybiBwYXlsb2FkDQoNCg0KZGVmIGdhdGVfaXNfZXhwbG9yYXRvcnkoY2ZnOiBWaXNhQ29uZmlnLCBjbGFzc2VzOiBsaXN0W3N0cl0sIGV4cGxvcmF0b3J5OiBib29sKSAtPiBib29sOg0KICAgIHBhdGggPSBnYXRlX3BhdGgoY2ZnKQ0KICAgIGlmIG5vdCBwYXRoLmlzX2ZpbGUoKToNCiAgICAgICAgaWYgZXhwbG9yYXRvcnk6DQogICAgICAgICAgICBMT0dHRVIud2FybmluZygiRTAgZ2F0ZSBpcyBtaXNzaW5nLiBUaGlzIHJ1biBpcyBleHBsb3JhdG9yeS4iKQ0KICAgICAgICAgICAgcmV0dXJuIFRydWUNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJFMCBnYXRlIGlzIG1pc3NpbmcuIFRyYWplY3RvcmllcyBzdGF5IGJsb2NrZWQgdW50aWwgZTAvZ2F0ZS5qc29uIGhhcyBzdGF0dXMgUEFTUy4iKQ0KICAgIGdhdGUgPSByZWFkX2pzb24ocGF0aCkNCiAgICBpZiBnYXRlLmdldCgiY29uZmlnX2hhc2giKSAhPSBjZmcuaGFzaDoNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJFMCBnYXRlIHdhcyB3cml0dGVuIGZvciBhIGRpZmZlcmVudCBjb25maWcgaGFzaC4iKQ0KICAgIG1pc3NpbmcgPSBbbmFtZSBmb3IgbmFtZSBpbiBjbGFzc2VzIGlmIG5hbWUgbm90IGluIHNldChnYXRlLmdldCgiY2xhc3NlcyIsIFtdKSldDQogICAgaWYgZ2F0ZS5nZXQoInN0YXR1cyIpID09ICJQQVNTIiBhbmQgZ2F0ZS5nZXQoInBhcnQiKSA9PSAiYWxsIiBhbmQgbm90IG1pc3Npbmc6DQogICAgICAgIHJldHVybiBGYWxzZQ0KICAgIGlmIGV4cGxvcmF0b3J5Og0KICAgICAgICBMT0dHRVIud2FybmluZygiRTAgZ2F0ZSBkb2VzIG5vdCBjb3ZlciB0aGlzIHJ1biAoJXMpLiBNYXJraW5nIGl0IGV4cGxvcmF0b3J5LiIsIGdhdGUuZ2V0KCJzdGF0dXMiKSkNCiAgICAgICAgcmV0dXJuIFRydWUNCiAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJFMCBnYXRlIHN0YXR1cyB7Z2F0ZS5nZXQoJ3N0YXR1cycpfSwgbWlzc2luZyB7bWlzc2luZ30uIFRyYWplY3RvcmllcyBzdGF5IGJsb2NrZWQuIikNCg==", "commissioning/visa/exchange.py": "IiIiU2VjdGlvbiA5OiBhcmUgb2ZmaWNpYWwgZ29vZCB0ZXN0IHNjb3JlcyBleGNoYW5nZWFibGUgd2l0aCBoZWxkLW91dCBnb29kIHNjb3Jlcz8gQmFuayA9IGFsbCBvZiBVLiIiIg0KDQpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zDQoNCmltcG9ydCBsb2dnaW5nDQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgNCg0KaW1wb3J0IG51bXB5IGFzIG5wDQppbXBvcnQgcGFuZGFzIGFzIHBkDQpmcm9tIHNjaXB5LnN0YXRzIGltcG9ydCBiaW5vbSwgbWFubndoaXRuZXl1DQpmcm9tIHNrbGVhcm4ubWV0cmljcyBpbXBvcnQgcm9jX2F1Y19zY29yZQ0KDQpmcm9tIGNvbW1pc3Npb25pbmcuaW9fdXRpbHMgaW1wb3J0IHJlYWRfanNvbiwgd3JpdGVfanNvbg0KZnJvbSBjb21taXNzaW9uaW5nLnBhdGNoY29yZSBpbXBvcnQgc2NvcmVzX2Zvcl9iYW5rDQpmcm9tIGNvbW1pc3Npb25pbmcuc3RhdHMgaW1wb3J0IGhvbG0NCmZyb20gY29tbWlzc2lvbmluZy52aXNhLmNhY2hlcyBpbXBvcnQgbG9hZF9kaXN0YW5jZXMNCmZyb20gY29tbWlzc2lvbmluZy52aXNhLmNvbmZpZyBpbXBvcnQgVmlzYUNvbmZpZw0KZnJvbSBjb21taXNzaW9uaW5nLnZpc2EuZGF0YSBpbXBvcnQgbG9hZF9zcGxpdCwgcm93X2Jsb2Nrcw0KDQpMT0dHRVIgPSBsb2dnaW5nLmdldExvZ2dlcihfX25hbWVfXykNCg0KDQpkZWYgcGFydF9wYXRoKGNmZzogVmlzYUNvbmZpZywgY2xhc3NfbmFtZTogc3RyKSAtPiBQYXRoOg0KICAgIHJldHVybiBjZmcud29ya19yb290IC8gImV4Y2hhbmdlIiAvIGYie2NsYXNzX25hbWV9Lmpzb24iDQoNCg0KZGVmIGV4Y2hhbmdlX3Njb3JlcyhkaXN0YW5jZTogbnAubmRhcnJheSwgc3BsaXQ6IGRpY3QpIC0+IGRpY3Rbc3RyLCBucC5uZGFycmF5XToNCiAgICBibG9ja3MgPSByb3dfYmxvY2tzKHNwbGl0KQ0KICAgIGJhbmsgPSBibG9ja3NbInVuaXZlcnNlIl0NCiAgICByZXR1cm4ge25hbWU6IHNjb3Jlc19mb3JfYmFuayhkaXN0YW5jZSwgYmxvY2tzW25hbWVdLCBiYW5rKSBmb3IgbmFtZSBpbiAoImhvbGRvdXQiLCAidGVzdF9nb29kIiwgInRlc3RfZGVmZWN0Iil9DQoNCg0KZGVmIGV4Y2hhbmdlX3N1bW1hcnkoY2ZnOiBWaXNhQ29uZmlnLCBjbGFzc19uYW1lOiBzdHIsIHNjb3JlczogZGljdFtzdHIsIG5wLm5kYXJyYXldKSAtPiBkaWN0Og0KICAgIGhvbGRvdXQsIGdvb2QsIGRlZmVjdCA9IHNjb3Jlc1siaG9sZG91dCJdLCBzY29yZXNbInRlc3RfZ29vZCJdLCBzY29yZXNbInRlc3RfZGVmZWN0Il0NCiAgICBsYWJlbHMgPSBucC5yX1tucC56ZXJvcyhsZW4oaG9sZG91dCkpLCBucC5vbmVzKGxlbihnb29kKSldDQogICAgY29uZm9ybWFsX3AgPSAoMS4wICsgKGhvbGRvdXRbTm9uZSwgOl0gPj0gZ29vZFs6LCBOb25lXSkuc3VtKGF4aXM9MSkpIC8gKGxlbihob2xkb3V0KSArIDEuMCkNCiAgICBuX2xvdyA9IGludChucC5zdW0oY29uZm9ybWFsX3AgPD0gY2ZnLmFscGhhKSkNCiAgICByZXR1cm4gew0KICAgICAgICAiY2xhc3NfbmFtZSI6IGNsYXNzX25hbWUsDQogICAgICAgICJncm91cCI6IGNmZy5ncm91cF9vZihjbGFzc19uYW1lKSwNCiAgICAgICAgIm5faG9sZG91dCI6IGludChsZW4oaG9sZG91dCkpLA0KICAgICAgICAibl90ZXN0X2dvb2QiOiBpbnQobGVuKGdvb2QpKSwNCiAgICAgICAgImdvb2Rfc2hpZnRfYXVyb2MiOiBmbG9hdChyb2NfYXVjX3Njb3JlKGxhYmVscywgbnAucl9baG9sZG91dCwgZ29vZF0pKSwNCiAgICAgICAgIm1hbm53aGl0bmV5X3AiOiBmbG9hdChtYW5ud2hpdG5leXUoZ29vZCwgaG9sZG91dCwgYWx0ZXJuYXRpdmU9InR3by1zaWRlZCIpLnB2YWx1ZSksDQogICAgICAgICJtZWRpYW5faG9sZG91dF9zY29yZSI6IGZsb2F0KG5wLm1lZGlhbihob2xkb3V0KSksDQogICAgICAgICJtZWRpYW5fdGVzdF9nb29kX3Njb3JlIjogZmxvYXQobnAubWVkaWFuKGdvb2QpKSwNCiAgICAgICAgImZyYWN0aW9uX2NvbmZvcm1hbF9wX2JlbG93X2FscGhhIjogbl9sb3cgLyBsZW4oZ29vZCksDQogICAgICAgICJiaW5vbWlhbF90YWlsX3AiOiBmbG9hdChiaW5vbS5zZihuX2xvdyAtIDEsIGxlbihnb29kKSwgY2ZnLmFscGhhKSksDQogICAgICAgICJkZWZlY3RfYXVyb2NfYmFua191IjogZmxvYXQoDQogICAgICAgICAgICByb2NfYXVjX3Njb3JlKG5wLnJfW25wLnplcm9zKGxlbihnb29kKSksIG5wLm9uZXMobGVuKGRlZmVjdCkpXSwgbnAucl9bZ29vZCwgZGVmZWN0XSkNCiAgICAgICAgKSwNCiAgICAgICAgImNvbmZpZ19oYXNoIjogY2ZnLmhhc2gsDQogICAgfQ0KDQoNCmRlZiBydW5fZXhjaGFuZ2UoY2ZnOiBWaXNhQ29uZmlnLCBjbGFzc2VzOiBsaXN0W3N0cl0pIC0+IHBkLkRhdGFGcmFtZToNCiAgICByb3dzID0gW10NCiAgICBmb3IgY2xhc3NfbmFtZSBpbiBjbGFzc2VzOg0KICAgICAgICBwYXRoID0gcGFydF9wYXRoKGNmZywgY2xhc3NfbmFtZSkNCiAgICAgICAgaWYgcGF0aC5pc19maWxlKCkgYW5kIHJlYWRfanNvbihwYXRoKS5nZXQoImNvbmZpZ19oYXNoIikgPT0gY2ZnLmhhc2g6DQogICAgICAgICAgICByb3dzLmFwcGVuZChyZWFkX2pzb24ocGF0aCkpDQogICAgICAgICAgICBjb250aW51ZQ0KICAgICAgICBMT0dHRVIuaW5mbygiRXhjaGFuZ2VhYmlsaXR5ICVzIiwgY2xhc3NfbmFtZSkNCiAgICAgICAgc3BsaXQgPSBsb2FkX3NwbGl0KGNmZywgY2xhc3NfbmFtZSwgMCkNCiAgICAgICAgZGlzdGFuY2UgPSBsb2FkX2Rpc3RhbmNlcyhjZmcsIGNsYXNzX25hbWUpWyJEIl0NCiAgICAgICAgc3VtbWFyeSA9IGV4Y2hhbmdlX3N1bW1hcnkoY2ZnLCBjbGFzc19uYW1lLCBleGNoYW5nZV9zY29yZXMoZGlzdGFuY2UsIHNwbGl0KSkNCiAgICAgICAgZGVsIGRpc3RhbmNlDQogICAgICAgIHdyaXRlX2pzb24ocGF0aCwgc3VtbWFyeSkNCiAgICAgICAgcm93cy5hcHBlbmQoc3VtbWFyeSkNCiAgICByZXR1cm4gZmluYWxpemUocm93cykNCg0KDQpkZWYgZmluYWxpemUocm93czogbGlzdFtkaWN0XSkgLT4gcGQuRGF0YUZyYW1lOg0KICAgIGZyYW1lID0gcGQuRGF0YUZyYW1lKHJvd3MpLnNvcnRfdmFsdWVzKCJjbGFzc19uYW1lIikucmVzZXRfaW5kZXgoZHJvcD1UcnVlKQ0KICAgIGZyYW1lWyJob2xtX3AiXSA9IGhvbG0oZnJhbWVbIm1hbm53aGl0bmV5X3AiXS50b2xpc3QoKSkNCiAgICBmcmFtZVsic2hpZnRfZGV0ZWN0ZWQiXSA9IGZyYW1lWyJob2xtX3AiXSA8IDAuMDUNCiAgICByZXR1cm4gZnJhbWUNCg0KDQpkZWYgbG9hZF9leGNoYW5nZShjZmc6IFZpc2FDb25maWcpIC0+IHBkLkRhdGFGcmFtZSB8IE5vbmU6DQogICAgcm93cyA9IFtdDQogICAgZm9yIGNsYXNzX25hbWUgaW4gY2ZnLmNsYXNzZXM6DQogICAgICAgIHBhdGggPSBwYXJ0X3BhdGgoY2ZnLCBjbGFzc19uYW1lKQ0KICAgICAgICBpZiBub3QgcGF0aC5pc19maWxlKCk6DQogICAgICAgICAgICByZXR1cm4gTm9uZQ0KICAgICAgICBwYXlsb2FkID0gcmVhZF9qc29uKHBhdGgpDQogICAgICAgIGlmIHBheWxvYWQuZ2V0KCJjb25maWdfaGFzaCIpICE9IGNmZy5oYXNoOg0KICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYie3BhdGh9IHdhcyB3cml0dGVuIGZvciBhbm90aGVyIGNvbmZpZyBoYXNoLiIpDQogICAgICAgIHJvd3MuYXBwZW5kKHBheWxvYWQpDQogICAgcmV0dXJuIGZpbmFsaXplKHJvd3MpDQo=", "commissioning/visa/report.py": "IiIiRmlndXJlcywgdGFibGVzLCBhbmQgYSBzZWxmLWNvbnRhaW5lZCBIVE1MIHJlcG9ydCBmb3IgdGhlIFZpc0EgcmVwbGljYXRpb24uIiIiDQoNCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMNCg0KaW1wb3J0IGh0bWwNCmltcG9ydCBqc29uDQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgNCg0KaW1wb3J0IG1hdHBsb3RsaWINCg0KbWF0cGxvdGxpYi51c2UoIkFnZyIpDQppbXBvcnQgbWF0cGxvdGxpYi5weXBsb3QgYXMgcGx0DQppbXBvcnQgbnVtcHkgYXMgbnANCmltcG9ydCBwYW5kYXMgYXMgcGQNCg0KZnJvbSBjb21taXNzaW9uaW5nLnJlcG9ydCBpbXBvcnQgS0NFTlRFUl9DT0xPUiwgUkFORE9NX0NPTE9SLCBTVFJBVEVHWV9DT0xPUiwgX2h0bWxfZnJhbWUsIF9pbmxpbmVfZmlndXJlcywgX3NhdmUsIF9zdHlsZQ0KDQpSVUxFX1NUWUxFID0geyJSMjAiOiAiLSIsICJSNjAiOiAiLS0ifQ0KR1JPVVBfQ09MT1IgPSB7Im11bHRpcGxlIjogUkFORE9NX0NPTE9SLCAic2luZ2xlIjogS0NFTlRFUl9DT0xPUiwgImNvbXBsZXgiOiAiIzJlN2Q0ZiJ9DQpUSEVPUlkgPSB7IlIyMCI6IDEgLyAyMSwgIlI2MCI6IDEgLyA2MX0NCg0KDQpkZWYgX3goZnJhbWU6IHBkLkRhdGFGcmFtZSkgLT4gbnAubmRhcnJheToNCiAgICByZXR1cm4gbnAubG9nMihmcmFtZVsiTiJdLnRvX251bXB5KGR0eXBlPWZsb2F0KSkNCg0KDQpkZWYgX3RpY2tzKGF4LCBncmlkOiBsaXN0W2ludF0pIC0+IE5vbmU6DQogICAgYXguc2V0X3h0aWNrcyhucC5sb2cyKGdyaWQpKQ0KICAgIGF4LnNldF94dGlja2xhYmVscyhbc3RyKG4pIGZvciBuIGluIGdyaWRdKQ0KICAgIGF4LnNldF94bGFiZWwoIk4gKGxvZyBzY2FsZSk7IGxhc3QgcG9pbnQgPSB8UHwiKQ0KDQoNCmRlZiBmaWd1cmVfdHByKG1ldHJpY3M6IHBkLkRhdGFGcmFtZSwgZGlyZWN0b3J5OiBQYXRoKSAtPiBOb25lOg0KICAgIGdyaWQgPSBzb3J0ZWQobWV0cmljc1siTiJdLnVuaXF1ZSgpKQ0KICAgIF9zdHlsZSgpDQogICAgZmlnLCBheGVzID0gcGx0LnN1YnBsb3RzKDEsIDIsIGZpZ3NpemU9KDEwLCAzLjgpLCBzaGFyZXk9VHJ1ZSkNCiAgICBmb3IgYXgsIHJ1bGUgaW4gemlwKGF4ZXMsICgiUjIwIiwgIlI2MCIpKToNCiAgICAgICAgZm9yIHN0cmF0ZWd5LCBjb2xvciBpbiBTVFJBVEVHWV9DT0xPUi5pdGVtcygpOg0KICAgICAgICAgICAgc3ViID0gbWV0cmljc1sobWV0cmljc1sicnVsZSJdID09IHJ1bGUpICYgKG1ldHJpY3NbInN0cmF0ZWd5Il0gPT0gc3RyYXRlZ3kpXQ0KICAgICAgICAgICAgcGVyX2NsYXNzID0gc3ViLmdyb3VwYnkoWyJOIiwgImNsYXNzX25hbWUiXSlbInRwciJdLm1lYW4oKS5yZXNldF9pbmRleCgpDQogICAgICAgICAgICBiYW5kID0gcGVyX2NsYXNzLmdyb3VwYnkoIk4iKVsidHByIl0uYWdnKA0KICAgICAgICAgICAgICAgIG1lZGlhbj0ibWVkaWFuIiwgcTE9bGFtYmRhIHg6IHgucXVhbnRpbGUoMC4yNSksIHEzPWxhbWJkYSB4OiB4LnF1YW50aWxlKDAuNzUpDQogICAgICAgICAgICApLnJlc2V0X2luZGV4KCkNCiAgICAgICAgICAgIHggPSBfeChiYW5kKQ0KICAgICAgICAgICAgYXgucGxvdCh4LCBiYW5kWyJtZWRpYW4iXSwgY29sb3I9Y29sb3IsIGx3PTIsIGxhYmVsPXN0cmF0ZWd5KQ0KICAgICAgICAgICAgYXguZmlsbF9iZXR3ZWVuKHgsIGJhbmRbInExIl0sIGJhbmRbInEzIl0sIGNvbG9yPWNvbG9yLCBhbHBoYT0wLjE4LCBsdz0wKQ0KICAgICAgICBheC5zZXRfdGl0bGUoZiJUUFIgdmVyc3VzIE4sIHRocmVzaG9sZCB7cnVsZX0iKQ0KICAgICAgICBfdGlja3MoYXgsIGdyaWQpDQogICAgYXhlc1swXS5zZXRfeWxhYmVsKCJUUFIgKG1lZGlhbiBvdmVyIGNsYXNzZXMsIElRUiBiYW5kKSIpDQogICAgYXhlc1swXS5sZWdlbmQoZnJhbWVvbj1GYWxzZSkNCiAgICBmaWcudGlnaHRfbGF5b3V0KCkNCiAgICBfc2F2ZShmaWcsIGRpcmVjdG9yeSwgImZpZ1YxX3RwciIpDQoNCg0KZGVmIGZpZ3VyZV9mcHIobWV0cmljczogcGQuRGF0YUZyYW1lLCBkaXJlY3Rvcnk6IFBhdGgpIC0+IE5vbmU6DQogICAgZ3JpZCA9IHNvcnRlZChtZXRyaWNzWyJOIl0udW5pcXVlKCkpDQogICAgX3N0eWxlKCkNCiAgICBmaWcsIGF4ZXMgPSBwbHQuc3VicGxvdHMoMSwgMiwgZmlnc2l6ZT0oMTAsIDMuOCksIHNoYXJleT1UcnVlKQ0KICAgIGZvciBheCwgKGNvbHVtbiwgdGl0bGUpIGluIHppcChheGVzLCAoKCJmcHJfaCIsICJoZWxkLW91dCBnb29kIEgiKSwgKCJmcHJfb2ZmIiwgIm9mZmljaWFsIGdvb2QgdGVzdCIpKSk6DQogICAgICAgIGZvciBydWxlIGluICgiUjIwIiwgIlI2MCIpOg0KICAgICAgICAgICAgc3ViID0gbWV0cmljc1ttZXRyaWNzWyJydWxlIl0gPT0gcnVsZV0uZ3JvdXBieSgiTiIpW2NvbHVtbl0ubWVhbigpLnJlc2V0X2luZGV4KCkNCiAgICAgICAgICAgIGF4LnBsb3QoX3goc3ViKSwgc3ViW2NvbHVtbl0sIGNvbG9yPSIjMzMzMzMzIiwgbHM9UlVMRV9TVFlMRVtydWxlXSwgbHc9MS44LCBsYWJlbD1mIntydWxlfSwgbWVhbiBvdmVyIGNsYXNzZXMiKQ0KICAgICAgICAgICAgYXguYXhobGluZShUSEVPUllbcnVsZV0sIGNvbG9yPSIjYTMzYjNiIiwgbHM9UlVMRV9TVFlMRVtydWxlXSwgbHc9MC45LCBsYWJlbD1mInRoZW9yeSAxL3tyb3VuZCgxIC8gVEhFT1JZW3J1bGVdKX0iKQ0KICAgICAgICBheC5zZXRfdGl0bGUoZiJSZWFsaXplZCBGUFIgb24ge3RpdGxlfSIpDQogICAgICAgIF90aWNrcyhheCwgZ3JpZCkNCiAgICBheGVzWzBdLnNldF95bGFiZWwoIkZQUiIpDQogICAgYXhlc1swXS5sZWdlbmQoZnJhbWVvbj1GYWxzZSwgZm9udHNpemU9OCkNCiAgICBmaWcudGlnaHRfbGF5b3V0KCkNCiAgICBfc2F2ZShmaWcsIGRpcmVjdG9yeSwgImZpZ1YyX2ZwciIpDQoNCg0KZGVmIGZpZ3VyZV90cHJfY29zdChzdGF0aXN0aWNzOiBkaWN0LCBkaXJlY3Rvcnk6IFBhdGgpIC0+IE5vbmU6DQogICAgcGVyX2NsYXNzID0gcGQuU2VyaWVzKHN0YXRpc3RpY3NbIkgzIl1bInBlcl9jbGFzcyJdKS5zb3J0X3ZhbHVlcygpDQogICAgX3N0eWxlKCkNCiAgICBmaWcsIGF4ID0gcGx0LnN1YnBsb3RzKGZpZ3NpemU9KDYuNCwgMy44KSkNCiAgICBheC5iYXJoKG5wLmFyYW5nZShsZW4ocGVyX2NsYXNzKSksIHBlcl9jbGFzcy50b19udW1weSgpLCBjb2xvcj1SQU5ET01fQ09MT1IsIGFscGhhPTAuODUpDQogICAgYXguYXh2bGluZShzdGF0aXN0aWNzWyJIMyJdWyJtYXJnaW4iXSwgY29sb3I9IiNhMzNiM2IiLCBscz0iLS0iLCBsdz0xLCBsYWJlbD0icHJhY3RpY2FsIG1hcmdpbiIpDQogICAgYXguYXh2bGluZShzdGF0aXN0aWNzWyJIMyJdWyJtZWRpYW4iXSwgY29sb3I9IiMzMzMzMzMiLCBsdz0xLCBsYWJlbD0ibWVkaWFuIG92ZXIgY2xhc3NlcyIpDQogICAgYXguc2V0X3l0aWNrcyhucC5hcmFuZ2UobGVuKHBlcl9jbGFzcykpKQ0KICAgIGF4LnNldF95dGlja2xhYmVscyhwZXJfY2xhc3MuaW5kZXgsIGZvbnRzaXplPTgpDQogICAgYXguc2V0X3hsYWJlbCgiVFBSKFIyMCkgLSBUUFIoUjYwKSBhdCBOID0gfFB8IikNCiAgICBheC5zZXRfdGl0bGUoIkgzOiBUUFIgY29zdCBvZiA2MCBpbnN0ZWFkIG9mIDIwIGNhbGlicmF0aW9uIGltYWdlcyIpDQogICAgYXgubGVnZW5kKGZyYW1lb249RmFsc2UsIGZvbnRzaXplPTgpDQogICAgZmlnLnRpZ2h0X2xheW91dCgpDQogICAgX3NhdmUoZmlnLCBkaXJlY3RvcnksICJmaWdWM190cHJfY29zdCIpDQoNCg0KZGVmIGZpZ3VyZV9mb3Jlc3Qoc3RhdGlzdGljczogZGljdCwgZGlyZWN0b3J5OiBQYXRoKSAtPiBOb25lOg0KICAgIHByaW1hcnkgPSBzdGF0aXN0aWNzWyJwcmltYXJ5Il1bIkgxX0gyX1IyMF9vZmZpY2lhbCJdDQogICAgX3N0eWxlKCkNCiAgICBmaWcsIGF4ZXMgPSBwbHQuc3VicGxvdHMoMSwgMiwgZmlnc2l6ZT0oOSwgMy44KSwgc2hhcmV5PVRydWUpDQogICAgZm9yIGF4LCBrZXksIGxhYmVsIGluIHppcChheGVzLCAoImF1bGNfdHByIiwgImF1bGNfZnByIiksICgizpRBVUxDX1RQUiIsICLOlEFVTENfRlBSIChvZmZpY2lhbCkiKSk6DQogICAgICAgIHZhbHVlcyA9IHBkLlNlcmllcyhwcmltYXJ5W2tleV1bInBlcl9jbGFzcyJdKS5zb3J0X2luZGV4KCkNCiAgICAgICAgYXguc2NhdHRlcih2YWx1ZXMudG9fbnVtcHkoKSwgbnAuYXJhbmdlKGxlbih2YWx1ZXMpKSwgY29sb3I9S0NFTlRFUl9DT0xPUiwgcz0yMikNCiAgICAgICAgYXguYXh2bGluZSgwLCBjb2xvcj0iIzU1NTU1NSIsIGx3PTEpDQogICAgICAgIGF4LnNldF90aXRsZShmIntsYWJlbH0sIGstY2VudGVyIOKIkiByYW5kb20iKQ0KICAgICAgICBheC5zZXRfeXRpY2tzKG5wLmFyYW5nZShsZW4odmFsdWVzKSkpDQogICAgICAgIGF4LnNldF95dGlja2xhYmVscyh2YWx1ZXMuaW5kZXgsIGZvbnRzaXplPTgpDQogICAgZmlnLnRpZ2h0X2xheW91dCgpDQogICAgX3NhdmUoZmlnLCBkaXJlY3RvcnksICJmaWdWNF9mb3Jlc3QiKQ0KDQoNCmRlZiBmaWd1cmVfc2hpZnQocDM6IHBkLkRhdGFGcmFtZSB8IE5vbmUsIGRpcmVjdG9yeTogUGF0aCkgLT4gYm9vbDoNCiAgICBpZiBwMyBpcyBOb25lIG9yIHAzLmVtcHR5Og0KICAgICAgICByZXR1cm4gRmFsc2UNCiAgICBvcmRlcmVkID0gcDMuc29ydF92YWx1ZXMoImdvb2Rfc2hpZnRfYXVyb2MiKQ0KICAgIF9zdHlsZSgpDQogICAgZmlnLCBheCA9IHBsdC5zdWJwbG90cyhmaWdzaXplPSg2LjYsIDMuOCkpDQogICAgeSA9IG5wLmFyYW5nZShsZW4ob3JkZXJlZCkpDQogICAgYXguYmFyaCh5LCBvcmRlcmVkWyJnb29kX3NoaWZ0X2F1cm9jIl0gLSAwLjUsIGxlZnQ9MC41LCBjb2xvcj1bR1JPVVBfQ09MT1JbZ10gZm9yIGcgaW4gb3JkZXJlZFsiZ3JvdXAiXV0sIGFscGhhPTAuODUpDQogICAgZm9yIHl5LCAoYXVyb2MsIGZsYWcpIGluIGVudW1lcmF0ZSh6aXAob3JkZXJlZFsiZ29vZF9zaGlmdF9hdXJvYyJdLCBvcmRlcmVkWyJzaGlmdF9kZXRlY3RlZCJdKSk6DQogICAgICAgIGlmIGZsYWc6DQogICAgICAgICAgICBheC50ZXh0KGF1cm9jICsgMC4wMSwgeXksICIqIiwgdmE9ImNlbnRlciIpDQogICAgYXguYXh2bGluZSgwLjUsIGNvbG9yPSIjNTU1NTU1IiwgbHc9MSkNCiAgICBheC5zZXRfeXRpY2tzKHkpDQogICAgYXguc2V0X3l0aWNrbGFiZWxzKG9yZGVyZWRbImNsYXNzX25hbWUiXSwgZm9udHNpemU9OCkNCiAgICBheC5zZXRfeGxhYmVsKCJBVVJPQyBvZiBvZmZpY2lhbCBnb29kIHZlcnN1cyBoZWxkLW91dCBnb29kIHNjb3JlcyAoMC41ID0gZXhjaGFuZ2VhYmxlKSIpDQogICAgYXguc2V0X3RpdGxlKCJTaGlmdCBhbW9uZyBnb29kIGltYWdlczsgKiBIb2xtIHAgPCAwLjA1IikNCiAgICBmaWcudGlnaHRfbGF5b3V0KCkNCiAgICBfc2F2ZShmaWcsIGRpcmVjdG9yeSwgImZpZ1Y1X3NoaWZ0IikNCiAgICByZXR1cm4gVHJ1ZQ0KDQoNCmRlZiBmaWd1cmVfYnVkZ2V0KGJlc3Q6IHBkLkRhdGFGcmFtZSB8IE5vbmUsIHN1bW1hcnlfcGF0aDogUGF0aCwgZGlyZWN0b3J5OiBQYXRoLCBtaW5fcGFzc2VzOiBpbnQpIC0+IGJvb2w6DQogICAgaWYgbm90IHN1bW1hcnlfcGF0aC5pc19maWxlKCk6DQogICAgICAgIHJldHVybiBGYWxzZQ0KICAgIHN1bW1hcnkgPSBwZC5yZWFkX2NzdihzdW1tYXJ5X3BhdGgpDQogICAgc3ViID0gc3VtbWFyeVtzdW1tYXJ5WyJydWxlIl0gPT0gInBhYyJdDQogICAgYnVkZ2V0cyA9IHNvcnRlZChzdWJbImJ1ZGdldCJdLnVuaXF1ZSgpKQ0KICAgIF9zdHlsZSgpDQogICAgZmlnLCBheGVzID0gcGx0LnN1YnBsb3RzKDEsIGxlbihidWRnZXRzKSwgZmlnc2l6ZT0oMy4yICogbGVuKGJ1ZGdldHMpLCAzLjQpLCBzaGFyZXk9VHJ1ZSwgc3F1ZWV6ZT1GYWxzZSkNCiAgICBmb3IgYXgsIGJ1ZGdldCBpbiB6aXAoYXhlc1swXSwgYnVkZ2V0cyk6DQogICAgICAgIHBhcnQgPSBzdWJbc3ViWyJidWRnZXQiXSA9PSBidWRnZXRdDQogICAgICAgIGZvciBfY2xhc3NfbmFtZSwgbGluZSBpbiBwYXJ0Lmdyb3VwYnkoImNsYXNzX25hbWUiKToNCiAgICAgICAgICAgIGxpbmUgPSBsaW5lLnNvcnRfdmFsdWVzKCJOIikNCiAgICAgICAgICAgIGF4LnBsb3QobnAubG9nMihsaW5lWyJOIl0pLCBsaW5lWyJwYXNzaW5nX29mZiJdLCBjb2xvcj1HUk9VUF9DT0xPUltsaW5lWyJncm91cCJdLmlsb2NbMF1dLCBsdz0xLCBhbHBoYT0wLjg1KQ0KICAgICAgICBheC5heGhsaW5lKG1pbl9wYXNzZXMsIGNvbG9yPSIjYTMzYjNiIiwgbHM9Ii0tIiwgbHc9MSkNCiAgICAgICAgYXguc2V0X3RpdGxlKGYiQiA9IHtidWRnZXR9IiwgZm9udHNpemU9OSkNCiAgICAgICAgYXguc2V0X3hsYWJlbCgibG9nMiBOOyBjYWxpYnJhdGlvbiA9IEIg4oiSIE4iKQ0KICAgIGF4ZXNbMCwgMF0uc2V0X3lsYWJlbCgiUGFzc2luZyByZXBldGl0aW9ucyBvZiAyMCAob2ZmaWNpYWwpIikNCiAgICBmaWcuc3VwdGl0bGUoIkJ1ZGdldCBzcGxpdCwgUEFDIHRocmVzaG9sZC4gQmx1ZSBtdWx0aXBsZSwgb3JhbmdlIHNpbmdsZSwgZ3JlZW4gUENCIikNCiAgICBmaWcudGlnaHRfbGF5b3V0KCkNCiAgICBfc2F2ZShmaWcsIGRpcmVjdG9yeSwgImZpZ1Y2X2J1ZGdldCIpDQogICAgcmV0dXJuIFRydWUNCg0KDQpkZWYgd3JpdGVfbGF0ZXgoZnJhbWU6IHBkLkRhdGFGcmFtZSwgcGF0aDogUGF0aCkgLT4gTm9uZToNCiAgICBkZWYgY2VsbCh2YWx1ZSkgLT4gc3RyOg0KICAgICAgICBpZiBpc2luc3RhbmNlKHZhbHVlLCAoZmxvYXQsIG5wLmZsb2F0aW5nKSk6DQogICAgICAgICAgICByZXR1cm4gZiJ7dmFsdWU6LjRmfSINCiAgICAgICAgcmV0dXJuIHN0cih2YWx1ZSkucmVwbGFjZSgiXyIsICJcXF8iKS5yZXBsYWNlKCLOlCIsICIkXFxEZWx0YSQiKQ0KDQogICAgbGluZXMgPSBbIlxcYmVnaW57dGFidWxhcn17IiArICJsIiAqIGxlbihmcmFtZS5jb2x1bW5zKSArICJ9IiwgIlxcdG9wcnVsZSJdDQogICAgbGluZXMuYXBwZW5kKCIgJiAiLmpvaW4oY2VsbChjKSBmb3IgYyBpbiBmcmFtZS5jb2x1bW5zKSArICIgXFxcXCIpDQogICAgbGluZXMuYXBwZW5kKCJcXG1pZHJ1bGUiKQ0KICAgIGxpbmVzLmV4dGVuZCgiICYgIi5qb2luKGNlbGwodikgZm9yIHYgaW4gcm93KSArICIgXFxcXCIgZm9yIHJvdyBpbiBmcmFtZS5pdGVydHVwbGVzKGluZGV4PUZhbHNlKSkNCiAgICBsaW5lcy5leHRlbmQoWyJcXGJvdHRvbXJ1bGUiLCAiXFxlbmR7dGFidWxhcn0iLCAiIl0pDQogICAgcGF0aC53cml0ZV90ZXh0KCJcbiIuam9pbihsaW5lcyksIGVuY29kaW5nPSJ1dGYtOCIpDQoNCg0KZGVmIF9zdGF0aXN0aWNzX3RhYmxlKHN0YXRpc3RpY3M6IGRpY3QpIC0+IHBkLkRhdGFGcmFtZToNCiAgICByb3dzID0gW10NCiAgICBibG9ja3MgPSB7InByaW1hcnkgUjIwIG9mZmljaWFsIjogc3RhdGlzdGljc1sicHJpbWFyeSJdWyJIMV9IMl9SMjBfb2ZmaWNpYWwiXX0NCiAgICBibG9ja3MudXBkYXRlKHtmInNlY29uZGFyeSB7a30iOiB2IGZvciBrLCB2IGluIHN0YXRpc3RpY3NbInNlY29uZGFyeSJdLml0ZW1zKCl9KQ0KICAgIGZvciBuYW1lLCBibG9jayBpbiBibG9ja3MuaXRlbXMoKToNCiAgICAgICAgZm9yIG1ldHJpYyBpbiAoImF1bGNfdHByIiwgImF1bGNfZnByIik6DQogICAgICAgICAgICBwYXJ0ID0gYmxvY2tbbWV0cmljXQ0KICAgICAgICAgICAgcm93cy5hcHBlbmQoDQogICAgICAgICAgICAgICAgew0KICAgICAgICAgICAgICAgICAgICAiY29tcGFyaXNvbiI6IG5hbWUsDQogICAgICAgICAgICAgICAgICAgICJtZXRyaWMiOiBtZXRyaWMsDQogICAgICAgICAgICAgICAgICAgICJtZWRpYW4gzpQiOiBwYXJ0WyJib290c3RyYXAiXVsibWVkaWFuIl0sDQogICAgICAgICAgICAgICAgICAgICJDSTk1IGxvdyI6IHBhcnRbImJvb3RzdHJhcCJdWyJjaTk1Il1bMF0sDQogICAgICAgICAgICAgICAgICAgICJDSTk1IGhpZ2giOiBwYXJ0WyJib290c3RyYXAiXVsiY2k5NSJdWzFdLA0KICAgICAgICAgICAgICAgICAgICAiV2lsY294b24gcCI6IHBhcnRbIndpbGNveG9uX3AiXSwNCiAgICAgICAgICAgICAgICAgICAgIkhvbG0gcCI6IHBhcnRbImhvbG1fcCJdLA0KICAgICAgICAgICAgICAgICAgICAiay1jZW50ZXIgYmV0dGVyIjogcGFydFsibl9rY2VudGVyX2JldHRlciJdLA0KICAgICAgICAgICAgICAgICAgICAicmFuZG9tIGJldHRlciI6IHBhcnRbIm5fcmFuZG9tX2JldHRlciJdLA0KICAgICAgICAgICAgICAgICAgICAicHJhY3RpY2FsIHJ1bGUiOiBibG9ja1sicHJhY3RpY2FsX2NyaXRlcmlvbl9wYXNzZWQiXSwNCiAgICAgICAgICAgICAgICB9DQogICAgICAgICAgICApDQogICAgcmV0dXJuIHBkLkRhdGFGcmFtZShyb3dzKQ0KDQoNCmRlZiBidWlsZF9yZXBvcnQoY2ZnLCBmcmFtZXM6IGRpY3QsIGdhdGU6IGRpY3QgfCBOb25lKSAtPiBQYXRoOg0KICAgIGRpcmVjdG9yeSA9IGNmZy53b3JrX3Jvb3QgLyAicmVwb3J0Ig0KICAgIGZpZ3VyZXMgPSBkaXJlY3RvcnkgLyAiZmlndXJlcyINCiAgICB0YWJsZXMgPSBkaXJlY3RvcnkgLyAidGFibGVzIg0KICAgIHRhYmxlcy5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpDQogICAgbWV0cmljcywgc3RhdGlzdGljcyA9IGZyYW1lc1sibWV0cmljcyJdLCBmcmFtZXNbInN0YXRpc3RpY3MiXQ0KICAgIGZpZ3VyZV90cHIobWV0cmljcywgZmlndXJlcykNCiAgICBmaWd1cmVfZnByKG1ldHJpY3MsIGZpZ3VyZXMpDQogICAgZmlndXJlX3Rwcl9jb3N0KHN0YXRpc3RpY3MsIGZpZ3VyZXMpDQogICAgZmlndXJlX2ZvcmVzdChzdGF0aXN0aWNzLCBmaWd1cmVzKQ0KICAgIHdyaXR0ZW4gPSBbImZpZ1YxX3RwciIsICJmaWdWMl9mcHIiLCAiZmlnVjNfdHByX2Nvc3QiLCAiZmlnVjRfZm9yZXN0Il0NCiAgICBpZiBmaWd1cmVfc2hpZnQoZnJhbWVzLmdldCgicDNfc2hpZnQiKSwgZmlndXJlcyk6DQogICAgICAgIHdyaXR0ZW4uYXBwZW5kKCJmaWdWNV9zaGlmdCIpDQogICAgaWYgZmlndXJlX2J1ZGdldChmcmFtZXMuZ2V0KCJidWRnZXRfYmVzdCIpLCBjZmcud29ya19yb290IC8gImFuYWx5c2lzIiAvICJidWRnZXRfc3VtbWFyeS5jc3YiLCBmaWd1cmVzLCBjZmcubl9zdGFyX21pbl9yZXBlYXRzKToNCiAgICAgICAgd3JpdHRlbi5hcHBlbmQoImZpZ1Y2X2J1ZGdldCIpDQogICAgc3RhdHNfdGFibGUgPSBfc3RhdGlzdGljc190YWJsZShzdGF0aXN0aWNzKQ0KICAgIHN0YXRzX3RhYmxlLnRvX2Nzdih0YWJsZXMgLyAic3RhdGlzdGljcy5jc3YiLCBpbmRleD1GYWxzZSkNCiAgICB3cml0ZV9sYXRleChzdGF0c190YWJsZSwgdGFibGVzIC8gInN0YXRpc3RpY3MudGV4IikNCiAgICBzZWN0aW9ucyA9IFtmJzxoMj57bmFtZX08L2gyPjxpbWcgc3JjPSJmaWd1cmVzL3tuYW1lfS5wbmciIGFsdD0ie25hbWV9Ij4nIGZvciBuYW1lIGluIHdyaXR0ZW5dDQogICAgZm9yIHRpdGxlLCBrZXkgaW4gKA0KICAgICAgICAoIk4qIGJ5IHJ1bGUgYW5kIEZQUiBzb3VyY2UiLCAibl9zdGFyIiksDQogICAgICAgICgiUDIgY2FsaWJyYXRpb24gb24gSCIsICJwMl9jYWxpYnJhdGlvbiIpLA0KICAgICAgICAoIlAzIHNoaWZ0IiwgInAzX3NoaWZ0IiksDQogICAgICAgICgiRmVhc2liaWxpdHkgY2VpbGluZyIsICJjZWlsaW5nIiksDQogICAgICAgICgiU3RvcHBpbmcgcnVsZSIsICJzdG9wcGluZ19zdW1tYXJ5IiksDQogICAgICAgICgiQnVkZ2V0IHNwbGl0LCBiZXN0IE4iLCAiYnVkZ2V0X2Jlc3QiKSwNCiAgICApOg0KICAgICAgICBmcmFtZSA9IGZyYW1lcy5nZXQoa2V5KQ0KICAgICAgICBpZiBmcmFtZSBpcyBub3QgTm9uZSBhbmQgbm90IGZyYW1lLmVtcHR5Og0KICAgICAgICAgICAgZnJhbWUudG9fY3N2KHRhYmxlcyAvIGYie2tleX0uY3N2IiwgaW5kZXg9RmFsc2UpDQogICAgICAgICAgICBzZWN0aW9ucy5hcHBlbmQoZiI8aDI+e2h0bWwuZXNjYXBlKHRpdGxlKX08L2gyPntfaHRtbF9mcmFtZShmcmFtZS5kcm9wKGNvbHVtbnM9Wydjb25maWdfaGFzaCddLCBlcnJvcnM9J2lnbm9yZScpKX0iKQ0KICAgIGdhdGVfdGV4dCA9ICJtaXNzaW5nIiBpZiBnYXRlIGlzIE5vbmUgZWxzZSBmIntnYXRlLmdldCgnc3RhdHVzJyl9IChFMGEgbWVhbiBBVVJPQyB7Z2F0ZS5nZXQoJ2UwYScsIHt9KSBhbmQgZ2F0ZVsnZTBhJ10uZ2V0KCdtZWFuX2F1cm9jJyl9KSINCiAgICBoMyA9IHN0YXRpc3RpY3NbIkgzIl0NCiAgICBwMiA9IHN0YXRpc3RpY3NbIlAyIl0NCiAgICBkb2N1bWVudCA9IGYiIiI8IURPQ1RZUEUgaHRtbD4NCjxodG1sIGxhbmc9ImVuIj48aGVhZD48bWV0YSBjaGFyc2V0PSJ1dGYtOCI+PHRpdGxlPlZpc0EgcmVwbGljYXRpb248L3RpdGxlPg0KPHN0eWxlPg0KYm9keSB7eyBmb250LWZhbWlseTogR2VvcmdpYSwgc2VyaWY7IG1hcmdpbjogMnJlbSBhdXRvOyBtYXgtd2lkdGg6IDEwMDBweDsgY29sb3I6ICMxYzFjMWM7IGxpbmUtaGVpZ2h0OiAxLjQ1OyB9fQ0KaW1nIHt7IG1heC13aWR0aDogMTAwJTsgaGVpZ2h0OiBhdXRvOyB9fQ0KdGFibGUge3sgYm9yZGVyLWNvbGxhcHNlOiBjb2xsYXBzZTsgd2lkdGg6IDEwMCU7IGZvbnQtc2l6ZTogMC44cmVtOyB9fQ0KdGgsIHRkIHt7IGJvcmRlci1ib3R0b206IDFweCBzb2xpZCAjZGRkOyBwYWRkaW5nOiAwLjNyZW0gMC40cmVtOyB0ZXh0LWFsaWduOiBsZWZ0OyB9fQ0KLmJhbm5lciB7eyBiYWNrZ3JvdW5kOiAjZWVmNGZiOyBib3JkZXI6IDFweCBzb2xpZCAjMWY0ZTc5OyBwYWRkaW5nOiAwLjZyZW0gMC44cmVtOyB9fQ0KPC9zdHlsZT48L2hlYWQ+PGJvZHk+DQo8aDE+Q29uZmlybWF0b3J5IHJlcGxpY2F0aW9uIG9uIFZpc0EgKHByZXJlZ2lzdHJhdGlvbl92aXNhLm1kKTwvaDE+DQo8cCBjbGFzcz0iYmFubmVyIj5Db25maWcgaGFzaCB7aHRtbC5lc2NhcGUoY2ZnLmhhc2gpfS4gRTAgZ2F0ZToge2h0bWwuZXNjYXBlKHN0cihnYXRlX3RleHQpKX0uIEV4cGxvcmF0b3J5IHJvd3M6IHtzdGF0aXN0aWNzLmdldCgnZXhwbG9yYXRvcnlfcm93cycpfS48L3A+DQo8aDI+UHJpbWFyeSBhbmQgc2Vjb25kYXJ5IGNvbXBhcmlzb25zIChrLWNlbnRlciDiiJIgcmFuZG9tKTwvaDI+e19odG1sX2ZyYW1lKHN0YXRzX3RhYmxlKX0NCjxoMj5IMzwvaDI+PHA+TWVkaWFuIFRQUiBjb3N0IG9mIFI2MCBhdCBOID0gfFB8OiB7aDNbJ21lZGlhbiddOi40Zn0gKENJOTUge2gzWydib290c3RyYXAnXVsnY2k5NSddWzBdOi40Zn0gdG8ge2gzWydib290c3RyYXAnXVsnY2k5NSddWzFdOi40Zn0pOyBtYXJnaW4ge2gzWydtYXJnaW4nXX07IFI2MCByZWNvbW1lbmRlZDoge2gzWydyNjBfcmVjb21tZW5kZWQnXX0uPC9wPg0KPGgyPlAyPC9oMj48cHJlPntodG1sLmVzY2FwZShqc29uLmR1bXBzKHtrOiB2IGZvciBrLCB2IGluIHAyLml0ZW1zKCkgaWYgayAhPSAnYnlfTid9LCBpbmRlbnQ9MikpfTwvcHJlPg0KPGgyPlAxIGFuZCBQMzwvaDI+PHByZT57aHRtbC5lc2NhcGUoanNvbi5kdW1wcyh7J1AxJzogc3RhdGlzdGljc1snUDEnXSwgJ1AzJzogc3RhdGlzdGljc1snUDMnXSwgJ2J1ZGdldCc6IHN0YXRpc3RpY3MuZ2V0KCdidWRnZXQnKX0sIGluZGVudD0yKSl9PC9wcmU+DQp7Jycuam9pbihzZWN0aW9ucyl9DQo8L2JvZHk+PC9odG1sPg0KIiIiDQogICAgcGF0aCA9IGRpcmVjdG9yeSAvICJpbmRleC5odG1sIg0KICAgIHBhdGgud3JpdGVfdGV4dChfaW5saW5lX2ZpZ3VyZXMoZG9jdW1lbnQsIGRpcmVjdG9yeSksIGVuY29kaW5nPSJ1dGYtOCIpDQogICAgcmV0dXJuIHBhdGgNCg==", "commissioning/visa/run.py": "IiIiMTIgeCAyMCB4IDIgbmVzdGVkIHRyYWplY3Rvcmllcy4gRWFjaCBncmlkIHBvaW50IGlzIGV2YWx1YXRlZCB1bmRlciBSMjAgYW5kIFI2MCBvbiB0aGUgc2FtZSBiYW5rLiIiIg0KDQpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zDQoNCmltcG9ydCBsb2dnaW5nDQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgNCg0KaW1wb3J0IG51bXB5IGFzIG5wDQppbXBvcnQgcGFuZGFzIGFzIHBkDQpmcm9tIHNrbGVhcm4ubWV0cmljcyBpbXBvcnQgcm9jX2F1Y19zY29yZQ0KDQpmcm9tIGNvbW1pc3Npb25pbmcuY2FsaWJyYXRpb24gaW1wb3J0ICgNCiAgICBfbWVldHMsDQogICAgY2xvcHBlcl9wZWFyc29uX2xvd2VyLA0KICAgIGNsb3BwZXJfcGVhcnNvbl91cHBlciwNCiAgICBmYWxzZV9yZWplY3RfYnVkZ2V0LA0KICAgIG9yYWNsZV9mcHJfYXRfdHByLA0KKQ0KZnJvbSBjb21taXNzaW9uaW5nLmlvX3V0aWxzIGltcG9ydCBhdG9taWNfcGFycXVldCwgYXRvbWljX3NhdmV6LCByZWFkX2pzb24NCmZyb20gY29tbWlzc2lvbmluZy5wYXRjaGNvcmUgaW1wb3J0IHNjb3Jlc19hbG9uZ19vcmRlcg0KZnJvbSBjb21taXNzaW9uaW5nLnNlbGVjdGlvbiBpbXBvcnQga2NlbnRlcl9vcmRlciwgcmFuZG9tX29yZGVyDQpmcm9tIGNvbW1pc3Npb25pbmcudmlzYS5jYWNoZXMgaW1wb3J0IGxvYWRfZGlzdGFuY2VzLCBsb2FkX2dhcA0KZnJvbSBjb21taXNzaW9uaW5nLnZpc2EuY29uZmlnIGltcG9ydCBWaXNhQ29uZmlnDQpmcm9tIGNvbW1pc3Npb25pbmcudmlzYS5kYXRhIGltcG9ydCBpbWFnZV9vcmRlciwgbG9hZF9zcGxpdCwgc3BsaXRfc2VlZHMNCmZyb20gY29tbWlzc2lvbmluZy52aXNhLmUwIGltcG9ydCBnYXRlX2lzX2V4cGxvcmF0b3J5DQoNCkxPR0dFUiA9IGxvZ2dpbmcuZ2V0TG9nZ2VyKF9fbmFtZV9fKQ0KDQoNCmRlZiB0cmFqZWN0b3J5X3BhdGhzKGNmZzogVmlzYUNvbmZpZywgY2xhc3NfbmFtZTogc3RyLCByZXA6IGludCwgc3RyYXRlZ3k6IHN0cikgLT4gdHVwbGVbUGF0aCwgUGF0aF06DQogICAgc3RlbSA9IGYie2NsYXNzX25hbWV9X3JlcHtyZXA6MDJkfV97c3RyYXRlZ3l9Ig0KICAgIHJldHVybiBjZmcud29ya19yb290IC8gInJ1bnMiIC8gInBhcnRzIiAvIGYie3N0ZW19LnBhcnF1ZXQiLCBjZmcud29ya19yb290IC8gInJ1bnMiIC8gInNjb3JlcyIgLyBmIntzdGVtfS5ucHoiDQoNCg0KZGVmIHJ1bGVfbWV0cmljcygNCiAgICB0aHJlc2hvbGQ6IGZsb2F0LA0KICAgIGRlZmVjdF9zY29yZXM6IG5wLm5kYXJyYXksDQogICAgZ29vZF9zY29yZXM6IG5wLm5kYXJyYXksDQogICAgaG9sZG91dF9zY29yZXM6IG5wLm5kYXJyYXksDQogICAgcHJhY3RpY2FsX3RwcjogZmxvYXQsDQogICAgcHJhY3RpY2FsX2ZwcjogZmxvYXQsDQopIC0+IGRpY3Q6DQogICAgIiIiUmVqZWN0IHdoZW4gdGhlIHNjb3JlIGlzIHN0cmljdGx5IGFib3ZlIHRoZSB0aHJlc2hvbGQuIiIiDQogICAgbl90cCA9IGludChucC5zdW0oZGVmZWN0X3Njb3JlcyA+IHRocmVzaG9sZCkpDQogICAgbl9mcF9vZmYgPSBpbnQobnAuc3VtKGdvb2Rfc2NvcmVzID4gdGhyZXNob2xkKSkNCiAgICBuX2ZwX2ggPSBpbnQobnAuc3VtKGhvbGRvdXRfc2NvcmVzID4gdGhyZXNob2xkKSkNCiAgICBuX2RlZmVjdCwgbl9nb29kLCBuX2ggPSBsZW4oZGVmZWN0X3Njb3JlcyksIGxlbihnb29kX3Njb3JlcyksIGxlbihob2xkb3V0X3Njb3JlcykNCiAgICB0cHJfb2sgPSBfbWVldHMobl90cCwgbl9kZWZlY3QsIHByYWN0aWNhbF90cHIsIGF0X2xlYXN0PVRydWUpDQogICAgcmV0dXJuIHsNCiAgICAgICAgInRocmVzaG9sZCI6IGZsb2F0KHRocmVzaG9sZCksDQogICAgICAgICJ0cHIiOiBuX3RwIC8gbl9kZWZlY3QsDQogICAgICAgICJ0cHJfbG93ZXIiOiBjbG9wcGVyX3BlYXJzb25fbG93ZXIobl90cCwgbl9kZWZlY3QpLA0KICAgICAgICAiZnByX29mZiI6IG5fZnBfb2ZmIC8gbl9nb29kLA0KICAgICAgICAiZnByX29mZl91cHBlciI6IGNsb3BwZXJfcGVhcnNvbl91cHBlcihuX2ZwX29mZiwgbl9nb29kKSwNCiAgICAgICAgImZwcl9oIjogbl9mcF9oIC8gbl9oLA0KICAgICAgICAiZnByX2hfdXBwZXIiOiBjbG9wcGVyX3BlYXJzb25fdXBwZXIobl9mcF9oLCBuX2gpLA0KICAgICAgICAibl90cCI6IG5fdHAsDQogICAgICAgICJuX2ZwX29mZiI6IG5fZnBfb2ZmLA0KICAgICAgICAibl9mcF9oIjogbl9mcF9oLA0KICAgICAgICAibl90ZXN0X2RlZmVjdCI6IG5fZGVmZWN0LA0KICAgICAgICAibl90ZXN0X2dvb2QiOiBuX2dvb2QsDQogICAgICAgICJuX2hvbGRvdXQiOiBuX2gsDQogICAgICAgICJidWRnZXRfb2ZmIjogZmFsc2VfcmVqZWN0X2J1ZGdldChuX2dvb2QpLA0KICAgICAgICAiYnVkZ2V0X2giOiBmYWxzZV9yZWplY3RfYnVkZ2V0KG5faCksDQogICAgICAgICJjcml0ZXJpb25fb2ZmIjogYm9vbCh0cHJfb2sgYW5kIF9tZWV0cyhuX2ZwX29mZiwgbl9nb29kLCBwcmFjdGljYWxfZnByLCBhdF9sZWFzdD1GYWxzZSkpLA0KICAgICAgICAiY3JpdGVyaW9uX2giOiBib29sKHRwcl9vayBhbmQgX21lZXRzKG5fZnBfaCwgbl9oLCBwcmFjdGljYWxfZnByLCBhdF9sZWFzdD1GYWxzZSkpLA0KICAgIH0NCg0KDQpkZWYgZXZhbHVhdGVfZ3JpZF9wb2ludCgNCiAgICBjZmc6IFZpc2FDb25maWcsDQogICAgY2FsaWJyYXRpb25fc2NvcmVzOiBucC5uZGFycmF5LA0KICAgIGhvbGRvdXRfc2NvcmVzOiBucC5uZGFycmF5LA0KICAgIHRlc3Rfc2NvcmVzOiBucC5uZGFycmF5LA0KICAgIGxhYmVsczogbnAubmRhcnJheSwNCikgLT4gbGlzdFtkaWN0XToNCiAgICB2YWx1ZXMgPSBbY2FsaWJyYXRpb25fc2NvcmVzLCBob2xkb3V0X3Njb3JlcywgdGVzdF9zY29yZXNdDQogICAgaWYgbm90IGFsbChucC5pc2Zpbml0ZSh2KS5hbGwoKSBmb3IgdiBpbiB2YWx1ZXMpOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIk5vbi1maW5pdGUgc2NvcmVzOyByZWZ1c2luZyB0byByZXBsYWNlIHRoZW0gd2l0aCBhIHNlbnRpbmVsIHZhbHVlLiIpDQogICAgaWYgbGVuKGNhbGlicmF0aW9uX3Njb3JlcykgIT0gY2ZnLm5fY2FsaWJyYXRpb25fbGFyZ2U6DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIkV4cGVjdGVkIHtjZmcubl9jYWxpYnJhdGlvbl9sYXJnZX0gY2FsaWJyYXRpb24gc2NvcmVzLCBnb3Qge2xlbihjYWxpYnJhdGlvbl9zY29yZXMpfSIpDQogICAgc21hbGwgPSBjYWxpYnJhdGlvbl9zY29yZXNbOiBjZmcubl9jYWxpYnJhdGlvbl9zbWFsbF0NCiAgICBzaGFyZWQgPSB7DQogICAgICAgICJhdXJvYyI6IGZsb2F0KHJvY19hdWNfc2NvcmUobGFiZWxzLCB0ZXN0X3Njb3JlcykpLA0KICAgICAgICAib3JhY2xlX2ZwciI6IG9yYWNsZV9mcHJfYXRfdHByKHRlc3Rfc2NvcmVzLCBsYWJlbHMsIGNmZy5wcmFjdGljYWxfdHByKSwNCiAgICAgICAgIm1lZGlhbl9jYWxpYnJhdGlvbl9jMjAiOiBmbG9hdChucC5tZWRpYW4oc21hbGwpKSwNCiAgICAgICAgIm1lZGlhbl9jYWxpYnJhdGlvbl9jNjAiOiBmbG9hdChucC5tZWRpYW4oY2FsaWJyYXRpb25fc2NvcmVzKSksDQogICAgfQ0KICAgIHJvd3MgPSBbXQ0KICAgIGZvciBydWxlLCB0aHJlc2hvbGQgaW4gKCgiUjIwIiwgZmxvYXQobnAubWF4KHNtYWxsKSkpLCAoIlI2MCIsIGZsb2F0KG5wLm1heChjYWxpYnJhdGlvbl9zY29yZXMpKSkpOg0KICAgICAgICBtZXRyaWNzID0gcnVsZV9tZXRyaWNzKA0KICAgICAgICAgICAgdGhyZXNob2xkLA0KICAgICAgICAgICAgdGVzdF9zY29yZXNbbGFiZWxzID09IDFdLA0KICAgICAgICAgICAgdGVzdF9zY29yZXNbbGFiZWxzID09IDBdLA0KICAgICAgICAgICAgaG9sZG91dF9zY29yZXMsDQogICAgICAgICAgICBjZmcucHJhY3RpY2FsX3RwciwNCiAgICAgICAgICAgIGNmZy5wcmFjdGljYWxfZnByLA0KICAgICAgICApDQogICAgICAgIHJvd3MuYXBwZW5kKHsicnVsZSI6IHJ1bGUsICoqc2hhcmVkLCAqKm1ldHJpY3N9KQ0KICAgIHJldHVybiByb3dzDQoNCg0KZGVmIHJ1bl9vbmUoDQogICAgY2ZnOiBWaXNhQ29uZmlnLA0KICAgIGNsYXNzX25hbWU6IHN0ciwNCiAgICByZXA6IGludCwNCiAgICBzdHJhdGVneTogc3RyLA0KICAgIGV4cGxvcmF0b3J5OiBib29sLA0KICAgIHBhdGhzOiBsaXN0W3N0cl0sDQogICAgZGlzdGFuY2U6IG5wLm5kYXJyYXksDQogICAgZ2FwOiBucC5uZGFycmF5LA0KKSAtPiBOb25lOg0KICAgIHRhYmxlX3BhdGgsIHNjb3Jlc19wYXRoID0gdHJhamVjdG9yeV9wYXRocyhjZmcsIGNsYXNzX25hbWUsIHJlcCwgc3RyYXRlZ3kpDQogICAgaWYgdGFibGVfcGF0aC5pc19maWxlKCkgYW5kIHNjb3Jlc19wYXRoLmlzX2ZpbGUoKToNCiAgICAgICAgaWYgc2V0KHBkLnJlYWRfcGFycXVldCh0YWJsZV9wYXRoKVsiY29uZmlnX2hhc2giXS5hc3R5cGUoc3RyKSkgIT0ge2NmZy5oYXNofToNCiAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIlJlZnVzaW5nIHRvIHJldXNlIGEgdHJhamVjdG9yeSBmcm9tIGFub3RoZXIgY29uZmlnOiB7dGFibGVfcGF0aH0iKQ0KICAgICAgICByZXR1cm4NCiAgICBzcGxpdCA9IGxvYWRfc3BsaXQoY2ZnLCBjbGFzc19uYW1lLCByZXApDQogICAgaWYgcGF0aHMgIT0gaW1hZ2Vfb3JkZXIoc3BsaXQpOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJ7Y2xhc3NfbmFtZX0gcmVwIHtyZXB9OiBjYWNoZSByb3dzIGRvIG5vdCBtYXRjaCB0aGUgc3BsaXQuIikNCiAgICBpbmRleCA9IHtwYXRoOiBwb3NpdGlvbiBmb3IgcG9zaXRpb24sIHBhdGggaW4gZW51bWVyYXRlKHBhdGhzKX0NCiAgICBjYWxpYnJhdGlvbiA9IFtpbmRleFtwXSBmb3IgcCBpbiBzcGxpdFsiY2FsaWJyYXRpb24iXV0NCiAgICBwb29sID0gW2luZGV4W3BdIGZvciBwIGluIHNwbGl0WyJwb29sIl1dDQogICAgaG9sZG91dCA9IFtpbmRleFtwXSBmb3IgcCBpbiBzcGxpdFsiaG9sZG91dCJdXQ0KICAgIHRlc3QgPSBbaW5kZXhbcF0gZm9yIHAgaW4gbGlzdChzcGxpdFsidGVzdF9nb29kIl0pICsgbGlzdChzcGxpdFsidGVzdF9kZWZlY3QiXSldDQogICAgbl91bml2ZXJzZSA9IGxlbihzcGxpdFsidW5pdmVyc2UiXSkNCiAgICBpZiBtYXgoY2FsaWJyYXRpb24gKyBwb29sKSA+PSBuX3VuaXZlcnNlIG9yIG1pbihob2xkb3V0ICsgdGVzdCkgPCBuX3VuaXZlcnNlOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJ7Y2xhc3NfbmFtZX06IEMgYW5kIFAgbXVzdCBiZSB1bml2ZXJzZSBjb2x1bW5zOyBIIGFuZCB0ZXN0IG11c3Qgbm90LiIpDQogICAgX3NwbGl0X3JuZywgcmFuZG9tX3JuZywga2NlbnRlcl9ybmcgPSBzcGxpdF9zZWVkcyhjZmcsIGNsYXNzX25hbWUsIHJlcCkNCiAgICBpZiBzdHJhdGVneSA9PSAicmFuZG9tIjoNCiAgICAgICAgb3JkZXIgPSByYW5kb21fb3JkZXIocG9vbCwgcmFuZG9tX3JuZykNCiAgICBlbGlmIHN0cmF0ZWd5ID09ICJrY2VudGVyIjoNCiAgICAgICAgb3JkZXIgPSBrY2VudGVyX29yZGVyKGdhcCwgcG9vbCwga2NlbnRlcl9ybmcpDQogICAgZWxzZToNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYiVW5rbm93biBzdHJhdGVneToge3N0cmF0ZWd5fSIpDQogICAgZ3JpZCA9IGNmZy5uX2dyaWQNCiAgICBpZiBncmlkWy0xXSAhPSBsZW4ocG9vbCk6DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIntjbGFzc19uYW1lfTogdGhlIGdyaWQgbXVzdCBlbmQgYXQgfFB8ID0ge2xlbihwb29sKX0iKQ0KICAgIHF1ZXJpZXMgPSBucC5hc2FycmF5KGNhbGlicmF0aW9uICsgaG9sZG91dCArIHRlc3QsIGR0eXBlPWludCkNCiAgICBzdGFja2VkID0gc2NvcmVzX2Fsb25nX29yZGVyKGRpc3RhbmNlLCBxdWVyaWVzLCBvcmRlciwgZ3JpZCkNCiAgICBuX2MsIG5faCA9IGxlbihjYWxpYnJhdGlvbiksIGxlbihob2xkb3V0KQ0KICAgIGxhYmVscyA9IG5wLmFycmF5KFswXSAqIGxlbihzcGxpdFsidGVzdF9nb29kIl0pICsgWzFdICogbGVuKHNwbGl0WyJ0ZXN0X2RlZmVjdCJdKSwgZHR5cGU9aW50KQ0KICAgIHJvd3MgPSBbXQ0KICAgIGZvciByb3dfaW5kZXgsIG4gaW4gZW51bWVyYXRlKGdyaWQpOg0KICAgICAgICBzY29yZXMgPSBzdGFja2VkW3Jvd19pbmRleF0NCiAgICAgICAgZm9yIHJlY29yZCBpbiBldmFsdWF0ZV9ncmlkX3BvaW50KGNmZywgc2NvcmVzWzpuX2NdLCBzY29yZXNbbl9jIDogbl9jICsgbl9oXSwgc2NvcmVzW25fYyArIG5faCA6XSwgbGFiZWxzKToNCiAgICAgICAgICAgIHJvd3MuYXBwZW5kKA0KICAgICAgICAgICAgICAgIHsNCiAgICAgICAgICAgICAgICAgICAgImNsYXNzX25hbWUiOiBjbGFzc19uYW1lLA0KICAgICAgICAgICAgICAgICAgICAiZ3JvdXAiOiBjZmcuZ3JvdXBfb2YoY2xhc3NfbmFtZSksDQogICAgICAgICAgICAgICAgICAgICJyZXAiOiBpbnQocmVwKSwNCiAgICAgICAgICAgICAgICAgICAgInN0cmF0ZWd5Ijogc3RyYXRlZ3ksDQogICAgICAgICAgICAgICAgICAgICJOIjogaW50KG4pLA0KICAgICAgICAgICAgICAgICAgICAicG9vbF9zaXplIjogbGVuKHBvb2wpLA0KICAgICAgICAgICAgICAgICAgICAiY29uZmlnX2hhc2giOiBjZmcuaGFzaCwNCiAgICAgICAgICAgICAgICAgICAgImV4cGxvcmF0b3J5IjogYm9vbChleHBsb3JhdG9yeSksDQogICAgICAgICAgICAgICAgICAgICoqcmVjb3JkLA0KICAgICAgICAgICAgICAgIH0NCiAgICAgICAgICAgICkNCiAgICBhdG9taWNfcGFycXVldCh0YWJsZV9wYXRoLCBwZC5EYXRhRnJhbWUocm93cykpDQogICAgYXRvbWljX3NhdmV6KA0KICAgICAgICBzY29yZXNfcGF0aCwNCiAgICAgICAgZ3JpZD1ucC5hc2FycmF5KGdyaWQsIGR0eXBlPW5wLmludDMyKSwNCiAgICAgICAgc2NvcmVzX2NhbGlicmF0aW9uPXN0YWNrZWRbOiwgOm5fY10uYXN0eXBlKG5wLmZsb2F0MzIpLA0KICAgICAgICBzY29yZXNfaG9sZG91dD1zdGFja2VkWzosIG5fYyA6IG5fYyArIG5faF0uYXN0eXBlKG5wLmZsb2F0MzIpLA0KICAgICAgICBzY29yZXNfdGVzdD1zdGFja2VkWzosIG5fYyArIG5faCA6XS5hc3R5cGUobnAuZmxvYXQzMiksDQogICAgICAgIHRlc3RfbGFiZWxzPWxhYmVscy5hc3R5cGUobnAuaW50OCksDQogICAgICAgIG9yZGVyPW9yZGVyLmFzdHlwZShucC5pbnQzMiksDQogICAgICAgIHRlc3RfcGF0aHM9bnAuYXNhcnJheShsaXN0KHNwbGl0WyJ0ZXN0X2dvb2QiXSkgKyBsaXN0KHNwbGl0WyJ0ZXN0X2RlZmVjdCJdKSksDQogICAgICAgIGNhbGlicmF0aW9uX3BhdGhzPW5wLmFzYXJyYXkoc3BsaXRbImNhbGlicmF0aW9uIl0pLA0KICAgICkNCiAgICBMT0dHRVIuaW5mbygiRmluaXNoZWQgJXMgcmVwICUwMmQgJXMiLCBjbGFzc19uYW1lLCByZXAsIHN0cmF0ZWd5KQ0KDQoNCmRlZiBydW4oY2ZnOiBWaXNhQ29uZmlnLCBjbGFzc2VzOiBsaXN0W3N0cl0sIGV4cGxvcmF0b3J5OiBib29sID0gRmFsc2UsIGZvcmNlX2V4cGxvcmF0b3J5OiBib29sID0gRmFsc2UpIC0+IE5vbmU6DQogICAgbWFyayA9IGdhdGVfaXNfZXhwbG9yYXRvcnkoY2ZnLCBjbGFzc2VzLCBleHBsb3JhdG9yeSkgb3IgZm9yY2VfZXhwbG9yYXRvcnkNCiAgICBmb3IgY2xhc3NfbmFtZSBpbiBjbGFzc2VzOg0KICAgICAgICBwZW5kaW5nID0gWw0KICAgICAgICAgICAgKHJlcCwgc3RyYXRlZ3kpDQogICAgICAgICAgICBmb3IgcmVwIGluIHJhbmdlKGNmZy5uX3JlcGVhdHMpDQogICAgICAgICAgICBmb3Igc3RyYXRlZ3kgaW4gY2ZnLnN0cmF0ZWdpZXMNCiAgICAgICAgICAgIGlmIG5vdCBhbGwocC5pc19maWxlKCkgZm9yIHAgaW4gdHJhamVjdG9yeV9wYXRocyhjZmcsIGNsYXNzX25hbWUsIHJlcCwgc3RyYXRlZ3kpKQ0KICAgICAgICBdDQogICAgICAgIGlmIG5vdCBwZW5kaW5nOg0KICAgICAgICAgICAgTE9HR0VSLmluZm8oIkFsbCB0cmFqZWN0b3JpZXMgZXhpc3QgZm9yICVzIiwgY2xhc3NfbmFtZSkNCiAgICAgICAgICAgIGNvbnRpbnVlDQogICAgICAgIGRpc3RhbmNlID0gbG9hZF9kaXN0YW5jZXMoY2ZnLCBjbGFzc19uYW1lKQ0KICAgICAgICBnYXAgPSBsb2FkX2dhcChjZmcsIGNsYXNzX25hbWUpDQogICAgICAgIGlmIGdhcFsicGF0aHMiXSAhPSBsaXN0KGRpc3RhbmNlWyJtZXRhIl1bInBhdGhzIl0pOg0KICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYie2NsYXNzX25hbWV9OiBHQVAgdGFibGUgYW5kIGRpc3RhbmNlIGNhY2hlIGFyZSBub3QgYWxpZ25lZC4iKQ0KICAgICAgICBmb3IgcmVwLCBzdHJhdGVneSBpbiBwZW5kaW5nOg0KICAgICAgICAgICAgcnVuX29uZShjZmcsIGNsYXNzX25hbWUsIHJlcCwgc3RyYXRlZ3ksIG1hcmssIGdhcFsicGF0aHMiXSwgZGlzdGFuY2VbIkQiXSwgZ2FwWyJnYXAiXSkNCiAgICAgICAgZGVsIGRpc3RhbmNlDQoNCg0KZGVmIGxvYWRfbWV0cmljcyhjZmc6IFZpc2FDb25maWcpIC0+IHBkLkRhdGFGcmFtZToNCiAgICBwYXJ0cyA9IHNvcnRlZCgoY2ZnLndvcmtfcm9vdCAvICJydW5zIiAvICJwYXJ0cyIpLmdsb2IoIioucGFycXVldCIpKQ0KICAgIGlmIG5vdCBwYXJ0czoNCiAgICAgICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoIk5vIHRyYWplY3RvcnkgdGFibGVzIGluIHJ1bnMvcGFydHMuIikNCiAgICBtZXRyaWNzID0gcGQuY29uY2F0KFtwZC5yZWFkX3BhcnF1ZXQocGF0aCkgZm9yIHBhdGggaW4gcGFydHNdLCBpZ25vcmVfaW5kZXg9VHJ1ZSkNCiAgICBpZiBzZXQobWV0cmljc1siY29uZmlnX2hhc2giXS5hc3R5cGUoc3RyKSkgIT0ge2NmZy5oYXNofToNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJUcmFqZWN0b3J5IHRhYmxlcyBtaXggY29uZmlnIGhhc2hlcy4iKQ0KICAgIHJldHVybiBtZXRyaWNzDQoNCg0KZGVmIHJlYWRfZ2F0ZShjZmc6IFZpc2FDb25maWcpIC0+IGRpY3QgfCBOb25lOg0KICAgIGZyb20gY29tbWlzc2lvbmluZy52aXNhLmUwIGltcG9ydCBnYXRlX3BhdGgNCg0KICAgIHBhdGggPSBnYXRlX3BhdGgoY2ZnKQ0KICAgIHJldHVybiByZWFkX2pzb24ocGF0aCkgaWYgcGF0aC5pc19maWxlKCkgZWxzZSBOb25lDQo="}

for relative, encoded in FILES.items():
    destination = PROJECT / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_bytes(base64.b64decode(encoded))
status = next(line for line in (PROJECT / "preregistration_visa.md").read_text(encoding="utf-8").splitlines() if line.startswith("Status:"))
print(f"Zapisano {len(FILES)} plików do {PROJECT}")
print("Preregistracja:", status)


In [ ]:
# KOMÓRKA 3/12 — biblioteki
%pip install -q "scipy>=1.11" "pandas>=2.1" "pyarrow>=14" "scikit-learn>=1.3" "matplotlib>=3.8" "PyYAML>=6" "Pillow>=10" "tqdm>=4.66" "timm>=0.9" "anomalib>=2.0"
import subprocess, sys
print(subprocess.check_output(
    [sys.executable, "-c", "import torch, anomalib; print('torch', torch.__version__, '| anomalib', anomalib.__version__, '| GPU', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'BRAK')"],
    text=True,
))
# Jeśli wypisało GPU BRAK: Runtime -> Change runtime type -> GPU, potem od komórki 1.

In [ ]:
# KOMÓRKA 4/12 — JEDYNA EDYCJA: ścieżki
import os, shutil, sys

VISA_TAR = os.environ["VISA_TAR"]
WORK = os.environ.get("VISA_WORK", "visa_runs")
RAW = "/content/VisA_raw"                    # rozpakowane archiwum, lokalnie
DATA = "/content/visa_1cls"                  # układ MVTec, lokalnie
CACHE = "/content/visa_cache"                # cechy i odległości, lokalnie

os.environ["VISA_DATA"] = DATA
os.environ["VISA_WORK"] = WORK
os.environ["VISA_CACHE"] = CACHE
os.chdir("/content/commissioning_project")
if "/content/commissioning_project" not in sys.path:
    sys.path.insert(0, "/content/commissioning_project")
if not os.path.isfile(VISA_TAR):
    raise FileNotFoundError(f"Nie widzę {VISA_TAR}. Popraw VISA_TAR.")

from commissioning.visa.config import load_visa_config
cfg = load_visa_config()
print("hash VisA", cfg.hash)
print("wyniki", cfg.work_root, "| cache", cfg.cache_root)
print("wolne miejsce na dysku lokalnym: %.0f GB (potrzeba ok. 50 GB)" % (shutil.disk_usage("/content").free / 1024**3))

In [ ]:
# KOMÓRKA 5/12 — rozpakowanie archiwum VisA na dysk lokalny (ok. 2–4 min)
import os, tarfile
if not os.path.isdir(os.path.join(RAW, "candle")):
    os.makedirs(RAW, exist_ok=True)
    with tarfile.open(VISA_TAR) as archive:
        archive.extractall(RAW)
# Archiwum może mieć katalog nadrzędny VisA/; konwersja potrzebuje folderu z candle/ i split_csv/.
root = RAW if os.path.isdir(os.path.join(RAW, "candle")) else os.path.join(RAW, os.listdir(RAW)[0])
print("katalog VisA:", root, "|", sorted(os.listdir(root))[:6])
RAW_ROOT = root

In [ ]:
# KOMÓRKA 6/12 — konwersja do układu MVTec według split_csv/1cls.csv (kopiowanie, bez zmiany plików)
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "commissioning.visa", "convert", "--raw-root", RAW_ROOT])

In [ ]:
# KOMÓRKA 7/12 — kontrola liczności i duplikatów, pula U, zbiór H, podziały C/P
# Nie liczy żadnych wyników. Zapisuje data_check.json i splits/ w VISA_WORK.
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'commissioning.visa', 'prepare'])

In [ ]:
# KOMÓRKA 8/12 — cechy Wide-ResNet-50-2, raz na klasę (GPU, ok. 1–2 min na klasę)
# Wolno uruchomić przed zamrożeniem preregistracji.
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'commissioning.visa', 'features'])

In [ ]:
# KOMÓRKA 9/12 — tensor odległości, raz na klasę (GPU, kilka minut na klasę)
# Od tej komórki preregistracja musi być zamrożona.
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'commissioning.visa', 'distances'])

In [ ]:
# KOMÓRKA 10/12 — bramka E0 (Anomalib na U)
# Bez PASS komórka 11 nie wystartuje.
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'commissioning.visa', 'e0'])

In [ ]:
# KOMÓRKA 11/12 — trajektorie, wymienialność, podział budżetu
import subprocess, sys
for stage_name in ("run", "exchange", "budget"):
    subprocess.check_call([sys.executable, "-m", "commissioning.visa", stage_name])
# Opcjonalnie, żeby zwolnić miejsce po E0: shutil.rmtree(CACHE + "/features")

In [ ]:
# KOMÓRKA 12/12 — statystyka i raport HTML
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "commissioning.visa", "analyze"])
subprocess.check_call([sys.executable, "-m", "commissioning.visa", "report"])
print("Raport:", WORK + "/report/index.html")